In [80]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        self.ln1 = nn.LayerNorm((context_size,head_size))
        self.ln2 = nn.LayerNorm((context_size,head_size))
        # self.ln1 = LayerNorm(head_size)
        # self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        self.ln1 = nn.LayerNorm(head_size)
        # self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  41,889
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3918  val: 4.386
train: 1.5908  val: 1.603
train: 0.9859  val: 1.033
train: 0.7461  val: 0.7851
train: 0.6235  val: 0.6596
done!


RuntimeError: Given normalized_shape=[8, 32], expected input with shape [*, 8, 32], but got input of size[1, 1, 32]

In [81]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        self.ln1 = nn.LayerNorm((context_size,head_size))
        self.ln2 = nn.LayerNorm((1,head_size))
        # self.ln1 = LayerNorm(head_size)
        # self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        self.ln1 = nn.LayerNorm(head_size)
        # self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  40,545
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False


RuntimeError: Given normalized_shape=[1, 32], expected input with shape [*, 1, 32], but got input of size[32, 8, 32]

In [82]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        self.ln1 = nn.LayerNorm((context_size,head_size))
        self.ln2 = nn.LayerNorm((context_size,head_size))
        # self.ln1 = LayerNorm(head_size)
        # self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        self.ln1 = nn.LayerNorm(head_size)
        # self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  41,889
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3918  val: 4.386
train: 1.5908  val: 1.603
train: 0.9859  val: 1.033
train: 0.7461  val: 0.7851
train: 0.6235  val: 0.6596
done!


RuntimeError: Given normalized_shape=[8, 32], expected input with shape [*, 8, 32], but got input of size[1, 1, 32]

In [83]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        self.ln1 = nn.LayerNorm((context_size, head_size))
        self.ln2 = nn.LayerNorm((context_size, head_size))
        # self.ln1 = LayerNorm(head_size)
        # self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        self.ln1 = nn.LayerNorm(head_size)
        # self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  41,889
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3918  val: 4.386
train: 1.5908  val: 1.603
train: 0.9859  val: 1.033
train: 0.7461  val: 0.7851
train: 0.6235  val: 0.6596
done!


RuntimeError: Given normalized_shape=[8, 32], expected input with shape [*, 8, 32], but got input of size[1, 1, 32]

In [84]:
initial_token = torch.zeros(size=(1,8)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 









HAUKINT:
Hatheno's whreongon.
Weber
wigit ancthasawsde ung thipe thind whepele,:
Hal,
thinodis,
Lond -rrad slofu say:
Wind talel win, nategan.

Miyrl.
Bur
YORET:
Min yeane, ondsld birss, ghanotin noumbrats hiss,
A yoce nardd is the
I krompish tord sotel ponlse nolsarder briten nglo,

neim miflreth ang ulir thetoor thall ange wicar meac'themd bus
Tof cou coml ol, es hlanet cea, tra hamee hest renoor nove coum
By fread.
Sarme stornterey an do novoes. minenr fangs't frorwstne, I lexer
nus'dene Marl


In [85]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-6) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm((8,300))
print(f'{ln_torch}')
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean 0 and var 1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean0var1, like wise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean0 var1
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

LayerNorm((8, 300), eps=1e-05, elementwise_affine=True)
our's: (-1.5894572324981482e-09, 1.0004158020019531)
torch: (0.0, 1.0004069805145264)


In [86]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-6) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
print(f'{ln_torch}')
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean 0 and var 1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean0var1, like wise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean0 var1
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

LayerNorm((300,), eps=1e-05, elementwise_affine=True)
our's: (-3.973643192267673e-09, 1.0004159212112427)
torch: (-5.563100202721216e-09, 1.000407099723816)


In [87]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-6) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean 0 and var 1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean0var1, like wise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean0 var1
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (-1.5894572324981482e-09, 1.0004159212112427)
torch: (-1.5894572324981482e-09, 1.0004066228866577)


In [88]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-6) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (5.9604645663569045e-09, 1.0004158020019531)
torch: (3.973643192267673e-09, 1.0004067420959473)


In [89]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-6) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=True)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (1.5894572324981482e-09, 0.9970810413360596)
torch: (-3.973643192267673e-09, 1.0004061460494995)


In [90]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-6) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=True)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (-2.1855035559070757e-09, 0.9970811009407043)
torch: (2.38418573772492e-09, 1.0004067420959473)


In [91]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3857  val: 4.38
train: 2.2392  val: 2.245
train: 2.1061  val: 2.152
train: 2.0343  val: 2.101
train: 1.9811  val: 2.068
done!


In [92]:
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 


For the that mad's why.

BERGORDW:

Yidi
Warct, saws
Eul: hellow thinde
To
Wleet bay,
thing many.
For-----
sluve ster:
I how light king, what wiln alch nare, near in yeane, of lady. Ase, ghuing I thou have his whath;
'TKISed is the
blinshil his lordsell pandse: your dother the defoorresim thele this:
What nog thy, thall ange wicHe ming,
Behatins
To ting coml?

LUCELLA:
Gor foin'sabhanke hear revoot we stelf:
But seades,
The strenced,
Live Hen woes.
IDCHAUAful,
Nt from sing, I
GEXENDYur'd nefork 


In [93]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3857  val: 4.38
train: 2.2392  val: 2.245
train: 2.1061  val: 2.152
train: 2.0343  val: 2.101
train: 1.9811  val: 2.068
done!

For the that mad's why.

BERGORDW:

Yidi
Warct, saws
Eul: hellow thinde
To
Wleet bay,
thing many.
For-----
sluve ster:
I how light king, what wiln alch nare, near in yeane, of lady. Ase, ghuing I thou have his whath;
'TKISed is the
blinshil his lordsell pandse: your dother the defoorresim thele this:
What nog thy, thall ange wicHe ming,
Behatins
To ting coml?

LUCELLA:
Gor foin'sabhanke hear revoot we stelf:
But seades,
The strenced,
Live Hen woes.
IDCHAUAful,
Nt from sing, I
GEXENDYur'd nefork 


In [94]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (2.38418573772492e-09, 1.0004067420959473)
torch: (9.53674295089968e-09, 1.0004067420959473)


In [95]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# for now i'll be using pytorch's layernorm until I figure out whats wrong with mycode!
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we didnt include the 0ths dimension like pytorch's and when we included
# that the result got similar. 
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3906  val: 4.385
train: 2.2373  val: 2.244
train: 2.1070  val: 2.155
train: 2.0339  val: 2.102
train: 1.9812  val: 2.069
done!

For the that mad'
With of on tith, luidie anct, saws
Eule at botht if thou
Wlefted' you windian.

FLO-wike sluke slen:
I have, ladik, anteraves,
norl wink
the bard a yeane, of lady. Whe, goshot I
thy brath his what:
Nown hondin the
blinsh, shat this the pandself livadet brives nefors.
ISIUS:
Aren that usir the of me
he hange wicHe mirs,
Behat!


TOLAT:
But ford, engelan this it'sabhanke hear rend these this in usse,--
Lord, strence'er thy?
In woes. IDandry,
Soset frongs neved
With haul'd nefonk 


In [96]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (2.38418573772492e-09, 1.0004067420959473)
torch: (9.53674295089968e-09, 1.0004067420959473)


In [97]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
# our model with eps:1e-5 (like pytorch's) 
#
#
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we were aggregating the last two dims like BN, whereas we should have only used the
# last dim, as we should not involve other samples in normalization. (I explained this thoroughly in the LayerNorm class)
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3906  val: 4.385
train: 2.2373  val: 2.244
train: 2.1070  val: 2.155
train: 2.0339  val: 2.102
train: 1.9812  val: 2.069
done!

For the that mad'
With of on tith, luidie anct, saws
Eule at botht if thou
Wlefted' you windian.

FLO-wike sluke slen:
I have, ladik, anteraves,
norl wink
the bard a yeane, of lady. Whe, goshot I
thy brath his what:
Nown hondin the
blinsh, shat this the pandself livadet brives nefors.
ISIUS:
Aren that usir the of me
he hange wicHe mirs,
Behat!


TOLAT:
But ford, engelan this it'sabhanke hear rend these this in usse,--
Lord, strence'er thy?
In woes. IDandry,
Soset frongs neved
With haul'd nefonk 


In [98]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 256
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

using more blocks with skip-connection-layernorm


RuntimeError: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero.

In [1]:
# in the name of God the most compassionate the most merciful
# in this section we will have a look at how a GPT model works
# in this section we will implement attention module, and see 
# how it works and lean more about its (sel-attention, cross
# attention, multi-head attention, etc)
# 
# so how are we going about this. we need to create a language model
# and then progressively imporve it with attention mechanism.
# we will basically be creating a kind of chatgpt (minus its chat capability and
# its obviously great perormance :)) 
# to keep this as simple as possible and not lose track of the important concepts involved,
# we can use our initial bigram model as the base model and work on improving that with attention.
# so lets start
#
# first lets import the basic stuff 

# for type hints
from collections.abc import Iterable

import random 
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# lets setup the manual seeds or determinstic output
torch.manual_seed(255)
np.random.seed(255)
random.seed(255)

# first lets read the dataset, we are going to use tinyshalespear which
# is around 1million characters (around 1MB in size), and our model is
# supposed to create texts resembling this dataset.
dataset = []
with open('./tiny_shakespear.txt','r') as file:
    # this time we read the whole text as one big str
    dataset = file.read()
    
print(f'{dataset[:100]=}')
# now lets create our atoi and itoa dictionaries for mapping
# lets create our unique character list, which is infact our vocabulary or vocab for short
vocab_list = sorted(set(''.join(dataset)))
vocab_size = len(vocab_list)
# ! explain token
# lets create our mapping dictionaries, we are basically going to use them
# for tokenization, converting our input into tokens which here are characters 
# and ultimately their integer representations
# so tokenization simply put, refers to converting an input into a list of numbers based on some creteria
# here, we just use a simple index number to map our vocabulary characters, which represent all character
# our model can use to generate text, to integer representation. 
# for realworld applications, we usually use libraries such as sentecepeace by google which works on a subword
# level which means, it doesnt encode the whole word, neither does it encode based on individual characters, it
# use sth in between. 
# (from its github repo: https://github.com/google/sentencepiece )
# SentencePiece is a re-implementation of sub-word units, an effective way to alleviate the open vocabulary 
# problems in neural machine translation. SentencePiece supports two segmentation algorithms, 
# byte-pair-encoding (BPE) [Sennrich et al.] and unigram language model [Kudo.]. 
#
# (For the reference:  
# Byte Pair Encoding (BPE) is a subword tokenization technique used in natural language processing (NLP) 
# and text processing tasks. It is a data compression algorithm that splits words into subword units. 
# BPE is commonly used in tasks such as machine translation, text generation, and language modeling.
# The basic idea behind BPE is to iteratively merge the most frequent pairs of characters or subword units
# in a corpus to create a new subword vocabulary. This merging process is based on the statistical properties
# of the corpus, specifically the frequency of character or subword pairs.
#
# Here's a high-level overview of the BPE algorithm:
# 1. Initialize the vocabulary with all the characters or subwords in the corpus.
# 2. Calculate the frequency of each character or subword in the corpus.
# 3. While the desired vocabulary size or a maximum number of iterations is not reached:
#    - Find the most frequent pair of characters or subwords in the corpus.
#    - Merge the pair into a new subword unit by concatenating them.
#    - Update the corpus by replacing occurrences of the merged pair with the new subword unit.
#    - Update the vocabulary and frequency counts based on the new corpus.
# 4. The final vocabulary is the set of subword units obtained after the desired number of iterations 
#    or vocabulary size is reached.
#
# BPE allows for the representation of both known and unknown words in a corpus. It is effective in 
# handling out-of-vocabulary (OOV) words and reducing the vocabulary size, which can improve the 
# efficiency and performance of NLP models. By breaking down words into subword units, BPE can capture
# morphological and semantic information more effectively, especially for languages with complex word 
# formations and agglutinative structures.
# 
# tiktokenize repo explains BPE in rather friendlier way: 
# Models don't see text like you and I, instead they see a sequence of numbers (known as tokens). 
# Byte pair encoding (BPE) is a way of converting text into tokens. It has a couple desirable properties:
# It's reversible and lossless, so you can convert tokens back into the original text
# It works on arbitrary text, even text that is not in the tokeniser's training data
# It compresses the text: the token sequence is shorter than the bytes corresponding to the original text. 
# On average, in practice, each token corresponds to about 4 bytes.
# It attempts to let the model see common subwords. For instance, "ing" is a common subword in English, 
# so BPE encodings will often split "encoding" into tokens like "encod" and "ing" 
# (instead of e.g. "enc" and "oding"). Because the model will then see the "ing" token again and again 
# in different contexts, it helps models generalise and better understand grammar.
#
# 
# A unigram language model however, is a type of statistical language model that predicts the probability 
# of each word in a sequence independently, based solely on the frequency of occurrence of individual 
# words in the training data. It does not consider the context or the order of the words in the sequence.
# In a unigram language model, the probability of a particular word is estimated by counting the frequency 
# of that word in the training corpus and normalizing it by the total number of words in the corpus. 
# The probability of a sequence of words is then calculated by multiplying the probabilities of each 
# individual word in the sequence.
# For example, consider the sentence "I love to eat pizza." In a unigram language model, the probability
# of this sentence would be calculated as the product of the probabilities of each word which would be: 
# P(I) * P(love) * P(to) * P(eat) * P(pizza).
# Unigram models are the simplest form of language models and do not capture any contextual information
# or dependencies between words. They are often used as a baseline or reference model in natural language 
# processing tasks. While unigram models are not very accurate in capturing the complexities of natural 
# language, they can be computationally efficient and useful in tasks where the context is less important, 
# such as certain text classification tasks or language generation tasks where only the frequency of 
# individual words matters.
# 
#
# Openai/chatgpt for example uses its own tokenizer, called tiktoken (https://github.com/openai/tiktoken)
# there are other tokenizers as well.
# huggingface has a good article on tokenizers (recommend it): https://huggingface.co/docs/transformers/main/tokenizer_summary 
# its a given that, each model only works with the tokenizer by which it was used to train. 
# so BERT, DistilBERT, and Electra only work with wordpiece, while chatgpt uses titoken and we! use
# simple characters-indexes! also note that the choice of tokenizer obviously affects our vocab size and
# model overhead/performance as well. 
# for example lets first implement our tokenizer and then compare it with sth like tiktoken module
atoi = {c:n for n,c in enumerate(vocab_list)}
itoa = {n:c for c,n in atoi.items()}
print(f'{vocab_size=}, {vocab_list=}')
print(f'{atoi=}')
print(f'{itoa=}')
# lets also create two helper functions to convert a list of these to the other part
def encode(characters:Iterable[str] ):
    return [atoi[c] for c in characters]

def decode(token_lst:Iterable[int]):
    return [itoa[n] for n in token_lst]

# lets test these 
print(f'{encode(dataset[:10])}')
print(f'{decode(encode(dataset[:10]))}')

# now lets try tiktoken
try: 
    import tiktoken
except:
    import os 
    os.system('pip install tiktoken')
    import tiktoken
# lets use the tokenizer for gpt2 model, this line downloads the gpt2 tokenizer and allows us to use it
encoder_gpt = tiktoken.get_encoding('gpt2')
# lets view its vocab size:
print(f'{encoder_gpt.n_vocab=:,}') # prints encoder_gpt.n_vocab=50,257
# now lets see how the encoding looks like here
print(f'{encoder_gpt.encode(dataset[:10])=}') # prints [5962, 327, 8846] 
# which is very intresting! while for us its 10 numbers/tokens for 10 characters(vocab size 65),
# this is 3 here with vocab size of 50,000!
# so we can have a short vocab size, at the expense of a larger sequence size, 
# or a large vocabsize and smaller sequence size.
# so thats why in practice, these subwords tokenizers are used for real world applications. but for our case
# we stick to our primitive tokenizer to keep things as simple as possible. 
# 
# Ok, so far so good. now we need to create a dataset, like before we need to have an input/label pair
# our input is a series of characters, (a sequence of some length), and our label is the next character
# sicne we are using a simple bigram model, given a single character, we want the probablity of what comes
# next. but, we also want to incorporate attention, and we want a context for our prediction, we want to
# be able to look at the past, and look at the past characters, and based on that do sth. this is the essence
# of attention (although this is not accurate, but for now this is the case, we will elaborate on this and expand
# this metaphor and reasoning inshallah)
# 
# lets first tokenize the whle dataset or corpus as its usually called in nlp nomenclature!
data = encode(dataset)
# since we are using pytorch lets convert that to a tensor
data_tensor = torch.tensor(data)
print(data_tensor[:100])
# now lets create a train/val split 
train_length = int(0.9 * len(data_tensor))
# note that we do not shuffle the data here like before, because we are not dealing with a list of samples!
# like names, here we are dealing with the whole text, and if we shuffle it like that, we just destroy it
# making it into a batch of random characters! which would not be useable for us anymore. we are trying to
# learn the underlying semantic and relationships hidden in our data, and randomizing them like that just 
# destroys those information! 
# to see this in action try this
_data_copy = data.copy()
random.shuffle(_data_copy)
print(''.join(decode(_data_copy[:100])))
# prints:
# Osetow Pr 
# oa geEeM rhnalelrt   aAdt Ktsw pTpeGxli
# oasEliMe
# vsieeose
# etttbSdnctiras  hnr:yhtayiuCv g
# so we simple divide the data normally
train_data = data_tensor[:train_length]
val_data = data_tensor[train_length:] 
# note that since we are planning on creating a simple transformer model, we usually dont feed the whole dataset
# becaue its prohibitevly computation intensive, instead, what happens in practice is that we, grab chunks 
# of data from the dataset and feed it to the transformer. and these chunks, of course has a length, what length?
# we usually specify a maximum_length for the input on which our transformer model works.
# this maximum_length is usually refered to as block_size or context_size.
# we had previously used different context_sizes and this is not really that different,
# lets for example define a context_size of 8
# block_size and context_size are interchangable
block_size = context_size = 8 
print(f'{train_data[:block_size]=}')
# which prints:
# train_data[:block_size]=tensor([18, 47, 56, 57, 58,  1, 15, 47])
# 
# one intresting observation we can make here is that, as simple as this seemingly ordindary list of numbers
# looks, this sequence of numbers, actually contains several examples. 
# if you think about it, each number, is a token, representing a character (or word, subword, etc),
# and it shows what comes after what. basically not only it shows several pairs so to speak, it also
# shows, which characters are more likely to come, before a specific character comes later. 
# what we are actually going to do is that, we are going to train all of these characters simultaneously
# notice that in this example, we have 7 examples in a sequence of 8 characters:
# lets elaborate on this more. 
# 1-in the context of 18, the next character is 47
# 2-in the context of 18,47, the next character is 56
# 3-in the context of 18,47,56, the next character is 57
# 4-in the context of 18,47,56,57 the next character is 58
# 5-in the context of 18,47,56,57,58 the next character is 1
# 6-in the context of 18,47,56,57,58,1 the next character is 15
# 7-and finally, in the context of 18,47,56,57,58,1,15 the next character is 47
# so this is infact 8 7 individual example embedded in a single context, 
# since we want the xontext_size to be 8, then we should grab one more character to have 
# 8 contexts
# lets visualize this in example in code
# lets have a typical sequence of size 8 
x = train_data[:block_size]
y = train_data[1:block_size+1]
# what would be our label? we want the next character so it would be the previous locations +1
# basically offset the input by 1!
# thats why we started from 1, becasue the input started from 0, and since the input ended at block_size
# its label(next character) would obviously be block_size+1, pretty obvious right?
#
# lets print this for better understanding 
for i in range(block_size):
    # note that since we are using slice, x[:i+1], gives us 0 up to ith element (inclusive)
    # this should be obvious! but I said it in case you forgot!!
    print(f'input is {x[:i+1].tolist()} label is {y[i]}')
# prints 
# input is [18] label is 47
# input is [18, 47] label is 56
# input is [18, 47, 56] label is 57
# input is [18, 47, 56, 57] label is 58
# input is [18, 47, 56, 57, 58] label is 1
# input is [18, 47, 56, 57, 58, 1] label is 15
# input is [18, 47, 56, 57, 58, 1, 15] label is 47
# input is [18, 47, 56, 57, 58, 1, 15, 47] label is 58
# so as you can see we started from context_size of 1 up to context_size of 8.
# note that we do not do this simply for the efficancy aspect of it, but also, when we feed these to
# our transformer model, we are making sure that the transformer model gets used to see all combinations
# of our input as well (from the context size of 1 up tp the context size of 8). 
# this way not only it sees the whole context_size as we initially expected
# but also all sequences before it, and basically what consituted to make the sample. 
# this allows us to later on, at test time be able to create sequences as small as context_size of only 1
# up to the max_length which is our context_size of 8 in our case.
# ! recheck and elaborate to clear any confusion
# !so by doing this, the transformer can learn how to predict/create/genrate text up to context_size, and after
# !it reached thta, we have to truncate it, becausse the transformer model never recieves more than the
# !context_size as input when its predicting the next character.
# so far what we covered here was the time dimension of our input. we have a sequence, and each entery
# basically denotes a time t dimension, at which, a character is introduced. 
# another imporatnt aspect we need to take care of is the batch dimension, cuz we are going to feed 
# multiple examples at once, we use this to harness the gpu parallilization capalibity in pytorch
# as without it, training this simple model would take a lot of time on cpu! 
#
# so lets create a function that gives us a batch of the inputs rather than a single list/tensor!
# 
batch_size = 4 
# our block_size
context_size = 8 
# 
def get_batch(split, batch_size):
    #lets grab the data basedo on the split
    data = train_data if split =='train' else val_data
    # we want a batch of 4 of 8 characters (context-size). 
    # to make a batch we can grab 4 random indices as input and then expand them
    # by adding the next 8(context_size) characters to them, in order not to go past the 
    # last index, we subtract the length of data(last valid index) from context size 
    idxs = torch.randint(low=0, high=len(data)-context_size, size=(batch_size,))
    # we have our indices, so lets create our samples
    # x = [data[i:i+context_size] for i in idxs]
    # and for labels we do the same thing but offset it by 1 so each characters label
    # becomes the next character
    # y = [data[i+1:i+context_size+1] for i in idxs]
    #
    # print(f'{x=}')
    # print(f'{y=}')
    # prints: 
    # x=[tensor([58, 39, 49, 43,  1, 51, 63,  1]), tensor([53, 59, 41, 46,  5, 42,  1, 61]), tensor([47, 53, 52,  1, 39, 57,  1, 63]), tensor([39, 57, 58,  1, 51, 63,  1, 50])]
    # y=[tensor([39, 49, 43,  1, 51, 63,  1, 54]), tensor([59, 41, 46,  5, 42,  1, 61, 47]), tensor([53, 52,  1, 39, 57,  1, 63, 53]), tensor([57, 58,  1, 51, 63,  1, 50, 53])]
    #
    # as you can see we have a list of tensors. since we want a batch, we just stack them or concat them
    # on top of each other
    # using stack!
    # x = torch.stack(x)
    # y = torch.stack(y)
    # stack() is intrestingly implemented by concat(), so if we want, we can do the same using concat!
    #
    # by default conact, concatenates all the tensors as one large tensor!
    # we need a reshape/view to get the right shape
    # since we are using view, no copy,etc is done, and its an efficient operation
    # x = torch.concat(x,dim=0).view(batch_size, context_size)
    # y = torch.concat(y,dim=0).view(batch_size, context_size)
    # 
    # however if we add an extra dim to our inputs we can remove the extra reshape/view
    # x = torch.concat([t.unsqueeze(0) for t in x], dim=0)
    # y = torch.concat([t.unsqueeze(0) for t in y], dim=0)
    #
    # By unsqueezing, we ensure that the tensors have the same number of dimensions before 
    # concatenating them with torch.cat/concat/concatenate.(side tip: concat and concatenate are aliases for cat) 
    # This effectively replicates the behavior of torch.stack.
    # see torch.cat concatenates tensors along an existing dimension, and we only have 1 dimensional
    # tensors, so it will concatenate them along that, effectively making one large 1 dimensional vector
    # however, when we use unsqueeze on each tensor, using torch.unsqueeze(0), we are adding a new dimension
    # (dim 0) to that tensor, making it 1,x instead of the original shape of (x,).  
    # So, by unsqueezing each tensor along the desired dimension, which for us is the 0ths dimension (or row dim) 
    # and then concatenating them with torch.cat, we achieve the same result as torch.stack.
    # 
    # in practice we'd like to do this all in one go!
    x = torch.stack([data[i:i+context_size] for i in idxs])
    y = torch.stack([data[i+1:i+context_size+1] for i in idxs])
    
    return x,y

x,y  = get_batch('train',batch_size=4)
print(f'{x.shape=}\n{y.shape=}')
print(f'{x=}\n{y=}')
# which prints 
# x.shape=torch.Size([4, 8])
# y.shape=torch.Size([4, 8])
# tensor([[58, 39, 49, 43,  1, 51, 63,  1],
#         [53, 59, 41, 46,  5, 42,  1, 61],
#         [47, 53, 52,  1, 39, 57,  1, 63],
#         [39, 57, 58,  1, 51, 63,  1, 50]])
# tensor([[39, 49, 43,  1, 51, 63,  1, 54],
#         [59, 41, 46,  5, 42,  1, 61, 47],
#         [53, 52,  1, 39, 57,  1, 63, 53],
#         [57, 58,  1, 51, 63,  1, 50, 53]])
#
# now this is our batch of data, 4 samples with 8 characters, bascially 32 examples, lets see that as well
for b in range(batch_size):
    # this is our time dimension
    for t in range(context_size):
        print(f'{x[b,:t+1]} --> {y[b,t:t+1].item()}')
#        
# prints : 
# tensor([58]) --> 39
# tensor([58, 39]) --> 49
# tensor([58, 39, 49]) --> 43
# tensor([58, 39, 49, 43]) --> 1
# tensor([58, 39, 49, 43,  1]) --> 51
# tensor([58, 39, 49, 43,  1, 51]) --> 63
# tensor([58, 39, 49, 43,  1, 51, 63]) --> 1
# tensor([58, 39, 49, 43,  1, 51, 63,  1]) --> 54
# tensor([53]) --> 59
# tensor([53, 59]) --> 41
# tensor([53, 59, 41]) --> 46
# tensor([53, 59, 41, 46]) --> 5
# tensor([53, 59, 41, 46,  5]) --> 42
# tensor([53, 59, 41, 46,  5, 42]) --> 1
# tensor([53, 59, 41, 46,  5, 42,  1]) --> 61
# tensor([53, 59, 41, 46,  5, 42,  1, 61]) --> 47
# tensor([47]) --> 53
# tensor([47, 53]) --> 52
# tensor([47, 53, 52]) --> 1
# tensor([47, 53, 52,  1]) --> 39
# tensor([47, 53, 52,  1, 39]) --> 57
# tensor([47, 53, 52,  1, 39, 57]) --> 1
# tensor([47, 53, 52,  1, 39, 57,  1]) --> 63
# tensor([47, 53, 52,  1, 39, 57,  1, 63]) --> 53
# tensor([39]) --> 57
# tensor([39, 57]) --> 58
# tensor([39, 57, 58]) --> 1
# tensor([39, 57, 58,  1]) --> 51
# tensor([39, 57, 58,  1, 51]) --> 63
# tensor([39, 57, 58,  1, 51, 63]) --> 1
# tensor([39, 57, 58,  1, 51, 63,  1]) --> 50
# tensor([39, 57, 58,  1, 51, 63,  1, 50]) --> 53 
#
# so now that we have the data sorted out, lets create our model. 
# as we said, we are going to use a bigram model and later add attention mechanism to it. 
# to make things easier and more self contained, lets add all the required logic to this model
# like when we do a forward, we be able to calculate loss as well if we are given the targets
# so lets go
class BigramModel(nn.Module):
    def __init__(self, vocab_size) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        # our bigram model was nothing more than a 2d array of vocab_size, we can achive that using 
        # a single weight matrix or torch.Embedding. we use torch.Embedding to not reinvent the wheel!
        self.token_embedding = torch.nn.Embedding(vocab_size, vocab_size)
    
    # since we want to be able to calculate loss, if there are labels, we get Y as well
    def __call__(self, inputs:torch.Tensor, labels:torch.Tensor=None) -> torch.Tensor:
        logits = self.token_embedding(inputs)
        loss = None
        if labels is not None:
            # calculate loss 
            # print(f'{logits.shape=}') # prints (4,8,65)
            # note that both inputs and labels have the shape (B,T)
            # but logits has the shape (B,T,C) which C here equals vocab_size 
            # this is an issue for torch.crossentropy, as it expects the input
            # to be in the form of (B,C,T), that is, the channels/embeddings dimension
            # need to be right after the batch dimension or otherwise it wont work.
            # so we need to account for that.
            # we can go on permute the dimensions, like this and make crossentropy happy 
            # logits = logits.permute((0,2,1))
            # however, if for some reason the output of logits in the form of (4,8,65) is not ideal, 
            # maybe for example because really what it is 32 examples arranged in a (4,8) shape and 
            # we rather a normal 2d tensor of shape (32,65), 
            # we can instead, do just that, flatten the two dimensions into one and carry on!
            # we basically are concatenating the batch and time dimensions
            # into one dimension (effectively, stacking samples on top of each other), instead of having 
            # four compartments, each having 8 segments, we are going to have 1 long compartment with 32 segments/rows
            # for the lack of better words!!
            # so lets first get the shapes 
            # B,T,C = logits.shape
            # logits = logits.view(B*T,C)
            # and we also need to do the same for the labels 
            # labels = labels.view(B*T)
            # we could also do,
            # labels = labels.view(-1)
            # but thats not really needed, so we just simply permute logits temporarily so the logits shape
            # stays the same regardless of calculating the loss or not 
            loss = F.cross_entropy(logits.permute((0,2,1)), labels)
        return logits, loss
    
    # now lets also add a generate method to generate texts
    def generate(self, idx, max_token_count):
        # before we implement this lets review what we expect this method to do
        # we want this to generate some characters, given an initial character
        # we also want to be able to control the length of the generated text
        # so we take a max_token_count.
        # we take the initial index, 
        # feed it to the model, 
        # get the logits,
        # turn logits to probs,
        # use torch.multinomial to sample from our probablity distribution
        # get the new character index, and add it to a list to gradually 
        # -create our final text output
        # so lets go 
        # our idx should have the shape [B,T]
        assert len(idx.shape) == 2, f'idx.shape({idx.shape}) should have (b,t) form.'
        #
        # lets generate as many characters/idx as max_token_count specifies
        # this is basically specifies the time/sequence dimensions
        for i in range(max_token_count):
            # since we are defining a class method, to call the callable, we simply use self()
            logits, _ = self(idx)
            # to get the probablities 
            # usually we would simply do 
            # probs = torch.softmax(logits, dim=1)
            # but here, we want to only focus on the next character because this is what comes next
            # each time obviously, so we only take the last timestep to see what the model predicted
            logits = logits[:,-1,:] # this now becomes (B,C) instead of the initial (B,T,C)
            probs = torch.softmax(logits, dim=-1) 
            # now lets sample from it
            idx_next_char = torch.multinomial(probs, num_samples=1, replacement=True) # shape is (B,1)
            # now lets add this to the next input to be fed to the model 
            # since idx has the shape(batch, T), we should add this tothe second dimension
            # to the time dimension/ or sequence dimension. this as the loop goes on, 
            # creates the shape (B,T+1) 
            #this doesnt make sense for this particular model, becasue we are always checking
            # the next character given the previous one, so all the concatenation we are doing
            # is just useless. the reason we are implementing this like this, is to create a 
            # base, so that we can improve upon it when we add attention later on which will use
            # the history of previous characters.
            idx = torch.cat((idx, idx_next_char), dim=1) 
        
        # and finally when all is done return the idx which by now should have the whole output
        return idx
    
model = BigramModel(vocab_size)
out,loss = model(x,y)
print(f'{out.shape} {loss}')
# prints:
# torch.Size([32, 65]) 4.574285984039307
# the loss is good, we learned previously that we can evaluate a base loss provided our number of classes
# since we have 65 classes (our vocab_size or number of characters involved) the uniform probability for each class
# would be 1/65 =0.015384615, which if we take its negative log would turn out to be -ln(1/65) = 4.17438727, 
# which is pretty close to the loss we got here, signifying its a pretty decent value to begin with.
# also lets see the generate method at work
# lets create a dummy input, basically a batch of 1 and sequence of 1 of zero!
# we do this to generate a text from scratch
inputs = torch.zeros(size=(1,1),dtype=torch.int32)
output = model.generate(inputs, max_token_count=100)
print(f'{output=}')
print(''.join(decode(output.squeeze(0).tolist())))
# prints 
# wUbN:i :kLnOqHCQTG;.MbupOWH Xfi!MSUalaQppNNqIoWfmuSIIfmuZqf-3NLt-YkOc3vC-YkBfEHr3pJodwVY.nw.,r!&,Clt
# which is expected since our model is not trained yet! 
# lets train the model now and see how it works

batch_size = 32
context_size = 8
vocab_size = len(vocab_list)
max_iter = 20000

model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device=device)

print(f'{torch.__config__.show()}')
print(f'{device=}')
print(f'{model=}')

model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        print(f'{loss}')
# prints 
# 4.760574817657471
# 3.727213144302368
# 3.0139858722686768
# 2.6749324798583984
# 2.5884087085723877
# 2.5262675285339355
# 2.4420430660247803
# 2.4787800312042236
# 2.4680707454681396
# 2.564662456512451
# 2.51802659034729
# 2.562812328338623
# 2.422783613204956
# 2.5023772716522217
# 2.501049518585205
# 2.411632537841797
# 2.4028165340423584
# 2.4135568141937256
# 2.50952410697937
# 2.574878215789795
# relying on single batch loss is not a good idea to measure the performance of a model. moreover
# relying on training loss, is not good either, so it would be much better if we considered more batches
# for loss and even better we could also investivate the models performance on our validation set. 
# so lets do just this and define a function that calculates loss for training and validation sets alike
# but considers more batches for loss calculation

@torch.no_grad()
def evaluate_loss (iterations, device=None):
    results={}
    # before calculating the loss, lets switch to eval mode,although for our specific case this doesnt matter
    # but its goo practice, as later on, we will add layers that their behavior do change depending on traing
    # val mode.  
    model.eval()
    if device is None:
        # use the device assigned to what model params are assigned
        device = next(model.parameters()).device
    # since we already used no_grad decorator, we dont need to use no_grad context manager here
    # with torch.no_grad():
    for split in ['train','val']:
        losses = torch.zeros(size=(iterations,), device=device)
        for i in range(iterations):
            x,y = get_batch(split,batch_size)
            x,y = tuple(t.to(device) for t in (x,y))
            logits, loss = model(x,y)
            losses[i] += loss
        results[split] = losses.mean(0)
    # since we want to use this inside training loop, make sure we set the model back to train mode
    # incase we use layers such as batchnorm,etc that the require being trained!
    model.train()
    return results

loss= evaluate_loss(200)
print(f'{loss["train"]=} {loss["val"]=}')
# so now that our function seems to be working lets use it in the training loop :
model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
model = model.to(device=device)
model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        # now instead of simply printing loss, lets use our new function!
        loss= evaluate_loss(200)
        print(f'train_loss: {loss["train"].item():.4f},  val_loss: {loss["val"].item():.4f}')
# which results in :
# train_loss: 4.7491,  val_loss: 4.7430
# train_loss: 3.6673,  val_loss: 3.6670
# train_loss: 3.0460,  val_loss: 3.0511
# train_loss: 2.7335,  val_loss: 2.7492
# train_loss: 2.5948,  val_loss: 2.6156
# train_loss: 2.5292,  val_loss: 2.5470
# train_loss: 2.4982,  val_loss: 2.5194
# train_loss: 2.4807,  val_loss: 2.4991
# train_loss: 2.4737,  val_loss: 2.5036
# train_loss: 2.4650,  val_loss: 2.4949
# train_loss: 2.4648,  val_loss: 2.4910
# train_loss: 2.4626,  val_loss: 2.4836
# train_loss: 2.4570,  val_loss: 2.4893
# train_loss: 2.4517,  val_loss: 2.4823
# train_loss: 2.4508,  val_loss: 2.4819
# train_loss: 2.4564,  val_loss: 2.4839
# train_loss: 2.4555,  val_loss: 2.4802
# train_loss: 2.4503,  val_loss: 2.4927
# train_loss: 2.4522,  val_loss: 2.4905
# train_loss: 2.4572,  val_loss: 2.4888
#
# now lets check its output again after some training 
inputs = torch.zeros(size=(1,1), device=device, dtype=torch.int32)
output = model.generate(inputs, max_token_count=500)
print(''.join(decode(output.squeeze(0).tolist())))
# prints :
# Ane pod BUL:
#
# Oringe
# nfote s Rinsou oe the,
# An ETwe
#
# PUSENI bmilft!'d ate t Ifur
# Athomeg?
# Whagre,
# TCo belangead ne bl e, bednth ftor veso.
# US tla lin:
# Yourthiee he w whetitrd;
# Angoknghothartro ll heshethor hie douluk t, s se, denopit s h wom uspouct blyowie s'nos t prou gmesat
# Shd, tieithy.
# Asiour hofo bay avear hanousthaie
# Calonth wolulo.
#
# The hancondors pthighar:
# WAME outhaser d!
# S:
# h im t w s, inowo ORILOUCHoou isecetoubecompis
# Ay gnd, teve luthorea, therpeclsthevecthede fichim?
# PUSa VI aken 
# 
# which looks much better than the initial random output we got earlier.

dataset[:100]='First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
vocab_size=65, vocab_list=['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
atoi={'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't':

In [2]:
torch.manual_seed(255)
# now lets add attention mechanism to our base model. 
# attention mechanism at its core tries to take advantage of the rich information embedded
# in the sequence. for our work, this is specifically about the past history but in general 
# attention can utilize both past and future connections/sequence tokens. what we described 
# here just now is not exactly accurate, but gives us a foundation to build our intuition as
# we continue on. we will elaborate more of course and hopefully get it all.
# before we venture any further into the crux of the matter, let us learn about a technique
# thats used to efficiently implement attention mechanism.
# for this purpose,lets imagine we have a simple input like the following: 
# lets create and input of the following shape
B,T,C = (4,8,2)
# to make it more intuitive lets make a tensor with known numbers andthen reshape it
x = torch.arange(0,64,dtype=torch.float).view(B,T,C)
# imagine we have an input like what we encountered previously in our examples. in this sample 
# input, we have a batch of 4 samples, each having 8 sequences with each sequence having a vector of 2 values
# what we are planning to do is to provide a way by which each token can communicate with other 
# tokens. we have 8 tokens in our sequence. so we want our tokens to be able to communicate with
# all previous tokens that came before it. the reason we are only looking in the past token is 
# simply becasue the we are trying to perdict the future, so it only makes sense to look at the
# past and current timestamp and infer on what to do for the future.  
# so the easiest way to implement a kind of communication between tokens could be to sum or average the
# values of all previous tokens plus the current one as a way of taking into account their contribution
# to the final answer.
# that is, lets say if we are currently at token 5, we take the average of 
# the current token and all previous tokens before it, effectively making a feature vector that
# reflects our current status of the sequence so far, having taken all previous tokens/steps up to now.
# note that as you may also have thought, summing or averaging arent the best way to model such interations.
# in fact they are an extremely weak form of interaction between tokens,
# this kind of communicating is extremely lossy so to speak, that is we lose a great deal of information 
# concerning the underlying relationships between tokens, their arrangements,their implicit interactions,
# semantics, etc. but for now this is ok. we will later on see how to bring back such information.
# so now what we want to do, is to calculate the sum or average of all tokens up to the current token in 
# all batches at the same time.
# a naive way would be to do sth like this using a for loop:
results = torch.zeros(size=(B,T,C))
for b in range(B):
    for t in range(T):
        results[b,t] = x[b,:t+1].mean(0)
print(f'{results=}')
# which prints 
# results=tensor(
#        [[[ 0.,  1.],
#          [ 1.,  2.],
#          [ 2.,  3.],
#          [ 3.,  4.],
#          [ 4.,  5.],
#          [ 5.,  6.],
#          [ 6.,  7.],
#          [ 7.,  8.]],

#         [[16., 17.],
#          [17., 18.],
#          [18., 19.],
#          [19., 20.],
#          [20., 21.],
#          [21., 22.],
#          [22., 23.],
#          [23., 24.]],

#         [[32., 33.],
#          [33., 34.],
#          [34., 35.],
#          [35., 36.],
#          [36., 37.],
#          [37., 38.],
#          [38., 39.],
#          [39., 40.]],

#         [[48., 49.],
#          [49., 50.],
#          [50., 51.],
#          [51., 52.],
#          [52., 53.],
#          [53., 54.],
#          [54., 55.],
#          [55., 56.]]])
#
# You may find out that, some researchers refer to this operation here as BoW, or bag of words. 
# We are effectively averaging embeddings here and averaging embeddings can be considered a form of 
# Bag of Words (BoW) representation. In BoW, the focus is on the occurrence and frequency of words, 
# rather than their order or structure. By averaging embeddings, we are essentially treating each word 
# as an independent feature and capturing its representation in the form of a numerical vector.
#
# While averaging embeddings does not capture the exact frequency of each word, it does capture the 
# overall distribution and semantic information present in the text. Similar to BoW, this approach 
# disregards word order and focuses on the presence and representation of words. However, it should be
# noted that averaging embeddings may preserve some semantic relationships between words, which BoW 
# representations might not capture as effectively.
# 
# Side note: 
# Bag of Words (BoW) is a commonly used technique in natural language processing (NLP) for representing text
# as a numerical feature vector. It disregards the order and structure of words in a document and focuses only
# on their occurrence and frequency.
# In the BoW model, a document or a piece of text is represented as a "bag" (unordered set) of words, where 
# each word is treated as an independent feature. The presence or absence of words in the document is encoded
# as a binary value (0 or 1), and the frequency of each word is often used as the value in the feature vector.
# 
# Here's a step-by-step overview of the BoW process:
# 1. Tokenization: The text is split into individual words or tokens. Punctuation marks, whitespace, and other
#    special characters are usually removed or treated as separate tokens.
# 2. Vocabulary Creation: A vocabulary is created by taking all unique words from the entire corpus 
#    (collection of documents). Each unique word is assigned a unique index or position in the vocabulary.
# 3. Vectorization: Each document is represented as a feature vector, typically a one-hot encoding or a count
#    vector. In a one-hot encoding, each word in the vocabulary corresponds to a binary feature, and the vector
#    contains 1s in the positions where the word occurs and 0s elsewhere. In a count vector, the value at each 
#    position represents the frequency of the corresponding word in the document.
# 4. Classification or Analysis: The resulting feature vectors can be used as input to machine learning models
#    for tasks such as text classification, sentiment analysis, document clustering, or information retrieval.
# Needless to say, BoW has some limitations. It does not capture the semantic meaning or context of words, as
# it treats each word independently. It also ignores the grammar and word order. However, BoW is simple, 
# efficient, and can be a useful baseline representation for various NLP tasks.
# 
# so to recap one more time, we are basiaclly treating each timestep/sequence dimension, as a word, so we have
# 8 tokens/words, and we are averaging them (we are infact averaging their embeddings, but thats obvious!)
# so we got ourselves bow representation!
# now back to our discussion, if we  try to visualize the results we get 
print(x[0])
print(results[0])
# prints:
# x[0]=
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
#         [ 6.,  7.],
#         [ 8.,  9.],
#         [10., 11.],
#         [12., 13.],
#         [14., 15.]])
# results[0]=
# tensor([[0., 1.],
#         [1., 2.],
#         [2., 3.],
#         [3., 4.],
#         [4., 5.],
#         [5., 6.],
#         [6., 7.],
#         [7., 8.]])
# if you look closely, you'll notice that each row, contains the mean of all the rows before it
# consider the first 3 rows in x[0], 
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
# now refer to the 3rd row in results[0] which is :
#         [2., 3.],
# likewise, consider the last row in results which contains the average for all the rows in x:
# 0+2+4+6+8+10+12+14 = 56 which when divided by their count, 8, results in 7, 
# and this is the same for the second column, thus we get:
# results[0,7] = [7., 8.]])
# this all good but the problem is using for loops to calculate this is very inefficient, it
# happens that this operation can be efficiently calculated using matrix multiplication.
# lets learn this trick using an example: 
# suppose we have the following as the input: 
a = torch.ones(size=(3,3))
b = torch.randint(0,10,size=(3,2)).float()
c = a@b
print(f'{a=}')
print(f'{b=}')
print(f'{c=}\n----')
# prints
# a=tensor(
#        [[1., 1., 1.],
#         [1., 1., 1.],
#         [1., 1., 1.]])
# b=tensor(
#        [[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c=tensor(
#        [[20., 13.],
#         [20., 13.],
#         [20., 13.]])
# nothing fancy here, we have matrix multiplication, the first row of 'a' is dot-producted by first col
# of 'b', then sumed, it makes up the first col of first row in c. likewise the first row of 'a' dot 
# the second col of 'b', then summed the results, makes up the second col of first row in c. and this goes on for the rest of the matrixes. this is
# what we learned back in higheschool, so what is it exactly that we are learning exactly?
# if you look closely, you'll notice that, the c cols are actually the sum of all the rows in b!
#        [9]
# b[:,0]=[5] 
#        [6]
# 9+5+6 is 20! likewise, 
#        [3]
# b[:,1]=[5] 
#        [5]
# 3+5+5 is 13!
# hence c = [20., 13.] which is repeated obviously because the second and third rows of a are all 1s as well.
#           [20., 13.]  
#           [20., 13.] 
# I guess you are now starting to get where we are going with this, if we can some how alter the 'a' matrix,
# we may very well be able to achieve our goal! how you may ask? the answer is using torch.tril!
# torch.tril() is a function that returns a matrix from a given tensor, so that half of it set to zero,
# basially it creates a triangular tensor, where the right half is just zeros! lets see how it works, 
# lets apply it on 'a'
a_tril = torch.tril(a)
print(f'a_tril:\n{a_tril}')
# it prints
# a_tril=tensor(
#        [[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# as you can see the the right half is set to zero and we are left with a triangle shape of 1s! on the left side
# now if we do a@b this time we get:
c = a_tril@b 
print(f'b:\n{b}')
print(f'c:\n{c}')
# a_tril:
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[ 9.,  3.],
#         [14.,  8.],
#         [20., 13.]])
# now if you look closely, you'll notice that, this time, each row in c, is effectively the sum of the previous
# rows in b, like the first row of 'a' is only 1 in the 0ths column, so the first row of b is copied in c intact
# (workout the math and see why). 
# the second row in 'a', now has two 1s in col 0 and 1 respectively, which effectively translates to summing the first
# two rows in b. (9+5 =14, 3+5=8). likewise, the third row in 'a' is all 1s, signfigying all rows in b will
# be summed which gives us (9+5+5=20, 3+5+5=13). 
# so basically we are doing sums here, becasue our tensor a is all ones. so if we want to somehow calculate the
# average, instead of sum, we can easily change 'a' by normalizing it so that the each row sums to 1 (i.e. all cols
# sum to 1), this way the end result will be the average (becasue the 'b' is multiplied by a fraction/scale and then summed)
# so if we scale 'a' by the sum of all its columns, we should get average instead
a = torch.ones(size=(3,3))
a = torch.tril(a)
a = a/a.sum(dim=1, keepdim=True)
print(f'a:\n{a}')
# a:
# tensor([[1.0000, 0.0000, 0.0000],
#         [0.5000, 0.5000, 0.0000],
#         [0.3333, 0.3333, 0.3333]])
#
# note that, now each row, sums to 1. the first row, the first element is 1, because the rest are 0s
# but in the second row, as there are two 1s, the probabality is divided between the two, each being 0.5
# likewise, in the third row, as there are 3 1s, the probablity is divided between all of them, making each
# to have the value 0.33
# and now if we try to multiply them, we get average as the result:
c = a@b 
print(f'calculating average:')
print(f'b:\n{b}')
print(f'c:\n{c}')
# we get 
# calculating average:
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[9.0000, 3.0000],
#         [7.0000, 4.0000],
#         [6.6667, 4.3333]])
# we see that, each row in c, is the average of all the rows before it. 
#
# so using this trick, we can take the incremental average of any matrix we like. 
# now that we learned the trick, lets go back and implement the bows for loops using this techique !
# prevbiously we had : 
# results = torch.zeros(size=(B,T,C))
# for b in range(B):
#     for t in range(T):
#         results[b,t] = x[b,:t+1].mean(0)
# which calculated the average for sequence dimensions (tokens) incrimentally
# lets do this now 
# we want a TxT weight becasue we want to average T timestep/tokens
weight =  torch.tril(torch.ones(size=(T,T)))
# remember to set keepdim=True, or otherwise, as we sum along dim=1, we lose that dim
# (it collapses, and then the broadcast will be wrong, it will infact make each column 
# have one probablity instead of each row, which is the exact opposite of what we want)
weight = weight/weight.sum(dim=1, keepdim=True)
print(f'weight:\n{weight}')
# and now we need to multiply this by x! 
bow_results= weight@x
print(f'bow_results:\n{bow_results}') 
# which prints 
# bow_results:
# shape: torch.Size([4, 8, 2])
# tensor([[[ 0.0000,  1.0000],
#          [ 1.0000,  2.0000],
#          [ 2.0000,  3.0000],
#          [ 3.0000,  4.0000],
#          [ 4.0000,  5.0000],
#          [ 5.0000,  6.0000],
#          [ 6.0000,  7.0000],
#          [ 7.0000,  8.0000]],

#         [[16.0000, 17.0000],
#          [17.0000, 18.0000],
#          [18.0000, 19.0000],
#          [19.0000, 20.0000],
#          [20.0000, 21.0000],
#          [21.0000, 22.0000],
#          [22.0000, 23.0000],
#          [23.0000, 24.0000]],

#         [[32.0000, 33.0000],
#          [33.0000, 34.0000],
#          [34.0000, 35.0000],
#          [35.0000, 36.0000],
#          [36.0000, 37.0000],
#          [37.0000, 38.0000],
#          [38.0000, 39.0000],
#          [39.0000, 40.0000]],

#         [[48.0000, 49.0000],
#          [49.0000, 50.0000],
#          [50.0000, 51.0000],
#          [51.0000, 52.0000],
#          [52.0000, 53.0000],
#          [53.0000, 54.0000],
#          [54.0000, 55.0000],
#          [55.0000, 56.0000]]])
# which gives us the same results as we expected.
# one more thing before we continue on, note that the weight matrix is TxT while 
# the input is (BxTxC). (T,T) and (B,T,C) are not compatible, so what happens is
# that (T,T) is reshaped and a batch dimension is added to (T,T),making it (1,T,T)
# and then this is broadcasted along the batch dimension (replicated) to become 
# (B,T,T), then this will be multiplied by the (B,T,C)( note that at this stage
# a@b will be a batch multiplication operation, if you set the batch dimension aside, youll
# see that the rest of the dimensions match up, we have (T,T) and (T,C) which will
# result in (T,C). now if we add the batches back in, we will endup with (B,T,C)
# so in practice, the multiplication is done B times and then results are stacked.
# the first batches will multiply each other (T,T)x(T,C) = (T,C)
# the second batches will then multiply each other as well, getting another (T,C)
# and this goes on until we get (B,T,C))
#
a = torch.arange(0,9).view(3,3)
# a = torch.ones((3,3)).long()
b = torch.arange(0,30).view(2,3,5)
print(f'a:\n{a}')
print(f'b:\n{b}')
c = a@b 
print(f'c=a@b:\n{c}')
# is equivalent to 
# add a batch dimension to a
a = a.view(1,*a.shape) # or a.unsqueeze(0)
print(f'a with batch dim:\n{a.shape=}')
# replicate along the batch dimension
a = torch.cat(tuple(a.clone() for i in range(len(b))), dim=0)
print(f'{a.shape=}')
print(f'a(after replication along dim=0):\n{a}')
print(f'b:\n{b}')
# and now we have a case of batch-multiplication, the batch is the same 
# and sub tensors also are compatible, we have (T,T) and (T,C) so we now
# individually multiply each sub-tensor
c2 = torch.zeros_like(b)
for i in range(b.shape[0]):
    c2[i,...] = a[i]@b[i] # (T,T) x (T,C) -> (T,C)
# and ultimatley the result will have the shape (B,T,C)
print((c==c2).all())
# so to recap this 
# When performing the matrix multiplication `c = a @ b` with the given tensors, 
# several broadcasting steps occur to align the dimensions properly. 
# Here's a how it happens:
# 1. Tensor a has the (shape: 3, 3):
# 2. Tensor b has the (shape: 2, 3, 5):

# 3. They are not compatible so we broadcast tensor a to match the shape of b. 
# tensor a is expanded to (1, 3, 3) to have a batch dimension, and replicated 
# along dim 0 to result in (2, 3, 3). now both tensors have a batch of 2, and 
# if we put aside the batch dimension for a second, we'll notice that the rest
# of the shapes are compatible (3,3) and (3,5). so when we bring back the batch 
# dimension, we see we have a case of batch multiplication, that is we have 2
# sets of compatible tensors that need to be multiplied together. so we use a 
# simple for loop, to do multiplication, and stack the results and we are done!
#
# now back to our discussion. as we just saw, the traingualr shape in our weighted
# sum matrix, allows that each token at t dimension, can only interact with the tokens
# before it.
# there is another way of implementing the same thing but a bit differently
# if you look closely you can see that, we are dealing with probablities, so
# we may verywell use softmax to simplify this further.
# first lets create our triangular weight matrix
tril_tensor = torch.tril(torch.ones(size=(T,T)))
# now lets create a mask
weight = torch.zeros_like(tril_tensor)
# now lets mask all the zeros to -inf, so when we do softmax, all those -infs
# become 0. (recall that exp^-inf is 0! while exp^inf is inf! so its important to 
# set -inf (and also exp^0 is 1))
# so effectively what happens here is that, we setting each entery in trail_tensor
# with zero to -inf, and leave the rest as zeros. when this tensor goes through softmax
# the 0s will be 1s and -infs will be 0s before they are normalized, when they are normalized
# the probablity of 1 will be split between all the enteries with the value of 1.
# so the first row has a single 1, so it will be 1.0, the second row has two 1s, so
# each one will take 0.5, the third will have 3 1s, so they each will become 0.333
# and so on.
weight=weight.masked_fill(tril_tensor==0, -torch.inf)
# now calculate the probs for each row, treating all cols as probs so their sum is 1
weight = weight.softmax(dim=-1)
bow_results2 = weight@x
print(f'{torch.all(bow_results==bow_results2)}')
# so you might ask, why would we want to make things more complicated like this to only
# achieve what we already achieved pretyy efficiently before?
# the answer is, this approach provides us with a flexibility that the previous ones
# wouldnt provide us withs. if you think about it for a moment, you'll notice that
# the 'weight' matrix can essentially be anything and not just zeros! 
# we have decoupled it from tril, which's job is to set the right half of the tensor
# to zero, so we actually get the expected behavior later on.(which is setting a constrain really (more layer on))
# if you havent yet figured it out, the 'wieght' matrix, can be truly a weight matrix
# which can show different strengths for each token, basically it can learn the interations
# between tokens and manifest them. currently it is us who sets it to all zeros, so we get
# uniform probablities, which inturn manifest itself as an average. but what if instead of 
# all zeros, we actually learn the values from the data itself? that would make sense, and 
# make it so that each token, can have a different connection(strength) to any other tokens
# now, and thus build semantic/meaningful relation. this is inafct what we are after. we want
# for tokens to learn associations and relations with other tokens based on the data present
# in the dataset, having uniform weight like what we initially did really is a far cry from
# what we intend, and therefore, we opt in to use this new approach that allows us to actually
# exploit this new capability. 
# This is infact the problem that attention solves, that is gathering
# information from the past but in a data driven manner.(side note, we are not limited to 'past' 
# information only per say, in this example, this is the case however, we will explain this 
# in more detail)), we will see how attention does this exactly in a moment. 
#
# but before we jump into attention implementation also note that the tril part, infact is a hard-constrain here that prevents tokens from
# the past from interacting with the tokens from the future. 
# so to recap here, basically the idea is, this triangular form, allows us to have 
# weighted aggregations of past elements. each element in the lower triangular part, 
# specifies, the degree by which it plays a rule in the said outcome. 
# that is how much of each element gets to fuse into this specific position(i.e. current token's)
# so now lets incorporate attention into our model
# Attention does its job by using two vectors called, key and query.
# basically every single token, emits two vectors called key and query, the query verctor
# as the name suggests, implies, what we are looking for, and the key vetcor, again as the
# name suggets, implies, the contents, what it contains. 
# the way we get our 'weights' for these tokens is we simply dotproduct them together, 
# the ones that yield a high output/value, signify they are related positively.
# so our query is dotproducted with all the token's(keys) and the outputs reveal their 
# closeness/relevancy/similarity so to speak(if they align so to speak, they result in larger number)
# so effectively, the ones with higher number, are more similar to the query, the highest, 
# obviously having the most relavancy/similarity to the query.s
# so lets implemenet this
# #%%
# we want to implement a single attention head, we can later use this to create 
# multi-attention-head which is basically several single attention heads working in 
# parallel. so how do we implement one! 
# first lets create some random input 
torch.manual_seed(255)
# lets specify the batch, context_size and vocab_size
B,T,C = 4,8,32
# lets create an input 
x = torch.randn(size=(B,T,C))
# we said that attention works with two vectors, key and query, lets implement them
# we can use nn.Linear to implement them but beore that, what are the dims of such vectors,
# we are dealing with text and thus our inputs are tokens/vocabs, so the first dim would be 
# our vocab_size to account for all tokens, the next dim, is sth called a head_sizes, which 
# specifies the size of the key/query output usually 16 is used a lot for head size so we 
# use that as well
head_size = 16
# lets not forget to set bias=False, so what it does is exactly dotproduct 
# (actually, some people still leave the bias enabled! 
# so we will test both cases later and see if it actually matters! and if so to what extend! ) 
key = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
# now lets get the output
k = key(x)      # shape : 4,8,16 or (B,T, head_size)
q = query(x)    # shape : 4,8,16 or (B,T, head_size)
 # so the dims arent compatible for batch-multiplication, so we need a transpose
 # k.T wouldnt work becasue we have a batch-dim, so instead we use .transpose and
 # explictily specify the dims we want to be transposed. 
 # this will result in 4,8,16 by 4,16,8 which would give us 4,8,8 which is (B,T,T)
 # really as the result
 #! check transpose result is it okto use 2,1 or -2,-1 or 1,2
weight_raw = q@k.transpose(2,1)
# now lets for a moment think about what is happening here, the key and query are applied
# on the input and each return an output of (B,T,head_size), they are in fact, processing
# all the tokens in the input, individually, simultaneously, all the same time. so each 
# token is both a query, and a key, and when we do a dotproduct, we are basically telling 
# it to reveal the relation/similarity/relevance of every token with every other tokens.
# and as we explained earlier, this is infact our weight matrix (which was initially zeros)
# but is now learned from the data!
# print(f'weight_raw\n{weight_raw}')
# now we can apply constrain on it so that tokens can only communicate with the past so 
# we use the tril trick now!
tril_constrain = torch.tril(torch.ones(size=(T,T)))
raw_wieghts_masked = weight_raw.masked_fill(tril_constrain==0, float('-inf'))
# apply sotmax to get probablity for each token
weight = raw_wieghts_masked.softmax(dim=2) # remember weight is (B,T,T)
# and finally we can apply our weight on the input (we called it raw for a reason, read on)
# (by the way this is also called self-attention!)
bow_raw = weight@x 
# lets print raw_weights and weights and have some intuitive observations 
print(f'weight_raw\n{weight_raw}')
print(f'raw_weights_masked: {raw_wieghts_masked}')
print(f'weight\n{weight}')
# prints
# raw_weights(unconstrained)
#weight_raw
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00, -1.7903e+00,  1.4650e-01, -1.4147e+00, -1.7452e+00, -4.5249e+00, -1.9763e+00,  1.8833e+00],
#          [-1.3490e+00,  9.4076e-01, -6.9690e-01,  7.9933e-01,  4.7634e-01,  4.1861e-01,  2.3663e-01,  4.5217e-01],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,  9.0092e-01,  1.4083e+00,  2.5488e+00,  1.6350e+00,  1.0048e+00],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01, -1.1213e+00,  2.7562e+00,  7.0136e-02, -2.0337e+00],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,  4.9236e+00,  3.9545e+00, -2.2600e-01],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01, -1.5979e+00,  8.4627e-01],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,  2.2338e+00],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00, -2.1323e+00, -1.3948e+00, -5.6973e-01, -2.7986e-01,  4.3579e+00,  4.8477e-01, -9.7559e-01],
#          [ 8.7786e-01, -2.3352e+00, -2.2988e+00,  1.0322e+00, -1.4231e-01,  5.5233e+00,  1.0819e+00, -2.4722e-01],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,  2.9387e-02, -5.6397e-01, -2.4025e-01, -4.2512e-02, -8.4302e-01],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01, -4.0429e+00, -1.0676e+00, -3.0694e+00, -1.6156e+00],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00, -4.5593e+00, -3.1755e-01, -1.0504e+00],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00, -3.5616e-01, -2.6951e+00],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,  4.9077e-02],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01, -4.7902e-01,  6.7590e-01, -2.3153e-02,  1.9763e-01,  4.4385e-01,  7.2986e-01, -3.8846e-01],
#          [-6.2810e-01, -8.3713e-01,  1.3468e-01, -5.1328e-01, -2.6653e-02,  3.4325e-01, -1.0015e+00, -1.0334e+00],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01, -1.9702e+00, -1.0641e+00,  1.7792e-01,  8.0194e-01, -1.5265e-01],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,  3.6759e-01,  1.0552e+00, -5.2949e-01, -1.7715e+00],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01, -3.0057e+00,  1.3784e+00, -1.2091e+00],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02, -1.4040e+00,  2.3786e+00],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00, -1.6068e+00],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<UnsafeViewBackward0>)
#
# raw_masked_wieghts(constrained):
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.3490e+00,  9.4076e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01,        -inf,        -inf,        -inf,        -inf],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,        -inf,        -inf,        -inf],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01,        -inf,        -inf],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,        -inf],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 8.7786e-01, -2.3352e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01,        -inf,        -inf,        -inf,        -inf],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00,        -inf,        -inf,        -inf],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00,        -inf,        -inf],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,        -inf],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-6.2810e-01, -8.3713e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,        -inf,        -inf,        -inf,        -inf],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01,        -inf,        -inf,        -inf],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02,        -inf,        -inf],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00,        -inf],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<MaskedFillBackward0>)
# 
# weight (final- normalized)
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.1976e-02, 9.0802e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9773e-01, 6.0261e-01, 1.9966e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.4181e-02, 3.4422e-02, 3.7221e-01, 5.7918e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [4.6023e-02, 1.9501e-03, 5.3236e-01, 3.5612e-01, 6.3550e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9494e-01, 1.9195e-01, 2.7619e-01, 2.8253e-02, 1.3648e-01, 1.7219e-01, 0.0000e+00, 0.0000e+00],
#          [4.5214e-01, 9.6007e-02, 7.3108e-02, 5.3035e-02, 2.8887e-01, 8.7621e-04, 3.5963e-02, 0.0000e+00],
#          [6.8046e-02, 5.1605e-01, 5.0474e-03, 1.5304e-02, 2.9133e-01, 3.8823e-04, 1.3775e-02, 9.0057e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.6132e-01, 3.8675e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.1870e-01, 3.3895e-01, 4.4235e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.2894e-02, 6.5398e-01, 2.4345e-01, 7.9674e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [3.7332e-02, 6.5690e-01, 7.8427e-02, 5.1206e-03, 2.2222e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.3611e-01, 1.7995e-02, 2.1887e-02, 1.1900e-01, 1.8219e-02, 6.8680e-01, 0.0000e+00, 0.0000e+00],
#          [9.8946e-02, 1.4120e-02, 1.8065e-02, 9.5010e-03, 3.2130e-01, 9.9891e-02, 4.3817e-01, 0.0000e+00],
#          [2.3306e-02, 3.5621e-02, 9.5163e-02, 1.5801e-01, 1.1530e-02, 1.7183e-01, 4.1141e-02, 4.6340e-01]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.5207e-01, 4.4793e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.8708e-02, 6.5816e-01, 2.4313e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.8422e-01, 9.1987e-02, 1.2557e-01, 5.9822e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.0870e-01, 8.3642e-02, 4.8896e-02, 3.2723e-01, 3.3154e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.4141e-02, 7.9271e-02, 2.1893e-01, 2.2608e-01, 2.3782e-01, 1.4376e-01, 0.0000e+00, 0.0000e+00],
#          [7.8726e-02, 1.1815e-01, 3.7190e-01, 1.2607e-02, 3.6387e-02, 1.1790e-01, 2.6434e-01, 0.0000e+00],
#          [5.6646e-02, 7.4575e-02, 6.8217e-02, 3.1568e-02, 5.9024e-02, 1.1073e-01, 3.1662e-02, 5.6757e-01]]], grad_fn=<SoftmaxBackward0>)
#
# as you can see, our weight is initialized for each batch based on the input data, and each token has its own
# weight, that is they are not uniform!, to get a better understanding lets consider the first batch of 
# raw-weights[0], raw_masked_wieghts[0] and weight[0]:
#
# raw_weights[0]
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
#
#
# raw_masked_wieghts[0]
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
# 
# weight[0]
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
# take the last token, which 8.6020e-02, notice this token, not only knows its content and own position in 
# the sequence(its the 8th token after all) but also knows which tokens comes before it and how much its 
# related to any of them.(basically when it knows which token comes before it, it creates its own query so
# to speak, and talks to every single previous token to find about which ones are more or less relavent to
# it and to what extend.) 
# For example,lets say, (since we are dealing with character level text generation!), the last token is a 
# vowel and says hey im a vowel and im looking for everyone else thats a vowel! and uses query to talk to 
# other tokens(keys). and intrestingly another token (lets say number 4) says im a vowel as well, and the 
# response is thus generate a larger number.(this is not a good example, words in sentence would make more sense
# !give a better example!)
# in our example, For the last token, it seems the 5th and 3rd token are particularly intresting/relavent/important,
# followed by the 7th token.
# likewise, this happens for every token, i.e. token number 4, says the same thing and searches in its past toekns
# so on and so forth! 
# so what happens next is that when we get a high relevancy scale /response in the weights, like we just described
# when we do a softmax it will assign a large probablity to them, and this instructs the network that, we need more
# information from them, effectively allowing for aggregating a lot of their information into our position(lets say e.g. 8th token. 
# and we happen to learn more about them this way.
# now in practice, we are not intrested in aggregating the x raw values per say, rather we want their information, 
# so instead of just using the raw values of x, we instead use a representation of them, so to speak. 
# this is achieved using a third vector known as, 'value' and is the last vector we use. 
# !explain when we say vector, note that, we are talking from the prespective of a token, (every token has some vector 
# !of information and it gets to aggregate information via a weighted sum from all the tokens that point to it, and it
# )# !in practice we use a matrix, and hence the linear module to implement this to run the operation for all tokens in parallell. 
# just like the key and query, we set its bias to False, so we only get a simple vector,
# and a dotproduct output (again this is the intuition, but how much adding a bias would
# affect this we will see later, for now we stick to the default no bias version)
value = torch.nn.Linear(C,head_size, bias=False) # produces (B,T,head_size) just like the other two key,query vectors
x_processed = value(x)
# this is not yet final final, we still need to do one more thing (read on!)
bow_final = weight@x_processed
#
#!check also note that, as we previously once pointed out, attention is a communication mechanism between tokens, and 
# by default there is nothing in this mechanism that provides a notion of space/position for tokens involved, i.e.
# by default these tokens/nodes/points, dont have any idea about where they are or how they are positioned (with resepect
# to others) etc, so we need to encode this information as well.(to better visualizing it, consider each token as 
# a node in a directed graph, each node has a connection to another node, (for example each node has a connection to
# itself, and another connection to other nodes,etc) and as you can see there is no notion of space here,the attention
# simply acts on a set of vectors in this graph, and thats why we need to encode them positionally as well, so they 
# have information about their position with regards to other tokens. 
# !check (compare this to the convolution case and images
# or even text where the spatial aspect of data is preserved, but in attention, as we just stated, there is no notion
# of position, its just a set of individual vectors being operated on)-note that the connection between nodes, do give
# us a sense of structure, but it may not be enough to infer the underlying semantic, imagine a case, where a word
# for example has several meaning, and may very well have high relevancy to some tokens at the same time, but without
# additional positional information, an ambiguous semantic can be infered between the tokens involved, however when
# positional information is also present, such ambiguity can be avoided)
# also note that, in attention, samples do not interact with each other at all. when we have a batch of 4, each sample
# is processed in isolation, but in parallell to other samples, based on our graph example earlier, we would have 4 
# graphs of 8 nodes for example for each input sample. 
#
# we said earlier that what we implemented here is known as self-attention, the reason it is called self attention
# is that the key and query and values are applied on the same input(the use the same source!), and hence the name,
# self attention.
# also note that, in our specific case, tokens/nodes are can not communicate with the future nodes, but in general
# this constraint can be removed (and infact is removed/not implemented for some applications) where its benificial
# to be able to communicate with all the tokens. one example is sentiment analysis, where you want all the tokens to
# able to communicate with eachother so you can get an accurate analysis. and for this case, we would use an encoder
# block, which is basically what we have here, minus the constraint section(tril/mask part), what we have implemented
# here is called a decoder block, where we are decoding bunch of tokens, and it makes sense that the previous tokens
# do not comunicate with the future ones(becasue they would give the answer! and it defeats the whole purpose here!),
# becasue its a given that only the previous tokens must be used to predict the future/next token, hence the filtering/constraint part to prevent tokens from 
# comunicating with the future nodes/tokens.
# !so far we explained about the self-attention, which we saw, is called that way solely for the fact that key, query
# and value use the same source. the attention mechanism as we briefly pointed out, is much more general and can be
# used in different ways. one of such ways, is what is used to create sth called cross-attention. 
# cross-attention basically refers to the case where we have an encoder/decoder blocks, in which the queries come from x
# but the key and value come from an external source and sometimes from the encoder block. so cross-attention is used
# when theres a separate source of information we would like to pool from and use it as well.
#
# so far we implemented the attention based on the original paper(there are some differences we get to later on)
# except the part where we need to divide by the sqrt of the head_size. that is called scaled-attention
# so lets talk about this, and see why its needed. 
# the reason we add this so called 'scale' to our computation, is that, without it, the probablities will be saturated
# and when we add this term to the mix, it will make the 'weight' matrix to be 'unit variance', when Q and K are unit variance
# and this allows softamx to stay diffuse and not saturate too much.
# in other words, if we simply multiply key and query like that, the variance of the resulting weight matrix will be
# around the head_size instead of 1 which is bad and makes optimization really hard.
# to see this effect consider the following example
k = torch.randn(size=(B,T,head_size))
q = torch.randn(size=(B,T,head_size))
w_unscaled = q@k.transpose(-2, -1) 
print(f'{k.var()=}')
print(f'{q.var()=}')
print(f'{w_unscaled.var()=}')
# now add the 1/sqrt(head_size)
w_scaled = q@k.transpose(-2, -1) * head_size**-0.5 
print(f'after applying 1/sqrt(head_size)')
# makes the weight variance 1!
print(f'{w_scaled.var()=}')
# why is it important? if you recall, the weight matrix is fed into softmax, so its really important, especially during
# initialization that weight matrix be fairly diffuse, if we look at weight matrix here, we'll notice that they are now
# fairly diffuse 
print(f'weight_scaled[0]:\n{w_scaled[0]}')
# prints 
# weight_scaled[0]:
# tensor([[ 0.3972, -2.0957, -0.5396, -1.4860, -0.8393,  0.6273, -0.0157,  0.6284],
#         [ 1.3588, -0.0440,  2.1040, -0.2796,  1.7797,  1.4362,  1.3843,  0.0368],
#         [ 0.8109,  1.7400,  1.3579,  0.2097,  1.7152, -1.1242, -0.4349, -0.5690],
#         [ 0.0513,  2.8303,  0.7332,  0.0041,  0.9688, -1.6174, -1.4255,  0.2869],
#         [-0.7819, -0.6691, -1.4017, -0.4155, -0.8568, -0.2704, -0.9453,  0.3763],
#         [ 0.4092, -0.3079,  0.8472, -1.1753, -0.7699,  0.5091,  0.6385,  0.9677],
#         [ 0.3043, -2.1402, -2.2582, -0.3573, -1.2838, -0.0260, -0.0513,  0.9151],
#         [ 0.4180, -2.8353, -0.6435,  0.4842, -3.0202,  1.7690,  1.8837, -0.4393]])
#
# when softmax is applied:
print(w_scaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.8026, 0.1974, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1901, 0.4814, 0.3285, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.0499, 0.8038, 0.0987, 0.0476, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1989, 0.2226, 0.1070, 0.2869, 0.1845, 0.0000, 0.0000, 0.0000],
#         [0.2148, 0.1049, 0.3329, 0.0440, 0.0661, 0.2374, 0.0000, 0.0000],
#         [0.3027, 0.0263, 0.0233, 0.1562, 0.0618, 0.2176, 0.2121, 0.0000],
#         [0.0901, 0.0035, 0.0312, 0.0962, 0.0029, 0.3478, 0.3901, 0.0382]])
#
# 
# now compare it with the unscaled weight : 
print(f'weight_unscaled[0]:\n{w_unscaled[0]}')
# weight_unscaled[0]:
# tensor([[  1.5889,  -8.3829,  -2.1583,  -5.9440,  -3.3573,   2.5092,  -0.0629,  2.5135],
#         [  5.4350,  -0.1759,   8.4159,  -1.1183,   7.1189,   5.7446,   5.5373,  0.1472],
#         [  3.2435,   6.9600,   5.4317,   0.8388,   6.8607,  -4.4969,  -1.7396, -2.2761],
#         [  0.2051,  11.3214,   2.9327,   0.0165,   3.8754,  -6.4696,  -5.7019,  1.1475],
#         [ -3.1278,  -2.6763,  -5.6069,  -1.6621,  -3.4272,  -1.0816,  -3.7811,  1.5053],
#         [  1.6368,  -1.2317,   3.3888,  -4.7012,  -3.0795,   2.0364,   2.5541,  3.8710],
#         [  1.2171,  -8.5610,  -9.0327,  -1.4293,  -5.1353,  -0.1038,  -0.2053,  3.6603],
#         [  1.6719, -11.3411,  -2.5740,   1.9369, -12.0810,   7.0760,   7.5347, -1.7573]])
#
# when softmax is applied:
print(w_unscaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [9.9636e-01, 3.6442e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.9592e-02, 8.0566e-01, 1.7475e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.4865e-05, 9.9975e-01, 2.2738e-04, 1.2309e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2943e-01, 2.0328e-01, 1.0848e-02, 5.6050e-01, 9.5940e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2012e-01, 6.8207e-03, 6.9264e-01, 2.1236e-04, 1.0748e-03, 1.7913e-01, 0.0000e+00, 0.0000e+00],
#         [6.3261e-01, 3.5856e-05, 2.2371e-05, 4.4856e-02, 1.1023e-03, 1.6883e-01, 1.5254e-01, 0.0000e+00],
#         [1.7351e-03, 3.8710e-09, 2.4850e-05, 2.2616e-03, 1.8472e-09, 3.8572e-01, 6.1020e-01, 5.6236e-05]])
#
# as you can see, some values are very negative, while others are very positive, for example, we have both 11 and -11
# and this is a recipe for disaster! the reason it is a problem is that, with softamx, when numbers take very positive
# and nagative values, softmax actually converges towards one-hot-vector. basically the values shrink towards the max.
# this can be seen the above, but the example below should demonstrate it more clearly.
# lets apply softmax on a list of numbers that are close to each other, we see the probablities are difuse and normal
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2]),dim=-1)}')
# we get a diffused probablity out of softmax
# tensor([0.2562, 0.3822, 0.1717, 0.1898])
# however if we increase the magnitude of the numbers(sharpen them) (and thus the difference between them) by like multiplying by 
# a number like 8 (just to simulate the effect here and so we can compare it with the previous case)
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2])*8,dim=-1)}')
# we see that the softmax, starts to sharpen towards the max, shrinking others except the 
# largest number, and effectively
# converging toward a one-hot-encoded vector.
# tensor([0.0390, 0.9559, 0.0016, 0.0035])
# so we dont want these values to be extreme, especially during initialization or otherwise, softmax will be way too picky!
# and we are basically aggregating the information from a single node instead of multiple ones (becasue one has the largest
# nvalue, its as if our sequence length is 1! and we lose access to the wealth of information the past history offers)
# so we want the probablities to be diffuse and not peaked like the second example here.
# so this scaling is used to retain the variance at a good value especially at initialization.
# so now that we are finally finished the self attention head, lets implement it as a module and incorporate everything
# we just discussed here.
class AttentionHead(nn.Module):
    def __init__(self, context_size, embd_size, head_size=16, use_bias=False) -> None:
        super().__init__()
        # we need context_size or block_size for creating the tril constrain
        self.context_size = context_size
        # we want this as the input dim for our key,query and value, this is 
        # infact the input_dim, since we plan on using the embeddings, we named
        # it embd_size
        self.embd_size = embd_size
        # head_size is the output dimension of our attnetion
        # !note that usually the head_size is equal to embd_size
        self.head_size = head_size
        self.key = nn.Linear(embd_size, head_size, bias=use_bias)
        self.query = nn.Linear(embd_size, head_size, bias=use_bias)
        self.value = nn.Linear(embd_size, head_size, bias=use_bias)

        # use buffer for trail, since this is a buffer, we can use the self.register_buffer which is
        # inherited from nn.Module class, to add it as buffer to the module, so it can be saved in the
        # state_dict when we save the model.  tril doesnt change, and is not updated(its required_grad is false),
        # so it being saved in state_dict doesnt matter to us, but to demonstrate this feature of pytorch, 
        # we are using it, and its a good practice, since pytorch knows how to deal with it and wont include it
        # in the computaion graph anyway!
        # 
        # This is typically used to register a buffer that should not to be considered a model parameter. 
        # For example, BatchNorm's running_mean is not a parameter, but is part of the module's state. 
        # Buffers, by default, are persistent and will be saved alongside parameters. This
        # behavior can be changed by setting persistent to False. The only difference between a persistent
        # buffer and a non-persistent buffer is that the latter will not be a part of this module's state_dict.
        # tril allows us to impose constrain by creating a lower traingualr matrix and setting
        # the rest entries to zero, effectively preventing tokens of the future from communicating with the past
        # each token can only communicate with the previous tokens that came before it.
        self.register_buffer('tril', torch.tril(torch.ones(context_size,context_size)))

    def __call__(self, inputs:torch.Tensor) -> torch.Tensor:
        #! check the shapes!! 
        B,T,C = inputs.shape # 4,8,16
        # print(f'{inputs.shape=}')
        # create the weight by using k,q,v
        k = self.key(inputs)    #! (B,T,C) or (B,E,16)? (4,8,16)
        q = self.query(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        v = self.value(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        # create the weight matrix and scale it by 1/sqrt(head_size) to keep weight unit variance 
        weight = q@k.transpose(-2,-1)* self.head_size**-0.5 # !(B,E,E)
        # weight_C = q@k.transpose(-2,-1)* C**-0.5 # !(B,E,E)
        # print(f'weight.var: {weight.var().item():.4f}')
        # print(f'weight_C.var: {weight_C.var().item():.4f}')
        # print(f'x:{tuple(inputs.shape)} k:{tuple(k.shape)} q:{tuple(q.shape)} v:{tuple(v.shape)} w:{tuple(weight.shape)} hs:{self.head_size}')
        # apply the tril constrain - (this makes this a decoder block!)
        # important note: notice we used tril[:T,:T] and not simply tril
        # this is because, when the input has a small context_size < self.context_size
        # like when we wantto generate inputs with sth like zeros((1,1)) which says
        # there is a single token (context_size 1) of 0 as the begining of the sequence
        # when this is input, the tril by default makes a context_size,context_size matrix
        # which will be different than the input context_size which is 1 e.g. or 2 e.g.
        # and it will fail becasue our weight would be 1,1,1 or 1,2,2, but trail is 1,8,8
        # and clearly this will cause an error. for this reason, we always create the 
        # tril check dynamcally by explicitly specifying the context_size based on the 
        # current input context_size so in case the context_size is smaller, tril is resized
        # dynamically accordingly. 
        # print(f'tril[:T,:T]==0: {(self.tril[:T,:T]==0).shape}')
        # print(f'tril==0: {(self.tril==0).shape}')
        weight = weight.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        # weight2 = weight.masked_fill(self.tril==0,float('-inf'))
        # print(f'{weight.shape=}')
        # print(f'{weight2.shape=}')
        # note that we are using batch, so instead of hardcodin 2,
        # we use -1 to refer to the last dim
        weight = weight.softmax(dim=-1)
        # finally apply the weight on the v
        bow = weight@v  # !(B,E,16)
        return bow        

# now lets add this to our model 
        
# so to recap, we first calculated the relavancy between all tokens against eachother, then constrained them so that 
# each token can only use the information from/interact with its past tokens. then since we needed probablity distribution so
# we then normalized it and then used that to pickout which tokens information(in the past) to aggregate/use with the inputs to 
# achieve our goal.(which is to predict the next character based on everything seen so far!)
# we dont want to aggregate inputs value (with respect to the weight matrix), we instead would like to use their representation
# so we use a new layer to do this, its called value, and we instead use its output instead of inputs raw value.
#
# 
#
# before we jump in and add the attention module, lets review our base model and see how we can 
# improve/prepare it before we incorporate the attention. 
# class BigramModelWithAttention(nn.Module):
#     def __init__(self, vocab_size, embd_size) -> None:
#         super().__init__()
#         self.vocab_size = vocab_size
#         self.embd_size = embd_size
#         # unlike the previous model, lets decouple the final logits from 
#         # the number of embeddings, because we are using attentions, and
#         # we want to have multiple operations inbetween obviously.
#         self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
#         # in order to get the final logits, we need a linea layer at end
#         self.fc = torch.nn.Linear(embd_size, vocab_size)
#        
#     def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
#         out = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
#         logits = self.fc(out)         # has the shape (B,T,C) c is vocabsize
#         loss = None
#         if labels is not None:
#             # recall that crossentropy likes its input to be B,C,T and we are B,T,C
#             # so lets permute and make it happy!
#             loss = F.cross_entropy(logits.permute(0,2,1), labels)
#         return logits, loss
#    
#     def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
#         # lets generate an output as long as num_max_token
#         for i in range(max_token_count):
#             # make sure idx is 2d
#             assert len(idxs) >1, f"idx.shape '({tuple(idxs.shape)})' is invalid. it must have the form (B,T)"
#             # now lets feed it to the model and sample from the probablities it produces
#             preds,_ = self(idxs)
#             # convert to probs 
#             probs = preds.softmax(dim=1)
#             # since we are bigram still, lets only get the last token as the next token predicted!
#             probs = probs[:,-1,:]
#             # now lets sample from it 
#             new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
#             # now concatenate the new token to the previous one and feed it back to the model
#             # for the next round of prediction
#             # also remember that we are creating a sequence, so we concat them at dim=1 to get 
#             # a longer sequence (we are gradually increasing the sequence length from 1 up to
#             # max_token_count)
#             idxs = torch.cat((idxs,new_idx), dim=1)
#            
#         return idxs

# this works fine, however, we can do better. here we just encoded the tokens, but
# as we already explained, attention has no notion of position or spatial structure like conv e.g.
# so we need to also incorporate the position information for each token as well
# since we are using the the attention head, we need more arguments
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, head_size, device, use_bias_att=False) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embd_size = embd_size
        self.context_size = context_size
        # for gpu/cpu acceleration during training
        self.device = device
        # unlike the previous model, lets decouple the final logits from 
        # the number of embeddings, because we are using attentions, and
        # we want to have multiple operations inbetween obviously.
        self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
        # lets now add the positional embedding as well 
        # using this, we try to retain the position embedding for our tokens
        # up to the current token in context_size
        self.position_embd = torch.nn.Embedding(context_size, embd_size)
        # add a self-attention head
        self.head = AttentionHead(context_size, embd_size, head_size, use_bias=use_bias_att)
        
        # in order to get the final logits, we need a linea layer at end
        # since we now have attention before this layer, the output dim of attention which is
        # head_size will be used here
        self.fc = torch.nn.Linear(head_size, vocab_size)
        
    def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
        # lets grab the shapes, since we will be using them 
        B,T = inputs.shape
        token_embeddings = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
        # since we have a position_embedding lets use that as well
        # and notice that we didnt use the inputs, but rather torch.arange(T)
        # this means, for each input, as we process it, we also get embeddings up to
        # the current token count as well, if the inputs has 3 tokens currently, we
        # will creeate position embeddings for 0,1 and 2, and for the next input
        # this continues likewise. this results in (T,E)
        position_embeddings = self.position_embd(torch.arange(T,device=self.device))
        # print(f'pos_embd:{position_embeddings.shape}')
        # and lets add the two embeddings together
        # this effectively gives us, not only the token embeddings(identity)
        # but also its position in the sequence. note that this doesnt really
        # help in a bigram model, but when it comes to attention it really does!
        embeddings = token_embeddings + position_embeddings
        # now lets feed this to attention head
        out_attention = self.head(embeddings)
        # lets feed this embedding to our fc at the end instead
        logits = self.fc(out_attention)         # has the shape (B,T,C) c is vocabsize
        loss = None
        if labels is not None:
            # recall that crossentropy likes its input to be B,C,T and we are B,T,C
            # so lets permute and make it happy!
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss
    
    def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
        # lets generate an output as long as num_max_token
        for i in range(max_token_count):
            # print(f'{i}/{max_token_count}) idxs: {tuple(idxs.shape)}')
            # make sure idx is 2d
            assert idxs.ndim >1, f"idx.shape '({tuple(idxs.shape)})' is invalid({idxs.ndim}). it must have the form (B,T)"
            # now lets feed it to the model and sample from the probablities it produces
            # but since, we now have postional embeddings as well, we can no longer have 
            # more than block_size/contex_size in, becasue if our idx is more than context_size
            # our positional embedding will go out of scope and error out
            # so here we are basically getting as many as context_size
            # note that we dont destroy the idxs! each time we get the last context_size tokens
            # from it and feed it to the model to generate the next token, and keep going
            # if we replace the idxs by sth like idx=idx[:,-self.context_size:], we would
            # only create a sequence of only self.context_size, no matter how many iterations
            # we do, we just repeat the same sequence again and again!
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are bigram still, lets only get the last token as the next token predicted!
            logits = logits[:,-1,:]
            # convert to probs 
            probs = logits.softmax(dim=-1)
            # now lets sample from it 
            new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
            # now concatenate the new token to the previous one and feed it back to the model
            # for the next round of prediction
            # also remember that we are creating a sequence, so we concat them at dim=1 to get 
            # a longer sequence (we are gradually increasing the sequence length from 1 up to
            # max_token_count)
            idxs = torch.cat((idxs,new_idx), dim=-1)
            
            
        return idxs
    
# now lets test this and see if it works 
x,y = get_batch('train',4)
model = BigramModelWithAttention(vocab_size, 
                                 context_size, 
                                 embd_size=16,
                                 head_size=16,
                                 device='cpu',
                                 use_bias_att=False)
# model.cuda()
# x,y = (t.cuda() for t in zip(x,y))
logits,loss = model(x,y)
print(f'{logits.shape=} {loss=:.4f}')
# and now we can train this : 
device='cpu'
batch_size = 32
head_size = 16
embd_size = 16 
context_size = 8
vocab_size = len(vocab_list)
max_iter = 5000
# only to test the effect of bias in k,q,v calculations
use_bias_attn=False
# attention requires much lower lr compared to plain bigram model
lr = 1e-3
model = BigramModelWithAttention(vocab_size=vocab_size,
                                 context_size=context_size,
                                 embd_size=embd_size,
                                 head_size=head_size,
                                 device=device,
                                 use_bias_att=use_bias_attn)

param_count = sum([p.nelement() for p in model.parameters()])
print(f'param count:  {param_count:,}')
print(f'head size:    {head_size}')
print(f'embd size:    {embd_size}')
print(f'context size: {context_size}')

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
model = model.to(device)
# set model to train mode explicitly
model.train()
for i in range(max_iter):
    # get the batch
    x,y = get_batch('train', batch_size=batch_size)
    # feed the model and get the logits
    logits, loss = model(x,y)
    # evaluate the model
    if i%1000==0:
        losses=evaluate_loss(100, device)
        print(f'train: {losses["train"]:.4f}  val: {losses["val"]:.4f}')
    # zeroout_grads
    model.zero_grad(True)
    loss.backward() 
    optimizer.step()
print(f'done!')

# now lets try its output
input = torch.zeros(size=(1,1)).int()
output = model.generate(input, 500).squeeze(0).tolist()
print(f"{''.join(decode(output))}")
# prints
#here are a few tries:
# param count:  3,041
# head size:    16
# embd size:    16
# context size: 8
# train: 4.2551  val: 4.2533
# train: 2.7175  val: 2.7354
# train: 2.5917  val: 2.5681
# train: 2.5299  val: 2.5265
# train: 2.4684  val: 2.4759
# done!

# Wonedimy
# Y:
# ARUS:
# LAnouver; pof th, sthe the mae,
# Wor ut barrioreche savarduncou?

# hiler bathakm,
# I O:
# AK:
# Acat ED:
# et alliro dow wowilou ttherss; whest owur, garsacwe Ie hithaceyidous sou f'ze iwot ry tohien fm.
# h.

# Ar'm-ore tr Fofhou
# Mr ojous omerrastheasncy yous Wowuce whor tu I I:
# Hbeotth spat fel woro bel, aneicudidy ktiferrd thout ngord. th Iid aral, be'nd.

# Pal tels Ms eimu Me myo on I teasthe del,
# AChinout,
# Wowure. pu tcher muss-renon; wingh ocet im
# r Ddr,
# Do ne thu,
# LI ir't
# Operst ry wu

# param count:  5,745
# head size:    16
# embd size:    32
# context size: 32
# train: 4.1374  val: 4.1414
# train: 2.6264  val: 2.6210
# train: 2.5100  val: 2.5092
# train: 2.4434  val: 2.4578
# train: 2.4140  val: 2.4252
# done!

# Thee to an tal my,
# Thake ced arce chart bre le be bizoresf thak des garg
# Lur Wentescaln
# Hooco beins pord sth ds hom span gr; ther. My Why ort thallils ofrof not tos gas hyine ind hand
# Thid One the shond nd,
# AFenosee omn st hy no fe.
# GEwank, iand!
# The sbe qurss yod hem snel.
# CLerle,
# Whow Vou bem!
# Theewourtoave, shougen t:
# Ber hangn toutince mor ed,
# AD Scheapy thatho ABEN:
# AD:
# SUEDithy nde ndel mmy Tongusate hfolit, gintoher, hat gwe tiet what whingimst hpoourde wowopal,
# The by br des qO: I no os,

# try2: decreasing context size
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

#try 3: decreasing embd_size, but keeping context_size 32
# param count:  2,265
# head size:    16
# embd size:    8
# context size: 32
# train: 4.2202  val: 4.2158
# train: 2.9840  val: 3.0093
# train: 2.7667  val: 2.7621
# train: 2.6355  val: 2.6411
# train: 2.6070  val: 2.6077
# done!

# o
# TI:
# BOGot be
# DOus ssh.
# Bea hashren v:
# ANULINII feso sid izeis by tofuso te I be fnowe?
# Wledu ha m, thope w,
# ARUTI land med cu
# SI bve fme
# ARIThilaserothan omyhee
# Oina c;
# UCAngose.
# Terve t thirre my tthunulpthemaf wof.

#  corlur yoNE:
# NHy w te hen cod, tonos vengo veschete tors t ithaimeqlutreqRoosoye chin y h thuasafaous.
# ANCathelo-
# The?
# Shen br uyh sbe ans I:
# Gouthyykey m Authirou ae tovas ber ut lnt:,

# TAMENI sf rousinthsin h'howenthor sg.
# hidandhe waresd!

# I f
# Se loner mrdin gsvette n thest?

# try 4: decreasing head_size, keeping embd_size and context_size the same
# param count:  4,457
# head size:    8
# embd size:    32
# context size: 32
# train: 4.3416  val: 4.3409
# train: 2.7354  val: 2.7333
# train: 2.6400  val: 2.6309
# train: 2.5716  val: 2.5692
# train: 2.5217  val: 2.5182
# done!

# IF
# S:
# To ofldickeens we fofane for o warus
# Herdliion bis, te,
# The ta, d ounor;
# Ad h ank; sor Cito-'d turses tierlllle,
# Whe pen, tint yt Ging sino leles fes hesr sst.
# Than hinean.


# Nofl an udn f hak,
# Wh stharovernes;
# And meno scerr tth Gbl y hy I t stit pand sy, pefif.
# US dik thith be tagag, harlovenppsh!
#  cegshil,
# Wheanst thiges eprine sy wabat tha.

# Wheas
# Wit:
# I
# Thouw'arsangort,
# Giseal?

# Wh; mo tands t's.

# NRNULECADSAxave!

# ANNDIEI Fh Fod meveeas pee be,
# Bes arner,

# Woo t'le.


# VIPORLEOG
# CAOO:

# try 5:
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

# DANGUTTA:
# Bert hatire on the wo A? sewas ker haywins wet.

# mousill akerd fous sther Le fke sth,, bepeere gn ssst I oun amand serewos, cobear song.


# AMEfediengg yit othesshint.

# Anc:
# A must shan ghety ak is, shet.

# MEO:
# Tom yot stths!
# NCIBus hand akt imy berdr,
# Thulll id mere thassut,
# Moureleray, fous mante. QBO decencepous ariveed hart;
# I; bifo wthen chant tht tho hy youn,
# Grolomy, ucel.

# BRIINE ERIYht I arum arw t; odeat ngher bly wehe foru thas sot our:
# Bn; sshiutithemowith her fourpler thant

#
# which looks depressing not gonna lie! but its better than the simple bigram model we 
# built earlier, anyway, can we improve it? certainly! 
# for example we could use dropout on our attention! we could use normalization layers
# and we could use multiple heads instead of just one! which takes us to the next
# subject which is multi-head attention! which is  really nothing except several normal!
# attention blocks run in parallell! 
# so lets implement these and see how much we can improve upon this

results=tensor([[[ 0.,  1.],
         [ 1.,  2.],
         [ 2.,  3.],
         [ 3.,  4.],
         [ 4.,  5.],
         [ 5.,  6.],
         [ 6.,  7.],
         [ 7.,  8.]],

        [[16., 17.],
         [17., 18.],
         [18., 19.],
         [19., 20.],
         [20., 21.],
         [21., 22.],
         [22., 23.],
         [23., 24.]],

        [[32., 33.],
         [33., 34.],
         [34., 35.],
         [35., 36.],
         [36., 37.],
         [37., 38.],
         [38., 39.],
         [39., 40.]],

        [[48., 49.],
         [49., 50.],
         [50., 51.],
         [51., 52.],
         [52., 53.],
         [53., 54.],
         [54., 55.],
         [55., 56.]]])
tensor([[ 0.,  1.],
        [ 2.,  3.],
        [ 4.,  5.],
        [ 6.,  7.],
        [ 8.,  9.],
        [10., 11.],
        [12., 13.],
        [14., 15.]])
tensor([[0., 1.],
        [1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.],
        [5., 6.],
        [6., 7.],
        [7., 8.]])
a=tens

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_head, head_size, embd_size, context_size,bias_attn=False) -> None:
        super().__init__()
        self.num_head = num_head
        self.head_size = head_size
        # we assume the head_size is already split between the num_heads and thus we dont split
        # it again
        # assert head_size//num_head == 0, f'head_size({head_size}) must be divisable by head_num({num_head})'
        # we need to create n heads so lets do it 
        # self.heads = [AttentionHead(context_size, 
        #                             embd_size, 
        #                             head_size, 
        #                             bias_attn) for _ in range(num_head)]
        # but we can also use pytorch's nn.ModuleList which is a better equivalent than list
        # note that, nn.Sequential cant be used, becasue it runs the modules in succesion
        # i.e. serially, one after the other (feeds the output of the previous module to 
        # the next module, etc) which is not what we want. we want to calculate each head
        # independetly and aggregate their outputs so, either a python list or torch moudle list
        # can be used
        # sidenote: this module that we are building, is also known as, masked multi-attention-head
        # becasue we are using the single-head self-attention which uses a constrain we imposed
        # by masking if you recall that!
        self.heads = torch.nn.ModuleList(AttentionHead(context_size, 
                                         embd_size, 
                                         head_size,
                                         bias_attn) for _ in range(num_head))
      
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # note that head_size is usually the embd_size, so it covers the whole 
        # embeddings obviously, and note that each head will work on a portion
        # of the given head_size, if e.g. we have head_size/embd_size = 16
        # if we had 1 head, the head_size would be 16. however if we had 2 heads
        # we had to split the head_size in half for each head, and later on
        # we had to concat them to get the full-size head_size/embd_size.
        # therefore here, we need to concat their results along the cols
        # so multi-head-attention is akin to group convolution, and thus the head_size
        # and num_head must align properly.
        outputs = torch.cat([head(inputs) for head in self.heads], dim=-1)
        return outputs
# lets test 
# note that we set the split value for head_size based on our num_head
m = MultiHeadAttention(num_head=4, head_size=16//4, embd_size=16, context_size=8, bias_attn=False)
logits = m(torch.randn(size=(1,8,16)))
print(f'{logits.shape=}')
# 
# now lets use this in our model
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 
# now lets train this model and see how it performs this time
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# outputs : 
# param_count  =  3,041
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2041  val: 4.204
# train: 2.6895  val: 2.685
# train: 2.5449  val: 2.528
# train: 2.4570  val: 2.477
# train: 2.4238  val: 2.433
# done!

# I'd In mame
# Ast owu hapland.

# Mayl thew youres?
# IRisire dis lhend gitim whorder, sphorgunct Beye skek boutrig!

# I, sho sciltr not bpreathl harr'd doess elowt cof or iny poovigre.
# Thalrit?

# HThary?

# Lerim he bamat:
# Thiet hey, krepably bose bour cave grosr benemece wleir hon'g.

# MADIUMI but th Mond im's veoo tit. IDf met Cerve's wue rrer hiftren'tr'd:
# Anoks:
# HOt
# We I phess chi bouine mares yon ther; tlotirtlinl therot:
# Io
# Wh per,
# Fous thayigso ceany pate.

# Annd wosh fong uat
# Now ICUSo CCgodertiove
#
# trying with larger embedding size seems to improve the results: 
# param_count  =  7,553
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1828  val: 4.186
# train: 2.4703  val: 2.472
# train: 2.3649  val: 2.367
# train: 2.3009  val: 2.32
# train: 2.2713  val: 2.288
# done!

# Cly Toste?
# IOK:
# Whan his me' tis it I ond ber'thse?

# PO:
# Theray delling, dothoinerk an to the, or
# E VINGON: rocO:
# Thes win now thatin knir:
# Wigh fy, bund werar is wet my;
# I ene:
# JUu'd sow prut hat amver'd ENDUE VUF E VO?

# Se gings nom and.
# CArwak mandesplay beesire brit mand.

# MED:
# Thou ased ith is whis wentemprt hits this in ther.
# Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
# LUT: am le the wids my thousecarto my haty wind WLIERME:
# Wher a and
# Whesh to wyt wir

# compared to single head attention, with the same hyper parameters, train: 2.4684  val: 2.4759
# and now we got train: 2.4238  val: 2.433 which is better(we also got train: 2.2713  val: 2.288 
# with just increasing the embedding_size, so playing with parameters even blindly making model bigger
# seems to give us a boost), so we had improvements, but we still need a long way ahead of us!
# 
# Ok, to improve upon our results, there are couple of more things we need to add to our attention 
# block. if you look at the paper, we'll see a few concepts that we havent talked about or implemented
# yet, including the feed-forward(position-wise feedforward network) and layernorm, so lets talk about 
# them.
# feed-forward network or as its called in the paper,'position-wise feedforward network', is simply
# a "fully connected network which is applied to each position separately and identically", this 
# consists of two linear layer with a relu activation function inbetween. 
# so its a linear layer with a relu activation function followed by another linear layer basically. 
# you may also hear this network be refered to as 'computation after communication' so to speak!
# signifying the fact that it runs a computation after the attention module.
# the idea behind this extra addition is simple, to provide a higher representation out of attention work
# this network, causes every single token to have a nonlinear transformation and achieve a higher abstraction
# possibly yielding new information benficial to the task. to put it in casual way, it can be seen as though
# the attention gatheres some stats/information, and this step is akin to looking into it and thinking about it
# comming up with some new findings, that is , if attention part is refered to as communication part, this is
# the computation part/ or thinking part for the lack of a better word.
# lets implement this network in our bigram model and see if this seemingly simple change, affects us at all
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # now let us create the feedforward network, which basically is a linear layer with relu
        # followed by another linear layer! for this so called network, the in_features and out_features
        # are simply the same, and is embd_size, because its a sandwich layer between our attention output
        # and the last layer. this layer usually is (embedsize,embed_size) if you recall head_size is usally
        # equal to embed_size. also note that in the paper the feedforward network is two linear layers with
        # a relu (one linear layer with a relu plus another linear layer).if you look closely you'll notice 
        # we already have a last layer after the attention, so we dont need to put a nother linear layer
        # afterward, becasue that one linear layer suffices (multiple linear layers, dont provide
        # any higher abstraction anyway, so its prefectly fine)
        # but on the other hand the paper's implementation has another change, it states 
        # the inner layers dim are increased by 4x, so to be faithful to the paper we also 
        # add the second linear layer, with the suggested change, this wont change the output 
        # shape, so we are fine. we just added an extra linear projection layer. 
        # this additional projection operation actually improves the result,however if we simply 
        # use the same dim linear layer (i.e. have sth like linear(head_size, head_size)) adds nothing
        # to the representational power of the network and you wont see anything substantial vs if you
        # completely remove this layer and only use linear/relu only.
        # why this works is becasue, we increase the nonlinear output neurons of the first layer by 4,
        # increasing its representational capacity, and then use the second linear layer to get the output
        # size compatible for the next layer (doing a linear projection), and hence our improvements lie
        # in the nonlinearity this addition provides. 
        self.feedforwardnet = nn.Sequential(nn.Linear(embd_size, head_size*4), nn.ReLU(),
                                            nn.Linear(head_size*4, head_size))
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # add our new addition, feedforwardnet
        out = self.feedforwardnet(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using feedforwardnet added')
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints : 
# using feedforwardnet added
# param_count  =  5,169
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1975  val: 4.198
# train: 2.6222  val: 2.634
# train: 2.4919  val: 2.481
# train: 2.4320  val: 2.435
# train: 2.3979  val: 2.392
# done!

# Selwils iur.

# AnNdt
# By thak, yue, nein ende vat Rout'w haler,
# The hot hes? is, whas bles. Yig,
# ROm
# QUS:
# Ay Whigice rea:
# Therer a youst um mith dins I dorseancer by gou:
# I sot the, aserer tom sne arobfs-Rid,
# This sas whe ou,
# Acet Yark no.

# OT:
# Fely ip you pel hivet seatly. hial pand,
# Thand repwat,
# I tarve ther se this ten thio drord dot tho,
# Rulee res le, il fet!
# Nord Wotiilet homom for my woue cuve wid.

# Angilene.

# Fhivy, ih:
# Boumlke uthereaerf
# Whe ithe hares.

# Whos sthik thenth
# Tait:
# She tove w

# the loss didnt get better! but maybe if we increase the context_size, and embedding_size, it 
# perorm better?
# using a larger embedding_size and thus larger model did improve the result, 
#
# using feedforwardnet added
# param_count  =  15,905
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1569  val: 4.159
# train: 2.3885  val: 2.396
# train: 2.2789  val: 2.312
# train: 2.2064  val: 2.249
# train: 2.1599  val: 2.198
# done!

# To get fer arast deve lad, I' fale are you up this riany;
# Weareien,
# Tho Guser, me ter is! Do lothe, to ling wo whe Pist to deave mat not, in and- air besface, y to fron; use ler weet herefalf thop an der pretrier
# nesly toide
# Thats ous.

# RENT:
# I for the haw to, to noss teave ler shat earelss ater, want Heanct herry deeter erry, heave masere dacke; nochat onsemy ning Meib,--?

# LUCLUER:
# If Feropeake mo shat onsivoole washy the well beWarquent;
# I on well he ome it you lare to lo to kit youser,
# Weves

# the larger network, seems to be doing much better than its counter part from before
# the one without the ffnet, we got train-loss: 2.1599  val-loss: 2.198 here whereas 
# previously we got train-loss: 2.2713  val-loss: 2.288. so we have improvements despite the 
# text still not being that good(but still better than before). so we need more enhancements)

logits.shape=torch.Size([1, 8, 16])
param_count  =  7,553
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1828  val: 4.186
train: 2.4703  val: 2.472
train: 2.3649  val: 2.367
train: 2.3009  val: 2.32
train: 2.2713  val: 2.288
done!

Cly Toste?
IOK:
Whan his me' tis it I ond ber'thse?

PO:
Theray delling, dothoinerk an to the, or
E VINGON: rocO:
Thes win now thatin knir:
Wigh fy, bund werar is wet my;
I ene:
JUu'd sow prut hat amver'd ENDUE VUF E VO?

Se gings nom and.
CArwak mandesplay beesire brit mand.

MED:
Thou ased ith is whis wentemprt hits this in ther.
Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
LUT: am le the wids my thousecarto my haty wind WLIERME:
Wher a and
Whesh to wyt wir
using feedforwardnet added
param_count  =  15,905
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1569  val: 4.1

In [4]:
# the next improvement, would be to use more of these, as its shown in the paper, if one block
# works, then adding more should work better right? we had single head, it improved our condition, 
# so we used more heads, and now lets more multi-attention heads! to make things easier, lets
# make a module out of it the same way we created previous modules. 

# first lets create our ffnet as a seprate block, so we use it after the attention
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        return self.ffnet(self.attn(inputs))

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks!')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks!
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2087  val: 4.211
# train: 2.6084  val: 2.606
# train: 2.3737  val: 2.39
# train: 2.2865  val: 2.307
# train: 2.2087  val: 2.237
# done!

# FAUKIN YRE:
# Dato':
# wis.

# BERDEWEE:

# Piwit ancon; awt
# Emuke this his fred not pe,
# Har,
# And him my bird-wray sluf. Ile:
# With tulel wik, antegatorg.

# EREN VELV:
# Whath a meane, ones
# sheirss,
# Bose til
# Ilf bodtund ba,
# Aripce nerod is the blilshil hit hit mef?

# GESINCE:
# Sard, with the deoorrnsicominle thid: as.

# ANRE:
# Hy prall anges what micciths delre
# To tike comf hon ebat'd mur. Kin'sa, Fharth:
# Af'?

# ROASA:
# Ect mily fledd.
# SHricest.
# At dey bompe novor where lisg.

# KETILNINCE:
# There gay his my nefonk
#
# as you can see, we got wrose results than before! despite making the network larger, our loss
# really didnt improve as we expected. 
# is our initial hypothesis that having more blocks and higher nonlinearity/representation is benificial
# wrong? or is there something else thats causing the issue? 
# as you might have guessed, its the latter. we are basically creating more layers, and with
# more layers, we face training issues that we discussed earlier. 
# so how should we tackle this, we cant use BN, becasue BatchNOrm, accumulates the statistics
# from different samples, this is a no no for us. we dont want other samples to interfer with 
# our sample (or basically each other!) in anyway, so what should we do? 
# the paper utilizes two mechanisms or operations to tackle this issue. one being skip-connections
# (also known as residual connction) and the other, layer-normalization. 
# you should be familiar with skip-connections as they are the founding factor or resenets
# and have been extremely influential. so to cut a long story short, skip connections are simply
# connections from input skipping the operations involved in the block they reside and directly 
# being added to the output and then returned the result.(basically F(x) + x)
# as we know, the gradients are distributed equally when they reach addition, so input gets the gradients
# without being weakened due to large depth of the network. 
# so before we implement the layer normalization part, lets see howmuch of a change adding skip-connection
# causes and whether it proves our initial hypothesis about depth and gradient signal weakening or not
# we add this to our AttentionBlock (but we could add this to any submodule)

using more blocks!
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.2087  val: 4.211
train: 2.6084  val: 2.606
train: 2.3737  val: 2.39
train: 2.2865  val: 2.307
train: 2.2087  val: 2.237
done!

FAUKIN YRE:
Dato':
wis.

BERDEWEE:

Piwit ancon; awt
Emuke this his fred not pe,
Har,
And him my bird-wray sluf. Ile:
With tulel wik, antegatorg.

EREN VELV:
Whath a meane, ones
sheirss,
Bose til
Ilf bodtund ba,
Aripce nerod is the blilshil hit hit mef?

GESINCE:
Sard, with the deoorrnsicominle thid: as.

ANRE:
Hy prall anges what micciths delre
To tike comf hon ebat'd mur. Kin'sa, Fharth:
Af'?

ROASA:
Ect mily fledd.
SHricest.
At dey bompe novor where lisg.

KETILNINCE:
There gay his my nefonk 


In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
# lets add a skip-connection to this block
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        out = self.attn(inputs) + inputs
        out =  self.ffnet(out)  + inputs
        return out

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5636  val: 4.553
# train: 2.2666  val: 2.281
# train: 2.1330  val: 2.183
# train: 2.0708  val: 2.14
# train: 2.0204  val: 2.109
# done!

# Zunt will jucked's whreef onst's so
# wippplalct, sawter.

# Firfor have deal pere,
# Shal,
# thich many.
# Frig.

# Thall have.

# WANWHAM:
# Sad king theave,
# Cnomlen nare, near was?


# CKINGlLOROMBOENGBETH:
# Dich loverd und by, may;
# ''Kh
# Godie the bline plock's minstell the denelsbad, with think
# What
# sicond lead this undrut, too, the but me my se ming'ld;
# Shing
# To tiot comford, engelan this it's heare: hear them the vinclamil.
# Is done, the stronced,
# Lion?
# Hartow where lisg.

# DOLILINA:
# Lord ISe and
# Lud'd nef hel

# and test with one skip-connection on the 'output only' to prove or assumption on skip-connections
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5616  val: 4.552
# train: 2.3196  val: 2.322
# train: 2.1862  val: 2.221
# train: 2.1175  val: 2.163
# train: 2.0640  val: 2.137
# done!

# FARY VI:
# A-bame 'tries.

# BERGEWES:

# PABLLO:
# Yon; aws
# Then ath-pmate for, not pe,
# PARILA:
# So man.

# FORY.

# TAsUS:
# I cenpine nou, lad king theak,
# Anny, In nartine-'

# An yeant, of ladie ust, good ting lover that by, may;
# And hordie the
# llils plove ond sofful and heave
# Take with the deo,
# And mort leath and usir the dom the but me my crombasill;
# Shim are, con comlmork engeland your, trabe,
# Beth:
# Yor?

# CORIAR EFt mily fle--spoles, strence'd ono do now, will?
# Mursice,
# Ntormorest
# The juge ernus'd neforki

# as you can see, it greatly improved our results, and the text also got much better!
# as we already pointed out, having skip-connections per modules, help much more than a single 
# per module output only, nevertheless, we notice, using skip-connection really improved our results.
# now lets add the second operation, layernorm. layernorm(https://arxiv.org/abs/1607.06450) is a normalization layer, just like batchnormalization
# came a year later than batchnormalization paper, but the difference between them is that, unline batchnormalization
# it works on a per sample basis and does not involve using othersamples to normalize a specific sample (basically
# samples dont affect eachother)
# As pytorch docs puts it:
# "Unlike Batch Normalization and Instance Normalization, 
#  which applies scalar scale and bias for each entire channel/plane with the affine option,
#  Layer Normalization applies per-element scale and bias with elementwise_affine"
# the implementation is similar to the batchnormalization, and it does not require calculating running_mean/var
# we can use the pytorch module just fine, but since its really similar to BN, lets implement it here 
# 

using more blocks with skip-connection
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.5636  val: 4.553
train: 2.2666  val: 2.281
train: 2.1330  val: 2.183
train: 2.0708  val: 2.14
train: 2.0204  val: 2.109
done!

Zunt will jucked's whreef onst's so
wippplalct, sawter.

Firfor have deal pere,
Shal,
thich many.
Frig.

Thall have.

WANWHAM:
Sad king theave,
Cnomlen nare, near was?


CKINGlLOROMBOENGBETH:
Dich loverd und by, may;
''Kh
Godie the bline plock's minstell the denelsbad, with think
What
sicond lead this undrut, too, the but me my se ming'ld;
Shing
To tiot comford, engelan this it's heare: hear them the vinclamil.
Is done, the stronced,
Lion?
Hartow where lisg.

DOLILINA:
Lord ISe and
Lud'd nef hel


In [6]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (2.38418573772492e-09, 1.0004067420959473)
torch: (9.53674295089968e-09, 1.0004067420959473)


In [7]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
#
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we were aggregating the last two dims like BN, whereas we should have only used the
# last dim, as we should not involve other samples in normalization. (I explained this thoroughly in the LayerNorm class)
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3906  val: 4.385
train: 2.2373  val: 2.244
train: 2.1070  val: 2.155
train: 2.0339  val: 2.102
train: 1.9812  val: 2.069
done!

For the that mad'
With of on tith, luidie anct, saws
Eule at botht if thou
Wlefted' you windian.

FLO-wike sluke slen:
I have, ladik, anteraves,
norl wink
the bard a yeane, of lady. Whe, goshot I
thy brath his what:
Nown hondin the
blinsh, shat this the pandself livadet brives nefors.
ISIUS:
Aren that usir the of me
he hange wicHe mirs,
Behat!


TOLAT:
But ford, engelan this it'sabhanke hear rend these this in usse,--
Lord, strence'er thy?
In woes. IDandry,
Soset frongs neved
With haul'd nefonk 


In [8]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 256
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

using more blocks with skip-connection-layernorm
param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [9]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 256
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')

model.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

using more blocks with skip-connection-layernorm
param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 67.81 MiB is free. Process 6645 has 227.69 MiB memory in use. Including non-PyTorch memory, this process has 9.21 GiB memory in use. Of the allocated memory 8.18 GiB is allocated by PyTorch, and 47.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [10]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 250
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')

model.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

using more blocks with skip-connection-layernorm
param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 94.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 47.56 MiB is free. Process 6645 has 227.69 MiB memory in use. Including non-PyTorch memory, this process has 9.23 GiB memory in use. Of the allocated memory 8.07 GiB is allocated by PyTorch, and 182.34 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [11]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

torch.cuda.empty_cache()

head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 250
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')

model.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

using more blocks with skip-connection-layernorm
param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 94.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 89.06 MiB is free. Process 6645 has 227.69 MiB memory in use. Including non-PyTorch memory, this process has 9.16 GiB memory in use. Of the allocated memory 8.02 GiB is allocated by PyTorch, and 156.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [12]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

torch.cuda.empty_cache()

using more blocks with skip-connection-layernorm


In [13]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)
del model
torch.cuda.empty_cache()

using more blocks with skip-connection-layernorm


In [14]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)
import gc 
del model
gc.collect()
torch.cuda.empty_cache()

using more blocks with skip-connection-layernorm


NameError: name 'model' is not defined

In [15]:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)
import gc 
with torch.no_grad():
    torch.cuda.empty_cache()

using more blocks with skip-connection-layernorm


In [1]:
# in the name of God the most compassionate the most merciful
# in this section we will have a look at how a GPT model works
# in this section we will implement attention module, and see 
# how it works and lean more about its (sel-attention, cross
# attention, multi-head attention, etc)
# 
# so how are we going about this. we need to create a language model
# and then progressively imporve it with attention mechanism.
# we will basically be creating a kind of chatgpt (minus its chat capability and
# its obviously great perormance :)) 
# to keep this as simple as possible and not lose track of the important concepts involved,
# we can use our initial bigram model as the base model and work on improving that with attention.
# so lets start
#
# first lets import the basic stuff 

# for type hints
from collections.abc import Iterable

import random 
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# lets setup the manual seeds or determinstic output
torch.manual_seed(255)
np.random.seed(255)
random.seed(255)

# first lets read the dataset, we are going to use tinyshalespear which
# is around 1million characters (around 1MB in size), and our model is
# supposed to create texts resembling this dataset.
dataset = []
with open('./tiny_shakespear.txt','r') as file:
    # this time we read the whole text as one big str
    dataset = file.read()
    
print(f'{dataset[:100]=}')
# now lets create our atoi and itoa dictionaries for mapping
# lets create our unique character list, which is infact our vocabulary or vocab for short
vocab_list = sorted(set(''.join(dataset)))
vocab_size = len(vocab_list)
# ! explain token
# lets create our mapping dictionaries, we are basically going to use them
# for tokenization, converting our input into tokens which here are characters 
# and ultimately their integer representations
# so tokenization simply put, refers to converting an input into a list of numbers based on some creteria
# here, we just use a simple index number to map our vocabulary characters, which represent all character
# our model can use to generate text, to integer representation. 
# for realworld applications, we usually use libraries such as sentecepeace by google which works on a subword
# level which means, it doesnt encode the whole word, neither does it encode based on individual characters, it
# use sth in between. 
# (from its github repo: https://github.com/google/sentencepiece )
# SentencePiece is a re-implementation of sub-word units, an effective way to alleviate the open vocabulary 
# problems in neural machine translation. SentencePiece supports two segmentation algorithms, 
# byte-pair-encoding (BPE) [Sennrich et al.] and unigram language model [Kudo.]. 
#
# (For the reference:  
# Byte Pair Encoding (BPE) is a subword tokenization technique used in natural language processing (NLP) 
# and text processing tasks. It is a data compression algorithm that splits words into subword units. 
# BPE is commonly used in tasks such as machine translation, text generation, and language modeling.
# The basic idea behind BPE is to iteratively merge the most frequent pairs of characters or subword units
# in a corpus to create a new subword vocabulary. This merging process is based on the statistical properties
# of the corpus, specifically the frequency of character or subword pairs.
#
# Here's a high-level overview of the BPE algorithm:
# 1. Initialize the vocabulary with all the characters or subwords in the corpus.
# 2. Calculate the frequency of each character or subword in the corpus.
# 3. While the desired vocabulary size or a maximum number of iterations is not reached:
#    - Find the most frequent pair of characters or subwords in the corpus.
#    - Merge the pair into a new subword unit by concatenating them.
#    - Update the corpus by replacing occurrences of the merged pair with the new subword unit.
#    - Update the vocabulary and frequency counts based on the new corpus.
# 4. The final vocabulary is the set of subword units obtained after the desired number of iterations 
#    or vocabulary size is reached.
#
# BPE allows for the representation of both known and unknown words in a corpus. It is effective in 
# handling out-of-vocabulary (OOV) words and reducing the vocabulary size, which can improve the 
# efficiency and performance of NLP models. By breaking down words into subword units, BPE can capture
# morphological and semantic information more effectively, especially for languages with complex word 
# formations and agglutinative structures.
# 
# tiktokenize repo explains BPE in rather friendlier way: 
# Models don't see text like you and I, instead they see a sequence of numbers (known as tokens). 
# Byte pair encoding (BPE) is a way of converting text into tokens. It has a couple desirable properties:
# It's reversible and lossless, so you can convert tokens back into the original text
# It works on arbitrary text, even text that is not in the tokeniser's training data
# It compresses the text: the token sequence is shorter than the bytes corresponding to the original text. 
# On average, in practice, each token corresponds to about 4 bytes.
# It attempts to let the model see common subwords. For instance, "ing" is a common subword in English, 
# so BPE encodings will often split "encoding" into tokens like "encod" and "ing" 
# (instead of e.g. "enc" and "oding"). Because the model will then see the "ing" token again and again 
# in different contexts, it helps models generalise and better understand grammar.
#
# 
# A unigram language model however, is a type of statistical language model that predicts the probability 
# of each word in a sequence independently, based solely on the frequency of occurrence of individual 
# words in the training data. It does not consider the context or the order of the words in the sequence.
# In a unigram language model, the probability of a particular word is estimated by counting the frequency 
# of that word in the training corpus and normalizing it by the total number of words in the corpus. 
# The probability of a sequence of words is then calculated by multiplying the probabilities of each 
# individual word in the sequence.
# For example, consider the sentence "I love to eat pizza." In a unigram language model, the probability
# of this sentence would be calculated as the product of the probabilities of each word which would be: 
# P(I) * P(love) * P(to) * P(eat) * P(pizza).
# Unigram models are the simplest form of language models and do not capture any contextual information
# or dependencies between words. They are often used as a baseline or reference model in natural language 
# processing tasks. While unigram models are not very accurate in capturing the complexities of natural 
# language, they can be computationally efficient and useful in tasks where the context is less important, 
# such as certain text classification tasks or language generation tasks where only the frequency of 
# individual words matters.
# 
#
# Openai/chatgpt for example uses its own tokenizer, called tiktoken (https://github.com/openai/tiktoken)
# there are other tokenizers as well.
# huggingface has a good article on tokenizers (recommend it): https://huggingface.co/docs/transformers/main/tokenizer_summary 
# its a given that, each model only works with the tokenizer by which it was used to train. 
# so BERT, DistilBERT, and Electra only work with wordpiece, while chatgpt uses titoken and we! use
# simple characters-indexes! also note that the choice of tokenizer obviously affects our vocab size and
# model overhead/performance as well. 
# for example lets first implement our tokenizer and then compare it with sth like tiktoken module
atoi = {c:n for n,c in enumerate(vocab_list)}
itoa = {n:c for c,n in atoi.items()}
print(f'{vocab_size=}, {vocab_list=}')
print(f'{atoi=}')
print(f'{itoa=}')
# lets also create two helper functions to convert a list of these to the other part
def encode(characters:Iterable[str] ):
    return [atoi[c] for c in characters]

def decode(token_lst:Iterable[int]):
    return [itoa[n] for n in token_lst]

# lets test these 
print(f'{encode(dataset[:10])}')
print(f'{decode(encode(dataset[:10]))}')

# now lets try tiktoken
try: 
    import tiktoken
except:
    import os 
    os.system('pip install tiktoken')
    import tiktoken
# lets use the tokenizer for gpt2 model, this line downloads the gpt2 tokenizer and allows us to use it
encoder_gpt = tiktoken.get_encoding('gpt2')
# lets view its vocab size:
print(f'{encoder_gpt.n_vocab=:,}') # prints encoder_gpt.n_vocab=50,257
# now lets see how the encoding looks like here
print(f'{encoder_gpt.encode(dataset[:10])=}') # prints [5962, 327, 8846] 
# which is very intresting! while for us its 10 numbers/tokens for 10 characters(vocab size 65),
# this is 3 here with vocab size of 50,000!
# so we can have a short vocab size, at the expense of a larger sequence size, 
# or a large vocabsize and smaller sequence size.
# so thats why in practice, these subwords tokenizers are used for real world applications. but for our case
# we stick to our primitive tokenizer to keep things as simple as possible. 
# 
# Ok, so far so good. now we need to create a dataset, like before we need to have an input/label pair
# our input is a series of characters, (a sequence of some length), and our label is the next character
# sicne we are using a simple bigram model, given a single character, we want the probablity of what comes
# next. but, we also want to incorporate attention, and we want a context for our prediction, we want to
# be able to look at the past, and look at the past characters, and based on that do sth. this is the essence
# of attention (although this is not accurate, but for now this is the case, we will elaborate on this and expand
# this metaphor and reasoning inshallah)
# 
# lets first tokenize the whle dataset or corpus as its usually called in nlp nomenclature!
data = encode(dataset)
# since we are using pytorch lets convert that to a tensor
data_tensor = torch.tensor(data)
print(data_tensor[:100])
# now lets create a train/val split 
train_length = int(0.9 * len(data_tensor))
# note that we do not shuffle the data here like before, because we are not dealing with a list of samples!
# like names, here we are dealing with the whole text, and if we shuffle it like that, we just destroy it
# making it into a batch of random characters! which would not be useable for us anymore. we are trying to
# learn the underlying semantic and relationships hidden in our data, and randomizing them like that just 
# destroys those information! 
# to see this in action try this
_data_copy = data.copy()
random.shuffle(_data_copy)
print(''.join(decode(_data_copy[:100])))
# prints:
# Osetow Pr 
# oa geEeM rhnalelrt   aAdt Ktsw pTpeGxli
# oasEliMe
# vsieeose
# etttbSdnctiras  hnr:yhtayiuCv g
# so we simple divide the data normally
train_data = data_tensor[:train_length]
val_data = data_tensor[train_length:] 
# note that since we are planning on creating a simple transformer model, we usually dont feed the whole dataset
# becaue its prohibitevly computation intensive, instead, what happens in practice is that we, grab chunks 
# of data from the dataset and feed it to the transformer. and these chunks, of course has a length, what length?
# we usually specify a maximum_length for the input on which our transformer model works.
# this maximum_length is usually refered to as block_size or context_size.
# we had previously used different context_sizes and this is not really that different,
# lets for example define a context_size of 8
# block_size and context_size are interchangable
block_size = context_size = 8 
print(f'{train_data[:block_size]=}')
# which prints:
# train_data[:block_size]=tensor([18, 47, 56, 57, 58,  1, 15, 47])
# 
# one intresting observation we can make here is that, as simple as this seemingly ordindary list of numbers
# looks, this sequence of numbers, actually contains several examples. 
# if you think about it, each number, is a token, representing a character (or word, subword, etc),
# and it shows what comes after what. basically not only it shows several pairs so to speak, it also
# shows, which characters are more likely to come, before a specific character comes later. 
# what we are actually going to do is that, we are going to train all of these characters simultaneously
# notice that in this example, we have 7 examples in a sequence of 8 characters:
# lets elaborate on this more. 
# 1-in the context of 18, the next character is 47
# 2-in the context of 18,47, the next character is 56
# 3-in the context of 18,47,56, the next character is 57
# 4-in the context of 18,47,56,57 the next character is 58
# 5-in the context of 18,47,56,57,58 the next character is 1
# 6-in the context of 18,47,56,57,58,1 the next character is 15
# 7-and finally, in the context of 18,47,56,57,58,1,15 the next character is 47
# so this is infact 8 7 individual example embedded in a single context, 
# since we want the xontext_size to be 8, then we should grab one more character to have 
# 8 contexts
# lets visualize this in example in code
# lets have a typical sequence of size 8 
x = train_data[:block_size]
y = train_data[1:block_size+1]
# what would be our label? we want the next character so it would be the previous locations +1
# basically offset the input by 1!
# thats why we started from 1, becasue the input started from 0, and since the input ended at block_size
# its label(next character) would obviously be block_size+1, pretty obvious right?
#
# lets print this for better understanding 
for i in range(block_size):
    # note that since we are using slice, x[:i+1], gives us 0 up to ith element (inclusive)
    # this should be obvious! but I said it in case you forgot!!
    print(f'input is {x[:i+1].tolist()} label is {y[i]}')
# prints 
# input is [18] label is 47
# input is [18, 47] label is 56
# input is [18, 47, 56] label is 57
# input is [18, 47, 56, 57] label is 58
# input is [18, 47, 56, 57, 58] label is 1
# input is [18, 47, 56, 57, 58, 1] label is 15
# input is [18, 47, 56, 57, 58, 1, 15] label is 47
# input is [18, 47, 56, 57, 58, 1, 15, 47] label is 58
# so as you can see we started from context_size of 1 up to context_size of 8.
# note that we do not do this simply for the efficancy aspect of it, but also, when we feed these to
# our transformer model, we are making sure that the transformer model gets used to see all combinations
# of our input as well (from the context size of 1 up tp the context size of 8). 
# this way not only it sees the whole context_size as we initially expected
# but also all sequences before it, and basically what consituted to make the sample. 
# this allows us to later on, at test time be able to create sequences as small as context_size of only 1
# up to the max_length which is our context_size of 8 in our case.
# ! recheck and elaborate to clear any confusion
# !so by doing this, the transformer can learn how to predict/create/genrate text up to context_size, and after
# !it reached thta, we have to truncate it, becausse the transformer model never recieves more than the
# !context_size as input when its predicting the next character.
# so far what we covered here was the time dimension of our input. we have a sequence, and each entery
# basically denotes a time t dimension, at which, a character is introduced. 
# another imporatnt aspect we need to take care of is the batch dimension, cuz we are going to feed 
# multiple examples at once, we use this to harness the gpu parallilization capalibity in pytorch
# as without it, training this simple model would take a lot of time on cpu! 
#
# so lets create a function that gives us a batch of the inputs rather than a single list/tensor!
# 
batch_size = 4 
# our block_size
context_size = 8 
# 
def get_batch(split, batch_size):
    #lets grab the data basedo on the split
    data = train_data if split =='train' else val_data
    # we want a batch of 4 of 8 characters (context-size). 
    # to make a batch we can grab 4 random indices as input and then expand them
    # by adding the next 8(context_size) characters to them, in order not to go past the 
    # last index, we subtract the length of data(last valid index) from context size 
    idxs = torch.randint(low=0, high=len(data)-context_size, size=(batch_size,))
    # we have our indices, so lets create our samples
    # x = [data[i:i+context_size] for i in idxs]
    # and for labels we do the same thing but offset it by 1 so each characters label
    # becomes the next character
    # y = [data[i+1:i+context_size+1] for i in idxs]
    #
    # print(f'{x=}')
    # print(f'{y=}')
    # prints: 
    # x=[tensor([58, 39, 49, 43,  1, 51, 63,  1]), tensor([53, 59, 41, 46,  5, 42,  1, 61]), tensor([47, 53, 52,  1, 39, 57,  1, 63]), tensor([39, 57, 58,  1, 51, 63,  1, 50])]
    # y=[tensor([39, 49, 43,  1, 51, 63,  1, 54]), tensor([59, 41, 46,  5, 42,  1, 61, 47]), tensor([53, 52,  1, 39, 57,  1, 63, 53]), tensor([57, 58,  1, 51, 63,  1, 50, 53])]
    #
    # as you can see we have a list of tensors. since we want a batch, we just stack them or concat them
    # on top of each other
    # using stack!
    # x = torch.stack(x)
    # y = torch.stack(y)
    # stack() is intrestingly implemented by concat(), so if we want, we can do the same using concat!
    #
    # by default conact, concatenates all the tensors as one large tensor!
    # we need a reshape/view to get the right shape
    # since we are using view, no copy,etc is done, and its an efficient operation
    # x = torch.concat(x,dim=0).view(batch_size, context_size)
    # y = torch.concat(y,dim=0).view(batch_size, context_size)
    # 
    # however if we add an extra dim to our inputs we can remove the extra reshape/view
    # x = torch.concat([t.unsqueeze(0) for t in x], dim=0)
    # y = torch.concat([t.unsqueeze(0) for t in y], dim=0)
    #
    # By unsqueezing, we ensure that the tensors have the same number of dimensions before 
    # concatenating them with torch.cat/concat/concatenate.(side tip: concat and concatenate are aliases for cat) 
    # This effectively replicates the behavior of torch.stack.
    # see torch.cat concatenates tensors along an existing dimension, and we only have 1 dimensional
    # tensors, so it will concatenate them along that, effectively making one large 1 dimensional vector
    # however, when we use unsqueeze on each tensor, using torch.unsqueeze(0), we are adding a new dimension
    # (dim 0) to that tensor, making it 1,x instead of the original shape of (x,).  
    # So, by unsqueezing each tensor along the desired dimension, which for us is the 0ths dimension (or row dim) 
    # and then concatenating them with torch.cat, we achieve the same result as torch.stack.
    # 
    # in practice we'd like to do this all in one go!
    x = torch.stack([data[i:i+context_size] for i in idxs])
    y = torch.stack([data[i+1:i+context_size+1] for i in idxs])
    
    return x,y

x,y  = get_batch('train',batch_size=4)
print(f'{x.shape=}\n{y.shape=}')
print(f'{x=}\n{y=}')
# which prints 
# x.shape=torch.Size([4, 8])
# y.shape=torch.Size([4, 8])
# tensor([[58, 39, 49, 43,  1, 51, 63,  1],
#         [53, 59, 41, 46,  5, 42,  1, 61],
#         [47, 53, 52,  1, 39, 57,  1, 63],
#         [39, 57, 58,  1, 51, 63,  1, 50]])
# tensor([[39, 49, 43,  1, 51, 63,  1, 54],
#         [59, 41, 46,  5, 42,  1, 61, 47],
#         [53, 52,  1, 39, 57,  1, 63, 53],
#         [57, 58,  1, 51, 63,  1, 50, 53]])
#
# now this is our batch of data, 4 samples with 8 characters, bascially 32 examples, lets see that as well
for b in range(batch_size):
    # this is our time dimension
    for t in range(context_size):
        print(f'{x[b,:t+1]} --> {y[b,t:t+1].item()}')
#        
# prints : 
# tensor([58]) --> 39
# tensor([58, 39]) --> 49
# tensor([58, 39, 49]) --> 43
# tensor([58, 39, 49, 43]) --> 1
# tensor([58, 39, 49, 43,  1]) --> 51
# tensor([58, 39, 49, 43,  1, 51]) --> 63
# tensor([58, 39, 49, 43,  1, 51, 63]) --> 1
# tensor([58, 39, 49, 43,  1, 51, 63,  1]) --> 54
# tensor([53]) --> 59
# tensor([53, 59]) --> 41
# tensor([53, 59, 41]) --> 46
# tensor([53, 59, 41, 46]) --> 5
# tensor([53, 59, 41, 46,  5]) --> 42
# tensor([53, 59, 41, 46,  5, 42]) --> 1
# tensor([53, 59, 41, 46,  5, 42,  1]) --> 61
# tensor([53, 59, 41, 46,  5, 42,  1, 61]) --> 47
# tensor([47]) --> 53
# tensor([47, 53]) --> 52
# tensor([47, 53, 52]) --> 1
# tensor([47, 53, 52,  1]) --> 39
# tensor([47, 53, 52,  1, 39]) --> 57
# tensor([47, 53, 52,  1, 39, 57]) --> 1
# tensor([47, 53, 52,  1, 39, 57,  1]) --> 63
# tensor([47, 53, 52,  1, 39, 57,  1, 63]) --> 53
# tensor([39]) --> 57
# tensor([39, 57]) --> 58
# tensor([39, 57, 58]) --> 1
# tensor([39, 57, 58,  1]) --> 51
# tensor([39, 57, 58,  1, 51]) --> 63
# tensor([39, 57, 58,  1, 51, 63]) --> 1
# tensor([39, 57, 58,  1, 51, 63,  1]) --> 50
# tensor([39, 57, 58,  1, 51, 63,  1, 50]) --> 53 
#
# so now that we have the data sorted out, lets create our model. 
# as we said, we are going to use a bigram model and later add attention mechanism to it. 
# to make things easier and more self contained, lets add all the required logic to this model
# like when we do a forward, we be able to calculate loss as well if we are given the targets
# so lets go
class BigramModel(nn.Module):
    def __init__(self, vocab_size) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        # our bigram model was nothing more than a 2d array of vocab_size, we can achive that using 
        # a single weight matrix or torch.Embedding. we use torch.Embedding to not reinvent the wheel!
        self.token_embedding = torch.nn.Embedding(vocab_size, vocab_size)
    
    # since we want to be able to calculate loss, if there are labels, we get Y as well
    def __call__(self, inputs:torch.Tensor, labels:torch.Tensor=None) -> torch.Tensor:
        logits = self.token_embedding(inputs)
        loss = None
        if labels is not None:
            # calculate loss 
            # print(f'{logits.shape=}') # prints (4,8,65)
            # note that both inputs and labels have the shape (B,T)
            # but logits has the shape (B,T,C) which C here equals vocab_size 
            # this is an issue for torch.crossentropy, as it expects the input
            # to be in the form of (B,C,T), that is, the channels/embeddings dimension
            # need to be right after the batch dimension or otherwise it wont work.
            # so we need to account for that.
            # we can go on permute the dimensions, like this and make crossentropy happy 
            # logits = logits.permute((0,2,1))
            # however, if for some reason the output of logits in the form of (4,8,65) is not ideal, 
            # maybe for example because really what it is 32 examples arranged in a (4,8) shape and 
            # we rather a normal 2d tensor of shape (32,65), 
            # we can instead, do just that, flatten the two dimensions into one and carry on!
            # we basically are concatenating the batch and time dimensions
            # into one dimension (effectively, stacking samples on top of each other), instead of having 
            # four compartments, each having 8 segments, we are going to have 1 long compartment with 32 segments/rows
            # for the lack of better words!!
            # so lets first get the shapes 
            # B,T,C = logits.shape
            # logits = logits.view(B*T,C)
            # and we also need to do the same for the labels 
            # labels = labels.view(B*T)
            # we could also do,
            # labels = labels.view(-1)
            # but thats not really needed, so we just simply permute logits temporarily so the logits shape
            # stays the same regardless of calculating the loss or not 
            loss = F.cross_entropy(logits.permute((0,2,1)), labels)
        return logits, loss
    
    # now lets also add a generate method to generate texts
    def generate(self, idx, max_token_count):
        # before we implement this lets review what we expect this method to do
        # we want this to generate some characters, given an initial character
        # we also want to be able to control the length of the generated text
        # so we take a max_token_count.
        # we take the initial index, 
        # feed it to the model, 
        # get the logits,
        # turn logits to probs,
        # use torch.multinomial to sample from our probablity distribution
        # get the new character index, and add it to a list to gradually 
        # -create our final text output
        # so lets go 
        # our idx should have the shape [B,T]
        assert len(idx.shape) == 2, f'idx.shape({idx.shape}) should have (b,t) form.'
        #
        # lets generate as many characters/idx as max_token_count specifies
        # this is basically specifies the time/sequence dimensions
        for i in range(max_token_count):
            # since we are defining a class method, to call the callable, we simply use self()
            logits, _ = self(idx)
            # to get the probablities 
            # usually we would simply do 
            # probs = torch.softmax(logits, dim=1)
            # but here, we want to only focus on the next character because this is what comes next
            # each time obviously, so we only take the last timestep to see what the model predicted
            logits = logits[:,-1,:] # this now becomes (B,C) instead of the initial (B,T,C)
            probs = torch.softmax(logits, dim=-1) 
            # now lets sample from it
            idx_next_char = torch.multinomial(probs, num_samples=1, replacement=True) # shape is (B,1)
            # now lets add this to the next input to be fed to the model 
            # since idx has the shape(batch, T), we should add this tothe second dimension
            # to the time dimension/ or sequence dimension. this as the loop goes on, 
            # creates the shape (B,T+1) 
            #this doesnt make sense for this particular model, becasue we are always checking
            # the next character given the previous one, so all the concatenation we are doing
            # is just useless. the reason we are implementing this like this, is to create a 
            # base, so that we can improve upon it when we add attention later on which will use
            # the history of previous characters.
            idx = torch.cat((idx, idx_next_char), dim=1) 
        
        # and finally when all is done return the idx which by now should have the whole output
        return idx
    
model = BigramModel(vocab_size)
out,loss = model(x,y)
print(f'{out.shape} {loss}')
# prints:
# torch.Size([32, 65]) 4.574285984039307
# the loss is good, we learned previously that we can evaluate a base loss provided our number of classes
# since we have 65 classes (our vocab_size or number of characters involved) the uniform probability for each class
# would be 1/65 =0.015384615, which if we take its negative log would turn out to be -ln(1/65) = 4.17438727, 
# which is pretty close to the loss we got here, signifying its a pretty decent value to begin with.
# also lets see the generate method at work
# lets create a dummy input, basically a batch of 1 and sequence of 1 of zero!
# we do this to generate a text from scratch
inputs = torch.zeros(size=(1,1),dtype=torch.int32)
output = model.generate(inputs, max_token_count=100)
print(f'{output=}')
print(''.join(decode(output.squeeze(0).tolist())))
# prints 
# wUbN:i :kLnOqHCQTG;.MbupOWH Xfi!MSUalaQppNNqIoWfmuSIIfmuZqf-3NLt-YkOc3vC-YkBfEHr3pJodwVY.nw.,r!&,Clt
# which is expected since our model is not trained yet! 
# lets train the model now and see how it works

batch_size = 32
context_size = 8
vocab_size = len(vocab_list)
max_iter = 20000

model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device=device)

print(f'{torch.__config__.show()}')
print(f'{device=}')
print(f'{model=}')

model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        print(f'{loss}')
# prints 
# 4.760574817657471
# 3.727213144302368
# 3.0139858722686768
# 2.6749324798583984
# 2.5884087085723877
# 2.5262675285339355
# 2.4420430660247803
# 2.4787800312042236
# 2.4680707454681396
# 2.564662456512451
# 2.51802659034729
# 2.562812328338623
# 2.422783613204956
# 2.5023772716522217
# 2.501049518585205
# 2.411632537841797
# 2.4028165340423584
# 2.4135568141937256
# 2.50952410697937
# 2.574878215789795
# relying on single batch loss is not a good idea to measure the performance of a model. moreover
# relying on training loss, is not good either, so it would be much better if we considered more batches
# for loss and even better we could also investivate the models performance on our validation set. 
# so lets do just this and define a function that calculates loss for training and validation sets alike
# but considers more batches for loss calculation

@torch.no_grad()
def evaluate_loss (iterations, device=None):
    results={}
    # before calculating the loss, lets switch to eval mode,although for our specific case this doesnt matter
    # but its goo practice, as later on, we will add layers that their behavior do change depending on traing
    # val mode.  
    model.eval()
    if device is None:
        # use the device assigned to what model params are assigned
        device = next(model.parameters()).device
    # since we already used no_grad decorator, we dont need to use no_grad context manager here
    # with torch.no_grad():
    for split in ['train','val']:
        losses = torch.zeros(size=(iterations,), device=device)
        for i in range(iterations):
            x,y = get_batch(split,batch_size)
            x,y = tuple(t.to(device) for t in (x,y))
            logits, loss = model(x,y)
            losses[i] += loss
        results[split] = losses.mean(0)
    # since we want to use this inside training loop, make sure we set the model back to train mode
    # incase we use layers such as batchnorm,etc that the require being trained!
    model.train()
    return results

loss= evaluate_loss(200)
print(f'{loss["train"]=} {loss["val"]=}')
# so now that our function seems to be working lets use it in the training loop :
model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
model = model.to(device=device)
model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        # now instead of simply printing loss, lets use our new function!
        loss= evaluate_loss(200)
        print(f'train_loss: {loss["train"].item():.4f},  val_loss: {loss["val"].item():.4f}')
# which results in :
# train_loss: 4.7491,  val_loss: 4.7430
# train_loss: 3.6673,  val_loss: 3.6670
# train_loss: 3.0460,  val_loss: 3.0511
# train_loss: 2.7335,  val_loss: 2.7492
# train_loss: 2.5948,  val_loss: 2.6156
# train_loss: 2.5292,  val_loss: 2.5470
# train_loss: 2.4982,  val_loss: 2.5194
# train_loss: 2.4807,  val_loss: 2.4991
# train_loss: 2.4737,  val_loss: 2.5036
# train_loss: 2.4650,  val_loss: 2.4949
# train_loss: 2.4648,  val_loss: 2.4910
# train_loss: 2.4626,  val_loss: 2.4836
# train_loss: 2.4570,  val_loss: 2.4893
# train_loss: 2.4517,  val_loss: 2.4823
# train_loss: 2.4508,  val_loss: 2.4819
# train_loss: 2.4564,  val_loss: 2.4839
# train_loss: 2.4555,  val_loss: 2.4802
# train_loss: 2.4503,  val_loss: 2.4927
# train_loss: 2.4522,  val_loss: 2.4905
# train_loss: 2.4572,  val_loss: 2.4888
#
# now lets check its output again after some training 
inputs = torch.zeros(size=(1,1), device=device, dtype=torch.int32)
output = model.generate(inputs, max_token_count=500)
print(''.join(decode(output.squeeze(0).tolist())))
# prints :
# Ane pod BUL:
#
# Oringe
# nfote s Rinsou oe the,
# An ETwe
#
# PUSENI bmilft!'d ate t Ifur
# Athomeg?
# Whagre,
# TCo belangead ne bl e, bednth ftor veso.
# US tla lin:
# Yourthiee he w whetitrd;
# Angoknghothartro ll heshethor hie douluk t, s se, denopit s h wom uspouct blyowie s'nos t prou gmesat
# Shd, tieithy.
# Asiour hofo bay avear hanousthaie
# Calonth wolulo.
#
# The hancondors pthighar:
# WAME outhaser d!
# S:
# h im t w s, inowo ORILOUCHoou isecetoubecompis
# Ay gnd, teve luthorea, therpeclsthevecthede fichim?
# PUSa VI aken 
# 
# which looks much better than the initial random output we got earlier.

dataset[:100]='First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
vocab_size=65, vocab_list=['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
atoi={'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't':

In [2]:
torch.manual_seed(255)
# now lets add attention mechanism to our base model. 
# attention mechanism at its core tries to take advantage of the rich information embedded
# in the sequence. for our work, this is specifically about the past history but in general 
# attention can utilize both past and future connections/sequence tokens. what we described 
# here just now is not exactly accurate, but gives us a foundation to build our intuition as
# we continue on. we will elaborate more of course and hopefully get it all.
# before we venture any further into the crux of the matter, let us learn about a technique
# thats used to efficiently implement attention mechanism.
# for this purpose,lets imagine we have a simple input like the following: 
# lets create and input of the following shape
B,T,C = (4,8,2)
# to make it more intuitive lets make a tensor with known numbers andthen reshape it
x = torch.arange(0,64,dtype=torch.float).view(B,T,C)
# imagine we have an input like what we encountered previously in our examples. in this sample 
# input, we have a batch of 4 samples, each having 8 sequences with each sequence having a vector of 2 values
# what we are planning to do is to provide a way by which each token can communicate with other 
# tokens. we have 8 tokens in our sequence. so we want our tokens to be able to communicate with
# all previous tokens that came before it. the reason we are only looking in the past token is 
# simply becasue the we are trying to perdict the future, so it only makes sense to look at the
# past and current timestamp and infer on what to do for the future.  
# so the easiest way to implement a kind of communication between tokens could be to sum or average the
# values of all previous tokens plus the current one as a way of taking into account their contribution
# to the final answer.
# that is, lets say if we are currently at token 5, we take the average of 
# the current token and all previous tokens before it, effectively making a feature vector that
# reflects our current status of the sequence so far, having taken all previous tokens/steps up to now.
# note that as you may also have thought, summing or averaging arent the best way to model such interations.
# in fact they are an extremely weak form of interaction between tokens,
# this kind of communicating is extremely lossy so to speak, that is we lose a great deal of information 
# concerning the underlying relationships between tokens, their arrangements,their implicit interactions,
# semantics, etc. but for now this is ok. we will later on see how to bring back such information.
# so now what we want to do, is to calculate the sum or average of all tokens up to the current token in 
# all batches at the same time.
# a naive way would be to do sth like this using a for loop:
results = torch.zeros(size=(B,T,C))
for b in range(B):
    for t in range(T):
        results[b,t] = x[b,:t+1].mean(0)
print(f'{results=}')
# which prints 
# results=tensor(
#        [[[ 0.,  1.],
#          [ 1.,  2.],
#          [ 2.,  3.],
#          [ 3.,  4.],
#          [ 4.,  5.],
#          [ 5.,  6.],
#          [ 6.,  7.],
#          [ 7.,  8.]],

#         [[16., 17.],
#          [17., 18.],
#          [18., 19.],
#          [19., 20.],
#          [20., 21.],
#          [21., 22.],
#          [22., 23.],
#          [23., 24.]],

#         [[32., 33.],
#          [33., 34.],
#          [34., 35.],
#          [35., 36.],
#          [36., 37.],
#          [37., 38.],
#          [38., 39.],
#          [39., 40.]],

#         [[48., 49.],
#          [49., 50.],
#          [50., 51.],
#          [51., 52.],
#          [52., 53.],
#          [53., 54.],
#          [54., 55.],
#          [55., 56.]]])
#
# You may find out that, some researchers refer to this operation here as BoW, or bag of words. 
# We are effectively averaging embeddings here and averaging embeddings can be considered a form of 
# Bag of Words (BoW) representation. In BoW, the focus is on the occurrence and frequency of words, 
# rather than their order or structure. By averaging embeddings, we are essentially treating each word 
# as an independent feature and capturing its representation in the form of a numerical vector.
#
# While averaging embeddings does not capture the exact frequency of each word, it does capture the 
# overall distribution and semantic information present in the text. Similar to BoW, this approach 
# disregards word order and focuses on the presence and representation of words. However, it should be
# noted that averaging embeddings may preserve some semantic relationships between words, which BoW 
# representations might not capture as effectively.
# 
# Side note: 
# Bag of Words (BoW) is a commonly used technique in natural language processing (NLP) for representing text
# as a numerical feature vector. It disregards the order and structure of words in a document and focuses only
# on their occurrence and frequency.
# In the BoW model, a document or a piece of text is represented as a "bag" (unordered set) of words, where 
# each word is treated as an independent feature. The presence or absence of words in the document is encoded
# as a binary value (0 or 1), and the frequency of each word is often used as the value in the feature vector.
# 
# Here's a step-by-step overview of the BoW process:
# 1. Tokenization: The text is split into individual words or tokens. Punctuation marks, whitespace, and other
#    special characters are usually removed or treated as separate tokens.
# 2. Vocabulary Creation: A vocabulary is created by taking all unique words from the entire corpus 
#    (collection of documents). Each unique word is assigned a unique index or position in the vocabulary.
# 3. Vectorization: Each document is represented as a feature vector, typically a one-hot encoding or a count
#    vector. In a one-hot encoding, each word in the vocabulary corresponds to a binary feature, and the vector
#    contains 1s in the positions where the word occurs and 0s elsewhere. In a count vector, the value at each 
#    position represents the frequency of the corresponding word in the document.
# 4. Classification or Analysis: The resulting feature vectors can be used as input to machine learning models
#    for tasks such as text classification, sentiment analysis, document clustering, or information retrieval.
# Needless to say, BoW has some limitations. It does not capture the semantic meaning or context of words, as
# it treats each word independently. It also ignores the grammar and word order. However, BoW is simple, 
# efficient, and can be a useful baseline representation for various NLP tasks.
# 
# so to recap one more time, we are basiaclly treating each timestep/sequence dimension, as a word, so we have
# 8 tokens/words, and we are averaging them (we are infact averaging their embeddings, but thats obvious!)
# so we got ourselves bow representation!
# now back to our discussion, if we  try to visualize the results we get 
print(x[0])
print(results[0])
# prints:
# x[0]=
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
#         [ 6.,  7.],
#         [ 8.,  9.],
#         [10., 11.],
#         [12., 13.],
#         [14., 15.]])
# results[0]=
# tensor([[0., 1.],
#         [1., 2.],
#         [2., 3.],
#         [3., 4.],
#         [4., 5.],
#         [5., 6.],
#         [6., 7.],
#         [7., 8.]])
# if you look closely, you'll notice that each row, contains the mean of all the rows before it
# consider the first 3 rows in x[0], 
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
# now refer to the 3rd row in results[0] which is :
#         [2., 3.],
# likewise, consider the last row in results which contains the average for all the rows in x:
# 0+2+4+6+8+10+12+14 = 56 which when divided by their count, 8, results in 7, 
# and this is the same for the second column, thus we get:
# results[0,7] = [7., 8.]])
# this all good but the problem is using for loops to calculate this is very inefficient, it
# happens that this operation can be efficiently calculated using matrix multiplication.
# lets learn this trick using an example: 
# suppose we have the following as the input: 
a = torch.ones(size=(3,3))
b = torch.randint(0,10,size=(3,2)).float()
c = a@b
print(f'{a=}')
print(f'{b=}')
print(f'{c=}\n----')
# prints
# a=tensor(
#        [[1., 1., 1.],
#         [1., 1., 1.],
#         [1., 1., 1.]])
# b=tensor(
#        [[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c=tensor(
#        [[20., 13.],
#         [20., 13.],
#         [20., 13.]])
# nothing fancy here, we have matrix multiplication, the first row of 'a' is dot-producted by first col
# of 'b', then sumed, it makes up the first col of first row in c. likewise the first row of 'a' dot 
# the second col of 'b', then summed the results, makes up the second col of first row in c. and this goes on for the rest of the matrixes. this is
# what we learned back in higheschool, so what is it exactly that we are learning exactly?
# if you look closely, you'll notice that, the c cols are actually the sum of all the rows in b!
#        [9]
# b[:,0]=[5] 
#        [6]
# 9+5+6 is 20! likewise, 
#        [3]
# b[:,1]=[5] 
#        [5]
# 3+5+5 is 13!
# hence c = [20., 13.] which is repeated obviously because the second and third rows of a are all 1s as well.
#           [20., 13.]  
#           [20., 13.] 
# I guess you are now starting to get where we are going with this, if we can some how alter the 'a' matrix,
# we may very well be able to achieve our goal! how you may ask? the answer is using torch.tril!
# torch.tril() is a function that returns a matrix from a given tensor, so that half of it set to zero,
# basially it creates a triangular tensor, where the right half is just zeros! lets see how it works, 
# lets apply it on 'a'
a_tril = torch.tril(a)
print(f'a_tril:\n{a_tril}')
# it prints
# a_tril=tensor(
#        [[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# as you can see the the right half is set to zero and we are left with a triangle shape of 1s! on the left side
# now if we do a@b this time we get:
c = a_tril@b 
print(f'b:\n{b}')
print(f'c:\n{c}')
# a_tril:
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[ 9.,  3.],
#         [14.,  8.],
#         [20., 13.]])
# now if you look closely, you'll notice that, this time, each row in c, is effectively the sum of the previous
# rows in b, like the first row of 'a' is only 1 in the 0ths column, so the first row of b is copied in c intact
# (workout the math and see why). 
# the second row in 'a', now has two 1s in col 0 and 1 respectively, which effectively translates to summing the first
# two rows in b. (9+5 =14, 3+5=8). likewise, the third row in 'a' is all 1s, signfigying all rows in b will
# be summed which gives us (9+5+5=20, 3+5+5=13). 
# so basically we are doing sums here, becasue our tensor a is all ones. so if we want to somehow calculate the
# average, instead of sum, we can easily change 'a' by normalizing it so that the each row sums to 1 (i.e. all cols
# sum to 1), this way the end result will be the average (becasue the 'b' is multiplied by a fraction/scale and then summed)
# so if we scale 'a' by the sum of all its columns, we should get average instead
a = torch.ones(size=(3,3))
a = torch.tril(a)
a = a/a.sum(dim=1, keepdim=True)
print(f'a:\n{a}')
# a:
# tensor([[1.0000, 0.0000, 0.0000],
#         [0.5000, 0.5000, 0.0000],
#         [0.3333, 0.3333, 0.3333]])
#
# note that, now each row, sums to 1. the first row, the first element is 1, because the rest are 0s
# but in the second row, as there are two 1s, the probabality is divided between the two, each being 0.5
# likewise, in the third row, as there are 3 1s, the probablity is divided between all of them, making each
# to have the value 0.33
# and now if we try to multiply them, we get average as the result:
c = a@b 
print(f'calculating average:')
print(f'b:\n{b}')
print(f'c:\n{c}')
# we get 
# calculating average:
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[9.0000, 3.0000],
#         [7.0000, 4.0000],
#         [6.6667, 4.3333]])
# we see that, each row in c, is the average of all the rows before it. 
#
# so using this trick, we can take the incremental average of any matrix we like. 
# now that we learned the trick, lets go back and implement the bows for loops using this techique !
# prevbiously we had : 
# results = torch.zeros(size=(B,T,C))
# for b in range(B):
#     for t in range(T):
#         results[b,t] = x[b,:t+1].mean(0)
# which calculated the average for sequence dimensions (tokens) incrimentally
# lets do this now 
# we want a TxT weight becasue we want to average T timestep/tokens
weight =  torch.tril(torch.ones(size=(T,T)))
# remember to set keepdim=True, or otherwise, as we sum along dim=1, we lose that dim
# (it collapses, and then the broadcast will be wrong, it will infact make each column 
# have one probablity instead of each row, which is the exact opposite of what we want)
weight = weight/weight.sum(dim=1, keepdim=True)
print(f'weight:\n{weight}')
# and now we need to multiply this by x! 
bow_results= weight@x
print(f'bow_results:\n{bow_results}') 
# which prints 
# bow_results:
# shape: torch.Size([4, 8, 2])
# tensor([[[ 0.0000,  1.0000],
#          [ 1.0000,  2.0000],
#          [ 2.0000,  3.0000],
#          [ 3.0000,  4.0000],
#          [ 4.0000,  5.0000],
#          [ 5.0000,  6.0000],
#          [ 6.0000,  7.0000],
#          [ 7.0000,  8.0000]],

#         [[16.0000, 17.0000],
#          [17.0000, 18.0000],
#          [18.0000, 19.0000],
#          [19.0000, 20.0000],
#          [20.0000, 21.0000],
#          [21.0000, 22.0000],
#          [22.0000, 23.0000],
#          [23.0000, 24.0000]],

#         [[32.0000, 33.0000],
#          [33.0000, 34.0000],
#          [34.0000, 35.0000],
#          [35.0000, 36.0000],
#          [36.0000, 37.0000],
#          [37.0000, 38.0000],
#          [38.0000, 39.0000],
#          [39.0000, 40.0000]],

#         [[48.0000, 49.0000],
#          [49.0000, 50.0000],
#          [50.0000, 51.0000],
#          [51.0000, 52.0000],
#          [52.0000, 53.0000],
#          [53.0000, 54.0000],
#          [54.0000, 55.0000],
#          [55.0000, 56.0000]]])
# which gives us the same results as we expected.
# one more thing before we continue on, note that the weight matrix is TxT while 
# the input is (BxTxC). (T,T) and (B,T,C) are not compatible, so what happens is
# that (T,T) is reshaped and a batch dimension is added to (T,T),making it (1,T,T)
# and then this is broadcasted along the batch dimension (replicated) to become 
# (B,T,T), then this will be multiplied by the (B,T,C)( note that at this stage
# a@b will be a batch multiplication operation, if you set the batch dimension aside, youll
# see that the rest of the dimensions match up, we have (T,T) and (T,C) which will
# result in (T,C). now if we add the batches back in, we will endup with (B,T,C)
# so in practice, the multiplication is done B times and then results are stacked.
# the first batches will multiply each other (T,T)x(T,C) = (T,C)
# the second batches will then multiply each other as well, getting another (T,C)
# and this goes on until we get (B,T,C))
#
a = torch.arange(0,9).view(3,3)
# a = torch.ones((3,3)).long()
b = torch.arange(0,30).view(2,3,5)
print(f'a:\n{a}')
print(f'b:\n{b}')
c = a@b 
print(f'c=a@b:\n{c}')
# is equivalent to 
# add a batch dimension to a
a = a.view(1,*a.shape) # or a.unsqueeze(0)
print(f'a with batch dim:\n{a.shape=}')
# replicate along the batch dimension
a = torch.cat(tuple(a.clone() for i in range(len(b))), dim=0)
print(f'{a.shape=}')
print(f'a(after replication along dim=0):\n{a}')
print(f'b:\n{b}')
# and now we have a case of batch-multiplication, the batch is the same 
# and sub tensors also are compatible, we have (T,T) and (T,C) so we now
# individually multiply each sub-tensor
c2 = torch.zeros_like(b)
for i in range(b.shape[0]):
    c2[i,...] = a[i]@b[i] # (T,T) x (T,C) -> (T,C)
# and ultimatley the result will have the shape (B,T,C)
print((c==c2).all())
# so to recap this 
# When performing the matrix multiplication `c = a @ b` with the given tensors, 
# several broadcasting steps occur to align the dimensions properly. 
# Here's a how it happens:
# 1. Tensor a has the (shape: 3, 3):
# 2. Tensor b has the (shape: 2, 3, 5):

# 3. They are not compatible so we broadcast tensor a to match the shape of b. 
# tensor a is expanded to (1, 3, 3) to have a batch dimension, and replicated 
# along dim 0 to result in (2, 3, 3). now both tensors have a batch of 2, and 
# if we put aside the batch dimension for a second, we'll notice that the rest
# of the shapes are compatible (3,3) and (3,5). so when we bring back the batch 
# dimension, we see we have a case of batch multiplication, that is we have 2
# sets of compatible tensors that need to be multiplied together. so we use a 
# simple for loop, to do multiplication, and stack the results and we are done!
#
# now back to our discussion. as we just saw, the traingualr shape in our weighted
# sum matrix, allows that each token at t dimension, can only interact with the tokens
# before it.
# there is another way of implementing the same thing but a bit differently
# if you look closely you can see that, we are dealing with probablities, so
# we may verywell use softmax to simplify this further.
# first lets create our triangular weight matrix
tril_tensor = torch.tril(torch.ones(size=(T,T)))
# now lets create a mask
weight = torch.zeros_like(tril_tensor)
# now lets mask all the zeros to -inf, so when we do softmax, all those -infs
# become 0. (recall that exp^-inf is 0! while exp^inf is inf! so its important to 
# set -inf (and also exp^0 is 1))
# so effectively what happens here is that, we setting each entery in trail_tensor
# with zero to -inf, and leave the rest as zeros. when this tensor goes through softmax
# the 0s will be 1s and -infs will be 0s before they are normalized, when they are normalized
# the probablity of 1 will be split between all the enteries with the value of 1.
# so the first row has a single 1, so it will be 1.0, the second row has two 1s, so
# each one will take 0.5, the third will have 3 1s, so they each will become 0.333
# and so on.
weight=weight.masked_fill(tril_tensor==0, -torch.inf)
# now calculate the probs for each row, treating all cols as probs so their sum is 1
weight = weight.softmax(dim=-1)
bow_results2 = weight@x
print(f'{torch.all(bow_results==bow_results2)}')
# so you might ask, why would we want to make things more complicated like this to only
# achieve what we already achieved pretyy efficiently before?
# the answer is, this approach provides us with a flexibility that the previous ones
# wouldnt provide us withs. if you think about it for a moment, you'll notice that
# the 'weight' matrix can essentially be anything and not just zeros! 
# we have decoupled it from tril, which's job is to set the right half of the tensor
# to zero, so we actually get the expected behavior later on.(which is setting a constrain really (more layer on))
# if you havent yet figured it out, the 'wieght' matrix, can be truly a weight matrix
# which can show different strengths for each token, basically it can learn the interations
# between tokens and manifest them. currently it is us who sets it to all zeros, so we get
# uniform probablities, which inturn manifest itself as an average. but what if instead of 
# all zeros, we actually learn the values from the data itself? that would make sense, and 
# make it so that each token, can have a different connection(strength) to any other tokens
# now, and thus build semantic/meaningful relation. this is inafct what we are after. we want
# for tokens to learn associations and relations with other tokens based on the data present
# in the dataset, having uniform weight like what we initially did really is a far cry from
# what we intend, and therefore, we opt in to use this new approach that allows us to actually
# exploit this new capability. 
# This is infact the problem that attention solves, that is gathering
# information from the past but in a data driven manner.(side note, we are not limited to 'past' 
# information only per say, in this example, this is the case however, we will explain this 
# in more detail)), we will see how attention does this exactly in a moment. 
#
# but before we jump into attention implementation also note that the tril part, infact is a hard-constrain here that prevents tokens from
# the past from interacting with the tokens from the future. 
# so to recap here, basically the idea is, this triangular form, allows us to have 
# weighted aggregations of past elements. each element in the lower triangular part, 
# specifies, the degree by which it plays a rule in the said outcome. 
# that is how much of each element gets to fuse into this specific position(i.e. current token's)
# so now lets incorporate attention into our model
# Attention does its job by using two vectors called, key and query.
# basically every single token, emits two vectors called key and query, the query verctor
# as the name suggests, implies, what we are looking for, and the key vetcor, again as the
# name suggets, implies, the contents, what it contains. 
# the way we get our 'weights' for these tokens is we simply dotproduct them together, 
# the ones that yield a high output/value, signify they are related positively.
# so our query is dotproducted with all the token's(keys) and the outputs reveal their 
# closeness/relevancy/similarity so to speak(if they align so to speak, they result in larger number)
# so effectively, the ones with higher number, are more similar to the query, the highest, 
# obviously having the most relavancy/similarity to the query.s
# so lets implemenet this
# #%%
# we want to implement a single attention head, we can later use this to create 
# multi-attention-head which is basically several single attention heads working in 
# parallel. so how do we implement one! 
# first lets create some random input 
torch.manual_seed(255)
# lets specify the batch, context_size and vocab_size
B,T,C = 4,8,32
# lets create an input 
x = torch.randn(size=(B,T,C))
# we said that attention works with two vectors, key and query, lets implement them
# we can use nn.Linear to implement them but beore that, what are the dims of such vectors,
# we are dealing with text and thus our inputs are tokens/vocabs, so the first dim would be 
# our vocab_size to account for all tokens, the next dim, is sth called a head_sizes, which 
# specifies the size of the key/query output usually 16 is used a lot for head size so we 
# use that as well
head_size = 16
# lets not forget to set bias=False, so what it does is exactly dotproduct 
# (actually, some people still leave the bias enabled! 
# so we will test both cases later and see if it actually matters! and if so to what extend! ) 
key = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
# now lets get the output
k = key(x)      # shape : 4,8,16 or (B,T, head_size)
q = query(x)    # shape : 4,8,16 or (B,T, head_size)
 # so the dims arent compatible for batch-multiplication, so we need a transpose
 # k.T wouldnt work becasue we have a batch-dim, so instead we use .transpose and
 # explictily specify the dims we want to be transposed. 
 # this will result in 4,8,16 by 4,16,8 which would give us 4,8,8 which is (B,T,T)
 # really as the result
 #! check transpose result is it okto use 2,1 or -2,-1 or 1,2
weight_raw = q@k.transpose(2,1)
# now lets for a moment think about what is happening here, the key and query are applied
# on the input and each return an output of (B,T,head_size), they are in fact, processing
# all the tokens in the input, individually, simultaneously, all the same time. so each 
# token is both a query, and a key, and when we do a dotproduct, we are basically telling 
# it to reveal the relation/similarity/relevance of every token with every other tokens.
# and as we explained earlier, this is infact our weight matrix (which was initially zeros)
# but is now learned from the data!
# print(f'weight_raw\n{weight_raw}')
# now we can apply constrain on it so that tokens can only communicate with the past so 
# we use the tril trick now!
tril_constrain = torch.tril(torch.ones(size=(T,T)))
raw_wieghts_masked = weight_raw.masked_fill(tril_constrain==0, float('-inf'))
# apply sotmax to get probablity for each token
weight = raw_wieghts_masked.softmax(dim=2) # remember weight is (B,T,T)
# and finally we can apply our weight on the input (we called it raw for a reason, read on)
# (by the way this is also called self-attention!)
bow_raw = weight@x 
# lets print raw_weights and weights and have some intuitive observations 
print(f'weight_raw\n{weight_raw}')
print(f'raw_weights_masked: {raw_wieghts_masked}')
print(f'weight\n{weight}')
# prints
# raw_weights(unconstrained)
#weight_raw
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00, -1.7903e+00,  1.4650e-01, -1.4147e+00, -1.7452e+00, -4.5249e+00, -1.9763e+00,  1.8833e+00],
#          [-1.3490e+00,  9.4076e-01, -6.9690e-01,  7.9933e-01,  4.7634e-01,  4.1861e-01,  2.3663e-01,  4.5217e-01],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,  9.0092e-01,  1.4083e+00,  2.5488e+00,  1.6350e+00,  1.0048e+00],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01, -1.1213e+00,  2.7562e+00,  7.0136e-02, -2.0337e+00],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,  4.9236e+00,  3.9545e+00, -2.2600e-01],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01, -1.5979e+00,  8.4627e-01],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,  2.2338e+00],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00, -2.1323e+00, -1.3948e+00, -5.6973e-01, -2.7986e-01,  4.3579e+00,  4.8477e-01, -9.7559e-01],
#          [ 8.7786e-01, -2.3352e+00, -2.2988e+00,  1.0322e+00, -1.4231e-01,  5.5233e+00,  1.0819e+00, -2.4722e-01],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,  2.9387e-02, -5.6397e-01, -2.4025e-01, -4.2512e-02, -8.4302e-01],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01, -4.0429e+00, -1.0676e+00, -3.0694e+00, -1.6156e+00],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00, -4.5593e+00, -3.1755e-01, -1.0504e+00],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00, -3.5616e-01, -2.6951e+00],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,  4.9077e-02],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01, -4.7902e-01,  6.7590e-01, -2.3153e-02,  1.9763e-01,  4.4385e-01,  7.2986e-01, -3.8846e-01],
#          [-6.2810e-01, -8.3713e-01,  1.3468e-01, -5.1328e-01, -2.6653e-02,  3.4325e-01, -1.0015e+00, -1.0334e+00],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01, -1.9702e+00, -1.0641e+00,  1.7792e-01,  8.0194e-01, -1.5265e-01],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,  3.6759e-01,  1.0552e+00, -5.2949e-01, -1.7715e+00],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01, -3.0057e+00,  1.3784e+00, -1.2091e+00],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02, -1.4040e+00,  2.3786e+00],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00, -1.6068e+00],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<UnsafeViewBackward0>)
#
# raw_masked_wieghts(constrained):
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.3490e+00,  9.4076e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01,        -inf,        -inf,        -inf,        -inf],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,        -inf,        -inf,        -inf],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01,        -inf,        -inf],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,        -inf],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 8.7786e-01, -2.3352e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01,        -inf,        -inf,        -inf,        -inf],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00,        -inf,        -inf,        -inf],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00,        -inf,        -inf],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,        -inf],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-6.2810e-01, -8.3713e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,        -inf,        -inf,        -inf,        -inf],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01,        -inf,        -inf,        -inf],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02,        -inf,        -inf],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00,        -inf],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<MaskedFillBackward0>)
# 
# weight (final- normalized)
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.1976e-02, 9.0802e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9773e-01, 6.0261e-01, 1.9966e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.4181e-02, 3.4422e-02, 3.7221e-01, 5.7918e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [4.6023e-02, 1.9501e-03, 5.3236e-01, 3.5612e-01, 6.3550e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9494e-01, 1.9195e-01, 2.7619e-01, 2.8253e-02, 1.3648e-01, 1.7219e-01, 0.0000e+00, 0.0000e+00],
#          [4.5214e-01, 9.6007e-02, 7.3108e-02, 5.3035e-02, 2.8887e-01, 8.7621e-04, 3.5963e-02, 0.0000e+00],
#          [6.8046e-02, 5.1605e-01, 5.0474e-03, 1.5304e-02, 2.9133e-01, 3.8823e-04, 1.3775e-02, 9.0057e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.6132e-01, 3.8675e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.1870e-01, 3.3895e-01, 4.4235e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.2894e-02, 6.5398e-01, 2.4345e-01, 7.9674e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [3.7332e-02, 6.5690e-01, 7.8427e-02, 5.1206e-03, 2.2222e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.3611e-01, 1.7995e-02, 2.1887e-02, 1.1900e-01, 1.8219e-02, 6.8680e-01, 0.0000e+00, 0.0000e+00],
#          [9.8946e-02, 1.4120e-02, 1.8065e-02, 9.5010e-03, 3.2130e-01, 9.9891e-02, 4.3817e-01, 0.0000e+00],
#          [2.3306e-02, 3.5621e-02, 9.5163e-02, 1.5801e-01, 1.1530e-02, 1.7183e-01, 4.1141e-02, 4.6340e-01]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.5207e-01, 4.4793e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.8708e-02, 6.5816e-01, 2.4313e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.8422e-01, 9.1987e-02, 1.2557e-01, 5.9822e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.0870e-01, 8.3642e-02, 4.8896e-02, 3.2723e-01, 3.3154e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.4141e-02, 7.9271e-02, 2.1893e-01, 2.2608e-01, 2.3782e-01, 1.4376e-01, 0.0000e+00, 0.0000e+00],
#          [7.8726e-02, 1.1815e-01, 3.7190e-01, 1.2607e-02, 3.6387e-02, 1.1790e-01, 2.6434e-01, 0.0000e+00],
#          [5.6646e-02, 7.4575e-02, 6.8217e-02, 3.1568e-02, 5.9024e-02, 1.1073e-01, 3.1662e-02, 5.6757e-01]]], grad_fn=<SoftmaxBackward0>)
#
# as you can see, our weight is initialized for each batch based on the input data, and each token has its own
# weight, that is they are not uniform!, to get a better understanding lets consider the first batch of 
# raw-weights[0], raw_masked_wieghts[0] and weight[0]:
#
# raw_weights[0]
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
#
#
# raw_masked_wieghts[0]
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
# 
# weight[0]
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
# take the last token, which 8.6020e-02, notice this token, not only knows its content and own position in 
# the sequence(its the 8th token after all) but also knows which tokens comes before it and how much its 
# related to any of them.(basically when it knows which token comes before it, it creates its own query so
# to speak, and talks to every single previous token to find about which ones are more or less relavent to
# it and to what extend.) 
# For example,lets say, (since we are dealing with character level text generation!), the last token is a 
# vowel and says hey im a vowel and im looking for everyone else thats a vowel! and uses query to talk to 
# other tokens(keys). and intrestingly another token (lets say number 4) says im a vowel as well, and the 
# response is thus generate a larger number.(this is not a good example, words in sentence would make more sense
# !give a better example!)
# in our example, For the last token, it seems the 5th and 3rd token are particularly intresting/relavent/important,
# followed by the 7th token.
# likewise, this happens for every token, i.e. token number 4, says the same thing and searches in its past toekns
# so on and so forth! 
# so what happens next is that when we get a high relevancy scale /response in the weights, like we just described
# when we do a softmax it will assign a large probablity to them, and this instructs the network that, we need more
# information from them, effectively allowing for aggregating a lot of their information into our position(lets say e.g. 8th token. 
# and we happen to learn more about them this way.
# now in practice, we are not intrested in aggregating the x raw values per say, rather we want their information, 
# so instead of just using the raw values of x, we instead use a representation of them, so to speak. 
# this is achieved using a third vector known as, 'value' and is the last vector we use. 
# !explain when we say vector, note that, we are talking from the prespective of a token, (every token has some vector 
# !of information and it gets to aggregate information via a weighted sum from all the tokens that point to it, and it
# )# !in practice we use a matrix, and hence the linear module to implement this to run the operation for all tokens in parallell. 
# just like the key and query, we set its bias to False, so we only get a simple vector,
# and a dotproduct output (again this is the intuition, but how much adding a bias would
# affect this we will see later, for now we stick to the default no bias version)
value = torch.nn.Linear(C,head_size, bias=False) # produces (B,T,head_size) just like the other two key,query vectors
x_processed = value(x)
# this is not yet final final, we still need to do one more thing (read on!)
bow_final = weight@x_processed
#
#!check also note that, as we previously once pointed out, attention is a communication mechanism between tokens, and 
# by default there is nothing in this mechanism that provides a notion of space/position for tokens involved, i.e.
# by default these tokens/nodes/points, dont have any idea about where they are or how they are positioned (with resepect
# to others) etc, so we need to encode this information as well.(to better visualizing it, consider each token as 
# a node in a directed graph, each node has a connection to another node, (for example each node has a connection to
# itself, and another connection to other nodes,etc) and as you can see there is no notion of space here,the attention
# simply acts on a set of vectors in this graph, and thats why we need to encode them positionally as well, so they 
# have information about their position with regards to other tokens. 
# !check (compare this to the convolution case and images
# or even text where the spatial aspect of data is preserved, but in attention, as we just stated, there is no notion
# of position, its just a set of individual vectors being operated on)-note that the connection between nodes, do give
# us a sense of structure, but it may not be enough to infer the underlying semantic, imagine a case, where a word
# for example has several meaning, and may very well have high relevancy to some tokens at the same time, but without
# additional positional information, an ambiguous semantic can be infered between the tokens involved, however when
# positional information is also present, such ambiguity can be avoided)
# also note that, in attention, samples do not interact with each other at all. when we have a batch of 4, each sample
# is processed in isolation, but in parallell to other samples, based on our graph example earlier, we would have 4 
# graphs of 8 nodes for example for each input sample. 
#
# we said earlier that what we implemented here is known as self-attention, the reason it is called self attention
# is that the key and query and values are applied on the same input(the use the same source!), and hence the name,
# self attention.
# also note that, in our specific case, tokens/nodes are can not communicate with the future nodes, but in general
# this constraint can be removed (and infact is removed/not implemented for some applications) where its benificial
# to be able to communicate with all the tokens. one example is sentiment analysis, where you want all the tokens to
# able to communicate with eachother so you can get an accurate analysis. and for this case, we would use an encoder
# block, which is basically what we have here, minus the constraint section(tril/mask part), what we have implemented
# here is called a decoder block, where we are decoding bunch of tokens, and it makes sense that the previous tokens
# do not comunicate with the future ones(becasue they would give the answer! and it defeats the whole purpose here!),
# becasue its a given that only the previous tokens must be used to predict the future/next token, hence the filtering/constraint part to prevent tokens from 
# comunicating with the future nodes/tokens.
# !so far we explained about the self-attention, which we saw, is called that way solely for the fact that key, query
# and value use the same source. the attention mechanism as we briefly pointed out, is much more general and can be
# used in different ways. one of such ways, is what is used to create sth called cross-attention. 
# cross-attention basically refers to the case where we have an encoder/decoder blocks, in which the queries come from x
# but the key and value come from an external source and sometimes from the encoder block. so cross-attention is used
# when theres a separate source of information we would like to pool from and use it as well.
#
# so far we implemented the attention based on the original paper(there are some differences we get to later on)
# except the part where we need to divide by the sqrt of the head_size. that is called scaled-attention
# so lets talk about this, and see why its needed. 
# the reason we add this so called 'scale' to our computation, is that, without it, the probablities will be saturated
# and when we add this term to the mix, it will make the 'weight' matrix to be 'unit variance', when Q and K are unit variance
# and this allows softamx to stay diffuse and not saturate too much.
# in other words, if we simply multiply key and query like that, the variance of the resulting weight matrix will be
# around the head_size instead of 1 which is bad and makes optimization really hard.
# to see this effect consider the following example
k = torch.randn(size=(B,T,head_size))
q = torch.randn(size=(B,T,head_size))
w_unscaled = q@k.transpose(-2, -1) 
print(f'{k.var()=}')
print(f'{q.var()=}')
print(f'{w_unscaled.var()=}')
# now add the 1/sqrt(head_size)
w_scaled = q@k.transpose(-2, -1) * head_size**-0.5 
print(f'after applying 1/sqrt(head_size)')
# makes the weight variance 1!
print(f'{w_scaled.var()=}')
# why is it important? if you recall, the weight matrix is fed into softmax, so its really important, especially during
# initialization that weight matrix be fairly diffuse, if we look at weight matrix here, we'll notice that they are now
# fairly diffuse 
print(f'weight_scaled[0]:\n{w_scaled[0]}')
# prints 
# weight_scaled[0]:
# tensor([[ 0.3972, -2.0957, -0.5396, -1.4860, -0.8393,  0.6273, -0.0157,  0.6284],
#         [ 1.3588, -0.0440,  2.1040, -0.2796,  1.7797,  1.4362,  1.3843,  0.0368],
#         [ 0.8109,  1.7400,  1.3579,  0.2097,  1.7152, -1.1242, -0.4349, -0.5690],
#         [ 0.0513,  2.8303,  0.7332,  0.0041,  0.9688, -1.6174, -1.4255,  0.2869],
#         [-0.7819, -0.6691, -1.4017, -0.4155, -0.8568, -0.2704, -0.9453,  0.3763],
#         [ 0.4092, -0.3079,  0.8472, -1.1753, -0.7699,  0.5091,  0.6385,  0.9677],
#         [ 0.3043, -2.1402, -2.2582, -0.3573, -1.2838, -0.0260, -0.0513,  0.9151],
#         [ 0.4180, -2.8353, -0.6435,  0.4842, -3.0202,  1.7690,  1.8837, -0.4393]])
#
# when softmax is applied:
print(w_scaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.8026, 0.1974, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1901, 0.4814, 0.3285, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.0499, 0.8038, 0.0987, 0.0476, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1989, 0.2226, 0.1070, 0.2869, 0.1845, 0.0000, 0.0000, 0.0000],
#         [0.2148, 0.1049, 0.3329, 0.0440, 0.0661, 0.2374, 0.0000, 0.0000],
#         [0.3027, 0.0263, 0.0233, 0.1562, 0.0618, 0.2176, 0.2121, 0.0000],
#         [0.0901, 0.0035, 0.0312, 0.0962, 0.0029, 0.3478, 0.3901, 0.0382]])
#
# 
# now compare it with the unscaled weight : 
print(f'weight_unscaled[0]:\n{w_unscaled[0]}')
# weight_unscaled[0]:
# tensor([[  1.5889,  -8.3829,  -2.1583,  -5.9440,  -3.3573,   2.5092,  -0.0629,  2.5135],
#         [  5.4350,  -0.1759,   8.4159,  -1.1183,   7.1189,   5.7446,   5.5373,  0.1472],
#         [  3.2435,   6.9600,   5.4317,   0.8388,   6.8607,  -4.4969,  -1.7396, -2.2761],
#         [  0.2051,  11.3214,   2.9327,   0.0165,   3.8754,  -6.4696,  -5.7019,  1.1475],
#         [ -3.1278,  -2.6763,  -5.6069,  -1.6621,  -3.4272,  -1.0816,  -3.7811,  1.5053],
#         [  1.6368,  -1.2317,   3.3888,  -4.7012,  -3.0795,   2.0364,   2.5541,  3.8710],
#         [  1.2171,  -8.5610,  -9.0327,  -1.4293,  -5.1353,  -0.1038,  -0.2053,  3.6603],
#         [  1.6719, -11.3411,  -2.5740,   1.9369, -12.0810,   7.0760,   7.5347, -1.7573]])
#
# when softmax is applied:
print(w_unscaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [9.9636e-01, 3.6442e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.9592e-02, 8.0566e-01, 1.7475e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.4865e-05, 9.9975e-01, 2.2738e-04, 1.2309e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2943e-01, 2.0328e-01, 1.0848e-02, 5.6050e-01, 9.5940e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2012e-01, 6.8207e-03, 6.9264e-01, 2.1236e-04, 1.0748e-03, 1.7913e-01, 0.0000e+00, 0.0000e+00],
#         [6.3261e-01, 3.5856e-05, 2.2371e-05, 4.4856e-02, 1.1023e-03, 1.6883e-01, 1.5254e-01, 0.0000e+00],
#         [1.7351e-03, 3.8710e-09, 2.4850e-05, 2.2616e-03, 1.8472e-09, 3.8572e-01, 6.1020e-01, 5.6236e-05]])
#
# as you can see, some values are very negative, while others are very positive, for example, we have both 11 and -11
# and this is a recipe for disaster! the reason it is a problem is that, with softamx, when numbers take very positive
# and nagative values, softmax actually converges towards one-hot-vector. basically the values shrink towards the max.
# this can be seen the above, but the example below should demonstrate it more clearly.
# lets apply softmax on a list of numbers that are close to each other, we see the probablities are difuse and normal
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2]),dim=-1)}')
# we get a diffused probablity out of softmax
# tensor([0.2562, 0.3822, 0.1717, 0.1898])
# however if we increase the magnitude of the numbers(sharpen them) (and thus the difference between them) by like multiplying by 
# a number like 8 (just to simulate the effect here and so we can compare it with the previous case)
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2])*8,dim=-1)}')
# we see that the softmax, starts to sharpen towards the max, shrinking others except the 
# largest number, and effectively
# converging toward a one-hot-encoded vector.
# tensor([0.0390, 0.9559, 0.0016, 0.0035])
# so we dont want these values to be extreme, especially during initialization or otherwise, softmax will be way too picky!
# and we are basically aggregating the information from a single node instead of multiple ones (becasue one has the largest
# nvalue, its as if our sequence length is 1! and we lose access to the wealth of information the past history offers)
# so we want the probablities to be diffuse and not peaked like the second example here.
# so this scaling is used to retain the variance at a good value especially at initialization.
# so now that we are finally finished the self attention head, lets implement it as a module and incorporate everything
# we just discussed here.
class AttentionHead(nn.Module):
    def __init__(self, context_size, embd_size, head_size=16, use_bias=False) -> None:
        super().__init__()
        # we need context_size or block_size for creating the tril constrain
        self.context_size = context_size
        # we want this as the input dim for our key,query and value, this is 
        # infact the input_dim, since we plan on using the embeddings, we named
        # it embd_size
        self.embd_size = embd_size
        # head_size is the output dimension of our attnetion
        # !note that usually the head_size is equal to embd_size
        self.head_size = head_size
        self.key = nn.Linear(embd_size, head_size, bias=use_bias)
        self.query = nn.Linear(embd_size, head_size, bias=use_bias)
        self.value = nn.Linear(embd_size, head_size, bias=use_bias)

        # use buffer for trail, since this is a buffer, we can use the self.register_buffer which is
        # inherited from nn.Module class, to add it as buffer to the module, so it can be saved in the
        # state_dict when we save the model.  tril doesnt change, and is not updated(its required_grad is false),
        # so it being saved in state_dict doesnt matter to us, but to demonstrate this feature of pytorch, 
        # we are using it, and its a good practice, since pytorch knows how to deal with it and wont include it
        # in the computaion graph anyway!
        # 
        # This is typically used to register a buffer that should not to be considered a model parameter. 
        # For example, BatchNorm's running_mean is not a parameter, but is part of the module's state. 
        # Buffers, by default, are persistent and will be saved alongside parameters. This
        # behavior can be changed by setting persistent to False. The only difference between a persistent
        # buffer and a non-persistent buffer is that the latter will not be a part of this module's state_dict.
        # tril allows us to impose constrain by creating a lower traingualr matrix and setting
        # the rest entries to zero, effectively preventing tokens of the future from communicating with the past
        # each token can only communicate with the previous tokens that came before it.
        self.register_buffer('tril', torch.tril(torch.ones(context_size,context_size)))

    def __call__(self, inputs:torch.Tensor) -> torch.Tensor:
        #! check the shapes!! 
        B,T,C = inputs.shape # 4,8,16
        # print(f'{inputs.shape=}')
        # create the weight by using k,q,v
        k = self.key(inputs)    #! (B,T,C) or (B,E,16)? (4,8,16)
        q = self.query(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        v = self.value(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        # create the weight matrix and scale it by 1/sqrt(head_size) to keep weight unit variance 
        weight = q@k.transpose(-2,-1)* self.head_size**-0.5 # !(B,E,E)
        # weight_C = q@k.transpose(-2,-1)* C**-0.5 # !(B,E,E)
        # print(f'weight.var: {weight.var().item():.4f}')
        # print(f'weight_C.var: {weight_C.var().item():.4f}')
        # print(f'x:{tuple(inputs.shape)} k:{tuple(k.shape)} q:{tuple(q.shape)} v:{tuple(v.shape)} w:{tuple(weight.shape)} hs:{self.head_size}')
        # apply the tril constrain - (this makes this a decoder block!)
        # important note: notice we used tril[:T,:T] and not simply tril
        # this is because, when the input has a small context_size < self.context_size
        # like when we wantto generate inputs with sth like zeros((1,1)) which says
        # there is a single token (context_size 1) of 0 as the begining of the sequence
        # when this is input, the tril by default makes a context_size,context_size matrix
        # which will be different than the input context_size which is 1 e.g. or 2 e.g.
        # and it will fail becasue our weight would be 1,1,1 or 1,2,2, but trail is 1,8,8
        # and clearly this will cause an error. for this reason, we always create the 
        # tril check dynamcally by explicitly specifying the context_size based on the 
        # current input context_size so in case the context_size is smaller, tril is resized
        # dynamically accordingly. 
        # print(f'tril[:T,:T]==0: {(self.tril[:T,:T]==0).shape}')
        # print(f'tril==0: {(self.tril==0).shape}')
        weight = weight.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        # weight2 = weight.masked_fill(self.tril==0,float('-inf'))
        # print(f'{weight.shape=}')
        # print(f'{weight2.shape=}')
        # note that we are using batch, so instead of hardcodin 2,
        # we use -1 to refer to the last dim
        weight = weight.softmax(dim=-1)
        # finally apply the weight on the v
        bow = weight@v  # !(B,E,16)
        return bow        

# now lets add this to our model 
        
# so to recap, we first calculated the relavancy between all tokens against eachother, then constrained them so that 
# each token can only use the information from/interact with its past tokens. then since we needed probablity distribution so
# we then normalized it and then used that to pickout which tokens information(in the past) to aggregate/use with the inputs to 
# achieve our goal.(which is to predict the next character based on everything seen so far!)
# we dont want to aggregate inputs value (with respect to the weight matrix), we instead would like to use their representation
# so we use a new layer to do this, its called value, and we instead use its output instead of inputs raw value.
#
# 
#
# before we jump in and add the attention module, lets review our base model and see how we can 
# improve/prepare it before we incorporate the attention. 
# class BigramModelWithAttention(nn.Module):
#     def __init__(self, vocab_size, embd_size) -> None:
#         super().__init__()
#         self.vocab_size = vocab_size
#         self.embd_size = embd_size
#         # unlike the previous model, lets decouple the final logits from 
#         # the number of embeddings, because we are using attentions, and
#         # we want to have multiple operations inbetween obviously.
#         self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
#         # in order to get the final logits, we need a linea layer at end
#         self.fc = torch.nn.Linear(embd_size, vocab_size)
#        
#     def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
#         out = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
#         logits = self.fc(out)         # has the shape (B,T,C) c is vocabsize
#         loss = None
#         if labels is not None:
#             # recall that crossentropy likes its input to be B,C,T and we are B,T,C
#             # so lets permute and make it happy!
#             loss = F.cross_entropy(logits.permute(0,2,1), labels)
#         return logits, loss
#    
#     def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
#         # lets generate an output as long as num_max_token
#         for i in range(max_token_count):
#             # make sure idx is 2d
#             assert len(idxs) >1, f"idx.shape '({tuple(idxs.shape)})' is invalid. it must have the form (B,T)"
#             # now lets feed it to the model and sample from the probablities it produces
#             preds,_ = self(idxs)
#             # convert to probs 
#             probs = preds.softmax(dim=1)
#             # since we are bigram still, lets only get the last token as the next token predicted!
#             probs = probs[:,-1,:]
#             # now lets sample from it 
#             new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
#             # now concatenate the new token to the previous one and feed it back to the model
#             # for the next round of prediction
#             # also remember that we are creating a sequence, so we concat them at dim=1 to get 
#             # a longer sequence (we are gradually increasing the sequence length from 1 up to
#             # max_token_count)
#             idxs = torch.cat((idxs,new_idx), dim=1)
#            
#         return idxs

# this works fine, however, we can do better. here we just encoded the tokens, but
# as we already explained, attention has no notion of position or spatial structure like conv e.g.
# so we need to also incorporate the position information for each token as well
# since we are using the the attention head, we need more arguments
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, head_size, device, use_bias_att=False) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embd_size = embd_size
        self.context_size = context_size
        # for gpu/cpu acceleration during training
        self.device = device
        # unlike the previous model, lets decouple the final logits from 
        # the number of embeddings, because we are using attentions, and
        # we want to have multiple operations inbetween obviously.
        self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
        # lets now add the positional embedding as well 
        # using this, we try to retain the position embedding for our tokens
        # up to the current token in context_size
        self.position_embd = torch.nn.Embedding(context_size, embd_size)
        # add a self-attention head
        self.head = AttentionHead(context_size, embd_size, head_size, use_bias=use_bias_att)
        
        # in order to get the final logits, we need a linea layer at end
        # since we now have attention before this layer, the output dim of attention which is
        # head_size will be used here
        self.fc = torch.nn.Linear(head_size, vocab_size)
        
    def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
        # lets grab the shapes, since we will be using them 
        B,T = inputs.shape
        token_embeddings = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
        # since we have a position_embedding lets use that as well
        # and notice that we didnt use the inputs, but rather torch.arange(T)
        # this means, for each input, as we process it, we also get embeddings up to
        # the current token count as well, if the inputs has 3 tokens currently, we
        # will creeate position embeddings for 0,1 and 2, and for the next input
        # this continues likewise. this results in (T,E)
        position_embeddings = self.position_embd(torch.arange(T,device=self.device))
        # print(f'pos_embd:{position_embeddings.shape}')
        # and lets add the two embeddings together
        # this effectively gives us, not only the token embeddings(identity)
        # but also its position in the sequence. note that this doesnt really
        # help in a bigram model, but when it comes to attention it really does!
        embeddings = token_embeddings + position_embeddings
        # now lets feed this to attention head
        out_attention = self.head(embeddings)
        # lets feed this embedding to our fc at the end instead
        logits = self.fc(out_attention)         # has the shape (B,T,C) c is vocabsize
        loss = None
        if labels is not None:
            # recall that crossentropy likes its input to be B,C,T and we are B,T,C
            # so lets permute and make it happy!
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss
    
    def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
        # lets generate an output as long as num_max_token
        for i in range(max_token_count):
            # print(f'{i}/{max_token_count}) idxs: {tuple(idxs.shape)}')
            # make sure idx is 2d
            assert idxs.ndim >1, f"idx.shape '({tuple(idxs.shape)})' is invalid({idxs.ndim}). it must have the form (B,T)"
            # now lets feed it to the model and sample from the probablities it produces
            # but since, we now have postional embeddings as well, we can no longer have 
            # more than block_size/contex_size in, becasue if our idx is more than context_size
            # our positional embedding will go out of scope and error out
            # so here we are basically getting as many as context_size
            # note that we dont destroy the idxs! each time we get the last context_size tokens
            # from it and feed it to the model to generate the next token, and keep going
            # if we replace the idxs by sth like idx=idx[:,-self.context_size:], we would
            # only create a sequence of only self.context_size, no matter how many iterations
            # we do, we just repeat the same sequence again and again!
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are bigram still, lets only get the last token as the next token predicted!
            logits = logits[:,-1,:]
            # convert to probs 
            probs = logits.softmax(dim=-1)
            # now lets sample from it 
            new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
            # now concatenate the new token to the previous one and feed it back to the model
            # for the next round of prediction
            # also remember that we are creating a sequence, so we concat them at dim=1 to get 
            # a longer sequence (we are gradually increasing the sequence length from 1 up to
            # max_token_count)
            idxs = torch.cat((idxs,new_idx), dim=-1)
            
            
        return idxs
    
# now lets test this and see if it works 
x,y = get_batch('train',4)
model = BigramModelWithAttention(vocab_size, 
                                 context_size, 
                                 embd_size=16,
                                 head_size=16,
                                 device='cpu',
                                 use_bias_att=False)
# model.cuda()
# x,y = (t.cuda() for t in zip(x,y))
logits,loss = model(x,y)
print(f'{logits.shape=} {loss=:.4f}')
# and now we can train this : 
device='cpu'
batch_size = 32
head_size = 16
embd_size = 16 
context_size = 8
vocab_size = len(vocab_list)
max_iter = 5000
# only to test the effect of bias in k,q,v calculations
use_bias_attn=False
# attention requires much lower lr compared to plain bigram model
lr = 1e-3
model = BigramModelWithAttention(vocab_size=vocab_size,
                                 context_size=context_size,
                                 embd_size=embd_size,
                                 head_size=head_size,
                                 device=device,
                                 use_bias_att=use_bias_attn)

param_count = sum([p.nelement() for p in model.parameters()])
print(f'param count:  {param_count:,}')
print(f'head size:    {head_size}')
print(f'embd size:    {embd_size}')
print(f'context size: {context_size}')

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
model = model.to(device)
# set model to train mode explicitly
model.train()
for i in range(max_iter):
    # get the batch
    x,y = get_batch('train', batch_size=batch_size)
    # feed the model and get the logits
    logits, loss = model(x,y)
    # evaluate the model
    if i%1000==0:
        losses=evaluate_loss(100, device)
        print(f'train: {losses["train"]:.4f}  val: {losses["val"]:.4f}')
    # zeroout_grads
    model.zero_grad(True)
    loss.backward() 
    optimizer.step()
print(f'done!')

# now lets try its output
input = torch.zeros(size=(1,1)).int()
output = model.generate(input, 500).squeeze(0).tolist()
print(f"{''.join(decode(output))}")
# prints
#here are a few tries:
# param count:  3,041
# head size:    16
# embd size:    16
# context size: 8
# train: 4.2551  val: 4.2533
# train: 2.7175  val: 2.7354
# train: 2.5917  val: 2.5681
# train: 2.5299  val: 2.5265
# train: 2.4684  val: 2.4759
# done!

# Wonedimy
# Y:
# ARUS:
# LAnouver; pof th, sthe the mae,
# Wor ut barrioreche savarduncou?

# hiler bathakm,
# I O:
# AK:
# Acat ED:
# et alliro dow wowilou ttherss; whest owur, garsacwe Ie hithaceyidous sou f'ze iwot ry tohien fm.
# h.

# Ar'm-ore tr Fofhou
# Mr ojous omerrastheasncy yous Wowuce whor tu I I:
# Hbeotth spat fel woro bel, aneicudidy ktiferrd thout ngord. th Iid aral, be'nd.

# Pal tels Ms eimu Me myo on I teasthe del,
# AChinout,
# Wowure. pu tcher muss-renon; wingh ocet im
# r Ddr,
# Do ne thu,
# LI ir't
# Operst ry wu

# param count:  5,745
# head size:    16
# embd size:    32
# context size: 32
# train: 4.1374  val: 4.1414
# train: 2.6264  val: 2.6210
# train: 2.5100  val: 2.5092
# train: 2.4434  val: 2.4578
# train: 2.4140  val: 2.4252
# done!

# Thee to an tal my,
# Thake ced arce chart bre le be bizoresf thak des garg
# Lur Wentescaln
# Hooco beins pord sth ds hom span gr; ther. My Why ort thallils ofrof not tos gas hyine ind hand
# Thid One the shond nd,
# AFenosee omn st hy no fe.
# GEwank, iand!
# The sbe qurss yod hem snel.
# CLerle,
# Whow Vou bem!
# Theewourtoave, shougen t:
# Ber hangn toutince mor ed,
# AD Scheapy thatho ABEN:
# AD:
# SUEDithy nde ndel mmy Tongusate hfolit, gintoher, hat gwe tiet what whingimst hpoourde wowopal,
# The by br des qO: I no os,

# try2: decreasing context size
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

#try 3: decreasing embd_size, but keeping context_size 32
# param count:  2,265
# head size:    16
# embd size:    8
# context size: 32
# train: 4.2202  val: 4.2158
# train: 2.9840  val: 3.0093
# train: 2.7667  val: 2.7621
# train: 2.6355  val: 2.6411
# train: 2.6070  val: 2.6077
# done!

# o
# TI:
# BOGot be
# DOus ssh.
# Bea hashren v:
# ANULINII feso sid izeis by tofuso te I be fnowe?
# Wledu ha m, thope w,
# ARUTI land med cu
# SI bve fme
# ARIThilaserothan omyhee
# Oina c;
# UCAngose.
# Terve t thirre my tthunulpthemaf wof.

#  corlur yoNE:
# NHy w te hen cod, tonos vengo veschete tors t ithaimeqlutreqRoosoye chin y h thuasafaous.
# ANCathelo-
# The?
# Shen br uyh sbe ans I:
# Gouthyykey m Authirou ae tovas ber ut lnt:,

# TAMENI sf rousinthsin h'howenthor sg.
# hidandhe waresd!

# I f
# Se loner mrdin gsvette n thest?

# try 4: decreasing head_size, keeping embd_size and context_size the same
# param count:  4,457
# head size:    8
# embd size:    32
# context size: 32
# train: 4.3416  val: 4.3409
# train: 2.7354  val: 2.7333
# train: 2.6400  val: 2.6309
# train: 2.5716  val: 2.5692
# train: 2.5217  val: 2.5182
# done!

# IF
# S:
# To ofldickeens we fofane for o warus
# Herdliion bis, te,
# The ta, d ounor;
# Ad h ank; sor Cito-'d turses tierlllle,
# Whe pen, tint yt Ging sino leles fes hesr sst.
# Than hinean.


# Nofl an udn f hak,
# Wh stharovernes;
# And meno scerr tth Gbl y hy I t stit pand sy, pefif.
# US dik thith be tagag, harlovenppsh!
#  cegshil,
# Wheanst thiges eprine sy wabat tha.

# Wheas
# Wit:
# I
# Thouw'arsangort,
# Giseal?

# Wh; mo tands t's.

# NRNULECADSAxave!

# ANNDIEI Fh Fod meveeas pee be,
# Bes arner,

# Woo t'le.


# VIPORLEOG
# CAOO:

# try 5:
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

# DANGUTTA:
# Bert hatire on the wo A? sewas ker haywins wet.

# mousill akerd fous sther Le fke sth,, bepeere gn ssst I oun amand serewos, cobear song.


# AMEfediengg yit othesshint.

# Anc:
# A must shan ghety ak is, shet.

# MEO:
# Tom yot stths!
# NCIBus hand akt imy berdr,
# Thulll id mere thassut,
# Moureleray, fous mante. QBO decencepous ariveed hart;
# I; bifo wthen chant tht tho hy youn,
# Grolomy, ucel.

# BRIINE ERIYht I arum arw t; odeat ngher bly wehe foru thas sot our:
# Bn; sshiutithemowith her fourpler thant

#
# which looks depressing not gonna lie! but its better than the simple bigram model we 
# built earlier, anyway, can we improve it? certainly! 
# for example we could use dropout on our attention! we could use normalization layers
# and we could use multiple heads instead of just one! which takes us to the next
# subject which is multi-head attention! which is  really nothing except several normal!
# attention blocks run in parallell! 
# so lets implement these and see how much we can improve upon this

results=tensor([[[ 0.,  1.],
         [ 1.,  2.],
         [ 2.,  3.],
         [ 3.,  4.],
         [ 4.,  5.],
         [ 5.,  6.],
         [ 6.,  7.],
         [ 7.,  8.]],

        [[16., 17.],
         [17., 18.],
         [18., 19.],
         [19., 20.],
         [20., 21.],
         [21., 22.],
         [22., 23.],
         [23., 24.]],

        [[32., 33.],
         [33., 34.],
         [34., 35.],
         [35., 36.],
         [36., 37.],
         [37., 38.],
         [38., 39.],
         [39., 40.]],

        [[48., 49.],
         [49., 50.],
         [50., 51.],
         [51., 52.],
         [52., 53.],
         [53., 54.],
         [54., 55.],
         [55., 56.]]])
tensor([[ 0.,  1.],
        [ 2.,  3.],
        [ 4.,  5.],
        [ 6.,  7.],
        [ 8.,  9.],
        [10., 11.],
        [12., 13.],
        [14., 15.]])
tensor([[0., 1.],
        [1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.],
        [5., 6.],
        [6., 7.],
        [7., 8.]])
a=tens

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_head, head_size, embd_size, context_size,bias_attn=False) -> None:
        super().__init__()
        self.num_head = num_head
        self.head_size = head_size
        # we assume the head_size is already split between the num_heads and thus we dont split
        # it again
        # assert head_size//num_head == 0, f'head_size({head_size}) must be divisable by head_num({num_head})'
        # we need to create n heads so lets do it 
        # self.heads = [AttentionHead(context_size, 
        #                             embd_size, 
        #                             head_size, 
        #                             bias_attn) for _ in range(num_head)]
        # but we can also use pytorch's nn.ModuleList which is a better equivalent than list
        # note that, nn.Sequential cant be used, becasue it runs the modules in succesion
        # i.e. serially, one after the other (feeds the output of the previous module to 
        # the next module, etc) which is not what we want. we want to calculate each head
        # independetly and aggregate their outputs so, either a python list or torch moudle list
        # can be used
        # sidenote: this module that we are building, is also known as, masked multi-attention-head
        # becasue we are using the single-head self-attention which uses a constrain we imposed
        # by masking if you recall that!
        self.heads = torch.nn.ModuleList(AttentionHead(context_size, 
                                         embd_size, 
                                         head_size,
                                         bias_attn) for _ in range(num_head))
      
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # note that head_size is usually the embd_size, so it covers the whole 
        # embeddings obviously, and note that each head will work on a portion
        # of the given head_size, if e.g. we have head_size/embd_size = 16
        # if we had 1 head, the head_size would be 16. however if we had 2 heads
        # we had to split the head_size in half for each head, and later on
        # we had to concat them to get the full-size head_size/embd_size.
        # therefore here, we need to concat their results along the cols
        # so multi-head-attention is akin to group convolution, and thus the head_size
        # and num_head must align properly.
        outputs = torch.cat([head(inputs) for head in self.heads], dim=-1)
        return outputs
# lets test 
# note that we set the split value for head_size based on our num_head
m = MultiHeadAttention(num_head=4, head_size=16//4, embd_size=16, context_size=8, bias_attn=False)
logits = m(torch.randn(size=(1,8,16)))
print(f'{logits.shape=}')
# 
# now lets use this in our model
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 
# now lets train this model and see how it performs this time
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# outputs : 
# param_count  =  3,041
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2041  val: 4.204
# train: 2.6895  val: 2.685
# train: 2.5449  val: 2.528
# train: 2.4570  val: 2.477
# train: 2.4238  val: 2.433
# done!

# I'd In mame
# Ast owu hapland.

# Mayl thew youres?
# IRisire dis lhend gitim whorder, sphorgunct Beye skek boutrig!

# I, sho sciltr not bpreathl harr'd doess elowt cof or iny poovigre.
# Thalrit?

# HThary?

# Lerim he bamat:
# Thiet hey, krepably bose bour cave grosr benemece wleir hon'g.

# MADIUMI but th Mond im's veoo tit. IDf met Cerve's wue rrer hiftren'tr'd:
# Anoks:
# HOt
# We I phess chi bouine mares yon ther; tlotirtlinl therot:
# Io
# Wh per,
# Fous thayigso ceany pate.

# Annd wosh fong uat
# Now ICUSo CCgodertiove
#
# trying with larger embedding size seems to improve the results: 
# param_count  =  7,553
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1828  val: 4.186
# train: 2.4703  val: 2.472
# train: 2.3649  val: 2.367
# train: 2.3009  val: 2.32
# train: 2.2713  val: 2.288
# done!

# Cly Toste?
# IOK:
# Whan his me' tis it I ond ber'thse?

# PO:
# Theray delling, dothoinerk an to the, or
# E VINGON: rocO:
# Thes win now thatin knir:
# Wigh fy, bund werar is wet my;
# I ene:
# JUu'd sow prut hat amver'd ENDUE VUF E VO?

# Se gings nom and.
# CArwak mandesplay beesire brit mand.

# MED:
# Thou ased ith is whis wentemprt hits this in ther.
# Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
# LUT: am le the wids my thousecarto my haty wind WLIERME:
# Wher a and
# Whesh to wyt wir

# compared to single head attention, with the same hyper parameters, train: 2.4684  val: 2.4759
# and now we got train: 2.4238  val: 2.433 which is better(we also got train: 2.2713  val: 2.288 
# with just increasing the embedding_size, so playing with parameters even blindly making model bigger
# seems to give us a boost), so we had improvements, but we still need a long way ahead of us!
# 
# Ok, to improve upon our results, there are couple of more things we need to add to our attention 
# block. if you look at the paper, we'll see a few concepts that we havent talked about or implemented
# yet, including the feed-forward(position-wise feedforward network) and layernorm, so lets talk about 
# them.
# feed-forward network or as its called in the paper,'position-wise feedforward network', is simply
# a "fully connected network which is applied to each position separately and identically", this 
# consists of two linear layer with a relu activation function inbetween. 
# so its a linear layer with a relu activation function followed by another linear layer basically. 
# you may also hear this network be refered to as 'computation after communication' so to speak!
# signifying the fact that it runs a computation after the attention module.
# the idea behind this extra addition is simple, to provide a higher representation out of attention work
# this network, causes every single token to have a nonlinear transformation and achieve a higher abstraction
# possibly yielding new information benficial to the task. to put it in casual way, it can be seen as though
# the attention gatheres some stats/information, and this step is akin to looking into it and thinking about it
# comming up with some new findings, that is , if attention part is refered to as communication part, this is
# the computation part/ or thinking part for the lack of a better word.
# lets implement this network in our bigram model and see if this seemingly simple change, affects us at all
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # now let us create the feedforward network, which basically is a linear layer with relu
        # followed by another linear layer! for this so called network, the in_features and out_features
        # are simply the same, and is embd_size, because its a sandwich layer between our attention output
        # and the last layer. this layer usually is (embedsize,embed_size) if you recall head_size is usally
        # equal to embed_size. also note that in the paper the feedforward network is two linear layers with
        # a relu (one linear layer with a relu plus another linear layer).if you look closely you'll notice 
        # we already have a last layer after the attention, so we dont need to put a nother linear layer
        # afterward, becasue that one linear layer suffices (multiple linear layers, dont provide
        # any higher abstraction anyway, so its prefectly fine)
        # but on the other hand the paper's implementation has another change, it states 
        # the inner layers dim are increased by 4x, so to be faithful to the paper we also 
        # add the second linear layer, with the suggested change, this wont change the output 
        # shape, so we are fine. we just added an extra linear projection layer. 
        # this additional projection operation actually improves the result,however if we simply 
        # use the same dim linear layer (i.e. have sth like linear(head_size, head_size)) adds nothing
        # to the representational power of the network and you wont see anything substantial vs if you
        # completely remove this layer and only use linear/relu only.
        # why this works is becasue, we increase the nonlinear output neurons of the first layer by 4,
        # increasing its representational capacity, and then use the second linear layer to get the output
        # size compatible for the next layer (doing a linear projection), and hence our improvements lie
        # in the nonlinearity this addition provides. 
        self.feedforwardnet = nn.Sequential(nn.Linear(embd_size, head_size*4), nn.ReLU(),
                                            nn.Linear(head_size*4, head_size))
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # add our new addition, feedforwardnet
        out = self.feedforwardnet(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using feedforwardnet added')
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints : 
# using feedforwardnet added
# param_count  =  5,169
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1975  val: 4.198
# train: 2.6222  val: 2.634
# train: 2.4919  val: 2.481
# train: 2.4320  val: 2.435
# train: 2.3979  val: 2.392
# done!

# Selwils iur.

# AnNdt
# By thak, yue, nein ende vat Rout'w haler,
# The hot hes? is, whas bles. Yig,
# ROm
# QUS:
# Ay Whigice rea:
# Therer a youst um mith dins I dorseancer by gou:
# I sot the, aserer tom sne arobfs-Rid,
# This sas whe ou,
# Acet Yark no.

# OT:
# Fely ip you pel hivet seatly. hial pand,
# Thand repwat,
# I tarve ther se this ten thio drord dot tho,
# Rulee res le, il fet!
# Nord Wotiilet homom for my woue cuve wid.

# Angilene.

# Fhivy, ih:
# Boumlke uthereaerf
# Whe ithe hares.

# Whos sthik thenth
# Tait:
# She tove w

# the loss didnt get better! but maybe if we increase the context_size, and embedding_size, it 
# perorm better?
# using a larger embedding_size and thus larger model did improve the result, 
#
# using feedforwardnet added
# param_count  =  15,905
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1569  val: 4.159
# train: 2.3885  val: 2.396
# train: 2.2789  val: 2.312
# train: 2.2064  val: 2.249
# train: 2.1599  val: 2.198
# done!

# To get fer arast deve lad, I' fale are you up this riany;
# Weareien,
# Tho Guser, me ter is! Do lothe, to ling wo whe Pist to deave mat not, in and- air besface, y to fron; use ler weet herefalf thop an der pretrier
# nesly toide
# Thats ous.

# RENT:
# I for the haw to, to noss teave ler shat earelss ater, want Heanct herry deeter erry, heave masere dacke; nochat onsemy ning Meib,--?

# LUCLUER:
# If Feropeake mo shat onsivoole washy the well beWarquent;
# I on well he ome it you lare to lo to kit youser,
# Weves

# the larger network, seems to be doing much better than its counter part from before
# the one without the ffnet, we got train-loss: 2.1599  val-loss: 2.198 here whereas 
# previously we got train-loss: 2.2713  val-loss: 2.288. so we have improvements despite the 
# text still not being that good(but still better than before). so we need more enhancements)

logits.shape=torch.Size([1, 8, 16])
param_count  =  7,553
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1828  val: 4.186
train: 2.4703  val: 2.472
train: 2.3649  val: 2.367
train: 2.3009  val: 2.32
train: 2.2713  val: 2.288
done!

Cly Toste?
IOK:
Whan his me' tis it I ond ber'thse?

PO:
Theray delling, dothoinerk an to the, or
E VINGON: rocO:
Thes win now thatin knir:
Wigh fy, bund werar is wet my;
I ene:
JUu'd sow prut hat amver'd ENDUE VUF E VO?

Se gings nom and.
CArwak mandesplay beesire brit mand.

MED:
Thou ased ith is whis wentemprt hits this in ther.
Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
LUT: am le the wids my thousecarto my haty wind WLIERME:
Wher a and
Whesh to wyt wir
using feedforwardnet added
param_count  =  15,905
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1569  val: 4.1

In [4]:
# the next improvement, would be to use more of these, as its shown in the paper, if one block
# works, then adding more should work better right? we had single head, it improved our condition, 
# so we used more heads, and now lets more multi-attention heads! to make things easier, lets
# make a module out of it the same way we created previous modules. 

# first lets create our ffnet as a seprate block, so we use it after the attention
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        return self.ffnet(self.attn(inputs))

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks!')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks!
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2087  val: 4.211
# train: 2.6084  val: 2.606
# train: 2.3737  val: 2.39
# train: 2.2865  val: 2.307
# train: 2.2087  val: 2.237
# done!

# FAUKIN YRE:
# Dato':
# wis.

# BERDEWEE:

# Piwit ancon; awt
# Emuke this his fred not pe,
# Har,
# And him my bird-wray sluf. Ile:
# With tulel wik, antegatorg.

# EREN VELV:
# Whath a meane, ones
# sheirss,
# Bose til
# Ilf bodtund ba,
# Aripce nerod is the blilshil hit hit mef?

# GESINCE:
# Sard, with the deoorrnsicominle thid: as.

# ANRE:
# Hy prall anges what micciths delre
# To tike comf hon ebat'd mur. Kin'sa, Fharth:
# Af'?

# ROASA:
# Ect mily fledd.
# SHricest.
# At dey bompe novor where lisg.

# KETILNINCE:
# There gay his my nefonk
#
# as you can see, we got wrose results than before! despite making the network larger, our loss
# really didnt improve as we expected. 
# is our initial hypothesis that having more blocks and higher nonlinearity/representation is benificial
# wrong? or is there something else thats causing the issue? 
# as you might have guessed, its the latter. we are basically creating more layers, and with
# more layers, we face training issues that we discussed earlier. 
# so how should we tackle this, we cant use BN, becasue BatchNOrm, accumulates the statistics
# from different samples, this is a no no for us. we dont want other samples to interfer with 
# our sample (or basically each other!) in anyway, so what should we do? 
# the paper utilizes two mechanisms or operations to tackle this issue. one being skip-connections
# (also known as residual connction) and the other, layer-normalization. 
# you should be familiar with skip-connections as they are the founding factor or resenets
# and have been extremely influential. so to cut a long story short, skip connections are simply
# connections from input skipping the operations involved in the block they reside and directly 
# being added to the output and then returned the result.(basically F(x) + x)
# as we know, the gradients are distributed equally when they reach addition, so input gets the gradients
# without being weakened due to large depth of the network. 
# so before we implement the layer normalization part, lets see howmuch of a change adding skip-connection
# causes and whether it proves our initial hypothesis about depth and gradient signal weakening or not
# we add this to our AttentionBlock (but we could add this to any submodule)

using more blocks!
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.2087  val: 4.211
train: 2.6084  val: 2.606
train: 2.3737  val: 2.39
train: 2.2865  val: 2.307
train: 2.2087  val: 2.237
done!

FAUKIN YRE:
Dato':
wis.

BERDEWEE:

Piwit ancon; awt
Emuke this his fred not pe,
Har,
And him my bird-wray sluf. Ile:
With tulel wik, antegatorg.

EREN VELV:
Whath a meane, ones
sheirss,
Bose til
Ilf bodtund ba,
Aripce nerod is the blilshil hit hit mef?

GESINCE:
Sard, with the deoorrnsicominle thid: as.

ANRE:
Hy prall anges what micciths delre
To tike comf hon ebat'd mur. Kin'sa, Fharth:
Af'?

ROASA:
Ect mily fledd.
SHricest.
At dey bompe novor where lisg.

KETILNINCE:
There gay his my nefonk 


In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
# lets add a skip-connection to this block
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        out = self.attn(inputs) + inputs
        out =  self.ffnet(out)  + inputs
        return out

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5636  val: 4.553
# train: 2.2666  val: 2.281
# train: 2.1330  val: 2.183
# train: 2.0708  val: 2.14
# train: 2.0204  val: 2.109
# done!

# Zunt will jucked's whreef onst's so
# wippplalct, sawter.

# Firfor have deal pere,
# Shal,
# thich many.
# Frig.

# Thall have.

# WANWHAM:
# Sad king theave,
# Cnomlen nare, near was?


# CKINGlLOROMBOENGBETH:
# Dich loverd und by, may;
# ''Kh
# Godie the bline plock's minstell the denelsbad, with think
# What
# sicond lead this undrut, too, the but me my se ming'ld;
# Shing
# To tiot comford, engelan this it's heare: hear them the vinclamil.
# Is done, the stronced,
# Lion?
# Hartow where lisg.

# DOLILINA:
# Lord ISe and
# Lud'd nef hel

# and test with one skip-connection on the 'output only' to prove or assumption on skip-connections
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5616  val: 4.552
# train: 2.3196  val: 2.322
# train: 2.1862  val: 2.221
# train: 2.1175  val: 2.163
# train: 2.0640  val: 2.137
# done!

# FARY VI:
# A-bame 'tries.

# BERGEWES:

# PABLLO:
# Yon; aws
# Then ath-pmate for, not pe,
# PARILA:
# So man.

# FORY.

# TAsUS:
# I cenpine nou, lad king theak,
# Anny, In nartine-'

# An yeant, of ladie ust, good ting lover that by, may;
# And hordie the
# llils plove ond sofful and heave
# Take with the deo,
# And mort leath and usir the dom the but me my crombasill;
# Shim are, con comlmork engeland your, trabe,
# Beth:
# Yor?

# CORIAR EFt mily fle--spoles, strence'd ono do now, will?
# Mursice,
# Ntormorest
# The juge ernus'd neforki

# as you can see, it greatly improved our results, and the text also got much better!
# as we already pointed out, having skip-connections per modules, help much more than a single 
# per module output only, nevertheless, we notice, using skip-connection really improved our results.
# now lets add the second operation, layernorm. layernorm(https://arxiv.org/abs/1607.06450) is a normalization layer, just like batchnormalization
# came a year later than batchnormalization paper, but the difference between them is that, unline batchnormalization
# it works on a per sample basis and does not involve using othersamples to normalize a specific sample (basically
# samples dont affect eachother)
# As pytorch docs puts it:
# "Unlike Batch Normalization and Instance Normalization, 
#  which applies scalar scale and bias for each entire channel/plane with the affine option,
#  Layer Normalization applies per-element scale and bias with elementwise_affine"
# the implementation is similar to the batchnormalization, and it does not require calculating running_mean/var
# we can use the pytorch module just fine, but since its really similar to BN, lets implement it here 
# 

using more blocks with skip-connection
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.5636  val: 4.553
train: 2.2666  val: 2.281
train: 2.1330  val: 2.183
train: 2.0708  val: 2.14
train: 2.0204  val: 2.109
done!

Zunt will jucked's whreef onst's so
wippplalct, sawter.

Firfor have deal pere,
Shal,
thich many.
Frig.

Thall have.

WANWHAM:
Sad king theave,
Cnomlen nare, near was?


CKINGlLOROMBOENGBETH:
Dich loverd und by, may;
''Kh
Godie the bline plock's minstell the denelsbad, with think
What
sicond lead this undrut, too, the but me my se ming'ld;
Shing
To tiot comford, engelan this it's heare: hear them the vinclamil.
Is done, the stronced,
Lion?
Hartow where lisg.

DOLILINA:
Lord ISe and
Lud'd nef hel


In [6]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (2.38418573772492e-09, 1.0004067420959473)
torch: (9.53674295089968e-09, 1.0004067420959473)


In [7]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
#
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we were aggregating the last two dims like BN, whereas we should have only used the
# last dim, as we should not involve other samples in normalization. (I explained this thoroughly in the LayerNorm class)
# 
# now how can we improve more? we implemented the paper, and what remains is to test with hyperparameters
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3906  val: 4.385
train: 2.2373  val: 2.244
train: 2.1070  val: 2.155
train: 2.0339  val: 2.102
train: 1.9812  val: 2.069
done!

For the that mad'
With of on tith, luidie anct, saws
Eule at botht if thou
Wlefted' you windian.

FLO-wike sluke slen:
I have, ladik, anteraves,
norl wink
the bard a yeane, of lady. Whe, goshot I
thy brath his what:
Nown hondin the
blinsh, shat this the pandself livadet brives nefors.
ISIUS:
Aren that usir the of me
he hange wicHe mirs,
Behat!


TOLAT:
But ford, engelan this it'sabhanke hear rend these this in usse,--
Lord, strence'er thy?
In woes. IDandry,
Soset frongs neved
With haul'd nefonk 


In [8]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
torch.manual_seed(255)
random.seed(255)


head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 250
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')

model.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

using more blocks with skip-connection-layernorm-beefed up!
param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 64.69 MiB is free. Process 6645 has 227.69 MiB memory in use. Including non-PyTorch memory, this process has 9.06 GiB memory in use. Of the allocated memory 7.96 GiB is allocated by PyTorch, and 117.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [9]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
torch.manual_seed(255)
random.seed(255)

del torch

using more blocks with skip-connection-layernorm-beefed up!


In [10]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
torch.manual_seed(255)
random.seed(255)
import gc 
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


NameError: name 'torch' is not defined

In [11]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
# torch.manual_seed(255)
random.seed(255)
import gc 
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


1547

In [12]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
torch.manual_seed(255)
random.seed(255)
import gc 
import torch
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


NameError: name 'torch' is not defined

In [13]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 

gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


488

In [14]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
del model
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


0

In [15]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
del x,y
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


0

In [16]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
torch.cuda.memory.reset_max_memory_allocated(0)
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


/home/hossein/anaconda3/lib/python3.11/site-packages/torch/cuda/memory.py:329: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


0

In [17]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
torch.cuda.memory.reset_max_memory_allocated(1)
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


RuntimeError: Invalid device argument 1: did you call init?

In [18]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
torch.cuda.memory.reset_max_memory_allocated()
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


1097

In [19]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.reset_max_memory_cached(0)
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


/home/hossein/anaconda3/lib/python3.11/site-packages/torch/cuda/memory.py:356: FutureWarning: torch.cuda.reset_max_memory_cached now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


0

In [20]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


0

In [21]:
head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')

model.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3845  val: 4.388
train: 1.9630  val: 2.045
train: 1.5727  val: 1.755
train: 1.4054  val: 1.632
train: 1.3029  val: 1.57
done!


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [22]:
model.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [23]:
model=model.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [24]:
model.cuda()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")


DUKE VINCENTIO:
Scent the Faulia.

KING EDWARD:
Sweet, good conceive these are to be.

CAPULET:
Not gentle Menenius, here well they pend lives and I
Rengeath one much buried.

HARCID:
O, God I'll: Bidish, Brobet, not--
Hath said, quotest me, is a done.

CORIOLANUS:
Then, no proud heret! what sat withile: woen so,
ha, wary thou comest, why thought that art thou nelsest;
yet fights on his dearth-hoad yoking hered.

KING EGBRY:
Very Grey, my lord?

CORIOLANUS:
The people.
Why let, sir, who have bee


In [25]:
model.cuda()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")


What, my Lady of Angelo?

BUCKINGHAM:
At at the untimely of doth bed, much,
I'll be a kind his worth sea of the
live mone conclaim my sorrow and amblitude
Dut jot the watching of goos, Caius York,
Whom hath power tire mild, I will leave you tell the
is it subjects.

GLOUCESTER:
So rass, swing play his manner great his love
Death is cloid of his fine.

POLIXENES:
No.

Ghost offence, sea, when we he is name
Where it were become me one years.

TLBONA:
Who, is he can you for the horse,
That he clave upon him. Grim to't her.

BRUTUS:
It be more fair climits night rest thinke I
not, and what all worse this.

SAETER:
Sir you are you must you wed?
Doth mendles when you have once descries them,
The harm of cullent you cursue for this:
Resole he must be abused you.

Clown:
Kill, by my your highne, and that all bitter thee.

JULIET:
Gaunt, I'll make thee.
Untired my brother birisment buriest!
I pray, minister in my eight; it is it he
not fastic that that fant for let thou
of thine, or sleepter.


In [26]:
model.cuda()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# What, my Lady of Angelo?

# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. but for cpu we had to do this , or make 
# positional_embedding to use models-device dynamically. 
model.cpu()
tuple(p.to('cpu') for p in model.position_embeddings.parameters())
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")



KING HENRY VI:
Hortense, from my feed dispatch! my mother cry pluck
Thrust you charged my heart desire my death.

EXTON:
O, Lauca, I may be a loud after since to
chall speak to the profession of the king; pray
your foot worship. Come to the my noble cush
At dways thy hand, in Romeo brief, Romeo,
Then rush'd me an one against Buckingham?

AUTo serve?

Patixtaes! let OF my person soldiers,
In courteous court's graciously curfe of a me,
Isely thoughts, as whether is care in Richmond.
Twarve and Perdum up my sovereign.

TYBALLO:
O, let me.

HESTRY:
Alan betholds, march he not inhance stronge
His uppon my lands plucking for my husband
These have his titures his mederitally
Is his losts of death. Help wa'd her but hell!

HASTINGS:
Verona's malice?

Nurse:
Herious, here live!
That is your deal of ministanor will
And give live both will have almost they have
piltual affire? rany her testroright he seem;
And it charagic you? am long-by.

HENRY BOLINGBROKE:
Why, madam, and you thus great time,

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [27]:
model.cuda()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# What, my Lady of Angelo?

# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. but for cpu we had to do this , or make 
# positional_embedding to use models-device dynamically. 
model.cpu()
for p in model.position_embeddings.parameters():
    p.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")


In from her bed to dequires us, we brail shelt,
And delight would have wretch it was servant
To die the grain him gone; were combricain us
And that innaud one displitature, and is done:
He must it had not set him thee to be graced.
Is this the my young spoul'n, if it have it
shapped him by the side and content and our
so youth; you well, in privates, by, the sunce will
have in you wear 'emoved men.
Dar, she show may less much drum for't:
And that would vengeant thou frant strong cau
And by this foratalish, the steel full free well.

First Servingman:
Who, have these reason! a raught with me. He complot
the return him: whith he, good George his mother?

Second Citizen:
This tell's be forces the house! my sovereique
And to the prettitable interutor,
Who goes on his protheteous will his ungrusited:
Comfortunes it our sweet arriable,
Wich case him slew'd them be pronom'd confessed?
If empty
Like with your truth, every shall as mouth'd:
I any would think out he where.
Wilt you moon more, w

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [28]:
model.cpu()
for p in model.position_embeddings.parameters():
    p.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [29]:
model.cpu()
for p in model.position_embeddings.modules():
    p.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [30]:
model.cpu()
for k,v in model.position_embeddings._modules.items():
    k.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [31]:
model.cpu()
for k,v in model.position_embeddings._modules.items():
    v.cpu()
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [32]:
# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=10000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# What, my Lady of Angelo?

# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. 



ANGELO:
Plead not battle well: I cannot do you brother?

ISABELLA:
Petruchs yourselves but say you not permit,
Is any will true i' the maders, sworn. O frailint
Of precial there take with a spoil, and again
Wilt a thwos hath rebuke. And the king is for the
shapen: now the looks war; if all that them
Christable a little faults and sweets show
The papent mercy and his abusey! where he's thousand?
Death author Bianca here no: yet, this lies:
Think we drunken, he will have ring again,
For mother finds to heaven! if thousand cries:
I was good as falsehood father, who, if an't
I must wiscasure you.

AUTo Clordion will the peyet to Take, to where,
Good night portently turn the time keeps.

Page:
The good pable one honour, each beary who dost who love
Or batter'd; with the occasion caused sights:
And whether affairs, best you tears for your oppress,
HE Master March-for will destay to King Lhamby
3 yoked for your highnest mother mistres hand.

PAULINA:
Ere your good Guilt's sweet: she doth us

In [33]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


4438

In [1]:
# in the name of God the most compassionate the most merciful
# in this section we will have a look at how a GPT model works
# in this section we will implement attention module, and see 
# how it works and lean more about its (sel-attention, cross
# attention, multi-head attention, etc)
# 
# so how are we going about this. we need to create a language model
# and then progressively imporve it with attention mechanism.
# we will basically be creating a kind of chatgpt (minus its chat capability and
# its obviously great perormance :)) 
# to keep this as simple as possible and not lose track of the important concepts involved,
# we can use our initial bigram model as the base model and work on improving that with attention.
# so lets start
#
# first lets import the basic stuff 

# for type hints
from collections.abc import Iterable

import random 
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# lets setup the manual seeds or determinstic output
torch.manual_seed(255)
np.random.seed(255)
random.seed(255)

# first lets read the dataset, we are going to use tinyshalespear which
# is around 1million characters (around 1MB in size), and our model is
# supposed to create texts resembling this dataset.
dataset = []
with open('./tiny_shakespear.txt','r') as file:
    # this time we read the whole text as one big str
    dataset = file.read()
    
print(f'{dataset[:100]=}')
# now lets create our atoi and itoa dictionaries for mapping
# lets create our unique character list, which is infact our vocabulary or vocab for short
vocab_list = sorted(set(''.join(dataset)))
vocab_size = len(vocab_list)
# ! explain token
# lets create our mapping dictionaries, we are basically going to use them
# for tokenization, converting our input into tokens which here are characters 
# and ultimately their integer representations
# so tokenization simply put, refers to converting an input into a list of numbers based on some creteria
# here, we just use a simple index number to map our vocabulary characters, which represent all character
# our model can use to generate text, to integer representation. 
# for realworld applications, we usually use libraries such as sentecepeace by google which works on a subword
# level which means, it doesnt encode the whole word, neither does it encode based on individual characters, it
# use sth in between. 
# (from its github repo: https://github.com/google/sentencepiece )
# SentencePiece is a re-implementation of sub-word units, an effective way to alleviate the open vocabulary 
# problems in neural machine translation. SentencePiece supports two segmentation algorithms, 
# byte-pair-encoding (BPE) [Sennrich et al.] and unigram language model [Kudo.]. 
#
# (For the reference:  
# Byte Pair Encoding (BPE) is a subword tokenization technique used in natural language processing (NLP) 
# and text processing tasks. It is a data compression algorithm that splits words into subword units. 
# BPE is commonly used in tasks such as machine translation, text generation, and language modeling.
# The basic idea behind BPE is to iteratively merge the most frequent pairs of characters or subword units
# in a corpus to create a new subword vocabulary. This merging process is based on the statistical properties
# of the corpus, specifically the frequency of character or subword pairs.
#
# Here's a high-level overview of the BPE algorithm:
# 1. Initialize the vocabulary with all the characters or subwords in the corpus.
# 2. Calculate the frequency of each character or subword in the corpus.
# 3. While the desired vocabulary size or a maximum number of iterations is not reached:
#    - Find the most frequent pair of characters or subwords in the corpus.
#    - Merge the pair into a new subword unit by concatenating them.
#    - Update the corpus by replacing occurrences of the merged pair with the new subword unit.
#    - Update the vocabulary and frequency counts based on the new corpus.
# 4. The final vocabulary is the set of subword units obtained after the desired number of iterations 
#    or vocabulary size is reached.
#
# BPE allows for the representation of both known and unknown words in a corpus. It is effective in 
# handling out-of-vocabulary (OOV) words and reducing the vocabulary size, which can improve the 
# efficiency and performance of NLP models. By breaking down words into subword units, BPE can capture
# morphological and semantic information more effectively, especially for languages with complex word 
# formations and agglutinative structures.
# 
# tiktokenize repo explains BPE in rather friendlier way: 
# Models don't see text like you and I, instead they see a sequence of numbers (known as tokens). 
# Byte pair encoding (BPE) is a way of converting text into tokens. It has a couple desirable properties:
# It's reversible and lossless, so you can convert tokens back into the original text
# It works on arbitrary text, even text that is not in the tokeniser's training data
# It compresses the text: the token sequence is shorter than the bytes corresponding to the original text. 
# On average, in practice, each token corresponds to about 4 bytes.
# It attempts to let the model see common subwords. For instance, "ing" is a common subword in English, 
# so BPE encodings will often split "encoding" into tokens like "encod" and "ing" 
# (instead of e.g. "enc" and "oding"). Because the model will then see the "ing" token again and again 
# in different contexts, it helps models generalise and better understand grammar.
#
# 
# A unigram language model however, is a type of statistical language model that predicts the probability 
# of each word in a sequence independently, based solely on the frequency of occurrence of individual 
# words in the training data. It does not consider the context or the order of the words in the sequence.
# In a unigram language model, the probability of a particular word is estimated by counting the frequency 
# of that word in the training corpus and normalizing it by the total number of words in the corpus. 
# The probability of a sequence of words is then calculated by multiplying the probabilities of each 
# individual word in the sequence.
# For example, consider the sentence "I love to eat pizza." In a unigram language model, the probability
# of this sentence would be calculated as the product of the probabilities of each word which would be: 
# P(I) * P(love) * P(to) * P(eat) * P(pizza).
# Unigram models are the simplest form of language models and do not capture any contextual information
# or dependencies between words. They are often used as a baseline or reference model in natural language 
# processing tasks. While unigram models are not very accurate in capturing the complexities of natural 
# language, they can be computationally efficient and useful in tasks where the context is less important, 
# such as certain text classification tasks or language generation tasks where only the frequency of 
# individual words matters.
# 
#
# Openai/chatgpt for example uses its own tokenizer, called tiktoken (https://github.com/openai/tiktoken)
# there are other tokenizers as well.
# huggingface has a good article on tokenizers (recommend it): https://huggingface.co/docs/transformers/main/tokenizer_summary 
# its a given that, each model only works with the tokenizer by which it was used to train. 
# so BERT, DistilBERT, and Electra only work with wordpiece, while chatgpt uses titoken and we! use
# simple characters-indexes! also note that the choice of tokenizer obviously affects our vocab size and
# model overhead/performance as well. 
# for example lets first implement our tokenizer and then compare it with sth like tiktoken module
atoi = {c:n for n,c in enumerate(vocab_list)}
itoa = {n:c for c,n in atoi.items()}
print(f'{vocab_size=}, {vocab_list=}')
print(f'{atoi=}')
print(f'{itoa=}')
# lets also create two helper functions to convert a list of these to the other part
def encode(characters:Iterable[str] ):
    return [atoi[c] for c in characters]

def decode(token_lst:Iterable[int]):
    return [itoa[n] for n in token_lst]

# lets test these 
print(f'{encode(dataset[:10])}')
print(f'{decode(encode(dataset[:10]))}')

# now lets try tiktoken
try: 
    import tiktoken
except:
    import os 
    os.system('pip install tiktoken')
    import tiktoken
# lets use the tokenizer for gpt2 model, this line downloads the gpt2 tokenizer and allows us to use it
encoder_gpt = tiktoken.get_encoding('gpt2')
# lets view its vocab size:
print(f'{encoder_gpt.n_vocab=:,}') # prints encoder_gpt.n_vocab=50,257
# now lets see how the encoding looks like here
print(f'{encoder_gpt.encode(dataset[:10])=}') # prints [5962, 327, 8846] 
# which is very intresting! while for us its 10 numbers/tokens for 10 characters(vocab size 65),
# this is 3 here with vocab size of 50,000!
# so we can have a short vocab size, at the expense of a larger sequence size, 
# or a large vocabsize and smaller sequence size.
# so thats why in practice, these subwords tokenizers are used for real world applications. but for our case
# we stick to our primitive tokenizer to keep things as simple as possible. 
# 
# Ok, so far so good. now we need to create a dataset, like before we need to have an input/label pair
# our input is a series of characters, (a sequence of some length), and our label is the next character
# sicne we are using a simple bigram model, given a single character, we want the probablity of what comes
# next. but, we also want to incorporate attention, and we want a context for our prediction, we want to
# be able to look at the past, and look at the past characters, and based on that do sth. this is the essence
# of attention (although this is not accurate, but for now this is the case, we will elaborate on this and expand
# this metaphor and reasoning inshallah)
# 
# lets first tokenize the whle dataset or corpus as its usually called in nlp nomenclature!
data = encode(dataset)
# since we are using pytorch lets convert that to a tensor
data_tensor = torch.tensor(data)
print(data_tensor[:100])
# now lets create a train/val split 
train_length = int(0.9 * len(data_tensor))
# note that we do not shuffle the data here like before, because we are not dealing with a list of samples!
# like names, here we are dealing with the whole text, and if we shuffle it like that, we just destroy it
# making it into a batch of random characters! which would not be useable for us anymore. we are trying to
# learn the underlying semantic and relationships hidden in our data, and randomizing them like that just 
# destroys those information! 
# to see this in action try this
_data_copy = data.copy()
random.shuffle(_data_copy)
print(''.join(decode(_data_copy[:100])))
# prints:
# Osetow Pr 
# oa geEeM rhnalelrt   aAdt Ktsw pTpeGxli
# oasEliMe
# vsieeose
# etttbSdnctiras  hnr:yhtayiuCv g
# so we simple divide the data normally
train_data = data_tensor[:train_length]
val_data = data_tensor[train_length:] 
# note that since we are planning on creating a simple transformer model, we usually dont feed the whole dataset
# becaue its prohibitevly computation intensive, instead, what happens in practice is that we, grab chunks 
# of data from the dataset and feed it to the transformer. and these chunks, of course has a length, what length?
# we usually specify a maximum_length for the input on which our transformer model works.
# this maximum_length is usually refered to as block_size or context_size.
# we had previously used different context_sizes and this is not really that different,
# lets for example define a context_size of 8
# block_size and context_size are interchangable
block_size = context_size = 8 
print(f'{train_data[:block_size]=}')
# which prints:
# train_data[:block_size]=tensor([18, 47, 56, 57, 58,  1, 15, 47])
# 
# one intresting observation we can make here is that, as simple as this seemingly ordindary list of numbers
# looks, this sequence of numbers, actually contains several examples. 
# if you think about it, each number, is a token, representing a character (or word, subword, etc),
# and it shows what comes after what. basically not only it shows several pairs so to speak, it also
# shows, which characters are more likely to come, before a specific character comes later. 
# what we are actually going to do is that, we are going to train all of these characters simultaneously
# notice that in this example, we have 7 examples in a sequence of 8 characters:
# lets elaborate on this more. 
# 1-in the context of 18, the next character is 47
# 2-in the context of 18,47, the next character is 56
# 3-in the context of 18,47,56, the next character is 57
# 4-in the context of 18,47,56,57 the next character is 58
# 5-in the context of 18,47,56,57,58 the next character is 1
# 6-in the context of 18,47,56,57,58,1 the next character is 15
# 7-and finally, in the context of 18,47,56,57,58,1,15 the next character is 47
# so this is infact 8 7 individual example embedded in a single context, 
# since we want the xontext_size to be 8, then we should grab one more character to have 
# 8 contexts
# lets visualize this in example in code
# lets have a typical sequence of size 8 
x = train_data[:block_size]
y = train_data[1:block_size+1]
# what would be our label? we want the next character so it would be the previous locations +1
# basically offset the input by 1!
# thats why we started from 1, becasue the input started from 0, and since the input ended at block_size
# its label(next character) would obviously be block_size+1, pretty obvious right?
#
# lets print this for better understanding 
for i in range(block_size):
    # note that since we are using slice, x[:i+1], gives us 0 up to ith element (inclusive)
    # this should be obvious! but I said it in case you forgot!!
    print(f'input is {x[:i+1].tolist()} label is {y[i]}')
# prints 
# input is [18] label is 47
# input is [18, 47] label is 56
# input is [18, 47, 56] label is 57
# input is [18, 47, 56, 57] label is 58
# input is [18, 47, 56, 57, 58] label is 1
# input is [18, 47, 56, 57, 58, 1] label is 15
# input is [18, 47, 56, 57, 58, 1, 15] label is 47
# input is [18, 47, 56, 57, 58, 1, 15, 47] label is 58
# so as you can see we started from context_size of 1 up to context_size of 8.
# note that we do not do this simply for the efficancy aspect of it, but also, when we feed these to
# our transformer model, we are making sure that the transformer model gets used to see all combinations
# of our input as well (from the context size of 1 up tp the context size of 8). 
# this way not only it sees the whole context_size as we initially expected
# but also all sequences before it, and basically what consituted to make the sample. 
# this allows us to later on, at test time be able to create sequences as small as context_size of only 1
# up to the max_length which is our context_size of 8 in our case.
# ! recheck and elaborate to clear any confusion
# !so by doing this, the transformer can learn how to predict/create/genrate text up to context_size, and after
# !it reached thta, we have to truncate it, becausse the transformer model never recieves more than the
# !context_size as input when its predicting the next character.
# so far what we covered here was the time dimension of our input. we have a sequence, and each entery
# basically denotes a time t dimension, at which, a character is introduced. 
# another imporatnt aspect we need to take care of is the batch dimension, cuz we are going to feed 
# multiple examples at once, we use this to harness the gpu parallilization capalibity in pytorch
# as without it, training this simple model would take a lot of time on cpu! 
#
# so lets create a function that gives us a batch of the inputs rather than a single list/tensor!
# 
batch_size = 4 
# our block_size
context_size = 8 
# 
def get_batch(split, batch_size):
    #lets grab the data basedo on the split
    data = train_data if split =='train' else val_data
    # we want a batch of 4 of 8 characters (context-size). 
    # to make a batch we can grab 4 random indices as input and then expand them
    # by adding the next 8(context_size) characters to them, in order not to go past the 
    # last index, we subtract the length of data(last valid index) from context size 
    idxs = torch.randint(low=0, high=len(data)-context_size, size=(batch_size,))
    # we have our indices, so lets create our samples
    # x = [data[i:i+context_size] for i in idxs]
    # and for labels we do the same thing but offset it by 1 so each characters label
    # becomes the next character
    # y = [data[i+1:i+context_size+1] for i in idxs]
    #
    # print(f'{x=}')
    # print(f'{y=}')
    # prints: 
    # x=[tensor([58, 39, 49, 43,  1, 51, 63,  1]), tensor([53, 59, 41, 46,  5, 42,  1, 61]), tensor([47, 53, 52,  1, 39, 57,  1, 63]), tensor([39, 57, 58,  1, 51, 63,  1, 50])]
    # y=[tensor([39, 49, 43,  1, 51, 63,  1, 54]), tensor([59, 41, 46,  5, 42,  1, 61, 47]), tensor([53, 52,  1, 39, 57,  1, 63, 53]), tensor([57, 58,  1, 51, 63,  1, 50, 53])]
    #
    # as you can see we have a list of tensors. since we want a batch, we just stack them or concat them
    # on top of each other
    # using stack!
    # x = torch.stack(x)
    # y = torch.stack(y)
    # stack() is intrestingly implemented by concat(), so if we want, we can do the same using concat!
    #
    # by default conact, concatenates all the tensors as one large tensor!
    # we need a reshape/view to get the right shape
    # since we are using view, no copy,etc is done, and its an efficient operation
    # x = torch.concat(x,dim=0).view(batch_size, context_size)
    # y = torch.concat(y,dim=0).view(batch_size, context_size)
    # 
    # however if we add an extra dim to our inputs we can remove the extra reshape/view
    # x = torch.concat([t.unsqueeze(0) for t in x], dim=0)
    # y = torch.concat([t.unsqueeze(0) for t in y], dim=0)
    #
    # By unsqueezing, we ensure that the tensors have the same number of dimensions before 
    # concatenating them with torch.cat/concat/concatenate.(side tip: concat and concatenate are aliases for cat) 
    # This effectively replicates the behavior of torch.stack.
    # see torch.cat concatenates tensors along an existing dimension, and we only have 1 dimensional
    # tensors, so it will concatenate them along that, effectively making one large 1 dimensional vector
    # however, when we use unsqueeze on each tensor, using torch.unsqueeze(0), we are adding a new dimension
    # (dim 0) to that tensor, making it 1,x instead of the original shape of (x,).  
    # So, by unsqueezing each tensor along the desired dimension, which for us is the 0ths dimension (or row dim) 
    # and then concatenating them with torch.cat, we achieve the same result as torch.stack.
    # 
    # in practice we'd like to do this all in one go!
    x = torch.stack([data[i:i+context_size] for i in idxs])
    y = torch.stack([data[i+1:i+context_size+1] for i in idxs])
    
    return x,y

x,y  = get_batch('train',batch_size=4)
print(f'{x.shape=}\n{y.shape=}')
print(f'{x=}\n{y=}')
# which prints 
# x.shape=torch.Size([4, 8])
# y.shape=torch.Size([4, 8])
# tensor([[58, 39, 49, 43,  1, 51, 63,  1],
#         [53, 59, 41, 46,  5, 42,  1, 61],
#         [47, 53, 52,  1, 39, 57,  1, 63],
#         [39, 57, 58,  1, 51, 63,  1, 50]])
# tensor([[39, 49, 43,  1, 51, 63,  1, 54],
#         [59, 41, 46,  5, 42,  1, 61, 47],
#         [53, 52,  1, 39, 57,  1, 63, 53],
#         [57, 58,  1, 51, 63,  1, 50, 53]])
#
# now this is our batch of data, 4 samples with 8 characters, bascially 32 examples, lets see that as well
for b in range(batch_size):
    # this is our time dimension
    for t in range(context_size):
        print(f'{x[b,:t+1]} --> {y[b,t:t+1].item()}')
#        
# prints : 
# tensor([58]) --> 39
# tensor([58, 39]) --> 49
# tensor([58, 39, 49]) --> 43
# tensor([58, 39, 49, 43]) --> 1
# tensor([58, 39, 49, 43,  1]) --> 51
# tensor([58, 39, 49, 43,  1, 51]) --> 63
# tensor([58, 39, 49, 43,  1, 51, 63]) --> 1
# tensor([58, 39, 49, 43,  1, 51, 63,  1]) --> 54
# tensor([53]) --> 59
# tensor([53, 59]) --> 41
# tensor([53, 59, 41]) --> 46
# tensor([53, 59, 41, 46]) --> 5
# tensor([53, 59, 41, 46,  5]) --> 42
# tensor([53, 59, 41, 46,  5, 42]) --> 1
# tensor([53, 59, 41, 46,  5, 42,  1]) --> 61
# tensor([53, 59, 41, 46,  5, 42,  1, 61]) --> 47
# tensor([47]) --> 53
# tensor([47, 53]) --> 52
# tensor([47, 53, 52]) --> 1
# tensor([47, 53, 52,  1]) --> 39
# tensor([47, 53, 52,  1, 39]) --> 57
# tensor([47, 53, 52,  1, 39, 57]) --> 1
# tensor([47, 53, 52,  1, 39, 57,  1]) --> 63
# tensor([47, 53, 52,  1, 39, 57,  1, 63]) --> 53
# tensor([39]) --> 57
# tensor([39, 57]) --> 58
# tensor([39, 57, 58]) --> 1
# tensor([39, 57, 58,  1]) --> 51
# tensor([39, 57, 58,  1, 51]) --> 63
# tensor([39, 57, 58,  1, 51, 63]) --> 1
# tensor([39, 57, 58,  1, 51, 63,  1]) --> 50
# tensor([39, 57, 58,  1, 51, 63,  1, 50]) --> 53 
#
# so now that we have the data sorted out, lets create our model. 
# as we said, we are going to use a bigram model and later add attention mechanism to it. 
# to make things easier and more self contained, lets add all the required logic to this model
# like when we do a forward, we be able to calculate loss as well if we are given the targets
# so lets go
class BigramModel(nn.Module):
    def __init__(self, vocab_size) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        # our bigram model was nothing more than a 2d array of vocab_size, we can achive that using 
        # a single weight matrix or torch.Embedding. we use torch.Embedding to not reinvent the wheel!
        self.token_embedding = torch.nn.Embedding(vocab_size, vocab_size)
    
    # since we want to be able to calculate loss, if there are labels, we get Y as well
    def __call__(self, inputs:torch.Tensor, labels:torch.Tensor=None) -> torch.Tensor:
        logits = self.token_embedding(inputs)
        loss = None
        if labels is not None:
            # calculate loss 
            # print(f'{logits.shape=}') # prints (4,8,65)
            # note that both inputs and labels have the shape (B,T)
            # but logits has the shape (B,T,C) which C here equals vocab_size 
            # this is an issue for torch.crossentropy, as it expects the input
            # to be in the form of (B,C,T), that is, the channels/embeddings dimension
            # need to be right after the batch dimension or otherwise it wont work.
            # so we need to account for that.
            # we can go on permute the dimensions, like this and make crossentropy happy 
            # logits = logits.permute((0,2,1))
            # however, if for some reason the output of logits in the form of (4,8,65) is not ideal, 
            # maybe for example because really what it is 32 examples arranged in a (4,8) shape and 
            # we rather a normal 2d tensor of shape (32,65), 
            # we can instead, do just that, flatten the two dimensions into one and carry on!
            # we basically are concatenating the batch and time dimensions
            # into one dimension (effectively, stacking samples on top of each other), instead of having 
            # four compartments, each having 8 segments, we are going to have 1 long compartment with 32 segments/rows
            # for the lack of better words!!
            # so lets first get the shapes 
            # B,T,C = logits.shape
            # logits = logits.view(B*T,C)
            # and we also need to do the same for the labels 
            # labels = labels.view(B*T)
            # we could also do,
            # labels = labels.view(-1)
            # but thats not really needed, so we just simply permute logits temporarily so the logits shape
            # stays the same regardless of calculating the loss or not 
            loss = F.cross_entropy(logits.permute((0,2,1)), labels)
        return logits, loss
    
    # now lets also add a generate method to generate texts
    def generate(self, idx, max_token_count):
        # before we implement this lets review what we expect this method to do
        # we want this to generate some characters, given an initial character
        # we also want to be able to control the length of the generated text
        # so we take a max_token_count.
        # we take the initial index, 
        # feed it to the model, 
        # get the logits,
        # turn logits to probs,
        # use torch.multinomial to sample from our probablity distribution
        # get the new character index, and add it to a list to gradually 
        # -create our final text output
        # so lets go 
        # our idx should have the shape [B,T]
        assert len(idx.shape) == 2, f'idx.shape({idx.shape}) should have (b,t) form.'
        #
        # lets generate as many characters/idx as max_token_count specifies
        # this is basically specifies the time/sequence dimensions
        for i in range(max_token_count):
            # since we are defining a class method, to call the callable, we simply use self()
            logits, _ = self(idx)
            # to get the probablities 
            # usually we would simply do 
            # probs = torch.softmax(logits, dim=1)
            # but here, we want to only focus on the next character because this is what comes next
            # each time obviously, so we only take the last timestep to see what the model predicted
            logits = logits[:,-1,:] # this now becomes (B,C) instead of the initial (B,T,C)
            probs = torch.softmax(logits, dim=-1) 
            # now lets sample from it
            idx_next_char = torch.multinomial(probs, num_samples=1, replacement=True) # shape is (B,1)
            # now lets add this to the next input to be fed to the model 
            # since idx has the shape(batch, T), we should add this tothe second dimension
            # to the time dimension/ or sequence dimension. this as the loop goes on, 
            # creates the shape (B,T+1) 
            #this doesnt make sense for this particular model, becasue we are always checking
            # the next character given the previous one, so all the concatenation we are doing
            # is just useless. the reason we are implementing this like this, is to create a 
            # base, so that we can improve upon it when we add attention later on which will use
            # the history of previous characters.
            idx = torch.cat((idx, idx_next_char), dim=1) 
        
        # and finally when all is done return the idx which by now should have the whole output
        return idx
    
model = BigramModel(vocab_size)
out,loss = model(x,y)
print(f'{out.shape} {loss}')
# prints:
# torch.Size([32, 65]) 4.574285984039307
# the loss is good, we learned previously that we can evaluate a base loss provided our number of classes
# since we have 65 classes (our vocab_size or number of characters involved) the uniform probability for each class
# would be 1/65 =0.015384615, which if we take its negative log would turn out to be -ln(1/65) = 4.17438727, 
# which is pretty close to the loss we got here, signifying its a pretty decent value to begin with.
# also lets see the generate method at work
# lets create a dummy input, basically a batch of 1 and sequence of 1 of zero!
# we do this to generate a text from scratch
inputs = torch.zeros(size=(1,1),dtype=torch.int32)
output = model.generate(inputs, max_token_count=100)
print(f'{output=}')
print(''.join(decode(output.squeeze(0).tolist())))
# prints 
# wUbN:i :kLnOqHCQTG;.MbupOWH Xfi!MSUalaQppNNqIoWfmuSIIfmuZqf-3NLt-YkOc3vC-YkBfEHr3pJodwVY.nw.,r!&,Clt
# which is expected since our model is not trained yet! 
# lets train the model now and see how it works

batch_size = 32
context_size = 8
vocab_size = len(vocab_list)
max_iter = 20000

model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device=device)

print(f'{torch.__config__.show()}')
print(f'{device=}')
print(f'{model=}')

model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        print(f'{loss}')
# prints 
# 4.760574817657471
# 3.727213144302368
# 3.0139858722686768
# 2.6749324798583984
# 2.5884087085723877
# 2.5262675285339355
# 2.4420430660247803
# 2.4787800312042236
# 2.4680707454681396
# 2.564662456512451
# 2.51802659034729
# 2.562812328338623
# 2.422783613204956
# 2.5023772716522217
# 2.501049518585205
# 2.411632537841797
# 2.4028165340423584
# 2.4135568141937256
# 2.50952410697937
# 2.574878215789795
# relying on single batch loss is not a good idea to measure the performance of a model. moreover
# relying on training loss, is not good either, so it would be much better if we considered more batches
# for loss and even better we could also investivate the models performance on our validation set. 
# so lets do just this and define a function that calculates loss for training and validation sets alike
# but considers more batches for loss calculation

@torch.no_grad()
def evaluate_loss (iterations, device=None):
    results={}
    # before calculating the loss, lets switch to eval mode,although for our specific case this doesnt matter
    # but its goo practice, as later on, we will add layers that their behavior do change depending on traing
    # val mode.  
    model.eval()
    if device is None:
        # use the device assigned to what model params are assigned
        device = next(model.parameters()).device
    # since we already used no_grad decorator, we dont need to use no_grad context manager here
    # with torch.no_grad():
    for split in ['train','val']:
        losses = torch.zeros(size=(iterations,), device=device)
        for i in range(iterations):
            x,y = get_batch(split,batch_size)
            x,y = tuple(t.to(device) for t in (x,y))
            logits, loss = model(x,y)
            losses[i] += loss
        results[split] = losses.mean(0)
    # since we want to use this inside training loop, make sure we set the model back to train mode
    # incase we use layers such as batchnorm,etc that the require being trained!
    model.train()
    return results

loss= evaluate_loss(200)
print(f'{loss["train"]=} {loss["val"]=}')
# so now that our function seems to be working lets use it in the training loop :
model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
model = model.to(device=device)
model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        # now instead of simply printing loss, lets use our new function!
        loss= evaluate_loss(200)
        print(f'train_loss: {loss["train"].item():.4f},  val_loss: {loss["val"].item():.4f}')
# which results in :
# train_loss: 4.7491,  val_loss: 4.7430
# train_loss: 3.6673,  val_loss: 3.6670
# train_loss: 3.0460,  val_loss: 3.0511
# train_loss: 2.7335,  val_loss: 2.7492
# train_loss: 2.5948,  val_loss: 2.6156
# train_loss: 2.5292,  val_loss: 2.5470
# train_loss: 2.4982,  val_loss: 2.5194
# train_loss: 2.4807,  val_loss: 2.4991
# train_loss: 2.4737,  val_loss: 2.5036
# train_loss: 2.4650,  val_loss: 2.4949
# train_loss: 2.4648,  val_loss: 2.4910
# train_loss: 2.4626,  val_loss: 2.4836
# train_loss: 2.4570,  val_loss: 2.4893
# train_loss: 2.4517,  val_loss: 2.4823
# train_loss: 2.4508,  val_loss: 2.4819
# train_loss: 2.4564,  val_loss: 2.4839
# train_loss: 2.4555,  val_loss: 2.4802
# train_loss: 2.4503,  val_loss: 2.4927
# train_loss: 2.4522,  val_loss: 2.4905
# train_loss: 2.4572,  val_loss: 2.4888
#
# now lets check its output again after some training 
inputs = torch.zeros(size=(1,1), device=device, dtype=torch.int32)
output = model.generate(inputs, max_token_count=500)
print(''.join(decode(output.squeeze(0).tolist())))
# prints :
# Ane pod BUL:
#
# Oringe
# nfote s Rinsou oe the,
# An ETwe
#
# PUSENI bmilft!'d ate t Ifur
# Athomeg?
# Whagre,
# TCo belangead ne bl e, bednth ftor veso.
# US tla lin:
# Yourthiee he w whetitrd;
# Angoknghothartro ll heshethor hie douluk t, s se, denopit s h wom uspouct blyowie s'nos t prou gmesat
# Shd, tieithy.
# Asiour hofo bay avear hanousthaie
# Calonth wolulo.
#
# The hancondors pthighar:
# WAME outhaser d!
# S:
# h im t w s, inowo ORILOUCHoou isecetoubecompis
# Ay gnd, teve luthorea, therpeclsthevecthede fichim?
# PUSa VI aken 
# 
# which looks much better than the initial random output we got earlier.

dataset[:100]='First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
vocab_size=65, vocab_list=['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
atoi={'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't':

In [2]:
torch.manual_seed(255)
# now lets add attention mechanism to our base model. 
# attention mechanism at its core tries to take advantage of the rich information embedded
# in the sequence. for our work, this is specifically about the past history but in general 
# attention can utilize both past and future connections/sequence tokens. what we described 
# here just now is not exactly accurate, but gives us a foundation to build our intuition as
# we continue on. we will elaborate more of course and hopefully get it all.
# before we venture any further into the crux of the matter, let us learn about a technique
# thats used to efficiently implement attention mechanism.
# for this purpose,lets imagine we have a simple input like the following: 
# lets create and input of the following shape
B,T,C = (4,8,2)
# to make it more intuitive lets make a tensor with known numbers andthen reshape it
x = torch.arange(0,64,dtype=torch.float).view(B,T,C)
# imagine we have an input like what we encountered previously in our examples. in this sample 
# input, we have a batch of 4 samples, each having 8 sequences with each sequence having a vector of 2 values
# what we are planning to do is to provide a way by which each token can communicate with other 
# tokens. we have 8 tokens in our sequence. so we want our tokens to be able to communicate with
# all previous tokens that came before it. the reason we are only looking in the past token is 
# simply becasue the we are trying to perdict the future, so it only makes sense to look at the
# past and current timestamp and infer on what to do for the future.  
# so the easiest way to implement a kind of communication between tokens could be to sum or average the
# values of all previous tokens plus the current one as a way of taking into account their contribution
# to the final answer.
# that is, lets say if we are currently at token 5, we take the average of 
# the current token and all previous tokens before it, effectively making a feature vector that
# reflects our current status of the sequence so far, having taken all previous tokens/steps up to now.
# note that as you may also have thought, summing or averaging arent the best way to model such interations.
# in fact they are an extremely weak form of interaction between tokens,
# this kind of communicating is extremely lossy so to speak, that is we lose a great deal of information 
# concerning the underlying relationships between tokens, their arrangements,their implicit interactions,
# semantics, etc. but for now this is ok. we will later on see how to bring back such information.
# so now what we want to do, is to calculate the sum or average of all tokens up to the current token in 
# all batches at the same time.
# a naive way would be to do sth like this using a for loop:
results = torch.zeros(size=(B,T,C))
for b in range(B):
    for t in range(T):
        results[b,t] = x[b,:t+1].mean(0)
print(f'{results=}')
# which prints 
# results=tensor(
#        [[[ 0.,  1.],
#          [ 1.,  2.],
#          [ 2.,  3.],
#          [ 3.,  4.],
#          [ 4.,  5.],
#          [ 5.,  6.],
#          [ 6.,  7.],
#          [ 7.,  8.]],

#         [[16., 17.],
#          [17., 18.],
#          [18., 19.],
#          [19., 20.],
#          [20., 21.],
#          [21., 22.],
#          [22., 23.],
#          [23., 24.]],

#         [[32., 33.],
#          [33., 34.],
#          [34., 35.],
#          [35., 36.],
#          [36., 37.],
#          [37., 38.],
#          [38., 39.],
#          [39., 40.]],

#         [[48., 49.],
#          [49., 50.],
#          [50., 51.],
#          [51., 52.],
#          [52., 53.],
#          [53., 54.],
#          [54., 55.],
#          [55., 56.]]])
#
# You may find out that, some researchers refer to this operation here as BoW, or bag of words. 
# We are effectively averaging embeddings here and averaging embeddings can be considered a form of 
# Bag of Words (BoW) representation. In BoW, the focus is on the occurrence and frequency of words, 
# rather than their order or structure. By averaging embeddings, we are essentially treating each word 
# as an independent feature and capturing its representation in the form of a numerical vector.
#
# While averaging embeddings does not capture the exact frequency of each word, it does capture the 
# overall distribution and semantic information present in the text. Similar to BoW, this approach 
# disregards word order and focuses on the presence and representation of words. However, it should be
# noted that averaging embeddings may preserve some semantic relationships between words, which BoW 
# representations might not capture as effectively.
# 
# Side note: 
# Bag of Words (BoW) is a commonly used technique in natural language processing (NLP) for representing text
# as a numerical feature vector. It disregards the order and structure of words in a document and focuses only
# on their occurrence and frequency.
# In the BoW model, a document or a piece of text is represented as a "bag" (unordered set) of words, where 
# each word is treated as an independent feature. The presence or absence of words in the document is encoded
# as a binary value (0 or 1), and the frequency of each word is often used as the value in the feature vector.
# 
# Here's a step-by-step overview of the BoW process:
# 1. Tokenization: The text is split into individual words or tokens. Punctuation marks, whitespace, and other
#    special characters are usually removed or treated as separate tokens.
# 2. Vocabulary Creation: A vocabulary is created by taking all unique words from the entire corpus 
#    (collection of documents). Each unique word is assigned a unique index or position in the vocabulary.
# 3. Vectorization: Each document is represented as a feature vector, typically a one-hot encoding or a count
#    vector. In a one-hot encoding, each word in the vocabulary corresponds to a binary feature, and the vector
#    contains 1s in the positions where the word occurs and 0s elsewhere. In a count vector, the value at each 
#    position represents the frequency of the corresponding word in the document.
# 4. Classification or Analysis: The resulting feature vectors can be used as input to machine learning models
#    for tasks such as text classification, sentiment analysis, document clustering, or information retrieval.
# Needless to say, BoW has some limitations. It does not capture the semantic meaning or context of words, as
# it treats each word independently. It also ignores the grammar and word order. However, BoW is simple, 
# efficient, and can be a useful baseline representation for various NLP tasks.
# 
# so to recap one more time, we are basiaclly treating each timestep/sequence dimension, as a word, so we have
# 8 tokens/words, and we are averaging them (we are infact averaging their embeddings, but thats obvious!)
# so we got ourselves bow representation!
# now back to our discussion, if we  try to visualize the results we get 
print(x[0])
print(results[0])
# prints:
# x[0]=
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
#         [ 6.,  7.],
#         [ 8.,  9.],
#         [10., 11.],
#         [12., 13.],
#         [14., 15.]])
# results[0]=
# tensor([[0., 1.],
#         [1., 2.],
#         [2., 3.],
#         [3., 4.],
#         [4., 5.],
#         [5., 6.],
#         [6., 7.],
#         [7., 8.]])
# if you look closely, you'll notice that each row, contains the mean of all the rows before it
# consider the first 3 rows in x[0], 
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
# now refer to the 3rd row in results[0] which is :
#         [2., 3.],
# likewise, consider the last row in results which contains the average for all the rows in x:
# 0+2+4+6+8+10+12+14 = 56 which when divided by their count, 8, results in 7, 
# and this is the same for the second column, thus we get:
# results[0,7] = [7., 8.]])
# this all good but the problem is using for loops to calculate this is very inefficient, it
# happens that this operation can be efficiently calculated using matrix multiplication.
# lets learn this trick using an example: 
# suppose we have the following as the input: 
a = torch.ones(size=(3,3))
b = torch.randint(0,10,size=(3,2)).float()
c = a@b
print(f'{a=}')
print(f'{b=}')
print(f'{c=}\n----')
# prints
# a=tensor(
#        [[1., 1., 1.],
#         [1., 1., 1.],
#         [1., 1., 1.]])
# b=tensor(
#        [[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c=tensor(
#        [[20., 13.],
#         [20., 13.],
#         [20., 13.]])
# nothing fancy here, we have matrix multiplication, the first row of 'a' is dot-producted by first col
# of 'b', then sumed, it makes up the first col of first row in c. likewise the first row of 'a' dot 
# the second col of 'b', then summed the results, makes up the second col of first row in c. and this goes on for the rest of the matrixes. this is
# what we learned back in higheschool, so what is it exactly that we are learning exactly?
# if you look closely, you'll notice that, the c cols are actually the sum of all the rows in b!
#        [9]
# b[:,0]=[5] 
#        [6]
# 9+5+6 is 20! likewise, 
#        [3]
# b[:,1]=[5] 
#        [5]
# 3+5+5 is 13!
# hence c = [20., 13.] which is repeated obviously because the second and third rows of a are all 1s as well.
#           [20., 13.]  
#           [20., 13.] 
# I guess you are now starting to get where we are going with this, if we can some how alter the 'a' matrix,
# we may very well be able to achieve our goal! how you may ask? the answer is using torch.tril!
# torch.tril() is a function that returns a matrix from a given tensor, so that half of it set to zero,
# basially it creates a triangular tensor, where the right half is just zeros! lets see how it works, 
# lets apply it on 'a'
a_tril = torch.tril(a)
print(f'a_tril:\n{a_tril}')
# it prints
# a_tril=tensor(
#        [[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# as you can see the the right half is set to zero and we are left with a triangle shape of 1s! on the left side
# now if we do a@b this time we get:
c = a_tril@b 
print(f'b:\n{b}')
print(f'c:\n{c}')
# a_tril:
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[ 9.,  3.],
#         [14.,  8.],
#         [20., 13.]])
# now if you look closely, you'll notice that, this time, each row in c, is effectively the sum of the previous
# rows in b, like the first row of 'a' is only 1 in the 0ths column, so the first row of b is copied in c intact
# (workout the math and see why). 
# the second row in 'a', now has two 1s in col 0 and 1 respectively, which effectively translates to summing the first
# two rows in b. (9+5 =14, 3+5=8). likewise, the third row in 'a' is all 1s, signfigying all rows in b will
# be summed which gives us (9+5+5=20, 3+5+5=13). 
# so basically we are doing sums here, becasue our tensor a is all ones. so if we want to somehow calculate the
# average, instead of sum, we can easily change 'a' by normalizing it so that the each row sums to 1 (i.e. all cols
# sum to 1), this way the end result will be the average (becasue the 'b' is multiplied by a fraction/scale and then summed)
# so if we scale 'a' by the sum of all its columns, we should get average instead
a = torch.ones(size=(3,3))
a = torch.tril(a)
a = a/a.sum(dim=1, keepdim=True)
print(f'a:\n{a}')
# a:
# tensor([[1.0000, 0.0000, 0.0000],
#         [0.5000, 0.5000, 0.0000],
#         [0.3333, 0.3333, 0.3333]])
#
# note that, now each row, sums to 1. the first row, the first element is 1, because the rest are 0s
# but in the second row, as there are two 1s, the probabality is divided between the two, each being 0.5
# likewise, in the third row, as there are 3 1s, the probablity is divided between all of them, making each
# to have the value 0.33
# and now if we try to multiply them, we get average as the result:
c = a@b 
print(f'calculating average:')
print(f'b:\n{b}')
print(f'c:\n{c}')
# we get 
# calculating average:
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[9.0000, 3.0000],
#         [7.0000, 4.0000],
#         [6.6667, 4.3333]])
# we see that, each row in c, is the average of all the rows before it. 
#
# so using this trick, we can take the incremental average of any matrix we like. 
# now that we learned the trick, lets go back and implement the bows for loops using this techique !
# prevbiously we had : 
# results = torch.zeros(size=(B,T,C))
# for b in range(B):
#     for t in range(T):
#         results[b,t] = x[b,:t+1].mean(0)
# which calculated the average for sequence dimensions (tokens) incrimentally
# lets do this now 
# we want a TxT weight becasue we want to average T timestep/tokens
weight =  torch.tril(torch.ones(size=(T,T)))
# remember to set keepdim=True, or otherwise, as we sum along dim=1, we lose that dim
# (it collapses, and then the broadcast will be wrong, it will infact make each column 
# have one probablity instead of each row, which is the exact opposite of what we want)
weight = weight/weight.sum(dim=1, keepdim=True)
print(f'weight:\n{weight}')
# and now we need to multiply this by x! 
bow_results= weight@x
print(f'bow_results:\n{bow_results}') 
# which prints 
# bow_results:
# shape: torch.Size([4, 8, 2])
# tensor([[[ 0.0000,  1.0000],
#          [ 1.0000,  2.0000],
#          [ 2.0000,  3.0000],
#          [ 3.0000,  4.0000],
#          [ 4.0000,  5.0000],
#          [ 5.0000,  6.0000],
#          [ 6.0000,  7.0000],
#          [ 7.0000,  8.0000]],

#         [[16.0000, 17.0000],
#          [17.0000, 18.0000],
#          [18.0000, 19.0000],
#          [19.0000, 20.0000],
#          [20.0000, 21.0000],
#          [21.0000, 22.0000],
#          [22.0000, 23.0000],
#          [23.0000, 24.0000]],

#         [[32.0000, 33.0000],
#          [33.0000, 34.0000],
#          [34.0000, 35.0000],
#          [35.0000, 36.0000],
#          [36.0000, 37.0000],
#          [37.0000, 38.0000],
#          [38.0000, 39.0000],
#          [39.0000, 40.0000]],

#         [[48.0000, 49.0000],
#          [49.0000, 50.0000],
#          [50.0000, 51.0000],
#          [51.0000, 52.0000],
#          [52.0000, 53.0000],
#          [53.0000, 54.0000],
#          [54.0000, 55.0000],
#          [55.0000, 56.0000]]])
# which gives us the same results as we expected.
# one more thing before we continue on, note that the weight matrix is TxT while 
# the input is (BxTxC). (T,T) and (B,T,C) are not compatible, so what happens is
# that (T,T) is reshaped and a batch dimension is added to (T,T),making it (1,T,T)
# and then this is broadcasted along the batch dimension (replicated) to become 
# (B,T,T), then this will be multiplied by the (B,T,C)( note that at this stage
# a@b will be a batch multiplication operation, if you set the batch dimension aside, youll
# see that the rest of the dimensions match up, we have (T,T) and (T,C) which will
# result in (T,C). now if we add the batches back in, we will endup with (B,T,C)
# so in practice, the multiplication is done B times and then results are stacked.
# the first batches will multiply each other (T,T)x(T,C) = (T,C)
# the second batches will then multiply each other as well, getting another (T,C)
# and this goes on until we get (B,T,C))
#
a = torch.arange(0,9).view(3,3)
# a = torch.ones((3,3)).long()
b = torch.arange(0,30).view(2,3,5)
print(f'a:\n{a}')
print(f'b:\n{b}')
c = a@b 
print(f'c=a@b:\n{c}')
# is equivalent to 
# add a batch dimension to a
a = a.view(1,*a.shape) # or a.unsqueeze(0)
print(f'a with batch dim:\n{a.shape=}')
# replicate along the batch dimension
a = torch.cat(tuple(a.clone() for i in range(len(b))), dim=0)
print(f'{a.shape=}')
print(f'a(after replication along dim=0):\n{a}')
print(f'b:\n{b}')
# and now we have a case of batch-multiplication, the batch is the same 
# and sub tensors also are compatible, we have (T,T) and (T,C) so we now
# individually multiply each sub-tensor
c2 = torch.zeros_like(b)
for i in range(b.shape[0]):
    c2[i,...] = a[i]@b[i] # (T,T) x (T,C) -> (T,C)
# and ultimatley the result will have the shape (B,T,C)
print((c==c2).all())
# so to recap this 
# When performing the matrix multiplication `c = a @ b` with the given tensors, 
# several broadcasting steps occur to align the dimensions properly. 
# Here's a how it happens:
# 1. Tensor a has the (shape: 3, 3):
# 2. Tensor b has the (shape: 2, 3, 5):

# 3. They are not compatible so we broadcast tensor a to match the shape of b. 
# tensor a is expanded to (1, 3, 3) to have a batch dimension, and replicated 
# along dim 0 to result in (2, 3, 3). now both tensors have a batch of 2, and 
# if we put aside the batch dimension for a second, we'll notice that the rest
# of the shapes are compatible (3,3) and (3,5). so when we bring back the batch 
# dimension, we see we have a case of batch multiplication, that is we have 2
# sets of compatible tensors that need to be multiplied together. so we use a 
# simple for loop, to do multiplication, and stack the results and we are done!
#
# now back to our discussion. as we just saw, the traingualr shape in our weighted
# sum matrix, allows that each token at t dimension, can only interact with the tokens
# before it.
# there is another way of implementing the same thing but a bit differently
# if you look closely you can see that, we are dealing with probablities, so
# we may verywell use softmax to simplify this further.
# first lets create our triangular weight matrix
tril_tensor = torch.tril(torch.ones(size=(T,T)))
# now lets create a mask
weight = torch.zeros_like(tril_tensor)
# now lets mask all the zeros to -inf, so when we do softmax, all those -infs
# become 0. (recall that exp^-inf is 0! while exp^inf is inf! so its important to 
# set -inf (and also exp^0 is 1))
# so effectively what happens here is that, we setting each entery in trail_tensor
# with zero to -inf, and leave the rest as zeros. when this tensor goes through softmax
# the 0s will be 1s and -infs will be 0s before they are normalized, when they are normalized
# the probablity of 1 will be split between all the enteries with the value of 1.
# so the first row has a single 1, so it will be 1.0, the second row has two 1s, so
# each one will take 0.5, the third will have 3 1s, so they each will become 0.333
# and so on.
weight=weight.masked_fill(tril_tensor==0, -torch.inf)
# now calculate the probs for each row, treating all cols as probs so their sum is 1
weight = weight.softmax(dim=-1)
bow_results2 = weight@x
print(f'{torch.all(bow_results==bow_results2)}')
# so you might ask, why would we want to make things more complicated like this to only
# achieve what we already achieved pretyy efficiently before?
# the answer is, this approach provides us with a flexibility that the previous ones
# wouldnt provide us withs. if you think about it for a moment, you'll notice that
# the 'weight' matrix can essentially be anything and not just zeros! 
# we have decoupled it from tril, which's job is to set the right half of the tensor
# to zero, so we actually get the expected behavior later on.(which is setting a constrain really (more layer on))
# if you havent yet figured it out, the 'wieght' matrix, can be truly a weight matrix
# which can show different strengths for each token, basically it can learn the interations
# between tokens and manifest them. currently it is us who sets it to all zeros, so we get
# uniform probablities, which inturn manifest itself as an average. but what if instead of 
# all zeros, we actually learn the values from the data itself? that would make sense, and 
# make it so that each token, can have a different connection(strength) to any other tokens
# now, and thus build semantic/meaningful relation. this is inafct what we are after. we want
# for tokens to learn associations and relations with other tokens based on the data present
# in the dataset, having uniform weight like what we initially did really is a far cry from
# what we intend, and therefore, we opt in to use this new approach that allows us to actually
# exploit this new capability. 
# This is infact the problem that attention solves, that is gathering
# information from the past but in a data driven manner.(side note, we are not limited to 'past' 
# information only per say, in this example, this is the case however, we will explain this 
# in more detail)), we will see how attention does this exactly in a moment. 
#
# but before we jump into attention implementation also note that the tril part, infact is a hard-constrain here that prevents tokens from
# the past from interacting with the tokens from the future. 
# so to recap here, basically the idea is, this triangular form, allows us to have 
# weighted aggregations of past elements. each element in the lower triangular part, 
# specifies, the degree by which it plays a rule in the said outcome. 
# that is how much of each element gets to fuse into this specific position(i.e. current token's)
# so now lets incorporate attention into our model
# Attention does its job by using two vectors called, key and query.
# basically every single token, emits two vectors called key and query, the query verctor
# as the name suggests, implies, what we are looking for, and the key vetcor, again as the
# name suggets, implies, the contents, what it contains. 
# the way we get our 'weights' for these tokens is we simply dotproduct them together, 
# the ones that yield a high output/value, signify they are related positively.
# so our query is dotproducted with all the token's(keys) and the outputs reveal their 
# closeness/relevancy/similarity so to speak(if they align so to speak, they result in larger number)
# so effectively, the ones with higher number, are more similar to the query, the highest, 
# obviously having the most relavancy/similarity to the query.s
# so lets implemenet this
# #%%
# we want to implement a single attention head, we can later use this to create 
# multi-attention-head which is basically several single attention heads working in 
# parallel. so how do we implement one! 
# first lets create some random input 
torch.manual_seed(255)
# lets specify the batch, context_size and vocab_size
B,T,C = 4,8,32
# lets create an input 
x = torch.randn(size=(B,T,C))
# we said that attention works with two vectors, key and query, lets implement them
# we can use nn.Linear to implement them but beore that, what are the dims of such vectors,
# we are dealing with text and thus our inputs are tokens/vocabs, so the first dim would be 
# our vocab_size to account for all tokens, the next dim, is sth called a head_sizes, which 
# specifies the size of the key/query output usually 16 is used a lot for head size so we 
# use that as well
head_size = 16
# lets not forget to set bias=False, so what it does is exactly dotproduct 
# (actually, some people still leave the bias enabled! 
# so we will test both cases later and see if it actually matters! and if so to what extend! ) 
key = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
# now lets get the output
k = key(x)      # shape : 4,8,16 or (B,T, head_size)
q = query(x)    # shape : 4,8,16 or (B,T, head_size)
 # so the dims arent compatible for batch-multiplication, so we need a transpose
 # k.T wouldnt work becasue we have a batch-dim, so instead we use .transpose and
 # explictily specify the dims we want to be transposed. 
 # this will result in 4,8,16 by 4,16,8 which would give us 4,8,8 which is (B,T,T)
 # really as the result
 #! check transpose result is it okto use 2,1 or -2,-1 or 1,2
weight_raw = q@k.transpose(2,1)
# now lets for a moment think about what is happening here, the key and query are applied
# on the input and each return an output of (B,T,head_size), they are in fact, processing
# all the tokens in the input, individually, simultaneously, all the same time. so each 
# token is both a query, and a key, and when we do a dotproduct, we are basically telling 
# it to reveal the relation/similarity/relevance of every token with every other tokens.
# and as we explained earlier, this is infact our weight matrix (which was initially zeros)
# but is now learned from the data!
# print(f'weight_raw\n{weight_raw}')
# now we can apply constrain on it so that tokens can only communicate with the past so 
# we use the tril trick now!
tril_constrain = torch.tril(torch.ones(size=(T,T)))
raw_wieghts_masked = weight_raw.masked_fill(tril_constrain==0, float('-inf'))
# apply sotmax to get probablity for each token
weight = raw_wieghts_masked.softmax(dim=2) # remember weight is (B,T,T)
# and finally we can apply our weight on the input (we called it raw for a reason, read on)
# (by the way this is also called self-attention!)
bow_raw = weight@x 
# lets print raw_weights and weights and have some intuitive observations 
print(f'weight_raw\n{weight_raw}')
print(f'raw_weights_masked: {raw_wieghts_masked}')
print(f'weight\n{weight}')
# prints
# raw_weights(unconstrained)
#weight_raw
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00, -1.7903e+00,  1.4650e-01, -1.4147e+00, -1.7452e+00, -4.5249e+00, -1.9763e+00,  1.8833e+00],
#          [-1.3490e+00,  9.4076e-01, -6.9690e-01,  7.9933e-01,  4.7634e-01,  4.1861e-01,  2.3663e-01,  4.5217e-01],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,  9.0092e-01,  1.4083e+00,  2.5488e+00,  1.6350e+00,  1.0048e+00],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01, -1.1213e+00,  2.7562e+00,  7.0136e-02, -2.0337e+00],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,  4.9236e+00,  3.9545e+00, -2.2600e-01],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01, -1.5979e+00,  8.4627e-01],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,  2.2338e+00],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00, -2.1323e+00, -1.3948e+00, -5.6973e-01, -2.7986e-01,  4.3579e+00,  4.8477e-01, -9.7559e-01],
#          [ 8.7786e-01, -2.3352e+00, -2.2988e+00,  1.0322e+00, -1.4231e-01,  5.5233e+00,  1.0819e+00, -2.4722e-01],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,  2.9387e-02, -5.6397e-01, -2.4025e-01, -4.2512e-02, -8.4302e-01],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01, -4.0429e+00, -1.0676e+00, -3.0694e+00, -1.6156e+00],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00, -4.5593e+00, -3.1755e-01, -1.0504e+00],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00, -3.5616e-01, -2.6951e+00],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,  4.9077e-02],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01, -4.7902e-01,  6.7590e-01, -2.3153e-02,  1.9763e-01,  4.4385e-01,  7.2986e-01, -3.8846e-01],
#          [-6.2810e-01, -8.3713e-01,  1.3468e-01, -5.1328e-01, -2.6653e-02,  3.4325e-01, -1.0015e+00, -1.0334e+00],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01, -1.9702e+00, -1.0641e+00,  1.7792e-01,  8.0194e-01, -1.5265e-01],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,  3.6759e-01,  1.0552e+00, -5.2949e-01, -1.7715e+00],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01, -3.0057e+00,  1.3784e+00, -1.2091e+00],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02, -1.4040e+00,  2.3786e+00],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00, -1.6068e+00],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<UnsafeViewBackward0>)
#
# raw_masked_wieghts(constrained):
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.3490e+00,  9.4076e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01,        -inf,        -inf,        -inf,        -inf],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,        -inf,        -inf,        -inf],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01,        -inf,        -inf],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,        -inf],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 8.7786e-01, -2.3352e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01,        -inf,        -inf,        -inf,        -inf],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00,        -inf,        -inf,        -inf],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00,        -inf,        -inf],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,        -inf],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-6.2810e-01, -8.3713e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,        -inf,        -inf,        -inf,        -inf],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01,        -inf,        -inf,        -inf],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02,        -inf,        -inf],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00,        -inf],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<MaskedFillBackward0>)
# 
# weight (final- normalized)
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.1976e-02, 9.0802e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9773e-01, 6.0261e-01, 1.9966e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.4181e-02, 3.4422e-02, 3.7221e-01, 5.7918e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [4.6023e-02, 1.9501e-03, 5.3236e-01, 3.5612e-01, 6.3550e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9494e-01, 1.9195e-01, 2.7619e-01, 2.8253e-02, 1.3648e-01, 1.7219e-01, 0.0000e+00, 0.0000e+00],
#          [4.5214e-01, 9.6007e-02, 7.3108e-02, 5.3035e-02, 2.8887e-01, 8.7621e-04, 3.5963e-02, 0.0000e+00],
#          [6.8046e-02, 5.1605e-01, 5.0474e-03, 1.5304e-02, 2.9133e-01, 3.8823e-04, 1.3775e-02, 9.0057e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.6132e-01, 3.8675e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.1870e-01, 3.3895e-01, 4.4235e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.2894e-02, 6.5398e-01, 2.4345e-01, 7.9674e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [3.7332e-02, 6.5690e-01, 7.8427e-02, 5.1206e-03, 2.2222e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.3611e-01, 1.7995e-02, 2.1887e-02, 1.1900e-01, 1.8219e-02, 6.8680e-01, 0.0000e+00, 0.0000e+00],
#          [9.8946e-02, 1.4120e-02, 1.8065e-02, 9.5010e-03, 3.2130e-01, 9.9891e-02, 4.3817e-01, 0.0000e+00],
#          [2.3306e-02, 3.5621e-02, 9.5163e-02, 1.5801e-01, 1.1530e-02, 1.7183e-01, 4.1141e-02, 4.6340e-01]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.5207e-01, 4.4793e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.8708e-02, 6.5816e-01, 2.4313e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.8422e-01, 9.1987e-02, 1.2557e-01, 5.9822e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.0870e-01, 8.3642e-02, 4.8896e-02, 3.2723e-01, 3.3154e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.4141e-02, 7.9271e-02, 2.1893e-01, 2.2608e-01, 2.3782e-01, 1.4376e-01, 0.0000e+00, 0.0000e+00],
#          [7.8726e-02, 1.1815e-01, 3.7190e-01, 1.2607e-02, 3.6387e-02, 1.1790e-01, 2.6434e-01, 0.0000e+00],
#          [5.6646e-02, 7.4575e-02, 6.8217e-02, 3.1568e-02, 5.9024e-02, 1.1073e-01, 3.1662e-02, 5.6757e-01]]], grad_fn=<SoftmaxBackward0>)
#
# as you can see, our weight is initialized for each batch based on the input data, and each token has its own
# weight, that is they are not uniform!, to get a better understanding lets consider the first batch of 
# raw-weights[0], raw_masked_wieghts[0] and weight[0]:
#
# raw_weights[0]
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
#
#
# raw_masked_wieghts[0]
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
# 
# weight[0]
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
# take the last token, which 8.6020e-02, notice this token, not only knows its content and own position in 
# the sequence(its the 8th token after all) but also knows which tokens comes before it and how much its 
# related to any of them.(basically when it knows which token comes before it, it creates its own query so
# to speak, and talks to every single previous token to find about which ones are more or less relavent to
# it and to what extend.) 
# For example,lets say, (since we are dealing with character level text generation!), the last token is a 
# vowel and says hey im a vowel and im looking for everyone else thats a vowel! and uses query to talk to 
# other tokens(keys). and intrestingly another token (lets say number 4) says im a vowel as well, and the 
# response is thus generate a larger number.(this is not a good example, words in sentence would make more sense
# !give a better example!)
# in our example, For the last token, it seems the 5th and 3rd token are particularly intresting/relavent/important,
# followed by the 7th token.
# likewise, this happens for every token, i.e. token number 4, says the same thing and searches in its past toekns
# so on and so forth! 
# so what happens next is that when we get a high relevancy scale /response in the weights, like we just described
# when we do a softmax it will assign a large probablity to them, and this instructs the network that, we need more
# information from them, effectively allowing for aggregating a lot of their information into our position(lets say e.g. 8th token. 
# and we happen to learn more about them this way.
# now in practice, we are not intrested in aggregating the x raw values per say, rather we want their information, 
# so instead of just using the raw values of x, we instead use a representation of them, so to speak. 
# this is achieved using a third vector known as, 'value' and is the last vector we use. 
# !explain when we say vector, note that, we are talking from the prespective of a token, (every token has some vector 
# !of information and it gets to aggregate information via a weighted sum from all the tokens that point to it, and it
# )# !in practice we use a matrix, and hence the linear module to implement this to run the operation for all tokens in parallell. 
# just like the key and query, we set its bias to False, so we only get a simple vector,
# and a dotproduct output (again this is the intuition, but how much adding a bias would
# affect this we will see later, for now we stick to the default no bias version)
value = torch.nn.Linear(C,head_size, bias=False) # produces (B,T,head_size) just like the other two key,query vectors
x_processed = value(x)
# this is not yet final final, we still need to do one more thing (read on!)
bow_final = weight@x_processed
#
#!check also note that, as we previously once pointed out, attention is a communication mechanism between tokens, and 
# by default there is nothing in this mechanism that provides a notion of space/position for tokens involved, i.e.
# by default these tokens/nodes/points, dont have any idea about where they are or how they are positioned (with resepect
# to others) etc, so we need to encode this information as well.(to better visualizing it, consider each token as 
# a node in a directed graph, each node has a connection to another node, (for example each node has a connection to
# itself, and another connection to other nodes,etc) and as you can see there is no notion of space here,the attention
# simply acts on a set of vectors in this graph, and thats why we need to encode them positionally as well, so they 
# have information about their position with regards to other tokens. 
# !check (compare this to the convolution case and images
# or even text where the spatial aspect of data is preserved, but in attention, as we just stated, there is no notion
# of position, its just a set of individual vectors being operated on)-note that the connection between nodes, do give
# us a sense of structure, but it may not be enough to infer the underlying semantic, imagine a case, where a word
# for example has several meaning, and may very well have high relevancy to some tokens at the same time, but without
# additional positional information, an ambiguous semantic can be infered between the tokens involved, however when
# positional information is also present, such ambiguity can be avoided)
# also note that, in attention, samples do not interact with each other at all. when we have a batch of 4, each sample
# is processed in isolation, but in parallell to other samples, based on our graph example earlier, we would have 4 
# graphs of 8 nodes for example for each input sample. 
#
# we said earlier that what we implemented here is known as self-attention, the reason it is called self attention
# is that the key and query and values are applied on the same input(the use the same source!), and hence the name,
# self attention.
# also note that, in our specific case, tokens/nodes are can not communicate with the future nodes, but in general
# this constraint can be removed (and infact is removed/not implemented for some applications) where its benificial
# to be able to communicate with all the tokens. one example is sentiment analysis, where you want all the tokens to
# able to communicate with eachother so you can get an accurate analysis. and for this case, we would use an encoder
# block, which is basically what we have here, minus the constraint section(tril/mask part), what we have implemented
# here is called a decoder block, where we are decoding bunch of tokens, and it makes sense that the previous tokens
# do not comunicate with the future ones(becasue they would give the answer! and it defeats the whole purpose here!),
# becasue its a given that only the previous tokens must be used to predict the future/next token, hence the filtering/constraint part to prevent tokens from 
# comunicating with the future nodes/tokens.
# !so far we explained about the self-attention, which we saw, is called that way solely for the fact that key, query
# and value use the same source. the attention mechanism as we briefly pointed out, is much more general and can be
# used in different ways. one of such ways, is what is used to create sth called cross-attention. 
# cross-attention basically refers to the case where we have an encoder/decoder blocks, in which the queries come from x
# but the key and value come from an external source and sometimes from the encoder block. so cross-attention is used
# when theres a separate source of information we would like to pool from and use it as well.
#
# so far we implemented the attention based on the original paper(there are some differences we get to later on)
# except the part where we need to divide by the sqrt of the head_size. that is called scaled-attention
# so lets talk about this, and see why its needed. 
# the reason we add this so called 'scale' to our computation, is that, without it, the probablities will be saturated
# and when we add this term to the mix, it will make the 'weight' matrix to be 'unit variance', when Q and K are unit variance
# and this allows softamx to stay diffuse and not saturate too much.
# in other words, if we simply multiply key and query like that, the variance of the resulting weight matrix will be
# around the head_size instead of 1 which is bad and makes optimization really hard.
# to see this effect consider the following example
k = torch.randn(size=(B,T,head_size))
q = torch.randn(size=(B,T,head_size))
w_unscaled = q@k.transpose(-2, -1) 
print(f'{k.var()=}')
print(f'{q.var()=}')
print(f'{w_unscaled.var()=}')
# now add the 1/sqrt(head_size)
w_scaled = q@k.transpose(-2, -1) * head_size**-0.5 
print(f'after applying 1/sqrt(head_size)')
# makes the weight variance 1!
print(f'{w_scaled.var()=}')
# why is it important? if you recall, the weight matrix is fed into softmax, so its really important, especially during
# initialization that weight matrix be fairly diffuse, if we look at weight matrix here, we'll notice that they are now
# fairly diffuse 
print(f'weight_scaled[0]:\n{w_scaled[0]}')
# prints 
# weight_scaled[0]:
# tensor([[ 0.3972, -2.0957, -0.5396, -1.4860, -0.8393,  0.6273, -0.0157,  0.6284],
#         [ 1.3588, -0.0440,  2.1040, -0.2796,  1.7797,  1.4362,  1.3843,  0.0368],
#         [ 0.8109,  1.7400,  1.3579,  0.2097,  1.7152, -1.1242, -0.4349, -0.5690],
#         [ 0.0513,  2.8303,  0.7332,  0.0041,  0.9688, -1.6174, -1.4255,  0.2869],
#         [-0.7819, -0.6691, -1.4017, -0.4155, -0.8568, -0.2704, -0.9453,  0.3763],
#         [ 0.4092, -0.3079,  0.8472, -1.1753, -0.7699,  0.5091,  0.6385,  0.9677],
#         [ 0.3043, -2.1402, -2.2582, -0.3573, -1.2838, -0.0260, -0.0513,  0.9151],
#         [ 0.4180, -2.8353, -0.6435,  0.4842, -3.0202,  1.7690,  1.8837, -0.4393]])
#
# when softmax is applied:
print(w_scaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.8026, 0.1974, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1901, 0.4814, 0.3285, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.0499, 0.8038, 0.0987, 0.0476, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1989, 0.2226, 0.1070, 0.2869, 0.1845, 0.0000, 0.0000, 0.0000],
#         [0.2148, 0.1049, 0.3329, 0.0440, 0.0661, 0.2374, 0.0000, 0.0000],
#         [0.3027, 0.0263, 0.0233, 0.1562, 0.0618, 0.2176, 0.2121, 0.0000],
#         [0.0901, 0.0035, 0.0312, 0.0962, 0.0029, 0.3478, 0.3901, 0.0382]])
#
# 
# now compare it with the unscaled weight : 
print(f'weight_unscaled[0]:\n{w_unscaled[0]}')
# weight_unscaled[0]:
# tensor([[  1.5889,  -8.3829,  -2.1583,  -5.9440,  -3.3573,   2.5092,  -0.0629,  2.5135],
#         [  5.4350,  -0.1759,   8.4159,  -1.1183,   7.1189,   5.7446,   5.5373,  0.1472],
#         [  3.2435,   6.9600,   5.4317,   0.8388,   6.8607,  -4.4969,  -1.7396, -2.2761],
#         [  0.2051,  11.3214,   2.9327,   0.0165,   3.8754,  -6.4696,  -5.7019,  1.1475],
#         [ -3.1278,  -2.6763,  -5.6069,  -1.6621,  -3.4272,  -1.0816,  -3.7811,  1.5053],
#         [  1.6368,  -1.2317,   3.3888,  -4.7012,  -3.0795,   2.0364,   2.5541,  3.8710],
#         [  1.2171,  -8.5610,  -9.0327,  -1.4293,  -5.1353,  -0.1038,  -0.2053,  3.6603],
#         [  1.6719, -11.3411,  -2.5740,   1.9369, -12.0810,   7.0760,   7.5347, -1.7573]])
#
# when softmax is applied:
print(w_unscaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [9.9636e-01, 3.6442e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.9592e-02, 8.0566e-01, 1.7475e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.4865e-05, 9.9975e-01, 2.2738e-04, 1.2309e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2943e-01, 2.0328e-01, 1.0848e-02, 5.6050e-01, 9.5940e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2012e-01, 6.8207e-03, 6.9264e-01, 2.1236e-04, 1.0748e-03, 1.7913e-01, 0.0000e+00, 0.0000e+00],
#         [6.3261e-01, 3.5856e-05, 2.2371e-05, 4.4856e-02, 1.1023e-03, 1.6883e-01, 1.5254e-01, 0.0000e+00],
#         [1.7351e-03, 3.8710e-09, 2.4850e-05, 2.2616e-03, 1.8472e-09, 3.8572e-01, 6.1020e-01, 5.6236e-05]])
#
# as you can see, some values are very negative, while others are very positive, for example, we have both 11 and -11
# and this is a recipe for disaster! the reason it is a problem is that, with softamx, when numbers take very positive
# and nagative values, softmax actually converges towards one-hot-vector. basically the values shrink towards the max.
# this can be seen the above, but the example below should demonstrate it more clearly.
# lets apply softmax on a list of numbers that are close to each other, we see the probablities are difuse and normal
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2]),dim=-1)}')
# we get a diffused probablity out of softmax
# tensor([0.2562, 0.3822, 0.1717, 0.1898])
# however if we increase the magnitude of the numbers(sharpen them) (and thus the difference between them) by like multiplying by 
# a number like 8 (just to simulate the effect here and so we can compare it with the previous case)
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2])*8,dim=-1)}')
# we see that the softmax, starts to sharpen towards the max, shrinking others except the 
# largest number, and effectively
# converging toward a one-hot-encoded vector.
# tensor([0.0390, 0.9559, 0.0016, 0.0035])
# so we dont want these values to be extreme, especially during initialization or otherwise, softmax will be way too picky!
# and we are basically aggregating the information from a single node instead of multiple ones (becasue one has the largest
# nvalue, its as if our sequence length is 1! and we lose access to the wealth of information the past history offers)
# so we want the probablities to be diffuse and not peaked like the second example here.
# so this scaling is used to retain the variance at a good value especially at initialization.
# so now that we are finally finished the self attention head, lets implement it as a module and incorporate everything
# we just discussed here.
class AttentionHead(nn.Module):
    def __init__(self, context_size, embd_size, head_size=16, use_bias=False) -> None:
        super().__init__()
        # we need context_size or block_size for creating the tril constrain
        self.context_size = context_size
        # we want this as the input dim for our key,query and value, this is 
        # infact the input_dim, since we plan on using the embeddings, we named
        # it embd_size
        self.embd_size = embd_size
        # head_size is the output dimension of our attnetion
        # !note that usually the head_size is equal to embd_size
        self.head_size = head_size
        self.key = nn.Linear(embd_size, head_size, bias=use_bias)
        self.query = nn.Linear(embd_size, head_size, bias=use_bias)
        self.value = nn.Linear(embd_size, head_size, bias=use_bias)

        # use buffer for trail, since this is a buffer, we can use the self.register_buffer which is
        # inherited from nn.Module class, to add it as buffer to the module, so it can be saved in the
        # state_dict when we save the model.  tril doesnt change, and is not updated(its required_grad is false),
        # so it being saved in state_dict doesnt matter to us, but to demonstrate this feature of pytorch, 
        # we are using it, and its a good practice, since pytorch knows how to deal with it and wont include it
        # in the computaion graph anyway!
        # 
        # This is typically used to register a buffer that should not to be considered a model parameter. 
        # For example, BatchNorm's running_mean is not a parameter, but is part of the module's state. 
        # Buffers, by default, are persistent and will be saved alongside parameters. This
        # behavior can be changed by setting persistent to False. The only difference between a persistent
        # buffer and a non-persistent buffer is that the latter will not be a part of this module's state_dict.
        # tril allows us to impose constrain by creating a lower traingualr matrix and setting
        # the rest entries to zero, effectively preventing tokens of the future from communicating with the past
        # each token can only communicate with the previous tokens that came before it.
        self.register_buffer('tril', torch.tril(torch.ones(context_size,context_size)))

    def __call__(self, inputs:torch.Tensor) -> torch.Tensor:
        #! check the shapes!! 
        B,T,C = inputs.shape # 4,8,16
        # print(f'{inputs.shape=}')
        # create the weight by using k,q,v
        k = self.key(inputs)    #! (B,T,C) or (B,E,16)? (4,8,16)
        q = self.query(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        v = self.value(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        # create the weight matrix and scale it by 1/sqrt(head_size) to keep weight unit variance 
        weight = q@k.transpose(-2,-1)* self.head_size**-0.5 # !(B,E,E)
        # weight_C = q@k.transpose(-2,-1)* C**-0.5 # !(B,E,E)
        # print(f'weight.var: {weight.var().item():.4f}')
        # print(f'weight_C.var: {weight_C.var().item():.4f}')
        # print(f'x:{tuple(inputs.shape)} k:{tuple(k.shape)} q:{tuple(q.shape)} v:{tuple(v.shape)} w:{tuple(weight.shape)} hs:{self.head_size}')
        # apply the tril constrain - (this makes this a decoder block!)
        # important note: notice we used tril[:T,:T] and not simply tril
        # this is because, when the input has a small context_size < self.context_size
        # like when we wantto generate inputs with sth like zeros((1,1)) which says
        # there is a single token (context_size 1) of 0 as the begining of the sequence
        # when this is input, the tril by default makes a context_size,context_size matrix
        # which will be different than the input context_size which is 1 e.g. or 2 e.g.
        # and it will fail becasue our weight would be 1,1,1 or 1,2,2, but trail is 1,8,8
        # and clearly this will cause an error. for this reason, we always create the 
        # tril check dynamcally by explicitly specifying the context_size based on the 
        # current input context_size so in case the context_size is smaller, tril is resized
        # dynamically accordingly. 
        # print(f'tril[:T,:T]==0: {(self.tril[:T,:T]==0).shape}')
        # print(f'tril==0: {(self.tril==0).shape}')
        weight = weight.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        # weight2 = weight.masked_fill(self.tril==0,float('-inf'))
        # print(f'{weight.shape=}')
        # print(f'{weight2.shape=}')
        # note that we are using batch, so instead of hardcodin 2,
        # we use -1 to refer to the last dim
        weight = weight.softmax(dim=-1)
        # finally apply the weight on the v
        bow = weight@v  # !(B,E,16)
        return bow        

# now lets add this to our model 
        
# so to recap, we first calculated the relavancy between all tokens against eachother, then constrained them so that 
# each token can only use the information from/interact with its past tokens. then since we needed probablity distribution so
# we then normalized it and then used that to pickout which tokens information(in the past) to aggregate/use with the inputs to 
# achieve our goal.(which is to predict the next character based on everything seen so far!)
# we dont want to aggregate inputs value (with respect to the weight matrix), we instead would like to use their representation
# so we use a new layer to do this, its called value, and we instead use its output instead of inputs raw value.
#
# 
#
# before we jump in and add the attention module, lets review our base model and see how we can 
# improve/prepare it before we incorporate the attention. 
# class BigramModelWithAttention(nn.Module):
#     def __init__(self, vocab_size, embd_size) -> None:
#         super().__init__()
#         self.vocab_size = vocab_size
#         self.embd_size = embd_size
#         # unlike the previous model, lets decouple the final logits from 
#         # the number of embeddings, because we are using attentions, and
#         # we want to have multiple operations inbetween obviously.
#         self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
#         # in order to get the final logits, we need a linea layer at end
#         self.fc = torch.nn.Linear(embd_size, vocab_size)
#        
#     def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
#         out = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
#         logits = self.fc(out)         # has the shape (B,T,C) c is vocabsize
#         loss = None
#         if labels is not None:
#             # recall that crossentropy likes its input to be B,C,T and we are B,T,C
#             # so lets permute and make it happy!
#             loss = F.cross_entropy(logits.permute(0,2,1), labels)
#         return logits, loss
#    
#     def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
#         # lets generate an output as long as num_max_token
#         for i in range(max_token_count):
#             # make sure idx is 2d
#             assert len(idxs) >1, f"idx.shape '({tuple(idxs.shape)})' is invalid. it must have the form (B,T)"
#             # now lets feed it to the model and sample from the probablities it produces
#             preds,_ = self(idxs)
#             # convert to probs 
#             probs = preds.softmax(dim=1)
#             # since we are bigram still, lets only get the last token as the next token predicted!
#             probs = probs[:,-1,:]
#             # now lets sample from it 
#             new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
#             # now concatenate the new token to the previous one and feed it back to the model
#             # for the next round of prediction
#             # also remember that we are creating a sequence, so we concat them at dim=1 to get 
#             # a longer sequence (we are gradually increasing the sequence length from 1 up to
#             # max_token_count)
#             idxs = torch.cat((idxs,new_idx), dim=1)
#            
#         return idxs

# this works fine, however, we can do better. here we just encoded the tokens, but
# as we already explained, attention has no notion of position or spatial structure like conv e.g.
# so we need to also incorporate the position information for each token as well
# since we are using the the attention head, we need more arguments
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, head_size, device, use_bias_att=False) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embd_size = embd_size
        self.context_size = context_size
        # for gpu/cpu acceleration during training
        self.device = device
        # unlike the previous model, lets decouple the final logits from 
        # the number of embeddings, because we are using attentions, and
        # we want to have multiple operations inbetween obviously.
        self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
        # lets now add the positional embedding as well 
        # using this, we try to retain the position embedding for our tokens
        # up to the current token in context_size
        self.position_embd = torch.nn.Embedding(context_size, embd_size)
        # add a self-attention head
        self.head = AttentionHead(context_size, embd_size, head_size, use_bias=use_bias_att)
        
        # in order to get the final logits, we need a linea layer at end
        # since we now have attention before this layer, the output dim of attention which is
        # head_size will be used here
        self.fc = torch.nn.Linear(head_size, vocab_size)
        
    def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
        # lets grab the shapes, since we will be using them 
        B,T = inputs.shape
        token_embeddings = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
        # since we have a position_embedding lets use that as well
        # and notice that we didnt use the inputs, but rather torch.arange(T)
        # this means, for each input, as we process it, we also get embeddings up to
        # the current token count as well, if the inputs has 3 tokens currently, we
        # will creeate position embeddings for 0,1 and 2, and for the next input
        # this continues likewise. this results in (T,E)
        position_embeddings = self.position_embd(torch.arange(T,device=self.device))
        # print(f'pos_embd:{position_embeddings.shape}')
        # and lets add the two embeddings together
        # this effectively gives us, not only the token embeddings(identity)
        # but also its position in the sequence. note that this doesnt really
        # help in a bigram model, but when it comes to attention it really does!
        embeddings = token_embeddings + position_embeddings
        # now lets feed this to attention head
        out_attention = self.head(embeddings)
        # lets feed this embedding to our fc at the end instead
        logits = self.fc(out_attention)         # has the shape (B,T,C) c is vocabsize
        loss = None
        if labels is not None:
            # recall that crossentropy likes its input to be B,C,T and we are B,T,C
            # so lets permute and make it happy!
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss
    
    def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
        # lets generate an output as long as num_max_token
        for i in range(max_token_count):
            # print(f'{i}/{max_token_count}) idxs: {tuple(idxs.shape)}')
            # make sure idx is 2d
            assert idxs.ndim >1, f"idx.shape '({tuple(idxs.shape)})' is invalid({idxs.ndim}). it must have the form (B,T)"
            # now lets feed it to the model and sample from the probablities it produces
            # but since, we now have postional embeddings as well, we can no longer have 
            # more than block_size/contex_size in, becasue if our idx is more than context_size
            # our positional embedding will go out of scope and error out
            # so here we are basically getting as many as context_size
            # note that we dont destroy the idxs! each time we get the last context_size tokens
            # from it and feed it to the model to generate the next token, and keep going
            # if we replace the idxs by sth like idx=idx[:,-self.context_size:], we would
            # only create a sequence of only self.context_size, no matter how many iterations
            # we do, we just repeat the same sequence again and again!
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are bigram still, lets only get the last token as the next token predicted!
            logits = logits[:,-1,:]
            # convert to probs 
            probs = logits.softmax(dim=-1)
            # now lets sample from it 
            new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
            # now concatenate the new token to the previous one and feed it back to the model
            # for the next round of prediction
            # also remember that we are creating a sequence, so we concat them at dim=1 to get 
            # a longer sequence (we are gradually increasing the sequence length from 1 up to
            # max_token_count)
            idxs = torch.cat((idxs,new_idx), dim=-1)
            
            
        return idxs
    
# now lets test this and see if it works 
x,y = get_batch('train',4)
model = BigramModelWithAttention(vocab_size, 
                                 context_size, 
                                 embd_size=16,
                                 head_size=16,
                                 device='cpu',
                                 use_bias_att=False)
# model.cuda()
# x,y = (t.cuda() for t in zip(x,y))
logits,loss = model(x,y)
print(f'{logits.shape=} {loss=:.4f}')
# and now we can train this : 
device='cpu'
batch_size = 32
head_size = 16
embd_size = 16 
context_size = 8
vocab_size = len(vocab_list)
max_iter = 5000
# only to test the effect of bias in k,q,v calculations
use_bias_attn=False
# attention requires much lower lr compared to plain bigram model
lr = 1e-3
model = BigramModelWithAttention(vocab_size=vocab_size,
                                 context_size=context_size,
                                 embd_size=embd_size,
                                 head_size=head_size,
                                 device=device,
                                 use_bias_att=use_bias_attn)

param_count = sum([p.nelement() for p in model.parameters()])
print(f'param count:  {param_count:,}')
print(f'head size:    {head_size}')
print(f'embd size:    {embd_size}')
print(f'context size: {context_size}')

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
model = model.to(device)
# set model to train mode explicitly
model.train()
for i in range(max_iter):
    # get the batch
    x,y = get_batch('train', batch_size=batch_size)
    # feed the model and get the logits
    logits, loss = model(x,y)
    # evaluate the model
    if i%1000==0:
        losses=evaluate_loss(100, device)
        print(f'train: {losses["train"]:.4f}  val: {losses["val"]:.4f}')
    # zeroout_grads
    model.zero_grad(True)
    loss.backward() 
    optimizer.step()
print(f'done!')

# now lets try its output
input = torch.zeros(size=(1,1)).int()
output = model.generate(input, 500).squeeze(0).tolist()
print(f"{''.join(decode(output))}")
# prints
#here are a few tries:
# param count:  3,041
# head size:    16
# embd size:    16
# context size: 8
# train: 4.2551  val: 4.2533
# train: 2.7175  val: 2.7354
# train: 2.5917  val: 2.5681
# train: 2.5299  val: 2.5265
# train: 2.4684  val: 2.4759
# done!

# Wonedimy
# Y:
# ARUS:
# LAnouver; pof th, sthe the mae,
# Wor ut barrioreche savarduncou?

# hiler bathakm,
# I O:
# AK:
# Acat ED:
# et alliro dow wowilou ttherss; whest owur, garsacwe Ie hithaceyidous sou f'ze iwot ry tohien fm.
# h.

# Ar'm-ore tr Fofhou
# Mr ojous omerrastheasncy yous Wowuce whor tu I I:
# Hbeotth spat fel woro bel, aneicudidy ktiferrd thout ngord. th Iid aral, be'nd.

# Pal tels Ms eimu Me myo on I teasthe del,
# AChinout,
# Wowure. pu tcher muss-renon; wingh ocet im
# r Ddr,
# Do ne thu,
# LI ir't
# Operst ry wu

# param count:  5,745
# head size:    16
# embd size:    32
# context size: 32
# train: 4.1374  val: 4.1414
# train: 2.6264  val: 2.6210
# train: 2.5100  val: 2.5092
# train: 2.4434  val: 2.4578
# train: 2.4140  val: 2.4252
# done!

# Thee to an tal my,
# Thake ced arce chart bre le be bizoresf thak des garg
# Lur Wentescaln
# Hooco beins pord sth ds hom span gr; ther. My Why ort thallils ofrof not tos gas hyine ind hand
# Thid One the shond nd,
# AFenosee omn st hy no fe.
# GEwank, iand!
# The sbe qurss yod hem snel.
# CLerle,
# Whow Vou bem!
# Theewourtoave, shougen t:
# Ber hangn toutince mor ed,
# AD Scheapy thatho ABEN:
# AD:
# SUEDithy nde ndel mmy Tongusate hfolit, gintoher, hat gwe tiet what whingimst hpoourde wowopal,
# The by br des qO: I no os,

# try2: decreasing context size
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

#try 3: decreasing embd_size, but keeping context_size 32
# param count:  2,265
# head size:    16
# embd size:    8
# context size: 32
# train: 4.2202  val: 4.2158
# train: 2.9840  val: 3.0093
# train: 2.7667  val: 2.7621
# train: 2.6355  val: 2.6411
# train: 2.6070  val: 2.6077
# done!

# o
# TI:
# BOGot be
# DOus ssh.
# Bea hashren v:
# ANULINII feso sid izeis by tofuso te I be fnowe?
# Wledu ha m, thope w,
# ARUTI land med cu
# SI bve fme
# ARIThilaserothan omyhee
# Oina c;
# UCAngose.
# Terve t thirre my tthunulpthemaf wof.

#  corlur yoNE:
# NHy w te hen cod, tonos vengo veschete tors t ithaimeqlutreqRoosoye chin y h thuasafaous.
# ANCathelo-
# The?
# Shen br uyh sbe ans I:
# Gouthyykey m Authirou ae tovas ber ut lnt:,

# TAMENI sf rousinthsin h'howenthor sg.
# hidandhe waresd!

# I f
# Se loner mrdin gsvette n thest?

# try 4: decreasing head_size, keeping embd_size and context_size the same
# param count:  4,457
# head size:    8
# embd size:    32
# context size: 32
# train: 4.3416  val: 4.3409
# train: 2.7354  val: 2.7333
# train: 2.6400  val: 2.6309
# train: 2.5716  val: 2.5692
# train: 2.5217  val: 2.5182
# done!

# IF
# S:
# To ofldickeens we fofane for o warus
# Herdliion bis, te,
# The ta, d ounor;
# Ad h ank; sor Cito-'d turses tierlllle,
# Whe pen, tint yt Ging sino leles fes hesr sst.
# Than hinean.


# Nofl an udn f hak,
# Wh stharovernes;
# And meno scerr tth Gbl y hy I t stit pand sy, pefif.
# US dik thith be tagag, harlovenppsh!
#  cegshil,
# Wheanst thiges eprine sy wabat tha.

# Wheas
# Wit:
# I
# Thouw'arsangort,
# Giseal?

# Wh; mo tands t's.

# NRNULECADSAxave!

# ANNDIEI Fh Fod meveeas pee be,
# Bes arner,

# Woo t'le.


# VIPORLEOG
# CAOO:

# try 5:
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

# DANGUTTA:
# Bert hatire on the wo A? sewas ker haywins wet.

# mousill akerd fous sther Le fke sth,, bepeere gn ssst I oun amand serewos, cobear song.


# AMEfediengg yit othesshint.

# Anc:
# A must shan ghety ak is, shet.

# MEO:
# Tom yot stths!
# NCIBus hand akt imy berdr,
# Thulll id mere thassut,
# Moureleray, fous mante. QBO decencepous ariveed hart;
# I; bifo wthen chant tht tho hy youn,
# Grolomy, ucel.

# BRIINE ERIYht I arum arw t; odeat ngher bly wehe foru thas sot our:
# Bn; sshiutithemowith her fourpler thant

#
# which looks depressing not gonna lie! but its better than the simple bigram model we 
# built earlier, anyway, can we improve it? certainly! 
# for example we could use dropout on our attention! we could use normalization layers
# and we could use multiple heads instead of just one! which takes us to the next
# subject which is multi-head attention! which is  really nothing except several normal!
# attention blocks run in parallell! 
# so lets implement these and see how much we can improve upon this

results=tensor([[[ 0.,  1.],
         [ 1.,  2.],
         [ 2.,  3.],
         [ 3.,  4.],
         [ 4.,  5.],
         [ 5.,  6.],
         [ 6.,  7.],
         [ 7.,  8.]],

        [[16., 17.],
         [17., 18.],
         [18., 19.],
         [19., 20.],
         [20., 21.],
         [21., 22.],
         [22., 23.],
         [23., 24.]],

        [[32., 33.],
         [33., 34.],
         [34., 35.],
         [35., 36.],
         [36., 37.],
         [37., 38.],
         [38., 39.],
         [39., 40.]],

        [[48., 49.],
         [49., 50.],
         [50., 51.],
         [51., 52.],
         [52., 53.],
         [53., 54.],
         [54., 55.],
         [55., 56.]]])
tensor([[ 0.,  1.],
        [ 2.,  3.],
        [ 4.,  5.],
        [ 6.,  7.],
        [ 8.,  9.],
        [10., 11.],
        [12., 13.],
        [14., 15.]])
tensor([[0., 1.],
        [1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.],
        [5., 6.],
        [6., 7.],
        [7., 8.]])
a=tens

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_head, head_size, embd_size, context_size,bias_attn=False) -> None:
        super().__init__()
        self.num_head = num_head
        self.head_size = head_size
        # we assume the head_size is already split between the num_heads and thus we dont split
        # it again
        # assert head_size//num_head == 0, f'head_size({head_size}) must be divisable by head_num({num_head})'
        # we need to create n heads so lets do it 
        # self.heads = [AttentionHead(context_size, 
        #                             embd_size, 
        #                             head_size, 
        #                             bias_attn) for _ in range(num_head)]
        # but we can also use pytorch's nn.ModuleList which is a better equivalent than list
        # note that, nn.Sequential cant be used, becasue it runs the modules in succesion
        # i.e. serially, one after the other (feeds the output of the previous module to 
        # the next module, etc) which is not what we want. we want to calculate each head
        # independetly and aggregate their outputs so, either a python list or torch moudle list
        # can be used
        # sidenote: this module that we are building, is also known as, masked multi-attention-head
        # becasue we are using the single-head self-attention which uses a constrain we imposed
        # by masking if you recall that!
        self.heads = torch.nn.ModuleList(AttentionHead(context_size, 
                                         embd_size, 
                                         head_size,
                                         bias_attn) for _ in range(num_head))
      
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # note that head_size is usually the embd_size, so it covers the whole 
        # embeddings obviously, and note that each head will work on a portion
        # of the given head_size, if e.g. we have head_size/embd_size = 16
        # if we had 1 head, the head_size would be 16. however if we had 2 heads
        # we had to split the head_size in half for each head, and later on
        # we had to concat them to get the full-size head_size/embd_size.
        # therefore here, we need to concat their results along the cols
        # so multi-head-attention is akin to group convolution, and thus the head_size
        # and num_head must align properly.
        outputs = torch.cat([head(inputs) for head in self.heads], dim=-1)
        return outputs
# lets test 
# note that we set the split value for head_size based on our num_head
m = MultiHeadAttention(num_head=4, head_size=16//4, embd_size=16, context_size=8, bias_attn=False)
logits = m(torch.randn(size=(1,8,16)))
print(f'{logits.shape=}')
# 
# now lets use this in our model
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 
# now lets train this model and see how it performs this time
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# outputs : 
# param_count  =  3,041
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2041  val: 4.204
# train: 2.6895  val: 2.685
# train: 2.5449  val: 2.528
# train: 2.4570  val: 2.477
# train: 2.4238  val: 2.433
# done!

# I'd In mame
# Ast owu hapland.

# Mayl thew youres?
# IRisire dis lhend gitim whorder, sphorgunct Beye skek boutrig!

# I, sho sciltr not bpreathl harr'd doess elowt cof or iny poovigre.
# Thalrit?

# HThary?

# Lerim he bamat:
# Thiet hey, krepably bose bour cave grosr benemece wleir hon'g.

# MADIUMI but th Mond im's veoo tit. IDf met Cerve's wue rrer hiftren'tr'd:
# Anoks:
# HOt
# We I phess chi bouine mares yon ther; tlotirtlinl therot:
# Io
# Wh per,
# Fous thayigso ceany pate.

# Annd wosh fong uat
# Now ICUSo CCgodertiove
#
# trying with larger embedding size seems to improve the results: 
# param_count  =  7,553
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1828  val: 4.186
# train: 2.4703  val: 2.472
# train: 2.3649  val: 2.367
# train: 2.3009  val: 2.32
# train: 2.2713  val: 2.288
# done!

# Cly Toste?
# IOK:
# Whan his me' tis it I ond ber'thse?

# PO:
# Theray delling, dothoinerk an to the, or
# E VINGON: rocO:
# Thes win now thatin knir:
# Wigh fy, bund werar is wet my;
# I ene:
# JUu'd sow prut hat amver'd ENDUE VUF E VO?

# Se gings nom and.
# CArwak mandesplay beesire brit mand.

# MED:
# Thou ased ith is whis wentemprt hits this in ther.
# Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
# LUT: am le the wids my thousecarto my haty wind WLIERME:
# Wher a and
# Whesh to wyt wir

# compared to single head attention, with the same hyper parameters, train: 2.4684  val: 2.4759
# and now we got train: 2.4238  val: 2.433 which is better(we also got train: 2.2713  val: 2.288 
# with just increasing the embedding_size, so playing with parameters even blindly making model bigger
# seems to give us a boost), so we had improvements, but we still need a long way ahead of us!
# 
# Ok, to improve upon our results, there are couple of more things we need to add to our attention 
# block. if you look at the paper, we'll see a few concepts that we havent talked about or implemented
# yet, including the feed-forward(position-wise feedforward network) and layernorm, so lets talk about 
# them.
# feed-forward network or as its called in the paper,'position-wise feedforward network', is simply
# a "fully connected network which is applied to each position separately and identically", this 
# consists of two linear layer with a relu activation function inbetween. 
# so its a linear layer with a relu activation function followed by another linear layer basically. 
# you may also hear this network be refered to as 'computation after communication' so to speak!
# signifying the fact that it runs a computation after the attention module.
# the idea behind this extra addition is simple, to provide a higher representation out of attention work
# this network, causes every single token to have a nonlinear transformation and achieve a higher abstraction
# possibly yielding new information benficial to the task. to put it in casual way, it can be seen as though
# the attention gatheres some stats/information, and this step is akin to looking into it and thinking about it
# comming up with some new findings, that is , if attention part is refered to as communication part, this is
# the computation part/ or thinking part for the lack of a better word.
# lets implement this network in our bigram model and see if this seemingly simple change, affects us at all
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # now let us create the feedforward network, which basically is a linear layer with relu
        # followed by another linear layer! for this so called network, the in_features and out_features
        # are simply the same, and is embd_size, because its a sandwich layer between our attention output
        # and the last layer. this layer usually is (embedsize,embed_size) if you recall head_size is usally
        # equal to embed_size. also note that in the paper the feedforward network is two linear layers with
        # a relu (one linear layer with a relu plus another linear layer).if you look closely you'll notice 
        # we already have a last layer after the attention, so we dont need to put a nother linear layer
        # afterward, becasue that one linear layer suffices (multiple linear layers, dont provide
        # any higher abstraction anyway, so its prefectly fine)
        # but on the other hand the paper's implementation has another change, it states 
        # the inner layers dim are increased by 4x, so to be faithful to the paper we also 
        # add the second linear layer, with the suggested change, this wont change the output 
        # shape, so we are fine. we just added an extra linear projection layer. 
        # this additional projection operation actually improves the result,however if we simply 
        # use the same dim linear layer (i.e. have sth like linear(head_size, head_size)) adds nothing
        # to the representational power of the network and you wont see anything substantial vs if you
        # completely remove this layer and only use linear/relu only.
        # why this works is becasue, we increase the nonlinear output neurons of the first layer by 4,
        # increasing its representational capacity, and then use the second linear layer to get the output
        # size compatible for the next layer (doing a linear projection), and hence our improvements lie
        # in the nonlinearity this addition provides. 
        self.feedforwardnet = nn.Sequential(nn.Linear(embd_size, head_size*4), nn.ReLU(),
                                            nn.Linear(head_size*4, head_size))
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # add our new addition, feedforwardnet
        out = self.feedforwardnet(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using feedforwardnet added')
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints : 
# using feedforwardnet added
# param_count  =  5,169
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1975  val: 4.198
# train: 2.6222  val: 2.634
# train: 2.4919  val: 2.481
# train: 2.4320  val: 2.435
# train: 2.3979  val: 2.392
# done!

# Selwils iur.

# AnNdt
# By thak, yue, nein ende vat Rout'w haler,
# The hot hes? is, whas bles. Yig,
# ROm
# QUS:
# Ay Whigice rea:
# Therer a youst um mith dins I dorseancer by gou:
# I sot the, aserer tom sne arobfs-Rid,
# This sas whe ou,
# Acet Yark no.

# OT:
# Fely ip you pel hivet seatly. hial pand,
# Thand repwat,
# I tarve ther se this ten thio drord dot tho,
# Rulee res le, il fet!
# Nord Wotiilet homom for my woue cuve wid.

# Angilene.

# Fhivy, ih:
# Boumlke uthereaerf
# Whe ithe hares.

# Whos sthik thenth
# Tait:
# She tove w

# the loss didnt get better! but maybe if we increase the context_size, and embedding_size, it 
# perorm better?
# using a larger embedding_size and thus larger model did improve the result, 
#
# using feedforwardnet added
# param_count  =  15,905
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1569  val: 4.159
# train: 2.3885  val: 2.396
# train: 2.2789  val: 2.312
# train: 2.2064  val: 2.249
# train: 2.1599  val: 2.198
# done!

# To get fer arast deve lad, I' fale are you up this riany;
# Weareien,
# Tho Guser, me ter is! Do lothe, to ling wo whe Pist to deave mat not, in and- air besface, y to fron; use ler weet herefalf thop an der pretrier
# nesly toide
# Thats ous.

# RENT:
# I for the haw to, to noss teave ler shat earelss ater, want Heanct herry deeter erry, heave masere dacke; nochat onsemy ning Meib,--?

# LUCLUER:
# If Feropeake mo shat onsivoole washy the well beWarquent;
# I on well he ome it you lare to lo to kit youser,
# Weves

# the larger network, seems to be doing much better than its counter part from before
# the one without the ffnet, we got train-loss: 2.1599  val-loss: 2.198 here whereas 
# previously we got train-loss: 2.2713  val-loss: 2.288. so we have improvements despite the 
# text still not being that good(but still better than before). so we need more enhancements)

logits.shape=torch.Size([1, 8, 16])
param_count  =  7,553
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1828  val: 4.186
train: 2.4703  val: 2.472
train: 2.3649  val: 2.367
train: 2.3009  val: 2.32
train: 2.2713  val: 2.288
done!

Cly Toste?
IOK:
Whan his me' tis it I ond ber'thse?

PO:
Theray delling, dothoinerk an to the, or
E VINGON: rocO:
Thes win now thatin knir:
Wigh fy, bund werar is wet my;
I ene:
JUu'd sow prut hat amver'd ENDUE VUF E VO?

Se gings nom and.
CArwak mandesplay beesire brit mand.

MED:
Thou ased ith is whis wentemprt hits this in ther.
Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
LUT: am le the wids my thousecarto my haty wind WLIERME:
Wher a and
Whesh to wyt wir
using feedforwardnet added
param_count  =  15,905
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1569  val: 4.1

In [4]:
# the next improvement, would be to use more of these, as its shown in the paper, if one block
# works, then adding more should work better right? we had single head, it improved our condition, 
# so we used more heads, and now lets more multi-attention heads! to make things easier, lets
# make a module out of it the same way we created previous modules. 

# first lets create our ffnet as a seprate block, so we use it after the attention
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        return self.ffnet(self.attn(inputs))

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks!')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks!
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2087  val: 4.211
# train: 2.6084  val: 2.606
# train: 2.3737  val: 2.39
# train: 2.2865  val: 2.307
# train: 2.2087  val: 2.237
# done!

# FAUKIN YRE:
# Dato':
# wis.

# BERDEWEE:

# Piwit ancon; awt
# Emuke this his fred not pe,
# Har,
# And him my bird-wray sluf. Ile:
# With tulel wik, antegatorg.

# EREN VELV:
# Whath a meane, ones
# sheirss,
# Bose til
# Ilf bodtund ba,
# Aripce nerod is the blilshil hit hit mef?

# GESINCE:
# Sard, with the deoorrnsicominle thid: as.

# ANRE:
# Hy prall anges what micciths delre
# To tike comf hon ebat'd mur. Kin'sa, Fharth:
# Af'?

# ROASA:
# Ect mily fledd.
# SHricest.
# At dey bompe novor where lisg.

# KETILNINCE:
# There gay his my nefonk
#
# as you can see, we got wrose results than before! despite making the network larger, our loss
# really didnt improve as we expected. 
# is our initial hypothesis that having more blocks and higher nonlinearity/representation is benificial
# wrong? or is there something else thats causing the issue? 
# as you might have guessed, its the latter. we are basically creating more layers, and with
# more layers, we face training issues that we discussed earlier. 
# so how should we tackle this, we cant use BN, becasue BatchNOrm, accumulates the statistics
# from different samples, this is a no no for us. we dont want other samples to interfer with 
# our sample (or basically each other!) in anyway, so what should we do? 
# the paper utilizes two mechanisms or operations to tackle this issue. one being skip-connections
# (also known as residual connction) and the other, layer-normalization. 
# you should be familiar with skip-connections as they are the founding factor or resenets
# and have been extremely influential. so to cut a long story short, skip connections are simply
# connections from input skipping the operations involved in the block they reside and directly 
# being added to the output and then returned the result.(basically F(x) + x)
# as we know, the gradients are distributed equally when they reach addition, so input gets the gradients
# without being weakened due to large depth of the network. 
# so before we implement the layer normalization part, lets see howmuch of a change adding skip-connection
# causes and whether it proves our initial hypothesis about depth and gradient signal weakening or not
# we add this to our AttentionBlock (but we could add this to any submodule)

using more blocks!
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.2087  val: 4.211
train: 2.6084  val: 2.606
train: 2.3737  val: 2.39
train: 2.2865  val: 2.307
train: 2.2087  val: 2.237
done!

FAUKIN YRE:
Dato':
wis.

BERDEWEE:

Piwit ancon; awt
Emuke this his fred not pe,
Har,
And him my bird-wray sluf. Ile:
With tulel wik, antegatorg.

EREN VELV:
Whath a meane, ones
sheirss,
Bose til
Ilf bodtund ba,
Aripce nerod is the blilshil hit hit mef?

GESINCE:
Sard, with the deoorrnsicominle thid: as.

ANRE:
Hy prall anges what micciths delre
To tike comf hon ebat'd mur. Kin'sa, Fharth:
Af'?

ROASA:
Ect mily fledd.
SHricest.
At dey bompe novor where lisg.

KETILNINCE:
There gay his my nefonk 


In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
# lets add a skip-connection to this block
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        out = self.attn(inputs) + inputs
        out =  self.ffnet(out)  + inputs
        return out

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5636  val: 4.553
# train: 2.2666  val: 2.281
# train: 2.1330  val: 2.183
# train: 2.0708  val: 2.14
# train: 2.0204  val: 2.109
# done!

# Zunt will jucked's whreef onst's so
# wippplalct, sawter.

# Firfor have deal pere,
# Shal,
# thich many.
# Frig.

# Thall have.

# WANWHAM:
# Sad king theave,
# Cnomlen nare, near was?


# CKINGlLOROMBOENGBETH:
# Dich loverd und by, may;
# ''Kh
# Godie the bline plock's minstell the denelsbad, with think
# What
# sicond lead this undrut, too, the but me my se ming'ld;
# Shing
# To tiot comford, engelan this it's heare: hear them the vinclamil.
# Is done, the stronced,
# Lion?
# Hartow where lisg.

# DOLILINA:
# Lord ISe and
# Lud'd nef hel

# and test with one skip-connection on the 'output only' to prove or assumption on skip-connections
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5616  val: 4.552
# train: 2.3196  val: 2.322
# train: 2.1862  val: 2.221
# train: 2.1175  val: 2.163
# train: 2.0640  val: 2.137
# done!

# FARY VI:
# A-bame 'tries.

# BERGEWES:

# PABLLO:
# Yon; aws
# Then ath-pmate for, not pe,
# PARILA:
# So man.

# FORY.

# TAsUS:
# I cenpine nou, lad king theak,
# Anny, In nartine-'

# An yeant, of ladie ust, good ting lover that by, may;
# And hordie the
# llils plove ond sofful and heave
# Take with the deo,
# And mort leath and usir the dom the but me my crombasill;
# Shim are, con comlmork engeland your, trabe,
# Beth:
# Yor?

# CORIAR EFt mily fle--spoles, strence'd ono do now, will?
# Mursice,
# Ntormorest
# The juge ernus'd neforki

# as you can see, it greatly improved our results, and the text also got much better!
# as we already pointed out, having skip-connections per modules, help much more than a single 
# per module output only, nevertheless, we notice, using skip-connection really improved our results.
# now lets add the second operation, layernorm. layernorm(https://arxiv.org/abs/1607.06450) is a normalization layer, just like batchnormalization
# came a year later than batchnormalization paper, but the difference between them is that, unline batchnormalization
# it works on a per sample basis and does not involve using othersamples to normalize a specific sample (basically
# samples dont affect eachother)
# As pytorch docs puts it:
# "Unlike Batch Normalization and Instance Normalization, 
#  which applies scalar scale and bias for each entire channel/plane with the affine option,
#  Layer Normalization applies per-element scale and bias with elementwise_affine"
# the implementation is similar to the batchnormalization, and it does not require calculating running_mean/var
# we can use the pytorch module just fine, but since its really similar to BN, lets implement it here 
# 

using more blocks with skip-connection
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.5636  val: 4.553
train: 2.2666  val: 2.281
train: 2.1330  val: 2.183
train: 2.0708  val: 2.14
train: 2.0204  val: 2.109
done!

Zunt will jucked's whreef onst's so
wippplalct, sawter.

Firfor have deal pere,
Shal,
thich many.
Frig.

Thall have.

WANWHAM:
Sad king theave,
Cnomlen nare, near was?


CKINGlLOROMBOENGBETH:
Dich loverd und by, may;
''Kh
Godie the bline plock's minstell the denelsbad, with think
What
sicond lead this undrut, too, the but me my se ming'ld;
Shing
To tiot comford, engelan this it's heare: hear them the vinclamil.
Is done, the stronced,
Lion?
Hartow where lisg.

DOLILINA:
Lord ISe and
Lud'd nef hel


In [6]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (2.38418573772492e-09, 1.0004067420959473)
torch: (9.53674295089968e-09, 1.0004067420959473)


In [7]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
#
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we were aggregating the last two dims like BN, whereas we should have only used the
# last dim, as we should not involve other samples in normalization. (I explained this thoroughly in the LayerNorm class)
# 
# now how can we improve more? we implemented the paper, basically we implemented transormer from scratch
# and what remains is to test with different hyperparameters to see how well it can generate texts similar 
# to our dataset. (we implemented the decore version, but the encoder as we explained earlier is the same
# without the constrains! well talk about this in a moment )
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3906  val: 4.385
train: 2.2373  val: 2.244
train: 2.1070  val: 2.155
train: 2.0339  val: 2.102
train: 1.9812  val: 2.069
done!

For the that mad'
With of on tith, luidie anct, saws
Eule at botht if thou
Wlefted' you windian.

FLO-wike sluke slen:
I have, ladik, anteraves,
norl wink
the bard a yeane, of lady. Whe, goshot I
thy brath his what:
Nown hondin the
blinsh, shat this the pandself livadet brives nefors.
ISIUS:
Aren that usir the of me
he hange wicHe mirs,
Behat!


TOLAT:
But ford, engelan this it'sabhanke hear rend these this in usse,--
Lord, strence'er thy?
In woes. IDandry,
Soset frongs neved
With haul'd nefonk 


In [8]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


/home/hossein/anaconda3/lib/python3.11/site-packages/torch/cuda/memory.py:329: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


46

In [9]:
head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')

param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3845  val: 4.388
train: 1.9630  val: 2.045
train: 1.5727  val: 1.755
train: 1.4054  val: 1.632
train: 1.3029  val: 1.57
done!


In [10]:
# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# What, my Lady of Angelo?

# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 


DUKE VINCENTIO:
Scent the Faulia.

KING EDWARD:
Sweet, good conceive these are to be.

CAPULET:
Not gentle Menenius, here well they pend lives and I
Rengeath one much buried.

HARCID:
O, God I'll: Bidish, Brobet, not--
Hath said, quotest me, is a done.

CORIOLANUS:
Then, no proud heret! what sat withile: woen so,
ha, wary thou comest, why thought that art thou nelsest;
yet fights on his dearth-hoad yoking hered.

KING EGBRY:
Very Grey, my lord?

CORIOLANUS:
The people.
Why let, sir, who have been sweet, Let to thee.

BARBIUNCENA:
In to the rest, my blood; Warwick's house.
Take you in his worth train'd then he mone.

ROMEO:
Those wasted damnable, Eauton,
Like, and brisothe comes do I nay,
and that you know I were, and you lead us
I tept brail, if such enemi: knack'd love from
Unsuing person, and never. O, you should woman joys;
But in this seven strem one of mine honestes
Their woman to die: nay, sir, and were back
if the imploteful of late, our chante faults
He that wretche with this.

In [11]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
print(f'{torch.cuda.memory_summary(0)}')
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
print(f'{torch.cuda.memory_summary(0)}')
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      | 191200 KiB |   6442 MiB | 207059 GiB | 207059 GiB |
|       from large pool | 137344 KiB |   6396 MiB | 206643 GiB | 206643 GiB |
|       from small pool |  53856 KiB |    129 MiB |    416 GiB |    416 GiB |
|---------------------------------------------------------------------------|
| Active memory         | 191200 KiB |   6442 MiB | 207059 GiB | 207059 GiB |
|   

248

In [12]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
print(f'{torch.cuda.memory_summary(0,True)}')
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
print(f'{torch.cuda.memory_summary(0,True)}')
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      | 191200 KiB | 191200 KiB | 207059 GiB | 207059 GiB |
|---------------------------------------------------------------------------|
| Active memory         | 191200 KiB | 191200 KiB | 207059 GiB | 207059 GiB |
|---------------------------------------------------------------------------|
| Requested memory      | 189405 KiB | 189405 KiB | 207053 GiB | 207052 GiB |
|---

248

In [13]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
print(f'{torch.cuda.memory_usage(0)}')
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
print(f'{torch.cuda.memory_usage(0)}')
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


ModuleNotFoundError: pynvml does not seem to be installed or it can't be imported.

In [14]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
print(f'{torch.cuda.memory_usage(0)}')
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
print(f'{torch.cuda.memory_usage(0)}')
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


ModuleNotFoundError: pynvml does not seem to be installed or it can't be imported.

In [15]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 
!pip install pynvml
print(f'{torch.cuda.memory_usage(0)}')
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
print(f'{torch.cuda.memory_usage(0)}')
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


ModuleNotFoundError: pynvml does not seem to be installed or it can't be imported.

In [16]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 

print(f'{torch.cuda.memory_stats(0)}')
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
print(f'{torch.cuda.memory_usage(0)}')
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!
OrderedDict([('active.all.allocated', 10034478), ('active.all.current', 694), ('active.all.freed', 10033784), ('active.all.peak', 694), ('active.large_pool.allocated', 6050018), ('active.large_pool.current', 51), ('active.large_pool.freed', 6049967), ('active.large_pool.peak', 51), ('active.small_pool.allocated', 3984460), ('active.small_pool.current', 643), ('active.small_pool.freed', 3983817), ('active.small_pool.peak', 643), ('active_bytes.all.allocated', 222328687562240), ('active_bytes.all.current', 195789312), ('active_bytes.all.freed', 222328491772928), ('active_bytes.all.peak', 195789312), ('active_bytes.large_pool.allocated', 221881831899136), ('active_bytes.large_pool.current', 140640256), ('active_bytes.large_pool.freed', 221881691258880), ('active_bytes.large_pool.peak', 140640256), ('active_bytes.small_pool.allocated', 446855663104), ('active_bytes.small_pool.current', 55149056), ('active_bytes.small_pool.freed', 

ModuleNotFoundError: pynvml does not seem to be installed or it can't be imported.

In [17]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import torch
torch.manual_seed(255)
random.seed(255)
import gc 

print(f'{torch.cuda.memory_stats(0)}')
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()
print(f'{torch.cuda.memory_stats(0)}')
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!
OrderedDict([('active.all.allocated', 10034478), ('active.all.current', 694), ('active.all.freed', 10033784), ('active.all.peak', 694), ('active.large_pool.allocated', 6050018), ('active.large_pool.current', 51), ('active.large_pool.freed', 6049967), ('active.large_pool.peak', 51), ('active.small_pool.allocated', 3984460), ('active.small_pool.current', 643), ('active.small_pool.freed', 3983817), ('active.small_pool.peak', 643), ('active_bytes.all.allocated', 222328687562240), ('active_bytes.all.current', 195789312), ('active_bytes.all.freed', 222328491772928), ('active_bytes.all.peak', 195789312), ('active_bytes.large_pool.allocated', 221881831899136), ('active_bytes.large_pool.current', 140640256), ('active_bytes.large_pool.freed', 221881691258880), ('active_bytes.large_pool.peak', 140640256), ('active_bytes.small_pool.allocated', 446855663104), ('active_bytes.small_pool.current', 55149056), ('active_bytes.small_pool.freed', 

2298

In [1]:
# in the name of God the most compassionate the most merciful
# in this section we will have a look at how a GPT model works
# in this section we will implement attention module, and see 
# how it works and lean more about its (sel-attention, cross
# attention, multi-head attention, etc)
# 
# so how are we going about this. we need to create a language model
# and then progressively imporve it with attention mechanism.
# we will basically be creating a kind of chatgpt (minus its chat capability and
# its obviously great perormance :)) 
# to keep this as simple as possible and not lose track of the important concepts involved,
# we can use our initial bigram model as the base model and work on improving that with attention.
# so lets start
#
# first lets import the basic stuff 

# for type hints
from collections.abc import Iterable

import random 
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# lets setup the manual seeds or determinstic output
torch.manual_seed(255)
np.random.seed(255)
random.seed(255)

# first lets read the dataset, we are going to use tinyshalespear which
# is around 1million characters (around 1MB in size), and our model is
# supposed to create texts resembling this dataset.
dataset = []
with open('./tiny_shakespear.txt','r') as file:
    # this time we read the whole text as one big str
    dataset = file.read()
    
print(f'{dataset[:100]=}')
# now lets create our atoi and itoa dictionaries for mapping
# lets create our unique character list, which is infact our vocabulary or vocab for short
vocab_list = sorted(set(''.join(dataset)))
vocab_size = len(vocab_list)
# ! explain token
# lets create our mapping dictionaries, we are basically going to use them
# for tokenization, converting our input into tokens which here are characters 
# and ultimately their integer representations
# so tokenization simply put, refers to converting an input into a list of numbers based on some creteria
# here, we just use a simple index number to map our vocabulary characters, which represent all character
# our model can use to generate text, to integer representation. 
# for realworld applications, we usually use libraries such as sentecepeace by google which works on a subword
# level which means, it doesnt encode the whole word, neither does it encode based on individual characters, it
# use sth in between. 
# (from its github repo: https://github.com/google/sentencepiece )
# SentencePiece is a re-implementation of sub-word units, an effective way to alleviate the open vocabulary 
# problems in neural machine translation. SentencePiece supports two segmentation algorithms, 
# byte-pair-encoding (BPE) [Sennrich et al.] and unigram language model [Kudo.]. 
#
# (For the reference:  
# Byte Pair Encoding (BPE) is a subword tokenization technique used in natural language processing (NLP) 
# and text processing tasks. It is a data compression algorithm that splits words into subword units. 
# BPE is commonly used in tasks such as machine translation, text generation, and language modeling.
# The basic idea behind BPE is to iteratively merge the most frequent pairs of characters or subword units
# in a corpus to create a new subword vocabulary. This merging process is based on the statistical properties
# of the corpus, specifically the frequency of character or subword pairs.
#
# Here's a high-level overview of the BPE algorithm:
# 1. Initialize the vocabulary with all the characters or subwords in the corpus.
# 2. Calculate the frequency of each character or subword in the corpus.
# 3. While the desired vocabulary size or a maximum number of iterations is not reached:
#    - Find the most frequent pair of characters or subwords in the corpus.
#    - Merge the pair into a new subword unit by concatenating them.
#    - Update the corpus by replacing occurrences of the merged pair with the new subword unit.
#    - Update the vocabulary and frequency counts based on the new corpus.
# 4. The final vocabulary is the set of subword units obtained after the desired number of iterations 
#    or vocabulary size is reached.
#
# BPE allows for the representation of both known and unknown words in a corpus. It is effective in 
# handling out-of-vocabulary (OOV) words and reducing the vocabulary size, which can improve the 
# efficiency and performance of NLP models. By breaking down words into subword units, BPE can capture
# morphological and semantic information more effectively, especially for languages with complex word 
# formations and agglutinative structures.
# 
# tiktokenize repo explains BPE in rather friendlier way: 
# Models don't see text like you and I, instead they see a sequence of numbers (known as tokens). 
# Byte pair encoding (BPE) is a way of converting text into tokens. It has a couple desirable properties:
# It's reversible and lossless, so you can convert tokens back into the original text
# It works on arbitrary text, even text that is not in the tokeniser's training data
# It compresses the text: the token sequence is shorter than the bytes corresponding to the original text. 
# On average, in practice, each token corresponds to about 4 bytes.
# It attempts to let the model see common subwords. For instance, "ing" is a common subword in English, 
# so BPE encodings will often split "encoding" into tokens like "encod" and "ing" 
# (instead of e.g. "enc" and "oding"). Because the model will then see the "ing" token again and again 
# in different contexts, it helps models generalise and better understand grammar.
#
# 
# A unigram language model however, is a type of statistical language model that predicts the probability 
# of each word in a sequence independently, based solely on the frequency of occurrence of individual 
# words in the training data. It does not consider the context or the order of the words in the sequence.
# In a unigram language model, the probability of a particular word is estimated by counting the frequency 
# of that word in the training corpus and normalizing it by the total number of words in the corpus. 
# The probability of a sequence of words is then calculated by multiplying the probabilities of each 
# individual word in the sequence.
# For example, consider the sentence "I love to eat pizza." In a unigram language model, the probability
# of this sentence would be calculated as the product of the probabilities of each word which would be: 
# P(I) * P(love) * P(to) * P(eat) * P(pizza).
# Unigram models are the simplest form of language models and do not capture any contextual information
# or dependencies between words. They are often used as a baseline or reference model in natural language 
# processing tasks. While unigram models are not very accurate in capturing the complexities of natural 
# language, they can be computationally efficient and useful in tasks where the context is less important, 
# such as certain text classification tasks or language generation tasks where only the frequency of 
# individual words matters.
# 
#
# Openai/chatgpt for example uses its own tokenizer, called tiktoken (https://github.com/openai/tiktoken)
# there are other tokenizers as well.
# huggingface has a good article on tokenizers (recommend it): https://huggingface.co/docs/transformers/main/tokenizer_summary 
# its a given that, each model only works with the tokenizer by which it was used to train. 
# so BERT, DistilBERT, and Electra only work with wordpiece, while chatgpt uses titoken and we! use
# simple characters-indexes! also note that the choice of tokenizer obviously affects our vocab size and
# model overhead/performance as well. 
# for example lets first implement our tokenizer and then compare it with sth like tiktoken module
atoi = {c:n for n,c in enumerate(vocab_list)}
itoa = {n:c for c,n in atoi.items()}
print(f'{vocab_size=}, {vocab_list=}')
print(f'{atoi=}')
print(f'{itoa=}')
# lets also create two helper functions to convert a list of these to the other part
def encode(characters:Iterable[str] ):
    return [atoi[c] for c in characters]

def decode(token_lst:Iterable[int]):
    return [itoa[n] for n in token_lst]

# lets test these 
print(f'{encode(dataset[:10])}')
print(f'{decode(encode(dataset[:10]))}')

# now lets try tiktoken
try: 
    import tiktoken
except:
    import os 
    os.system('pip install tiktoken')
    import tiktoken
# lets use the tokenizer for gpt2 model, this line downloads the gpt2 tokenizer and allows us to use it
encoder_gpt = tiktoken.get_encoding('gpt2')
# lets view its vocab size:
print(f'{encoder_gpt.n_vocab=:,}') # prints encoder_gpt.n_vocab=50,257
# now lets see how the encoding looks like here
print(f'{encoder_gpt.encode(dataset[:10])=}') # prints [5962, 327, 8846] 
# which is very intresting! while for us its 10 numbers/tokens for 10 characters(vocab size 65),
# this is 3 here with vocab size of 50,000!
# so we can have a short vocab size, at the expense of a larger sequence size, 
# or a large vocabsize and smaller sequence size.
# so thats why in practice, these subwords tokenizers are used for real world applications. but for our case
# we stick to our primitive tokenizer to keep things as simple as possible. 
# 
# Ok, so far so good. now we need to create a dataset, like before we need to have an input/label pair
# our input is a series of characters, (a sequence of some length), and our label is the next character
# sicne we are using a simple bigram model, given a single character, we want the probablity of what comes
# next. but, we also want to incorporate attention, and we want a context for our prediction, we want to
# be able to look at the past, and look at the past characters, and based on that do sth. this is the essence
# of attention (although this is not accurate, but for now this is the case, we will elaborate on this and expand
# this metaphor and reasoning inshallah)
# 
# lets first tokenize the whle dataset or corpus as its usually called in nlp nomenclature!
data = encode(dataset)
# since we are using pytorch lets convert that to a tensor
data_tensor = torch.tensor(data)
print(data_tensor[:100])
# now lets create a train/val split 
train_length = int(0.9 * len(data_tensor))
# note that we do not shuffle the data here like before, because we are not dealing with a list of samples!
# like names, here we are dealing with the whole text, and if we shuffle it like that, we just destroy it
# making it into a batch of random characters! which would not be useable for us anymore. we are trying to
# learn the underlying semantic and relationships hidden in our data, and randomizing them like that just 
# destroys those information! 
# to see this in action try this
_data_copy = data.copy()
random.shuffle(_data_copy)
print(''.join(decode(_data_copy[:100])))
# prints:
# Osetow Pr 
# oa geEeM rhnalelrt   aAdt Ktsw pTpeGxli
# oasEliMe
# vsieeose
# etttbSdnctiras  hnr:yhtayiuCv g
# so we simple divide the data normally
train_data = data_tensor[:train_length]
val_data = data_tensor[train_length:] 
# note that since we are planning on creating a simple transformer model, we usually dont feed the whole dataset
# becaue its prohibitevly computation intensive, instead, what happens in practice is that we, grab chunks 
# of data from the dataset and feed it to the transformer. and these chunks, of course has a length, what length?
# we usually specify a maximum_length for the input on which our transformer model works.
# this maximum_length is usually refered to as block_size or context_size.
# we had previously used different context_sizes and this is not really that different,
# lets for example define a context_size of 8
# block_size and context_size are interchangable
block_size = context_size = 8 
print(f'{train_data[:block_size]=}')
# which prints:
# train_data[:block_size]=tensor([18, 47, 56, 57, 58,  1, 15, 47])
# 
# one intresting observation we can make here is that, as simple as this seemingly ordindary list of numbers
# looks, this sequence of numbers, actually contains several examples. 
# if you think about it, each number, is a token, representing a character (or word, subword, etc),
# and it shows what comes after what. basically not only it shows several pairs so to speak, it also
# shows, which characters are more likely to come, before a specific character comes later. 
# what we are actually going to do is that, we are going to train all of these characters simultaneously
# notice that in this example, we have 7 examples in a sequence of 8 characters:
# lets elaborate on this more. 
# 1-in the context of 18, the next character is 47
# 2-in the context of 18,47, the next character is 56
# 3-in the context of 18,47,56, the next character is 57
# 4-in the context of 18,47,56,57 the next character is 58
# 5-in the context of 18,47,56,57,58 the next character is 1
# 6-in the context of 18,47,56,57,58,1 the next character is 15
# 7-and finally, in the context of 18,47,56,57,58,1,15 the next character is 47
# so this is infact 8 7 individual example embedded in a single context, 
# since we want the xontext_size to be 8, then we should grab one more character to have 
# 8 contexts
# lets visualize this in example in code
# lets have a typical sequence of size 8 
x = train_data[:block_size]
y = train_data[1:block_size+1]
# what would be our label? we want the next character so it would be the previous locations +1
# basically offset the input by 1!
# thats why we started from 1, becasue the input started from 0, and since the input ended at block_size
# its label(next character) would obviously be block_size+1, pretty obvious right?
#
# lets print this for better understanding 
for i in range(block_size):
    # note that since we are using slice, x[:i+1], gives us 0 up to ith element (inclusive)
    # this should be obvious! but I said it in case you forgot!!
    print(f'input is {x[:i+1].tolist()} label is {y[i]}')
# prints 
# input is [18] label is 47
# input is [18, 47] label is 56
# input is [18, 47, 56] label is 57
# input is [18, 47, 56, 57] label is 58
# input is [18, 47, 56, 57, 58] label is 1
# input is [18, 47, 56, 57, 58, 1] label is 15
# input is [18, 47, 56, 57, 58, 1, 15] label is 47
# input is [18, 47, 56, 57, 58, 1, 15, 47] label is 58
# so as you can see we started from context_size of 1 up to context_size of 8.
# note that we do not do this simply for the efficancy aspect of it, but also, when we feed these to
# our transformer model, we are making sure that the transformer model gets used to see all combinations
# of our input as well (from the context size of 1 up tp the context size of 8). 
# this way not only it sees the whole context_size as we initially expected
# but also all sequences before it, and basically what consituted to make the sample. 
# this allows us to later on, at test time be able to create sequences as small as context_size of only 1
# up to the max_length which is our context_size of 8 in our case.
# ! recheck and elaborate to clear any confusion
# !so by doing this, the transformer can learn how to predict/create/genrate text up to context_size, and after
# !it reached thta, we have to truncate it, becausse the transformer model never recieves more than the
# !context_size as input when its predicting the next character.
# so far what we covered here was the time dimension of our input. we have a sequence, and each entery
# basically denotes a time t dimension, at which, a character is introduced. 
# another imporatnt aspect we need to take care of is the batch dimension, cuz we are going to feed 
# multiple examples at once, we use this to harness the gpu parallilization capalibity in pytorch
# as without it, training this simple model would take a lot of time on cpu! 
#
# so lets create a function that gives us a batch of the inputs rather than a single list/tensor!
# 
batch_size = 4 
# our block_size
context_size = 8 
# 
def get_batch(split, batch_size):
    #lets grab the data basedo on the split
    data = train_data if split =='train' else val_data
    # we want a batch of 4 of 8 characters (context-size). 
    # to make a batch we can grab 4 random indices as input and then expand them
    # by adding the next 8(context_size) characters to them, in order not to go past the 
    # last index, we subtract the length of data(last valid index) from context size 
    idxs = torch.randint(low=0, high=len(data)-context_size, size=(batch_size,))
    # we have our indices, so lets create our samples
    # x = [data[i:i+context_size] for i in idxs]
    # and for labels we do the same thing but offset it by 1 so each characters label
    # becomes the next character
    # y = [data[i+1:i+context_size+1] for i in idxs]
    #
    # print(f'{x=}')
    # print(f'{y=}')
    # prints: 
    # x=[tensor([58, 39, 49, 43,  1, 51, 63,  1]), tensor([53, 59, 41, 46,  5, 42,  1, 61]), tensor([47, 53, 52,  1, 39, 57,  1, 63]), tensor([39, 57, 58,  1, 51, 63,  1, 50])]
    # y=[tensor([39, 49, 43,  1, 51, 63,  1, 54]), tensor([59, 41, 46,  5, 42,  1, 61, 47]), tensor([53, 52,  1, 39, 57,  1, 63, 53]), tensor([57, 58,  1, 51, 63,  1, 50, 53])]
    #
    # as you can see we have a list of tensors. since we want a batch, we just stack them or concat them
    # on top of each other
    # using stack!
    # x = torch.stack(x)
    # y = torch.stack(y)
    # stack() is intrestingly implemented by concat(), so if we want, we can do the same using concat!
    #
    # by default conact, concatenates all the tensors as one large tensor!
    # we need a reshape/view to get the right shape
    # since we are using view, no copy,etc is done, and its an efficient operation
    # x = torch.concat(x,dim=0).view(batch_size, context_size)
    # y = torch.concat(y,dim=0).view(batch_size, context_size)
    # 
    # however if we add an extra dim to our inputs we can remove the extra reshape/view
    # x = torch.concat([t.unsqueeze(0) for t in x], dim=0)
    # y = torch.concat([t.unsqueeze(0) for t in y], dim=0)
    #
    # By unsqueezing, we ensure that the tensors have the same number of dimensions before 
    # concatenating them with torch.cat/concat/concatenate.(side tip: concat and concatenate are aliases for cat) 
    # This effectively replicates the behavior of torch.stack.
    # see torch.cat concatenates tensors along an existing dimension, and we only have 1 dimensional
    # tensors, so it will concatenate them along that, effectively making one large 1 dimensional vector
    # however, when we use unsqueeze on each tensor, using torch.unsqueeze(0), we are adding a new dimension
    # (dim 0) to that tensor, making it 1,x instead of the original shape of (x,).  
    # So, by unsqueezing each tensor along the desired dimension, which for us is the 0ths dimension (or row dim) 
    # and then concatenating them with torch.cat, we achieve the same result as torch.stack.
    # 
    # in practice we'd like to do this all in one go!
    x = torch.stack([data[i:i+context_size] for i in idxs])
    y = torch.stack([data[i+1:i+context_size+1] for i in idxs])
    
    return x,y

x,y  = get_batch('train',batch_size=4)
print(f'{x.shape=}\n{y.shape=}')
print(f'{x=}\n{y=}')
# which prints 
# x.shape=torch.Size([4, 8])
# y.shape=torch.Size([4, 8])
# tensor([[58, 39, 49, 43,  1, 51, 63,  1],
#         [53, 59, 41, 46,  5, 42,  1, 61],
#         [47, 53, 52,  1, 39, 57,  1, 63],
#         [39, 57, 58,  1, 51, 63,  1, 50]])
# tensor([[39, 49, 43,  1, 51, 63,  1, 54],
#         [59, 41, 46,  5, 42,  1, 61, 47],
#         [53, 52,  1, 39, 57,  1, 63, 53],
#         [57, 58,  1, 51, 63,  1, 50, 53]])
#
# now this is our batch of data, 4 samples with 8 characters, bascially 32 examples, lets see that as well
for b in range(batch_size):
    # this is our time dimension
    for t in range(context_size):
        print(f'{x[b,:t+1]} --> {y[b,t:t+1].item()}')
#        
# prints : 
# tensor([58]) --> 39
# tensor([58, 39]) --> 49
# tensor([58, 39, 49]) --> 43
# tensor([58, 39, 49, 43]) --> 1
# tensor([58, 39, 49, 43,  1]) --> 51
# tensor([58, 39, 49, 43,  1, 51]) --> 63
# tensor([58, 39, 49, 43,  1, 51, 63]) --> 1
# tensor([58, 39, 49, 43,  1, 51, 63,  1]) --> 54
# tensor([53]) --> 59
# tensor([53, 59]) --> 41
# tensor([53, 59, 41]) --> 46
# tensor([53, 59, 41, 46]) --> 5
# tensor([53, 59, 41, 46,  5]) --> 42
# tensor([53, 59, 41, 46,  5, 42]) --> 1
# tensor([53, 59, 41, 46,  5, 42,  1]) --> 61
# tensor([53, 59, 41, 46,  5, 42,  1, 61]) --> 47
# tensor([47]) --> 53
# tensor([47, 53]) --> 52
# tensor([47, 53, 52]) --> 1
# tensor([47, 53, 52,  1]) --> 39
# tensor([47, 53, 52,  1, 39]) --> 57
# tensor([47, 53, 52,  1, 39, 57]) --> 1
# tensor([47, 53, 52,  1, 39, 57,  1]) --> 63
# tensor([47, 53, 52,  1, 39, 57,  1, 63]) --> 53
# tensor([39]) --> 57
# tensor([39, 57]) --> 58
# tensor([39, 57, 58]) --> 1
# tensor([39, 57, 58,  1]) --> 51
# tensor([39, 57, 58,  1, 51]) --> 63
# tensor([39, 57, 58,  1, 51, 63]) --> 1
# tensor([39, 57, 58,  1, 51, 63,  1]) --> 50
# tensor([39, 57, 58,  1, 51, 63,  1, 50]) --> 53 
#
# so now that we have the data sorted out, lets create our model. 
# as we said, we are going to use a bigram model and later add attention mechanism to it. 
# to make things easier and more self contained, lets add all the required logic to this model
# like when we do a forward, we be able to calculate loss as well if we are given the targets
# so lets go
class BigramModel(nn.Module):
    def __init__(self, vocab_size) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        # our bigram model was nothing more than a 2d array of vocab_size, we can achive that using 
        # a single weight matrix or torch.Embedding. we use torch.Embedding to not reinvent the wheel!
        self.token_embedding = torch.nn.Embedding(vocab_size, vocab_size)
    
    # since we want to be able to calculate loss, if there are labels, we get Y as well
    def __call__(self, inputs:torch.Tensor, labels:torch.Tensor=None) -> torch.Tensor:
        logits = self.token_embedding(inputs)
        loss = None
        if labels is not None:
            # calculate loss 
            # print(f'{logits.shape=}') # prints (4,8,65)
            # note that both inputs and labels have the shape (B,T)
            # but logits has the shape (B,T,C) which C here equals vocab_size 
            # this is an issue for torch.crossentropy, as it expects the input
            # to be in the form of (B,C,T), that is, the channels/embeddings dimension
            # need to be right after the batch dimension or otherwise it wont work.
            # so we need to account for that.
            # we can go on permute the dimensions, like this and make crossentropy happy 
            # logits = logits.permute((0,2,1))
            # however, if for some reason the output of logits in the form of (4,8,65) is not ideal, 
            # maybe for example because really what it is 32 examples arranged in a (4,8) shape and 
            # we rather a normal 2d tensor of shape (32,65), 
            # we can instead, do just that, flatten the two dimensions into one and carry on!
            # we basically are concatenating the batch and time dimensions
            # into one dimension (effectively, stacking samples on top of each other), instead of having 
            # four compartments, each having 8 segments, we are going to have 1 long compartment with 32 segments/rows
            # for the lack of better words!!
            # so lets first get the shapes 
            # B,T,C = logits.shape
            # logits = logits.view(B*T,C)
            # and we also need to do the same for the labels 
            # labels = labels.view(B*T)
            # we could also do,
            # labels = labels.view(-1)
            # but thats not really needed, so we just simply permute logits temporarily so the logits shape
            # stays the same regardless of calculating the loss or not 
            loss = F.cross_entropy(logits.permute((0,2,1)), labels)
        return logits, loss
    
    # now lets also add a generate method to generate texts
    def generate(self, idx, max_token_count):
        # before we implement this lets review what we expect this method to do
        # we want this to generate some characters, given an initial character
        # we also want to be able to control the length of the generated text
        # so we take a max_token_count.
        # we take the initial index, 
        # feed it to the model, 
        # get the logits,
        # turn logits to probs,
        # use torch.multinomial to sample from our probablity distribution
        # get the new character index, and add it to a list to gradually 
        # -create our final text output
        # so lets go 
        # our idx should have the shape [B,T]
        assert len(idx.shape) == 2, f'idx.shape({idx.shape}) should have (b,t) form.'
        #
        # lets generate as many characters/idx as max_token_count specifies
        # this is basically specifies the time/sequence dimensions
        for i in range(max_token_count):
            # since we are defining a class method, to call the callable, we simply use self()
            logits, _ = self(idx)
            # to get the probablities 
            # usually we would simply do 
            # probs = torch.softmax(logits, dim=1)
            # but here, we want to only focus on the next character because this is what comes next
            # each time obviously, so we only take the last timestep to see what the model predicted
            logits = logits[:,-1,:] # this now becomes (B,C) instead of the initial (B,T,C)
            probs = torch.softmax(logits, dim=-1) 
            # now lets sample from it
            idx_next_char = torch.multinomial(probs, num_samples=1, replacement=True) # shape is (B,1)
            # now lets add this to the next input to be fed to the model 
            # since idx has the shape(batch, T), we should add this tothe second dimension
            # to the time dimension/ or sequence dimension. this as the loop goes on, 
            # creates the shape (B,T+1) 
            #this doesnt make sense for this particular model, becasue we are always checking
            # the next character given the previous one, so all the concatenation we are doing
            # is just useless. the reason we are implementing this like this, is to create a 
            # base, so that we can improve upon it when we add attention later on which will use
            # the history of previous characters.
            idx = torch.cat((idx, idx_next_char), dim=1) 
        
        # and finally when all is done return the idx which by now should have the whole output
        return idx
    
model = BigramModel(vocab_size)
out,loss = model(x,y)
print(f'{out.shape} {loss}')
# prints:
# torch.Size([32, 65]) 4.574285984039307
# the loss is good, we learned previously that we can evaluate a base loss provided our number of classes
# since we have 65 classes (our vocab_size or number of characters involved) the uniform probability for each class
# would be 1/65 =0.015384615, which if we take its negative log would turn out to be -ln(1/65) = 4.17438727, 
# which is pretty close to the loss we got here, signifying its a pretty decent value to begin with.
# also lets see the generate method at work
# lets create a dummy input, basically a batch of 1 and sequence of 1 of zero!
# we do this to generate a text from scratch
inputs = torch.zeros(size=(1,1),dtype=torch.int32)
output = model.generate(inputs, max_token_count=100)
print(f'{output=}')
print(''.join(decode(output.squeeze(0).tolist())))
# prints 
# wUbN:i :kLnOqHCQTG;.MbupOWH Xfi!MSUalaQppNNqIoWfmuSIIfmuZqf-3NLt-YkOc3vC-YkBfEHr3pJodwVY.nw.,r!&,Clt
# which is expected since our model is not trained yet! 
# lets train the model now and see how it works

batch_size = 32
context_size = 8
vocab_size = len(vocab_list)
max_iter = 20000

model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device=device)

print(f'{torch.__config__.show()}')
print(f'{device=}')
print(f'{model=}')

model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        print(f'{loss}')
# prints 
# 4.760574817657471
# 3.727213144302368
# 3.0139858722686768
# 2.6749324798583984
# 2.5884087085723877
# 2.5262675285339355
# 2.4420430660247803
# 2.4787800312042236
# 2.4680707454681396
# 2.564662456512451
# 2.51802659034729
# 2.562812328338623
# 2.422783613204956
# 2.5023772716522217
# 2.501049518585205
# 2.411632537841797
# 2.4028165340423584
# 2.4135568141937256
# 2.50952410697937
# 2.574878215789795
# relying on single batch loss is not a good idea to measure the performance of a model. moreover
# relying on training loss, is not good either, so it would be much better if we considered more batches
# for loss and even better we could also investivate the models performance on our validation set. 
# so lets do just this and define a function that calculates loss for training and validation sets alike
# but considers more batches for loss calculation

@torch.no_grad()
def evaluate_loss (iterations, device=None):
    results={}
    # before calculating the loss, lets switch to eval mode,although for our specific case this doesnt matter
    # but its goo practice, as later on, we will add layers that their behavior do change depending on traing
    # val mode.  
    model.eval()
    if device is None:
        # use the device assigned to what model params are assigned
        device = next(model.parameters()).device
    # since we already used no_grad decorator, we dont need to use no_grad context manager here
    # with torch.no_grad():
    for split in ['train','val']:
        losses = torch.zeros(size=(iterations,), device=device)
        for i in range(iterations):
            x,y = get_batch(split,batch_size)
            x,y = tuple(t.to(device) for t in (x,y))
            logits, loss = model(x,y)
            losses[i] += loss
        results[split] = losses.mean(0)
    # since we want to use this inside training loop, make sure we set the model back to train mode
    # incase we use layers such as batchnorm,etc that the require being trained!
    model.train()
    return results

loss= evaluate_loss(200)
print(f'{loss["train"]=} {loss["val"]=}')
# so now that our function seems to be working lets use it in the training loop :
model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
model = model.to(device=device)
model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        # now instead of simply printing loss, lets use our new function!
        loss= evaluate_loss(200)
        print(f'train_loss: {loss["train"].item():.4f},  val_loss: {loss["val"].item():.4f}')
# which results in :
# train_loss: 4.7491,  val_loss: 4.7430
# train_loss: 3.6673,  val_loss: 3.6670
# train_loss: 3.0460,  val_loss: 3.0511
# train_loss: 2.7335,  val_loss: 2.7492
# train_loss: 2.5948,  val_loss: 2.6156
# train_loss: 2.5292,  val_loss: 2.5470
# train_loss: 2.4982,  val_loss: 2.5194
# train_loss: 2.4807,  val_loss: 2.4991
# train_loss: 2.4737,  val_loss: 2.5036
# train_loss: 2.4650,  val_loss: 2.4949
# train_loss: 2.4648,  val_loss: 2.4910
# train_loss: 2.4626,  val_loss: 2.4836
# train_loss: 2.4570,  val_loss: 2.4893
# train_loss: 2.4517,  val_loss: 2.4823
# train_loss: 2.4508,  val_loss: 2.4819
# train_loss: 2.4564,  val_loss: 2.4839
# train_loss: 2.4555,  val_loss: 2.4802
# train_loss: 2.4503,  val_loss: 2.4927
# train_loss: 2.4522,  val_loss: 2.4905
# train_loss: 2.4572,  val_loss: 2.4888
#
# now lets check its output again after some training 
inputs = torch.zeros(size=(1,1), device=device, dtype=torch.int32)
output = model.generate(inputs, max_token_count=500)
print(''.join(decode(output.squeeze(0).tolist())))
# prints :
# Ane pod BUL:
#
# Oringe
# nfote s Rinsou oe the,
# An ETwe
#
# PUSENI bmilft!'d ate t Ifur
# Athomeg?
# Whagre,
# TCo belangead ne bl e, bednth ftor veso.
# US tla lin:
# Yourthiee he w whetitrd;
# Angoknghothartro ll heshethor hie douluk t, s se, denopit s h wom uspouct blyowie s'nos t prou gmesat
# Shd, tieithy.
# Asiour hofo bay avear hanousthaie
# Calonth wolulo.
#
# The hancondors pthighar:
# WAME outhaser d!
# S:
# h im t w s, inowo ORILOUCHoou isecetoubecompis
# Ay gnd, teve luthorea, therpeclsthevecthede fichim?
# PUSa VI aken 
# 
# which looks much better than the initial random output we got earlier.

dataset[:100]='First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
vocab_size=65, vocab_list=['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
atoi={'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't':

In [2]:
torch.manual_seed(255)
# now lets add attention mechanism to our base model. 
# attention mechanism at its core tries to take advantage of the rich information embedded
# in the sequence. for our work, this is specifically about the past history but in general 
# attention can utilize both past and future connections/sequence tokens. what we described 
# here just now is not exactly accurate, but gives us a foundation to build our intuition as
# we continue on. we will elaborate more of course and hopefully get it all.
# before we venture any further into the crux of the matter, let us learn about a technique
# thats used to efficiently implement attention mechanism.
# for this purpose,lets imagine we have a simple input like the following: 
# lets create and input of the following shape
B,T,C = (4,8,2)
# to make it more intuitive lets make a tensor with known numbers andthen reshape it
x = torch.arange(0,64,dtype=torch.float).view(B,T,C)
# imagine we have an input like what we encountered previously in our examples. in this sample 
# input, we have a batch of 4 samples, each having 8 sequences with each sequence having a vector of 2 values
# what we are planning to do is to provide a way by which each token can communicate with other 
# tokens. we have 8 tokens in our sequence. so we want our tokens to be able to communicate with
# all previous tokens that came before it. the reason we are only looking in the past token is 
# simply becasue the we are trying to perdict the future, so it only makes sense to look at the
# past and current timestamp and infer on what to do for the future.  
# so the easiest way to implement a kind of communication between tokens could be to sum or average the
# values of all previous tokens plus the current one as a way of taking into account their contribution
# to the final answer.
# that is, lets say if we are currently at token 5, we take the average of 
# the current token and all previous tokens before it, effectively making a feature vector that
# reflects our current status of the sequence so far, having taken all previous tokens/steps up to now.
# note that as you may also have thought, summing or averaging arent the best way to model such interations.
# in fact they are an extremely weak form of interaction between tokens,
# this kind of communicating is extremely lossy so to speak, that is we lose a great deal of information 
# concerning the underlying relationships between tokens, their arrangements,their implicit interactions,
# semantics, etc. but for now this is ok. we will later on see how to bring back such information.
# so now what we want to do, is to calculate the sum or average of all tokens up to the current token in 
# all batches at the same time.
# a naive way would be to do sth like this using a for loop:
results = torch.zeros(size=(B,T,C))
for b in range(B):
    for t in range(T):
        results[b,t] = x[b,:t+1].mean(0)
print(f'{results=}')
# which prints 
# results=tensor(
#        [[[ 0.,  1.],
#          [ 1.,  2.],
#          [ 2.,  3.],
#          [ 3.,  4.],
#          [ 4.,  5.],
#          [ 5.,  6.],
#          [ 6.,  7.],
#          [ 7.,  8.]],

#         [[16., 17.],
#          [17., 18.],
#          [18., 19.],
#          [19., 20.],
#          [20., 21.],
#          [21., 22.],
#          [22., 23.],
#          [23., 24.]],

#         [[32., 33.],
#          [33., 34.],
#          [34., 35.],
#          [35., 36.],
#          [36., 37.],
#          [37., 38.],
#          [38., 39.],
#          [39., 40.]],

#         [[48., 49.],
#          [49., 50.],
#          [50., 51.],
#          [51., 52.],
#          [52., 53.],
#          [53., 54.],
#          [54., 55.],
#          [55., 56.]]])
#
# You may find out that, some researchers refer to this operation here as BoW, or bag of words. 
# We are effectively averaging embeddings here and averaging embeddings can be considered a form of 
# Bag of Words (BoW) representation. In BoW, the focus is on the occurrence and frequency of words, 
# rather than their order or structure. By averaging embeddings, we are essentially treating each word 
# as an independent feature and capturing its representation in the form of a numerical vector.
#
# While averaging embeddings does not capture the exact frequency of each word, it does capture the 
# overall distribution and semantic information present in the text. Similar to BoW, this approach 
# disregards word order and focuses on the presence and representation of words. However, it should be
# noted that averaging embeddings may preserve some semantic relationships between words, which BoW 
# representations might not capture as effectively.
# 
# Side note: 
# Bag of Words (BoW) is a commonly used technique in natural language processing (NLP) for representing text
# as a numerical feature vector. It disregards the order and structure of words in a document and focuses only
# on their occurrence and frequency.
# In the BoW model, a document or a piece of text is represented as a "bag" (unordered set) of words, where 
# each word is treated as an independent feature. The presence or absence of words in the document is encoded
# as a binary value (0 or 1), and the frequency of each word is often used as the value in the feature vector.
# 
# Here's a step-by-step overview of the BoW process:
# 1. Tokenization: The text is split into individual words or tokens. Punctuation marks, whitespace, and other
#    special characters are usually removed or treated as separate tokens.
# 2. Vocabulary Creation: A vocabulary is created by taking all unique words from the entire corpus 
#    (collection of documents). Each unique word is assigned a unique index or position in the vocabulary.
# 3. Vectorization: Each document is represented as a feature vector, typically a one-hot encoding or a count
#    vector. In a one-hot encoding, each word in the vocabulary corresponds to a binary feature, and the vector
#    contains 1s in the positions where the word occurs and 0s elsewhere. In a count vector, the value at each 
#    position represents the frequency of the corresponding word in the document.
# 4. Classification or Analysis: The resulting feature vectors can be used as input to machine learning models
#    for tasks such as text classification, sentiment analysis, document clustering, or information retrieval.
# Needless to say, BoW has some limitations. It does not capture the semantic meaning or context of words, as
# it treats each word independently. It also ignores the grammar and word order. However, BoW is simple, 
# efficient, and can be a useful baseline representation for various NLP tasks.
# 
# so to recap one more time, we are basiaclly treating each timestep/sequence dimension, as a word, so we have
# 8 tokens/words, and we are averaging them (we are infact averaging their embeddings, but thats obvious!)
# so we got ourselves bow representation!
# now back to our discussion, if we  try to visualize the results we get 
print(x[0])
print(results[0])
# prints:
# x[0]=
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
#         [ 6.,  7.],
#         [ 8.,  9.],
#         [10., 11.],
#         [12., 13.],
#         [14., 15.]])
# results[0]=
# tensor([[0., 1.],
#         [1., 2.],
#         [2., 3.],
#         [3., 4.],
#         [4., 5.],
#         [5., 6.],
#         [6., 7.],
#         [7., 8.]])
# if you look closely, you'll notice that each row, contains the mean of all the rows before it
# consider the first 3 rows in x[0], 
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
# now refer to the 3rd row in results[0] which is :
#         [2., 3.],
# likewise, consider the last row in results which contains the average for all the rows in x:
# 0+2+4+6+8+10+12+14 = 56 which when divided by their count, 8, results in 7, 
# and this is the same for the second column, thus we get:
# results[0,7] = [7., 8.]])
# this all good but the problem is using for loops to calculate this is very inefficient, it
# happens that this operation can be efficiently calculated using matrix multiplication.
# lets learn this trick using an example: 
# suppose we have the following as the input: 
a = torch.ones(size=(3,3))
b = torch.randint(0,10,size=(3,2)).float()
c = a@b
print(f'{a=}')
print(f'{b=}')
print(f'{c=}\n----')
# prints
# a=tensor(
#        [[1., 1., 1.],
#         [1., 1., 1.],
#         [1., 1., 1.]])
# b=tensor(
#        [[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c=tensor(
#        [[20., 13.],
#         [20., 13.],
#         [20., 13.]])
# nothing fancy here, we have matrix multiplication, the first row of 'a' is dot-producted by first col
# of 'b', then sumed, it makes up the first col of first row in c. likewise the first row of 'a' dot 
# the second col of 'b', then summed the results, makes up the second col of first row in c. and this goes on for the rest of the matrixes. this is
# what we learned back in higheschool, so what is it exactly that we are learning exactly?
# if you look closely, you'll notice that, the c cols are actually the sum of all the rows in b!
#        [9]
# b[:,0]=[5] 
#        [6]
# 9+5+6 is 20! likewise, 
#        [3]
# b[:,1]=[5] 
#        [5]
# 3+5+5 is 13!
# hence c = [20., 13.] which is repeated obviously because the second and third rows of a are all 1s as well.
#           [20., 13.]  
#           [20., 13.] 
# I guess you are now starting to get where we are going with this, if we can some how alter the 'a' matrix,
# we may very well be able to achieve our goal! how you may ask? the answer is using torch.tril!
# torch.tril() is a function that returns a matrix from a given tensor, so that half of it set to zero,
# basially it creates a triangular tensor, where the right half is just zeros! lets see how it works, 
# lets apply it on 'a'
a_tril = torch.tril(a)
print(f'a_tril:\n{a_tril}')
# it prints
# a_tril=tensor(
#        [[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# as you can see the the right half is set to zero and we are left with a triangle shape of 1s! on the left side
# now if we do a@b this time we get:
c = a_tril@b 
print(f'b:\n{b}')
print(f'c:\n{c}')
# a_tril:
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[ 9.,  3.],
#         [14.,  8.],
#         [20., 13.]])
# now if you look closely, you'll notice that, this time, each row in c, is effectively the sum of the previous
# rows in b, like the first row of 'a' is only 1 in the 0ths column, so the first row of b is copied in c intact
# (workout the math and see why). 
# the second row in 'a', now has two 1s in col 0 and 1 respectively, which effectively translates to summing the first
# two rows in b. (9+5 =14, 3+5=8). likewise, the third row in 'a' is all 1s, signfigying all rows in b will
# be summed which gives us (9+5+5=20, 3+5+5=13). 
# so basically we are doing sums here, becasue our tensor a is all ones. so if we want to somehow calculate the
# average, instead of sum, we can easily change 'a' by normalizing it so that the each row sums to 1 (i.e. all cols
# sum to 1), this way the end result will be the average (becasue the 'b' is multiplied by a fraction/scale and then summed)
# so if we scale 'a' by the sum of all its columns, we should get average instead
a = torch.ones(size=(3,3))
a = torch.tril(a)
a = a/a.sum(dim=1, keepdim=True)
print(f'a:\n{a}')
# a:
# tensor([[1.0000, 0.0000, 0.0000],
#         [0.5000, 0.5000, 0.0000],
#         [0.3333, 0.3333, 0.3333]])
#
# note that, now each row, sums to 1. the first row, the first element is 1, because the rest are 0s
# but in the second row, as there are two 1s, the probabality is divided between the two, each being 0.5
# likewise, in the third row, as there are 3 1s, the probablity is divided between all of them, making each
# to have the value 0.33
# and now if we try to multiply them, we get average as the result:
c = a@b 
print(f'calculating average:')
print(f'b:\n{b}')
print(f'c:\n{c}')
# we get 
# calculating average:
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[9.0000, 3.0000],
#         [7.0000, 4.0000],
#         [6.6667, 4.3333]])
# we see that, each row in c, is the average of all the rows before it. 
#
# so using this trick, we can take the incremental average of any matrix we like. 
# now that we learned the trick, lets go back and implement the bows for loops using this techique !
# prevbiously we had : 
# results = torch.zeros(size=(B,T,C))
# for b in range(B):
#     for t in range(T):
#         results[b,t] = x[b,:t+1].mean(0)
# which calculated the average for sequence dimensions (tokens) incrimentally
# lets do this now 
# we want a TxT weight becasue we want to average T timestep/tokens
weight =  torch.tril(torch.ones(size=(T,T)))
# remember to set keepdim=True, or otherwise, as we sum along dim=1, we lose that dim
# (it collapses, and then the broadcast will be wrong, it will infact make each column 
# have one probablity instead of each row, which is the exact opposite of what we want)
weight = weight/weight.sum(dim=1, keepdim=True)
print(f'weight:\n{weight}')
# and now we need to multiply this by x! 
bow_results= weight@x
print(f'bow_results:\n{bow_results}') 
# which prints 
# bow_results:
# shape: torch.Size([4, 8, 2])
# tensor([[[ 0.0000,  1.0000],
#          [ 1.0000,  2.0000],
#          [ 2.0000,  3.0000],
#          [ 3.0000,  4.0000],
#          [ 4.0000,  5.0000],
#          [ 5.0000,  6.0000],
#          [ 6.0000,  7.0000],
#          [ 7.0000,  8.0000]],

#         [[16.0000, 17.0000],
#          [17.0000, 18.0000],
#          [18.0000, 19.0000],
#          [19.0000, 20.0000],
#          [20.0000, 21.0000],
#          [21.0000, 22.0000],
#          [22.0000, 23.0000],
#          [23.0000, 24.0000]],

#         [[32.0000, 33.0000],
#          [33.0000, 34.0000],
#          [34.0000, 35.0000],
#          [35.0000, 36.0000],
#          [36.0000, 37.0000],
#          [37.0000, 38.0000],
#          [38.0000, 39.0000],
#          [39.0000, 40.0000]],

#         [[48.0000, 49.0000],
#          [49.0000, 50.0000],
#          [50.0000, 51.0000],
#          [51.0000, 52.0000],
#          [52.0000, 53.0000],
#          [53.0000, 54.0000],
#          [54.0000, 55.0000],
#          [55.0000, 56.0000]]])
# which gives us the same results as we expected.
# one more thing before we continue on, note that the weight matrix is TxT while 
# the input is (BxTxC). (T,T) and (B,T,C) are not compatible, so what happens is
# that (T,T) is reshaped and a batch dimension is added to (T,T),making it (1,T,T)
# and then this is broadcasted along the batch dimension (replicated) to become 
# (B,T,T), then this will be multiplied by the (B,T,C)( note that at this stage
# a@b will be a batch multiplication operation, if you set the batch dimension aside, youll
# see that the rest of the dimensions match up, we have (T,T) and (T,C) which will
# result in (T,C). now if we add the batches back in, we will endup with (B,T,C)
# so in practice, the multiplication is done B times and then results are stacked.
# the first batches will multiply each other (T,T)x(T,C) = (T,C)
# the second batches will then multiply each other as well, getting another (T,C)
# and this goes on until we get (B,T,C))
#
a = torch.arange(0,9).view(3,3)
# a = torch.ones((3,3)).long()
b = torch.arange(0,30).view(2,3,5)
print(f'a:\n{a}')
print(f'b:\n{b}')
c = a@b 
print(f'c=a@b:\n{c}')
# is equivalent to 
# add a batch dimension to a
a = a.view(1,*a.shape) # or a.unsqueeze(0)
print(f'a with batch dim:\n{a.shape=}')
# replicate along the batch dimension
a = torch.cat(tuple(a.clone() for i in range(len(b))), dim=0)
print(f'{a.shape=}')
print(f'a(after replication along dim=0):\n{a}')
print(f'b:\n{b}')
# and now we have a case of batch-multiplication, the batch is the same 
# and sub tensors also are compatible, we have (T,T) and (T,C) so we now
# individually multiply each sub-tensor
c2 = torch.zeros_like(b)
for i in range(b.shape[0]):
    c2[i,...] = a[i]@b[i] # (T,T) x (T,C) -> (T,C)
# and ultimatley the result will have the shape (B,T,C)
print((c==c2).all())
# so to recap this 
# When performing the matrix multiplication `c = a @ b` with the given tensors, 
# several broadcasting steps occur to align the dimensions properly. 
# Here's a how it happens:
# 1. Tensor a has the (shape: 3, 3):
# 2. Tensor b has the (shape: 2, 3, 5):

# 3. They are not compatible so we broadcast tensor a to match the shape of b. 
# tensor a is expanded to (1, 3, 3) to have a batch dimension, and replicated 
# along dim 0 to result in (2, 3, 3). now both tensors have a batch of 2, and 
# if we put aside the batch dimension for a second, we'll notice that the rest
# of the shapes are compatible (3,3) and (3,5). so when we bring back the batch 
# dimension, we see we have a case of batch multiplication, that is we have 2
# sets of compatible tensors that need to be multiplied together. so we use a 
# simple for loop, to do multiplication, and stack the results and we are done!
#
# now back to our discussion. as we just saw, the traingualr shape in our weighted
# sum matrix, allows that each token at t dimension, can only interact with the tokens
# before it.
# there is another way of implementing the same thing but a bit differently
# if you look closely you can see that, we are dealing with probablities, so
# we may verywell use softmax to simplify this further.
# first lets create our triangular weight matrix
tril_tensor = torch.tril(torch.ones(size=(T,T)))
# now lets create a mask
weight = torch.zeros_like(tril_tensor)
# now lets mask all the zeros to -inf, so when we do softmax, all those -infs
# become 0. (recall that exp^-inf is 0! while exp^inf is inf! so its important to 
# set -inf (and also exp^0 is 1))
# so effectively what happens here is that, we setting each entery in trail_tensor
# with zero to -inf, and leave the rest as zeros. when this tensor goes through softmax
# the 0s will be 1s and -infs will be 0s before they are normalized, when they are normalized
# the probablity of 1 will be split between all the enteries with the value of 1.
# so the first row has a single 1, so it will be 1.0, the second row has two 1s, so
# each one will take 0.5, the third will have 3 1s, so they each will become 0.333
# and so on.
weight=weight.masked_fill(tril_tensor==0, -torch.inf)
# now calculate the probs for each row, treating all cols as probs so their sum is 1
weight = weight.softmax(dim=-1)
bow_results2 = weight@x
print(f'{torch.all(bow_results==bow_results2)}')
# so you might ask, why would we want to make things more complicated like this to only
# achieve what we already achieved pretyy efficiently before?
# the answer is, this approach provides us with a flexibility that the previous ones
# wouldnt provide us withs. if you think about it for a moment, you'll notice that
# the 'weight' matrix can essentially be anything and not just zeros! 
# we have decoupled it from tril, which's job is to set the right half of the tensor
# to zero, so we actually get the expected behavior later on.(which is setting a constrain really (more layer on))
# if you havent yet figured it out, the 'wieght' matrix, can be truly a weight matrix
# which can show different strengths for each token, basically it can learn the interations
# between tokens and manifest them. currently it is us who sets it to all zeros, so we get
# uniform probablities, which inturn manifest itself as an average. but what if instead of 
# all zeros, we actually learn the values from the data itself? that would make sense, and 
# make it so that each token, can have a different connection(strength) to any other tokens
# now, and thus build semantic/meaningful relation. this is inafct what we are after. we want
# for tokens to learn associations and relations with other tokens based on the data present
# in the dataset, having uniform weight like what we initially did really is a far cry from
# what we intend, and therefore, we opt in to use this new approach that allows us to actually
# exploit this new capability. 
# This is infact the problem that attention solves, that is gathering
# information from the past but in a data driven manner.(side note, we are not limited to 'past' 
# information only per say, in this example, this is the case however, we will explain this 
# in more detail)), we will see how attention does this exactly in a moment. 
#
# but before we jump into attention implementation also note that the tril part, infact is a hard-constrain here that prevents tokens from
# the past from interacting with the tokens from the future. 
# so to recap here, basically the idea is, this triangular form, allows us to have 
# weighted aggregations of past elements. each element in the lower triangular part, 
# specifies, the degree by which it plays a rule in the said outcome. 
# that is how much of each element gets to fuse into this specific position(i.e. current token's)
# so now lets incorporate attention into our model
# Attention does its job by using two vectors called, key and query.
# basically every single token, emits two vectors called key and query, the query verctor
# as the name suggests, implies, what we are looking for, and the key vetcor, again as the
# name suggets, implies, the contents, what it contains. 
# the way we get our 'weights' for these tokens is we simply dotproduct them together, 
# the ones that yield a high output/value, signify they are related positively.
# so our query is dotproducted with all the token's(keys) and the outputs reveal their 
# closeness/relevancy/similarity so to speak(if they align so to speak, they result in larger number)
# so effectively, the ones with higher number, are more similar to the query, the highest, 
# obviously having the most relavancy/similarity to the query.s
# so lets implemenet this
# #%%
# we want to implement a single attention head, we can later use this to create 
# multi-attention-head which is basically several single attention heads working in 
# parallel. so how do we implement one! 
# first lets create some random input 
torch.manual_seed(255)
# lets specify the batch, context_size and vocab_size
B,T,C = 4,8,32
# lets create an input 
x = torch.randn(size=(B,T,C))
# we said that attention works with two vectors, key and query, lets implement them
# we can use nn.Linear to implement them but beore that, what are the dims of such vectors,
# we are dealing with text and thus our inputs are tokens/vocabs, so the first dim would be 
# our vocab_size to account for all tokens, the next dim, is sth called a head_sizes, which 
# specifies the size of the key/query output usually 16 is used a lot for head size so we 
# use that as well
head_size = 16
# lets not forget to set bias=False, so what it does is exactly dotproduct 
# (actually, some people still leave the bias enabled! 
# so we will test both cases later and see if it actually matters! and if so to what extend! ) 
key = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
# now lets get the output
k = key(x)      # shape : 4,8,16 or (B,T, head_size)
q = query(x)    # shape : 4,8,16 or (B,T, head_size)
 # so the dims arent compatible for batch-multiplication, so we need a transpose
 # k.T wouldnt work becasue we have a batch-dim, so instead we use .transpose and
 # explictily specify the dims we want to be transposed. 
 # this will result in 4,8,16 by 4,16,8 which would give us 4,8,8 which is (B,T,T)
 # really as the result
 #! check transpose result is it okto use 2,1 or -2,-1 or 1,2
weight_raw = q@k.transpose(2,1)
# now lets for a moment think about what is happening here, the key and query are applied
# on the input and each return an output of (B,T,head_size), they are in fact, processing
# all the tokens in the input, individually, simultaneously, all the same time. so each 
# token is both a query, and a key, and when we do a dotproduct, we are basically telling 
# it to reveal the relation/similarity/relevance of every token with every other tokens.
# and as we explained earlier, this is infact our weight matrix (which was initially zeros)
# but is now learned from the data!
# print(f'weight_raw\n{weight_raw}')
# now we can apply constrain on it so that tokens can only communicate with the past so 
# we use the tril trick now!
tril_constrain = torch.tril(torch.ones(size=(T,T)))
raw_wieghts_masked = weight_raw.masked_fill(tril_constrain==0, float('-inf'))
# apply sotmax to get probablity for each token
weight = raw_wieghts_masked.softmax(dim=2) # remember weight is (B,T,T)
# and finally we can apply our weight on the input (we called it raw for a reason, read on)
# (by the way this is also called self-attention!)
bow_raw = weight@x 
# lets print raw_weights and weights and have some intuitive observations 
print(f'weight_raw\n{weight_raw}')
print(f'raw_weights_masked: {raw_wieghts_masked}')
print(f'weight\n{weight}')
# prints
# raw_weights(unconstrained)
#weight_raw
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00, -1.7903e+00,  1.4650e-01, -1.4147e+00, -1.7452e+00, -4.5249e+00, -1.9763e+00,  1.8833e+00],
#          [-1.3490e+00,  9.4076e-01, -6.9690e-01,  7.9933e-01,  4.7634e-01,  4.1861e-01,  2.3663e-01,  4.5217e-01],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,  9.0092e-01,  1.4083e+00,  2.5488e+00,  1.6350e+00,  1.0048e+00],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01, -1.1213e+00,  2.7562e+00,  7.0136e-02, -2.0337e+00],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,  4.9236e+00,  3.9545e+00, -2.2600e-01],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01, -1.5979e+00,  8.4627e-01],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,  2.2338e+00],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00, -2.1323e+00, -1.3948e+00, -5.6973e-01, -2.7986e-01,  4.3579e+00,  4.8477e-01, -9.7559e-01],
#          [ 8.7786e-01, -2.3352e+00, -2.2988e+00,  1.0322e+00, -1.4231e-01,  5.5233e+00,  1.0819e+00, -2.4722e-01],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,  2.9387e-02, -5.6397e-01, -2.4025e-01, -4.2512e-02, -8.4302e-01],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01, -4.0429e+00, -1.0676e+00, -3.0694e+00, -1.6156e+00],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00, -4.5593e+00, -3.1755e-01, -1.0504e+00],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00, -3.5616e-01, -2.6951e+00],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,  4.9077e-02],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01, -4.7902e-01,  6.7590e-01, -2.3153e-02,  1.9763e-01,  4.4385e-01,  7.2986e-01, -3.8846e-01],
#          [-6.2810e-01, -8.3713e-01,  1.3468e-01, -5.1328e-01, -2.6653e-02,  3.4325e-01, -1.0015e+00, -1.0334e+00],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01, -1.9702e+00, -1.0641e+00,  1.7792e-01,  8.0194e-01, -1.5265e-01],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,  3.6759e-01,  1.0552e+00, -5.2949e-01, -1.7715e+00],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01, -3.0057e+00,  1.3784e+00, -1.2091e+00],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02, -1.4040e+00,  2.3786e+00],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00, -1.6068e+00],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<UnsafeViewBackward0>)
#
# raw_masked_wieghts(constrained):
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.3490e+00,  9.4076e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01,        -inf,        -inf,        -inf,        -inf],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,        -inf,        -inf,        -inf],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01,        -inf,        -inf],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,        -inf],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 8.7786e-01, -2.3352e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01,        -inf,        -inf,        -inf,        -inf],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00,        -inf,        -inf,        -inf],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00,        -inf,        -inf],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,        -inf],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-6.2810e-01, -8.3713e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,        -inf,        -inf,        -inf,        -inf],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01,        -inf,        -inf,        -inf],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02,        -inf,        -inf],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00,        -inf],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<MaskedFillBackward0>)
# 
# weight (final- normalized)
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.1976e-02, 9.0802e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9773e-01, 6.0261e-01, 1.9966e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.4181e-02, 3.4422e-02, 3.7221e-01, 5.7918e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [4.6023e-02, 1.9501e-03, 5.3236e-01, 3.5612e-01, 6.3550e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9494e-01, 1.9195e-01, 2.7619e-01, 2.8253e-02, 1.3648e-01, 1.7219e-01, 0.0000e+00, 0.0000e+00],
#          [4.5214e-01, 9.6007e-02, 7.3108e-02, 5.3035e-02, 2.8887e-01, 8.7621e-04, 3.5963e-02, 0.0000e+00],
#          [6.8046e-02, 5.1605e-01, 5.0474e-03, 1.5304e-02, 2.9133e-01, 3.8823e-04, 1.3775e-02, 9.0057e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.6132e-01, 3.8675e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.1870e-01, 3.3895e-01, 4.4235e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.2894e-02, 6.5398e-01, 2.4345e-01, 7.9674e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [3.7332e-02, 6.5690e-01, 7.8427e-02, 5.1206e-03, 2.2222e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.3611e-01, 1.7995e-02, 2.1887e-02, 1.1900e-01, 1.8219e-02, 6.8680e-01, 0.0000e+00, 0.0000e+00],
#          [9.8946e-02, 1.4120e-02, 1.8065e-02, 9.5010e-03, 3.2130e-01, 9.9891e-02, 4.3817e-01, 0.0000e+00],
#          [2.3306e-02, 3.5621e-02, 9.5163e-02, 1.5801e-01, 1.1530e-02, 1.7183e-01, 4.1141e-02, 4.6340e-01]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.5207e-01, 4.4793e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.8708e-02, 6.5816e-01, 2.4313e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.8422e-01, 9.1987e-02, 1.2557e-01, 5.9822e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.0870e-01, 8.3642e-02, 4.8896e-02, 3.2723e-01, 3.3154e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.4141e-02, 7.9271e-02, 2.1893e-01, 2.2608e-01, 2.3782e-01, 1.4376e-01, 0.0000e+00, 0.0000e+00],
#          [7.8726e-02, 1.1815e-01, 3.7190e-01, 1.2607e-02, 3.6387e-02, 1.1790e-01, 2.6434e-01, 0.0000e+00],
#          [5.6646e-02, 7.4575e-02, 6.8217e-02, 3.1568e-02, 5.9024e-02, 1.1073e-01, 3.1662e-02, 5.6757e-01]]], grad_fn=<SoftmaxBackward0>)
#
# as you can see, our weight is initialized for each batch based on the input data, and each token has its own
# weight, that is they are not uniform!, to get a better understanding lets consider the first batch of 
# raw-weights[0], raw_masked_wieghts[0] and weight[0]:
#
# raw_weights[0]
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
#
#
# raw_masked_wieghts[0]
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
# 
# weight[0]
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
# take the last token, which 8.6020e-02, notice this token, not only knows its content and own position in 
# the sequence(its the 8th token after all) but also knows which tokens comes before it and how much its 
# related to any of them.(basically when it knows which token comes before it, it creates its own query so
# to speak, and talks to every single previous token to find about which ones are more or less relavent to
# it and to what extend.) 
# For example,lets say, (since we are dealing with character level text generation!), the last token is a 
# vowel and says hey im a vowel and im looking for everyone else thats a vowel! and uses query to talk to 
# other tokens(keys). and intrestingly another token (lets say number 4) says im a vowel as well, and the 
# response is thus generate a larger number.(this is not a good example, words in sentence would make more sense
# !give a better example!)
# in our example, For the last token, it seems the 5th and 3rd token are particularly intresting/relavent/important,
# followed by the 7th token.
# likewise, this happens for every token, i.e. token number 4, says the same thing and searches in its past toekns
# so on and so forth! 
# so what happens next is that when we get a high relevancy scale /response in the weights, like we just described
# when we do a softmax it will assign a large probablity to them, and this instructs the network that, we need more
# information from them, effectively allowing for aggregating a lot of their information into our position(lets say e.g. 8th token. 
# and we happen to learn more about them this way.
# now in practice, we are not intrested in aggregating the x raw values per say, rather we want their information, 
# so instead of just using the raw values of x, we instead use a representation of them, so to speak. 
# this is achieved using a third vector known as, 'value' and is the last vector we use. 
# !explain when we say vector, note that, we are talking from the prespective of a token, (every token has some vector 
# !of information and it gets to aggregate information via a weighted sum from all the tokens that point to it, and it
# )# !in practice we use a matrix, and hence the linear module to implement this to run the operation for all tokens in parallell. 
# just like the key and query, we set its bias to False, so we only get a simple vector,
# and a dotproduct output (again this is the intuition, but how much adding a bias would
# affect this we will see later, for now we stick to the default no bias version)
value = torch.nn.Linear(C,head_size, bias=False) # produces (B,T,head_size) just like the other two key,query vectors
x_processed = value(x)
# this is not yet final final, we still need to do one more thing (read on!)
bow_final = weight@x_processed
#
#!check also note that, as we previously once pointed out, attention is a communication mechanism between tokens, and 
# by default there is nothing in this mechanism that provides a notion of space/position for tokens involved, i.e.
# by default these tokens/nodes/points, dont have any idea about where they are or how they are positioned (with resepect
# to others) etc, so we need to encode this information as well.(to better visualizing it, consider each token as 
# a node in a directed graph, each node has a connection to another node, (for example each node has a connection to
# itself, and another connection to other nodes,etc) and as you can see there is no notion of space here,the attention
# simply acts on a set of vectors in this graph, and thats why we need to encode them positionally as well, so they 
# have information about their position with regards to other tokens. 
# !check (compare this to the convolution case and images
# or even text where the spatial aspect of data is preserved, but in attention, as we just stated, there is no notion
# of position, its just a set of individual vectors being operated on)-note that the connection between nodes, do give
# us a sense of structure, but it may not be enough to infer the underlying semantic, imagine a case, where a word
# for example has several meaning, and may very well have high relevancy to some tokens at the same time, but without
# additional positional information, an ambiguous semantic can be infered between the tokens involved, however when
# positional information is also present, such ambiguity can be avoided)
# also note that, in attention, samples do not interact with each other at all. when we have a batch of 4, each sample
# is processed in isolation, but in parallell to other samples, based on our graph example earlier, we would have 4 
# graphs of 8 nodes for example for each input sample. 
#
# we said earlier that what we implemented here is known as self-attention, the reason it is called self attention
# is that the key and query and values are applied on the same input(the use the same source!), and hence the name,
# self attention.
# also note that, in our specific case, tokens/nodes are can not communicate with the future nodes, but in general
# this constraint can be removed (and infact is removed/not implemented for some applications) where its benificial
# to be able to communicate with all the tokens. one example is sentiment analysis, where you want all the tokens to
# able to communicate with eachother so you can get an accurate analysis. and for this case, we would use an encoder
# block, which is basically what we have here, minus the constraint section(tril/mask part), what we have implemented
# here is called a decoder block, where we are decoding bunch of tokens, and it makes sense that the previous tokens
# do not comunicate with the future ones(becasue they would give the answer! and it defeats the whole purpose here!),
# becasue its a given that only the previous tokens must be used to predict the future/next token, hence the filtering/constraint part to prevent tokens from 
# comunicating with the future nodes/tokens.
# !so far we explained about the self-attention, which we saw, is called that way solely for the fact that key, query
# and value use the same source. the attention mechanism as we briefly pointed out, is much more general and can be
# used in different ways. one of such ways, is what is used to create sth called cross-attention. 
# cross-attention basically refers to the case where we have an encoder/decoder blocks, in which the queries come from x
# but the key and value come from an external source and sometimes from the encoder block. so cross-attention is used
# when theres a separate source of information we would like to pool from and use it as well.
#
# so far we implemented the attention based on the original paper(there are some differences we get to later on)
# except the part where we need to divide by the sqrt of the head_size. that is called scaled-attention
# so lets talk about this, and see why its needed. 
# the reason we add this so called 'scale' to our computation, is that, without it, the probablities will be saturated
# and when we add this term to the mix, it will make the 'weight' matrix to be 'unit variance', when Q and K are unit variance
# and this allows softamx to stay diffuse and not saturate too much.
# in other words, if we simply multiply key and query like that, the variance of the resulting weight matrix will be
# around the head_size instead of 1 which is bad and makes optimization really hard.
# to see this effect consider the following example
k = torch.randn(size=(B,T,head_size))
q = torch.randn(size=(B,T,head_size))
w_unscaled = q@k.transpose(-2, -1) 
print(f'{k.var()=}')
print(f'{q.var()=}')
print(f'{w_unscaled.var()=}')
# now add the 1/sqrt(head_size)
w_scaled = q@k.transpose(-2, -1) * head_size**-0.5 
print(f'after applying 1/sqrt(head_size)')
# makes the weight variance 1!
print(f'{w_scaled.var()=}')
# why is it important? if you recall, the weight matrix is fed into softmax, so its really important, especially during
# initialization that weight matrix be fairly diffuse, if we look at weight matrix here, we'll notice that they are now
# fairly diffuse 
print(f'weight_scaled[0]:\n{w_scaled[0]}')
# prints 
# weight_scaled[0]:
# tensor([[ 0.3972, -2.0957, -0.5396, -1.4860, -0.8393,  0.6273, -0.0157,  0.6284],
#         [ 1.3588, -0.0440,  2.1040, -0.2796,  1.7797,  1.4362,  1.3843,  0.0368],
#         [ 0.8109,  1.7400,  1.3579,  0.2097,  1.7152, -1.1242, -0.4349, -0.5690],
#         [ 0.0513,  2.8303,  0.7332,  0.0041,  0.9688, -1.6174, -1.4255,  0.2869],
#         [-0.7819, -0.6691, -1.4017, -0.4155, -0.8568, -0.2704, -0.9453,  0.3763],
#         [ 0.4092, -0.3079,  0.8472, -1.1753, -0.7699,  0.5091,  0.6385,  0.9677],
#         [ 0.3043, -2.1402, -2.2582, -0.3573, -1.2838, -0.0260, -0.0513,  0.9151],
#         [ 0.4180, -2.8353, -0.6435,  0.4842, -3.0202,  1.7690,  1.8837, -0.4393]])
#
# when softmax is applied:
print(w_scaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.8026, 0.1974, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1901, 0.4814, 0.3285, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.0499, 0.8038, 0.0987, 0.0476, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1989, 0.2226, 0.1070, 0.2869, 0.1845, 0.0000, 0.0000, 0.0000],
#         [0.2148, 0.1049, 0.3329, 0.0440, 0.0661, 0.2374, 0.0000, 0.0000],
#         [0.3027, 0.0263, 0.0233, 0.1562, 0.0618, 0.2176, 0.2121, 0.0000],
#         [0.0901, 0.0035, 0.0312, 0.0962, 0.0029, 0.3478, 0.3901, 0.0382]])
#
# 
# now compare it with the unscaled weight : 
print(f'weight_unscaled[0]:\n{w_unscaled[0]}')
# weight_unscaled[0]:
# tensor([[  1.5889,  -8.3829,  -2.1583,  -5.9440,  -3.3573,   2.5092,  -0.0629,  2.5135],
#         [  5.4350,  -0.1759,   8.4159,  -1.1183,   7.1189,   5.7446,   5.5373,  0.1472],
#         [  3.2435,   6.9600,   5.4317,   0.8388,   6.8607,  -4.4969,  -1.7396, -2.2761],
#         [  0.2051,  11.3214,   2.9327,   0.0165,   3.8754,  -6.4696,  -5.7019,  1.1475],
#         [ -3.1278,  -2.6763,  -5.6069,  -1.6621,  -3.4272,  -1.0816,  -3.7811,  1.5053],
#         [  1.6368,  -1.2317,   3.3888,  -4.7012,  -3.0795,   2.0364,   2.5541,  3.8710],
#         [  1.2171,  -8.5610,  -9.0327,  -1.4293,  -5.1353,  -0.1038,  -0.2053,  3.6603],
#         [  1.6719, -11.3411,  -2.5740,   1.9369, -12.0810,   7.0760,   7.5347, -1.7573]])
#
# when softmax is applied:
print(w_unscaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [9.9636e-01, 3.6442e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.9592e-02, 8.0566e-01, 1.7475e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.4865e-05, 9.9975e-01, 2.2738e-04, 1.2309e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2943e-01, 2.0328e-01, 1.0848e-02, 5.6050e-01, 9.5940e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2012e-01, 6.8207e-03, 6.9264e-01, 2.1236e-04, 1.0748e-03, 1.7913e-01, 0.0000e+00, 0.0000e+00],
#         [6.3261e-01, 3.5856e-05, 2.2371e-05, 4.4856e-02, 1.1023e-03, 1.6883e-01, 1.5254e-01, 0.0000e+00],
#         [1.7351e-03, 3.8710e-09, 2.4850e-05, 2.2616e-03, 1.8472e-09, 3.8572e-01, 6.1020e-01, 5.6236e-05]])
#
# as you can see, some values are very negative, while others are very positive, for example, we have both 11 and -11
# and this is a recipe for disaster! the reason it is a problem is that, with softamx, when numbers take very positive
# and nagative values, softmax actually converges towards one-hot-vector. basically the values shrink towards the max.
# this can be seen the above, but the example below should demonstrate it more clearly.
# lets apply softmax on a list of numbers that are close to each other, we see the probablities are difuse and normal
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2]),dim=-1)}')
# we get a diffused probablity out of softmax
# tensor([0.2562, 0.3822, 0.1717, 0.1898])
# however if we increase the magnitude of the numbers(sharpen them) (and thus the difference between them) by like multiplying by 
# a number like 8 (just to simulate the effect here and so we can compare it with the previous case)
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2])*8,dim=-1)}')
# we see that the softmax, starts to sharpen towards the max, shrinking others except the 
# largest number, and effectively
# converging toward a one-hot-encoded vector.
# tensor([0.0390, 0.9559, 0.0016, 0.0035])
# so we dont want these values to be extreme, especially during initialization or otherwise, softmax will be way too picky!
# and we are basically aggregating the information from a single node instead of multiple ones (becasue one has the largest
# nvalue, its as if our sequence length is 1! and we lose access to the wealth of information the past history offers)
# so we want the probablities to be diffuse and not peaked like the second example here.
# so this scaling is used to retain the variance at a good value especially at initialization.
# so now that we are finally finished the self attention head, lets implement it as a module and incorporate everything
# we just discussed here.
class AttentionHead(nn.Module):
    def __init__(self, context_size, embd_size, head_size=16, use_bias=False) -> None:
        super().__init__()
        # we need context_size or block_size for creating the tril constrain
        self.context_size = context_size
        # we want this as the input dim for our key,query and value, this is 
        # infact the input_dim, since we plan on using the embeddings, we named
        # it embd_size
        self.embd_size = embd_size
        # head_size is the output dimension of our attnetion
        # !note that usually the head_size is equal to embd_size
        self.head_size = head_size
        self.key = nn.Linear(embd_size, head_size, bias=use_bias)
        self.query = nn.Linear(embd_size, head_size, bias=use_bias)
        self.value = nn.Linear(embd_size, head_size, bias=use_bias)

        # use buffer for trail, since this is a buffer, we can use the self.register_buffer which is
        # inherited from nn.Module class, to add it as buffer to the module, so it can be saved in the
        # state_dict when we save the model.  tril doesnt change, and is not updated(its required_grad is false),
        # so it being saved in state_dict doesnt matter to us, but to demonstrate this feature of pytorch, 
        # we are using it, and its a good practice, since pytorch knows how to deal with it and wont include it
        # in the computaion graph anyway!
        # 
        # This is typically used to register a buffer that should not to be considered a model parameter. 
        # For example, BatchNorm's running_mean is not a parameter, but is part of the module's state. 
        # Buffers, by default, are persistent and will be saved alongside parameters. This
        # behavior can be changed by setting persistent to False. The only difference between a persistent
        # buffer and a non-persistent buffer is that the latter will not be a part of this module's state_dict.
        # tril allows us to impose constrain by creating a lower traingualr matrix and setting
        # the rest entries to zero, effectively preventing tokens of the future from communicating with the past
        # each token can only communicate with the previous tokens that came before it.
        self.register_buffer('tril', torch.tril(torch.ones(context_size,context_size)))

    def __call__(self, inputs:torch.Tensor) -> torch.Tensor:
        #! check the shapes!! 
        B,T,C = inputs.shape # 4,8,16
        # print(f'{inputs.shape=}')
        # create the weight by using k,q,v
        k = self.key(inputs)    #! (B,T,C) or (B,E,16)? (4,8,16)
        q = self.query(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        v = self.value(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        # create the weight matrix and scale it by 1/sqrt(head_size) to keep weight unit variance 
        weight = q@k.transpose(-2,-1)* self.head_size**-0.5 # !(B,E,E)
        # weight_C = q@k.transpose(-2,-1)* C**-0.5 # !(B,E,E)
        # print(f'weight.var: {weight.var().item():.4f}')
        # print(f'weight_C.var: {weight_C.var().item():.4f}')
        # print(f'x:{tuple(inputs.shape)} k:{tuple(k.shape)} q:{tuple(q.shape)} v:{tuple(v.shape)} w:{tuple(weight.shape)} hs:{self.head_size}')
        # apply the tril constrain - (this makes this a decoder block!)
        # important note: notice we used tril[:T,:T] and not simply tril
        # this is because, when the input has a small context_size < self.context_size
        # like when we wantto generate inputs with sth like zeros((1,1)) which says
        # there is a single token (context_size 1) of 0 as the begining of the sequence
        # when this is input, the tril by default makes a context_size,context_size matrix
        # which will be different than the input context_size which is 1 e.g. or 2 e.g.
        # and it will fail becasue our weight would be 1,1,1 or 1,2,2, but trail is 1,8,8
        # and clearly this will cause an error. for this reason, we always create the 
        # tril check dynamcally by explicitly specifying the context_size based on the 
        # current input context_size so in case the context_size is smaller, tril is resized
        # dynamically accordingly. 
        # print(f'tril[:T,:T]==0: {(self.tril[:T,:T]==0).shape}')
        # print(f'tril==0: {(self.tril==0).shape}')
        weight = weight.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        # weight2 = weight.masked_fill(self.tril==0,float('-inf'))
        # print(f'{weight.shape=}')
        # print(f'{weight2.shape=}')
        # note that we are using batch, so instead of hardcodin 2,
        # we use -1 to refer to the last dim
        weight = weight.softmax(dim=-1)
        # finally apply the weight on the v
        bow = weight@v  # !(B,E,16)
        return bow        

# now lets add this to our model 
        
# so to recap, we first calculated the relavancy between all tokens against eachother, then constrained them so that 
# each token can only use the information from/interact with its past tokens. then since we needed probablity distribution so
# we then normalized it and then used that to pickout which tokens information(in the past) to aggregate/use with the inputs to 
# achieve our goal.(which is to predict the next character based on everything seen so far!)
# we dont want to aggregate inputs value (with respect to the weight matrix), we instead would like to use their representation
# so we use a new layer to do this, its called value, and we instead use its output instead of inputs raw value.
#
# 
#
# before we jump in and add the attention module, lets review our base model and see how we can 
# improve/prepare it before we incorporate the attention. 
# class BigramModelWithAttention(nn.Module):
#     def __init__(self, vocab_size, embd_size) -> None:
#         super().__init__()
#         self.vocab_size = vocab_size
#         self.embd_size = embd_size
#         # unlike the previous model, lets decouple the final logits from 
#         # the number of embeddings, because we are using attentions, and
#         # we want to have multiple operations inbetween obviously.
#         self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
#         # in order to get the final logits, we need a linea layer at end
#         self.fc = torch.nn.Linear(embd_size, vocab_size)
#        
#     def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
#         out = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
#         logits = self.fc(out)         # has the shape (B,T,C) c is vocabsize
#         loss = None
#         if labels is not None:
#             # recall that crossentropy likes its input to be B,C,T and we are B,T,C
#             # so lets permute and make it happy!
#             loss = F.cross_entropy(logits.permute(0,2,1), labels)
#         return logits, loss
#    
#     def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
#         # lets generate an output as long as num_max_token
#         for i in range(max_token_count):
#             # make sure idx is 2d
#             assert len(idxs) >1, f"idx.shape '({tuple(idxs.shape)})' is invalid. it must have the form (B,T)"
#             # now lets feed it to the model and sample from the probablities it produces
#             preds,_ = self(idxs)
#             # convert to probs 
#             probs = preds.softmax(dim=1)
#             # since we are bigram still, lets only get the last token as the next token predicted!
#             probs = probs[:,-1,:]
#             # now lets sample from it 
#             new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
#             # now concatenate the new token to the previous one and feed it back to the model
#             # for the next round of prediction
#             # also remember that we are creating a sequence, so we concat them at dim=1 to get 
#             # a longer sequence (we are gradually increasing the sequence length from 1 up to
#             # max_token_count)
#             idxs = torch.cat((idxs,new_idx), dim=1)
#            
#         return idxs

# this works fine, however, we can do better. here we just encoded the tokens, but
# as we already explained, attention has no notion of position or spatial structure like conv e.g.
# so we need to also incorporate the position information for each token as well
# since we are using the the attention head, we need more arguments
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, head_size, device, use_bias_att=False) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embd_size = embd_size
        self.context_size = context_size
        # for gpu/cpu acceleration during training
        self.device = device
        # unlike the previous model, lets decouple the final logits from 
        # the number of embeddings, because we are using attentions, and
        # we want to have multiple operations inbetween obviously.
        self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
        # lets now add the positional embedding as well 
        # using this, we try to retain the position embedding for our tokens
        # up to the current token in context_size
        self.position_embd = torch.nn.Embedding(context_size, embd_size)
        # add a self-attention head
        self.head = AttentionHead(context_size, embd_size, head_size, use_bias=use_bias_att)
        
        # in order to get the final logits, we need a linea layer at end
        # since we now have attention before this layer, the output dim of attention which is
        # head_size will be used here
        self.fc = torch.nn.Linear(head_size, vocab_size)
        
    def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
        # lets grab the shapes, since we will be using them 
        B,T = inputs.shape
        token_embeddings = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
        # since we have a position_embedding lets use that as well
        # and notice that we didnt use the inputs, but rather torch.arange(T)
        # this means, for each input, as we process it, we also get embeddings up to
        # the current token count as well, if the inputs has 3 tokens currently, we
        # will creeate position embeddings for 0,1 and 2, and for the next input
        # this continues likewise. this results in (T,E)
        position_embeddings = self.position_embd(torch.arange(T,device=self.device))
        # print(f'pos_embd:{position_embeddings.shape}')
        # and lets add the two embeddings together
        # this effectively gives us, not only the token embeddings(identity)
        # but also its position in the sequence. note that this doesnt really
        # help in a bigram model, but when it comes to attention it really does!
        embeddings = token_embeddings + position_embeddings
        # now lets feed this to attention head
        out_attention = self.head(embeddings)
        # lets feed this embedding to our fc at the end instead
        logits = self.fc(out_attention)         # has the shape (B,T,C) c is vocabsize
        loss = None
        if labels is not None:
            # recall that crossentropy likes its input to be B,C,T and we are B,T,C
            # so lets permute and make it happy!
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss
    
    def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
        # lets generate an output as long as num_max_token
        for i in range(max_token_count):
            # print(f'{i}/{max_token_count}) idxs: {tuple(idxs.shape)}')
            # make sure idx is 2d
            assert idxs.ndim >1, f"idx.shape '({tuple(idxs.shape)})' is invalid({idxs.ndim}). it must have the form (B,T)"
            # now lets feed it to the model and sample from the probablities it produces
            # but since, we now have postional embeddings as well, we can no longer have 
            # more than block_size/contex_size in, becasue if our idx is more than context_size
            # our positional embedding will go out of scope and error out
            # so here we are basically getting as many as context_size
            # note that we dont destroy the idxs! each time we get the last context_size tokens
            # from it and feed it to the model to generate the next token, and keep going
            # if we replace the idxs by sth like idx=idx[:,-self.context_size:], we would
            # only create a sequence of only self.context_size, no matter how many iterations
            # we do, we just repeat the same sequence again and again!
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are bigram still, lets only get the last token as the next token predicted!
            logits = logits[:,-1,:]
            # convert to probs 
            probs = logits.softmax(dim=-1)
            # now lets sample from it 
            new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
            # now concatenate the new token to the previous one and feed it back to the model
            # for the next round of prediction
            # also remember that we are creating a sequence, so we concat them at dim=1 to get 
            # a longer sequence (we are gradually increasing the sequence length from 1 up to
            # max_token_count)
            idxs = torch.cat((idxs,new_idx), dim=-1)
            
            
        return idxs
    
# now lets test this and see if it works 
x,y = get_batch('train',4)
model = BigramModelWithAttention(vocab_size, 
                                 context_size, 
                                 embd_size=16,
                                 head_size=16,
                                 device='cpu',
                                 use_bias_att=False)
# model.cuda()
# x,y = (t.cuda() for t in zip(x,y))
logits,loss = model(x,y)
print(f'{logits.shape=} {loss=:.4f}')
# and now we can train this : 
device='cpu'
batch_size = 32
head_size = 16
embd_size = 16 
context_size = 8
vocab_size = len(vocab_list)
max_iter = 5000
# only to test the effect of bias in k,q,v calculations
use_bias_attn=False
# attention requires much lower lr compared to plain bigram model
lr = 1e-3
model = BigramModelWithAttention(vocab_size=vocab_size,
                                 context_size=context_size,
                                 embd_size=embd_size,
                                 head_size=head_size,
                                 device=device,
                                 use_bias_att=use_bias_attn)

param_count = sum([p.nelement() for p in model.parameters()])
print(f'param count:  {param_count:,}')
print(f'head size:    {head_size}')
print(f'embd size:    {embd_size}')
print(f'context size: {context_size}')

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
model = model.to(device)
# set model to train mode explicitly
model.train()
for i in range(max_iter):
    # get the batch
    x,y = get_batch('train', batch_size=batch_size)
    # feed the model and get the logits
    logits, loss = model(x,y)
    # evaluate the model
    if i%1000==0:
        losses=evaluate_loss(100, device)
        print(f'train: {losses["train"]:.4f}  val: {losses["val"]:.4f}')
    # zeroout_grads
    model.zero_grad(True)
    loss.backward() 
    optimizer.step()
print(f'done!')

# now lets try its output
input = torch.zeros(size=(1,1)).int()
output = model.generate(input, 500).squeeze(0).tolist()
print(f"{''.join(decode(output))}")
# prints
#here are a few tries:
# param count:  3,041
# head size:    16
# embd size:    16
# context size: 8
# train: 4.2551  val: 4.2533
# train: 2.7175  val: 2.7354
# train: 2.5917  val: 2.5681
# train: 2.5299  val: 2.5265
# train: 2.4684  val: 2.4759
# done!

# Wonedimy
# Y:
# ARUS:
# LAnouver; pof th, sthe the mae,
# Wor ut barrioreche savarduncou?

# hiler bathakm,
# I O:
# AK:
# Acat ED:
# et alliro dow wowilou ttherss; whest owur, garsacwe Ie hithaceyidous sou f'ze iwot ry tohien fm.
# h.

# Ar'm-ore tr Fofhou
# Mr ojous omerrastheasncy yous Wowuce whor tu I I:
# Hbeotth spat fel woro bel, aneicudidy ktiferrd thout ngord. th Iid aral, be'nd.

# Pal tels Ms eimu Me myo on I teasthe del,
# AChinout,
# Wowure. pu tcher muss-renon; wingh ocet im
# r Ddr,
# Do ne thu,
# LI ir't
# Operst ry wu

# param count:  5,745
# head size:    16
# embd size:    32
# context size: 32
# train: 4.1374  val: 4.1414
# train: 2.6264  val: 2.6210
# train: 2.5100  val: 2.5092
# train: 2.4434  val: 2.4578
# train: 2.4140  val: 2.4252
# done!

# Thee to an tal my,
# Thake ced arce chart bre le be bizoresf thak des garg
# Lur Wentescaln
# Hooco beins pord sth ds hom span gr; ther. My Why ort thallils ofrof not tos gas hyine ind hand
# Thid One the shond nd,
# AFenosee omn st hy no fe.
# GEwank, iand!
# The sbe qurss yod hem snel.
# CLerle,
# Whow Vou bem!
# Theewourtoave, shougen t:
# Ber hangn toutince mor ed,
# AD Scheapy thatho ABEN:
# AD:
# SUEDithy nde ndel mmy Tongusate hfolit, gintoher, hat gwe tiet what whingimst hpoourde wowopal,
# The by br des qO: I no os,

# try2: decreasing context size
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

#try 3: decreasing embd_size, but keeping context_size 32
# param count:  2,265
# head size:    16
# embd size:    8
# context size: 32
# train: 4.2202  val: 4.2158
# train: 2.9840  val: 3.0093
# train: 2.7667  val: 2.7621
# train: 2.6355  val: 2.6411
# train: 2.6070  val: 2.6077
# done!

# o
# TI:
# BOGot be
# DOus ssh.
# Bea hashren v:
# ANULINII feso sid izeis by tofuso te I be fnowe?
# Wledu ha m, thope w,
# ARUTI land med cu
# SI bve fme
# ARIThilaserothan omyhee
# Oina c;
# UCAngose.
# Terve t thirre my tthunulpthemaf wof.

#  corlur yoNE:
# NHy w te hen cod, tonos vengo veschete tors t ithaimeqlutreqRoosoye chin y h thuasafaous.
# ANCathelo-
# The?
# Shen br uyh sbe ans I:
# Gouthyykey m Authirou ae tovas ber ut lnt:,

# TAMENI sf rousinthsin h'howenthor sg.
# hidandhe waresd!

# I f
# Se loner mrdin gsvette n thest?

# try 4: decreasing head_size, keeping embd_size and context_size the same
# param count:  4,457
# head size:    8
# embd size:    32
# context size: 32
# train: 4.3416  val: 4.3409
# train: 2.7354  val: 2.7333
# train: 2.6400  val: 2.6309
# train: 2.5716  val: 2.5692
# train: 2.5217  val: 2.5182
# done!

# IF
# S:
# To ofldickeens we fofane for o warus
# Herdliion bis, te,
# The ta, d ounor;
# Ad h ank; sor Cito-'d turses tierlllle,
# Whe pen, tint yt Ging sino leles fes hesr sst.
# Than hinean.


# Nofl an udn f hak,
# Wh stharovernes;
# And meno scerr tth Gbl y hy I t stit pand sy, pefif.
# US dik thith be tagag, harlovenppsh!
#  cegshil,
# Wheanst thiges eprine sy wabat tha.

# Wheas
# Wit:
# I
# Thouw'arsangort,
# Giseal?

# Wh; mo tands t's.

# NRNULECADSAxave!

# ANNDIEI Fh Fod meveeas pee be,
# Bes arner,

# Woo t'le.


# VIPORLEOG
# CAOO:

# try 5:
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

# DANGUTTA:
# Bert hatire on the wo A? sewas ker haywins wet.

# mousill akerd fous sther Le fke sth,, bepeere gn ssst I oun amand serewos, cobear song.


# AMEfediengg yit othesshint.

# Anc:
# A must shan ghety ak is, shet.

# MEO:
# Tom yot stths!
# NCIBus hand akt imy berdr,
# Thulll id mere thassut,
# Moureleray, fous mante. QBO decencepous ariveed hart;
# I; bifo wthen chant tht tho hy youn,
# Grolomy, ucel.

# BRIINE ERIYht I arum arw t; odeat ngher bly wehe foru thas sot our:
# Bn; sshiutithemowith her fourpler thant

#
# which looks depressing not gonna lie! but its better than the simple bigram model we 
# built earlier, anyway, can we improve it? certainly! 
# for example we could use dropout on our attention! we could use normalization layers
# and we could use multiple heads instead of just one! which takes us to the next
# subject which is multi-head attention! which is  really nothing except several normal!
# attention blocks run in parallell! 
# so lets implement these and see how much we can improve upon this

results=tensor([[[ 0.,  1.],
         [ 1.,  2.],
         [ 2.,  3.],
         [ 3.,  4.],
         [ 4.,  5.],
         [ 5.,  6.],
         [ 6.,  7.],
         [ 7.,  8.]],

        [[16., 17.],
         [17., 18.],
         [18., 19.],
         [19., 20.],
         [20., 21.],
         [21., 22.],
         [22., 23.],
         [23., 24.]],

        [[32., 33.],
         [33., 34.],
         [34., 35.],
         [35., 36.],
         [36., 37.],
         [37., 38.],
         [38., 39.],
         [39., 40.]],

        [[48., 49.],
         [49., 50.],
         [50., 51.],
         [51., 52.],
         [52., 53.],
         [53., 54.],
         [54., 55.],
         [55., 56.]]])
tensor([[ 0.,  1.],
        [ 2.,  3.],
        [ 4.,  5.],
        [ 6.,  7.],
        [ 8.,  9.],
        [10., 11.],
        [12., 13.],
        [14., 15.]])
tensor([[0., 1.],
        [1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.],
        [5., 6.],
        [6., 7.],
        [7., 8.]])
a=tens

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_head, head_size, embd_size, context_size,bias_attn=False) -> None:
        super().__init__()
        self.num_head = num_head
        self.head_size = head_size
        # we assume the head_size is already split between the num_heads and thus we dont split
        # it again
        # assert head_size//num_head == 0, f'head_size({head_size}) must be divisable by head_num({num_head})'
        # we need to create n heads so lets do it 
        # self.heads = [AttentionHead(context_size, 
        #                             embd_size, 
        #                             head_size, 
        #                             bias_attn) for _ in range(num_head)]
        # but we can also use pytorch's nn.ModuleList which is a better equivalent than list
        # note that, nn.Sequential cant be used, becasue it runs the modules in succesion
        # i.e. serially, one after the other (feeds the output of the previous module to 
        # the next module, etc) which is not what we want. we want to calculate each head
        # independetly and aggregate their outputs so, either a python list or torch moudle list
        # can be used
        # sidenote: this module that we are building, is also known as, masked multi-attention-head
        # becasue we are using the single-head self-attention which uses a constrain we imposed
        # by masking if you recall that!
        self.heads = torch.nn.ModuleList(AttentionHead(context_size, 
                                         embd_size, 
                                         head_size,
                                         bias_attn) for _ in range(num_head))
      
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # note that head_size is usually the embd_size, so it covers the whole 
        # embeddings obviously, and note that each head will work on a portion
        # of the given head_size, if e.g. we have head_size/embd_size = 16
        # if we had 1 head, the head_size would be 16. however if we had 2 heads
        # we had to split the head_size in half for each head, and later on
        # we had to concat them to get the full-size head_size/embd_size.
        # therefore here, we need to concat their results along the cols
        # so multi-head-attention is akin to group convolution, and thus the head_size
        # and num_head must align properly.
        outputs = torch.cat([head(inputs) for head in self.heads], dim=-1)
        return outputs
# lets test 
# note that we set the split value for head_size based on our num_head
m = MultiHeadAttention(num_head=4, head_size=16//4, embd_size=16, context_size=8, bias_attn=False)
logits = m(torch.randn(size=(1,8,16)))
print(f'{logits.shape=}')
# 
# now lets use this in our model
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 
# now lets train this model and see how it performs this time
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# outputs : 
# param_count  =  3,041
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2041  val: 4.204
# train: 2.6895  val: 2.685
# train: 2.5449  val: 2.528
# train: 2.4570  val: 2.477
# train: 2.4238  val: 2.433
# done!

# I'd In mame
# Ast owu hapland.

# Mayl thew youres?
# IRisire dis lhend gitim whorder, sphorgunct Beye skek boutrig!

# I, sho sciltr not bpreathl harr'd doess elowt cof or iny poovigre.
# Thalrit?

# HThary?

# Lerim he bamat:
# Thiet hey, krepably bose bour cave grosr benemece wleir hon'g.

# MADIUMI but th Mond im's veoo tit. IDf met Cerve's wue rrer hiftren'tr'd:
# Anoks:
# HOt
# We I phess chi bouine mares yon ther; tlotirtlinl therot:
# Io
# Wh per,
# Fous thayigso ceany pate.

# Annd wosh fong uat
# Now ICUSo CCgodertiove
#
# trying with larger embedding size seems to improve the results: 
# param_count  =  7,553
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1828  val: 4.186
# train: 2.4703  val: 2.472
# train: 2.3649  val: 2.367
# train: 2.3009  val: 2.32
# train: 2.2713  val: 2.288
# done!

# Cly Toste?
# IOK:
# Whan his me' tis it I ond ber'thse?

# PO:
# Theray delling, dothoinerk an to the, or
# E VINGON: rocO:
# Thes win now thatin knir:
# Wigh fy, bund werar is wet my;
# I ene:
# JUu'd sow prut hat amver'd ENDUE VUF E VO?

# Se gings nom and.
# CArwak mandesplay beesire brit mand.

# MED:
# Thou ased ith is whis wentemprt hits this in ther.
# Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
# LUT: am le the wids my thousecarto my haty wind WLIERME:
# Wher a and
# Whesh to wyt wir

# compared to single head attention, with the same hyper parameters, train: 2.4684  val: 2.4759
# and now we got train: 2.4238  val: 2.433 which is better(we also got train: 2.2713  val: 2.288 
# with just increasing the embedding_size, so playing with parameters even blindly making model bigger
# seems to give us a boost), so we had improvements, but we still need a long way ahead of us!
# 
# Ok, to improve upon our results, there are couple of more things we need to add to our attention 
# block. if you look at the paper, we'll see a few concepts that we havent talked about or implemented
# yet, including the feed-forward(position-wise feedforward network) and layernorm, so lets talk about 
# them.
# feed-forward network or as its called in the paper,'position-wise feedforward network', is simply
# a "fully connected network which is applied to each position separately and identically", this 
# consists of two linear layer with a relu activation function inbetween. 
# so its a linear layer with a relu activation function followed by another linear layer basically. 
# you may also hear this network be refered to as 'computation after communication' so to speak!
# signifying the fact that it runs a computation after the attention module.
# the idea behind this extra addition is simple, to provide a higher representation out of attention work
# this network, causes every single token to have a nonlinear transformation and achieve a higher abstraction
# possibly yielding new information benficial to the task. to put it in casual way, it can be seen as though
# the attention gatheres some stats/information, and this step is akin to looking into it and thinking about it
# comming up with some new findings, that is , if attention part is refered to as communication part, this is
# the computation part/ or thinking part for the lack of a better word.
# lets implement this network in our bigram model and see if this seemingly simple change, affects us at all
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # now let us create the feedforward network, which basically is a linear layer with relu
        # followed by another linear layer! for this so called network, the in_features and out_features
        # are simply the same, and is embd_size, because its a sandwich layer between our attention output
        # and the last layer. this layer usually is (embedsize,embed_size) if you recall head_size is usally
        # equal to embed_size. also note that in the paper the feedforward network is two linear layers with
        # a relu (one linear layer with a relu plus another linear layer).if you look closely you'll notice 
        # we already have a last layer after the attention, so we dont need to put a nother linear layer
        # afterward, becasue that one linear layer suffices (multiple linear layers, dont provide
        # any higher abstraction anyway, so its prefectly fine)
        # but on the other hand the paper's implementation has another change, it states 
        # the inner layers dim are increased by 4x, so to be faithful to the paper we also 
        # add the second linear layer, with the suggested change, this wont change the output 
        # shape, so we are fine. we just added an extra linear projection layer. 
        # this additional projection operation actually improves the result,however if we simply 
        # use the same dim linear layer (i.e. have sth like linear(head_size, head_size)) adds nothing
        # to the representational power of the network and you wont see anything substantial vs if you
        # completely remove this layer and only use linear/relu only.
        # why this works is becasue, we increase the nonlinear output neurons of the first layer by 4,
        # increasing its representational capacity, and then use the second linear layer to get the output
        # size compatible for the next layer (doing a linear projection), and hence our improvements lie
        # in the nonlinearity this addition provides. 
        self.feedforwardnet = nn.Sequential(nn.Linear(embd_size, head_size*4), nn.ReLU(),
                                            nn.Linear(head_size*4, head_size))
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # add our new addition, feedforwardnet
        out = self.feedforwardnet(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using feedforwardnet added')
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints : 
# using feedforwardnet added
# param_count  =  5,169
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1975  val: 4.198
# train: 2.6222  val: 2.634
# train: 2.4919  val: 2.481
# train: 2.4320  val: 2.435
# train: 2.3979  val: 2.392
# done!

# Selwils iur.

# AnNdt
# By thak, yue, nein ende vat Rout'w haler,
# The hot hes? is, whas bles. Yig,
# ROm
# QUS:
# Ay Whigice rea:
# Therer a youst um mith dins I dorseancer by gou:
# I sot the, aserer tom sne arobfs-Rid,
# This sas whe ou,
# Acet Yark no.

# OT:
# Fely ip you pel hivet seatly. hial pand,
# Thand repwat,
# I tarve ther se this ten thio drord dot tho,
# Rulee res le, il fet!
# Nord Wotiilet homom for my woue cuve wid.

# Angilene.

# Fhivy, ih:
# Boumlke uthereaerf
# Whe ithe hares.

# Whos sthik thenth
# Tait:
# She tove w

# the loss didnt get better! but maybe if we increase the context_size, and embedding_size, it 
# perorm better?
# using a larger embedding_size and thus larger model did improve the result, 
#
# using feedforwardnet added
# param_count  =  15,905
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1569  val: 4.159
# train: 2.3885  val: 2.396
# train: 2.2789  val: 2.312
# train: 2.2064  val: 2.249
# train: 2.1599  val: 2.198
# done!

# To get fer arast deve lad, I' fale are you up this riany;
# Weareien,
# Tho Guser, me ter is! Do lothe, to ling wo whe Pist to deave mat not, in and- air besface, y to fron; use ler weet herefalf thop an der pretrier
# nesly toide
# Thats ous.

# RENT:
# I for the haw to, to noss teave ler shat earelss ater, want Heanct herry deeter erry, heave masere dacke; nochat onsemy ning Meib,--?

# LUCLUER:
# If Feropeake mo shat onsivoole washy the well beWarquent;
# I on well he ome it you lare to lo to kit youser,
# Weves

# the larger network, seems to be doing much better than its counter part from before
# the one without the ffnet, we got train-loss: 2.1599  val-loss: 2.198 here whereas 
# previously we got train-loss: 2.2713  val-loss: 2.288. so we have improvements despite the 
# text still not being that good(but still better than before). so we need more enhancements)

logits.shape=torch.Size([1, 8, 16])
param_count  =  7,553
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1828  val: 4.186
train: 2.4703  val: 2.472
train: 2.3649  val: 2.367
train: 2.3009  val: 2.32
train: 2.2713  val: 2.288
done!

Cly Toste?
IOK:
Whan his me' tis it I ond ber'thse?

PO:
Theray delling, dothoinerk an to the, or
E VINGON: rocO:
Thes win now thatin knir:
Wigh fy, bund werar is wet my;
I ene:
JUu'd sow prut hat amver'd ENDUE VUF E VO?

Se gings nom and.
CArwak mandesplay beesire brit mand.

MED:
Thou ased ith is whis wentemprt hits this in ther.
Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
LUT: am le the wids my thousecarto my haty wind WLIERME:
Wher a and
Whesh to wyt wir
using feedforwardnet added
param_count  =  15,905
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1569  val: 4.1

In [4]:
# the next improvement, would be to use more of these, as its shown in the paper, if one block
# works, then adding more should work better right? we had single head, it improved our condition, 
# so we used more heads, and now lets more multi-attention heads! to make things easier, lets
# make a module out of it the same way we created previous modules. 

# first lets create our ffnet as a seprate block, so we use it after the attention
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        return self.ffnet(self.attn(inputs))

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks!')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks!
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2087  val: 4.211
# train: 2.6084  val: 2.606
# train: 2.3737  val: 2.39
# train: 2.2865  val: 2.307
# train: 2.2087  val: 2.237
# done!

# FAUKIN YRE:
# Dato':
# wis.

# BERDEWEE:

# Piwit ancon; awt
# Emuke this his fred not pe,
# Har,
# And him my bird-wray sluf. Ile:
# With tulel wik, antegatorg.

# EREN VELV:
# Whath a meane, ones
# sheirss,
# Bose til
# Ilf bodtund ba,
# Aripce nerod is the blilshil hit hit mef?

# GESINCE:
# Sard, with the deoorrnsicominle thid: as.

# ANRE:
# Hy prall anges what micciths delre
# To tike comf hon ebat'd mur. Kin'sa, Fharth:
# Af'?

# ROASA:
# Ect mily fledd.
# SHricest.
# At dey bompe novor where lisg.

# KETILNINCE:
# There gay his my nefonk
#
# as you can see, we got wrose results than before! despite making the network larger, our loss
# really didnt improve as we expected. 
# is our initial hypothesis that having more blocks and higher nonlinearity/representation is benificial
# wrong? or is there something else thats causing the issue? 
# as you might have guessed, its the latter. we are basically creating more layers, and with
# more layers, we face training issues that we discussed earlier. 
# so how should we tackle this, we cant use BN, becasue BatchNOrm, accumulates the statistics
# from different samples, this is a no no for us. we dont want other samples to interfer with 
# our sample (or basically each other!) in anyway, so what should we do? 
# the paper utilizes two mechanisms or operations to tackle this issue. one being skip-connections
# (also known as residual connction) and the other, layer-normalization. 
# you should be familiar with skip-connections as they are the founding factor or resenets
# and have been extremely influential. so to cut a long story short, skip connections are simply
# connections from input skipping the operations involved in the block they reside and directly 
# being added to the output and then returned the result.(basically F(x) + x)
# as we know, the gradients are distributed equally when they reach addition, so input gets the gradients
# without being weakened due to large depth of the network. 
# so before we implement the layer normalization part, lets see howmuch of a change adding skip-connection
# causes and whether it proves our initial hypothesis about depth and gradient signal weakening or not
# we add this to our AttentionBlock (but we could add this to any submodule)

using more blocks!
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.2087  val: 4.211
train: 2.6084  val: 2.606
train: 2.3737  val: 2.39
train: 2.2865  val: 2.307
train: 2.2087  val: 2.237
done!

FAUKIN YRE:
Dato':
wis.

BERDEWEE:

Piwit ancon; awt
Emuke this his fred not pe,
Har,
And him my bird-wray sluf. Ile:
With tulel wik, antegatorg.

EREN VELV:
Whath a meane, ones
sheirss,
Bose til
Ilf bodtund ba,
Aripce nerod is the blilshil hit hit mef?

GESINCE:
Sard, with the deoorrnsicominle thid: as.

ANRE:
Hy prall anges what micciths delre
To tike comf hon ebat'd mur. Kin'sa, Fharth:
Af'?

ROASA:
Ect mily fledd.
SHricest.
At dey bompe novor where lisg.

KETILNINCE:
There gay his my nefonk 


In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
# lets add a skip-connection to this block
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        out = self.attn(inputs) + inputs
        out =  self.ffnet(out)  + inputs
        return out

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5636  val: 4.553
# train: 2.2666  val: 2.281
# train: 2.1330  val: 2.183
# train: 2.0708  val: 2.14
# train: 2.0204  val: 2.109
# done!

# Zunt will jucked's whreef onst's so
# wippplalct, sawter.

# Firfor have deal pere,
# Shal,
# thich many.
# Frig.

# Thall have.

# WANWHAM:
# Sad king theave,
# Cnomlen nare, near was?


# CKINGlLOROMBOENGBETH:
# Dich loverd und by, may;
# ''Kh
# Godie the bline plock's minstell the denelsbad, with think
# What
# sicond lead this undrut, too, the but me my se ming'ld;
# Shing
# To tiot comford, engelan this it's heare: hear them the vinclamil.
# Is done, the stronced,
# Lion?
# Hartow where lisg.

# DOLILINA:
# Lord ISe and
# Lud'd nef hel

# and test with one skip-connection on the 'output only' to prove or assumption on skip-connections
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5616  val: 4.552
# train: 2.3196  val: 2.322
# train: 2.1862  val: 2.221
# train: 2.1175  val: 2.163
# train: 2.0640  val: 2.137
# done!

# FARY VI:
# A-bame 'tries.

# BERGEWES:

# PABLLO:
# Yon; aws
# Then ath-pmate for, not pe,
# PARILA:
# So man.

# FORY.

# TAsUS:
# I cenpine nou, lad king theak,
# Anny, In nartine-'

# An yeant, of ladie ust, good ting lover that by, may;
# And hordie the
# llils plove ond sofful and heave
# Take with the deo,
# And mort leath and usir the dom the but me my crombasill;
# Shim are, con comlmork engeland your, trabe,
# Beth:
# Yor?

# CORIAR EFt mily fle--spoles, strence'd ono do now, will?
# Mursice,
# Ntormorest
# The juge ernus'd neforki

# as you can see, it greatly improved our results, and the text also got much better!
# as we already pointed out, having skip-connections per modules, help much more than a single 
# per module output only, nevertheless, we notice, using skip-connection really improved our results.
# now lets add the second operation, layernorm. layernorm(https://arxiv.org/abs/1607.06450) is a normalization layer, just like batchnormalization
# came a year later than batchnormalization paper, but the difference between them is that, unline batchnormalization
# it works on a per sample basis and does not involve using othersamples to normalize a specific sample (basically
# samples dont affect eachother)
# As pytorch docs puts it:
# "Unlike Batch Normalization and Instance Normalization, 
#  which applies scalar scale and bias for each entire channel/plane with the affine option,
#  Layer Normalization applies per-element scale and bias with elementwise_affine"
# the implementation is similar to the batchnormalization, and it does not require calculating running_mean/var
# we can use the pytorch module just fine, but since its really similar to BN, lets implement it here 
# 

using more blocks with skip-connection
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.5636  val: 4.553
train: 2.2666  val: 2.281
train: 2.1330  val: 2.183
train: 2.0708  val: 2.14
train: 2.0204  val: 2.109
done!

Zunt will jucked's whreef onst's so
wippplalct, sawter.

Firfor have deal pere,
Shal,
thich many.
Frig.

Thall have.

WANWHAM:
Sad king theave,
Cnomlen nare, near was?


CKINGlLOROMBOENGBETH:
Dich loverd und by, may;
''Kh
Godie the bline plock's minstell the denelsbad, with think
What
sicond lead this undrut, too, the but me my se ming'ld;
Shing
To tiot comford, engelan this it's heare: hear them the vinclamil.
Is done, the stronced,
Lion?
Hartow where lisg.

DOLILINA:
Lord ISe and
Lud'd nef hel


In [6]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (2.38418573772492e-09, 1.0004067420959473)
torch: (9.53674295089968e-09, 1.0004067420959473)


In [7]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
#
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we were aggregating the last two dims like BN, whereas we should have only used the
# last dim, as we should not involve other samples in normalization. (I explained this thoroughly in the LayerNorm class)
# 
# now how can we improve more? we implemented the paper, basically we implemented transormer from scratch
# and what remains is to test with different hyperparameters to see how well it can generate texts similar 
# to our dataset. (we implemented the decore version, but the encoder as we explained earlier is the same
# without the constrains! well talk about this in a moment )
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3906  val: 4.385
train: 2.2373  val: 2.244
train: 2.1070  val: 2.155
train: 2.0339  val: 2.102
train: 1.9812  val: 2.069
done!

For the that mad'
With of on tith, luidie anct, saws
Eule at botht if thou
Wlefted' you windian.

FLO-wike sluke slen:
I have, ladik, anteraves,
norl wink
the bard a yeane, of lady. Whe, goshot I
thy brath his what:
Nown hondin the
blinsh, shat this the pandself livadet brives nefors.
ISIUS:
Aren that usir the of me
he hange wicHe mirs,
Behat!


TOLAT:
But ford, engelan this it'sabhanke hear rend these this in usse,--
Lord, strence'er thy?
In woes. IDandry,
Soset frongs neved
With haul'd nefonk 


In [8]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)
start = time.time()
torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()

using more blocks with skip-connection-layernorm-beefed up!


/home/hossein/anaconda3/lib/python3.11/site-packages/torch/cuda/memory.py:329: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [9]:
head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 8 # 16 # 32 # 64 # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'{time.time() - start} ')

param_count  =  9,806,657
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  8
device       =  cuda
use_bias_attn=  False
train: 4.3274  val: 4.327
train: 1.8653  val: 1.976
train: 1.7488  val: 1.886
train: 1.6929  val: 1.841
train: 1.6657  val: 1.82
done!
82.66008353233337 


In [10]:
# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 


DUKE OF AUMERLE:
Tellgege, there of the fort; bid my lords power: here had a and not.

CAPULET:
So so just be;
My directain so try, my dalicy of Her man; without much burn dible,
How thy oad.
Desk, is she is befool.
-alourish through the betrious done.

COMINIUS:
Yea you no poor reburth o' the king-jude
Thee one,
Each are that I have, why dash contrary.

MARIANA:
Peace, for drease you;
But justicy a grief,
Thereining
To be him; for for my spuile is the scan: all in my thy slewing. But, my lord, masway.

LEONTES:
Yet babelds on grief to the received blood; Warrich'd he, finia;
Has tid his words that one of him more conclaim good roose that marrial
Duke on York, that I hear my done.

YORK:
I bring I proud
Mistress, all you laid
Your temper tail, is it's onal: knave's lord.

BRUTUS:
'Tear me, some needs. O my can thee,
If I justic of it will seem my removed on my lire if is is a you no good in master
As a time.

KING RICHARD III:
Hare the woin,
There win thy times thee? 'fore but the lav

In [11]:
start = time.time()
head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 16 # 32 # 64 # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,809,729
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  16
device       =  cuda
use_bias_attn=  False
train: 4.3911  val: 4.381
train: 1.7263  val: 1.86
train: 1.5795  val: 1.746
train: 1.5157  val: 1.696
train: 1.4712  val: 1.656
done!
elapsed: 122.30294823646545 

AUTOLYCUS:
I will, my cack out as thou hast defend this same, how
Of monioner well, and what all
wiss girly.
The tickles of my fiery.
Thou hast you to go,--
To make your high;
One date to be fought,
Death in him, thy deckness for think what they are the heaven'd.

ANGELO:
Mine and Bolingbrok'd with all unsomb'd.'

CLAUDIO:
As free him: and that it, to
wave a right in ship to bind. Still not as
peace at ord, by me; either it is bear an happy of their valousne follow like more shall old noble lord
In confound, when I was, by Cominion.

PARIS:
Still thy mind lely the revel:
By my gracious my counsel of death. Bad my Lancaster such a biracle confess sin the corse, si

In [12]:
start = time.time()
head_num = 6
block_num = 6
head_size = 384
embd_size = 384
context_size = 32 #8 # 16 # 32 # 64 # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,815,873
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  32
device       =  cuda
use_bias_attn=  False
train: 4.3443  val: 4.345
train: 1.6658  val: 1.817
train: 1.4801  val: 1.664
train: 1.4008  val: 1.609
train: 1.3445  val: 1.585
done!
elapsed: 220.91568803787231 

With unlooking the sentences of his penince
Have ass on the hours.

First Lord:
Perhaps,'d and pain upon me; and which his son.

BIANCA:
Pardon, his young into his limet.

BRUTUS:
Many lost on advices; sweet 'Tis place,
Plantagenet, set thou to malk a take order bite.

Second Citizen:
That's our loves in golden and deep!
Welvers and kneel, no 's looking pains.

Citizens:
And hark you, but dread'st hour of Lancaster;
I'll take its up an yourself I lack.

DERBY:
Your love and in claudio much is too loyal.

SAMPSON:
In am tastirretion.

CAMILLO:
Clifford, will in the wretch of the senate thy did trumpetite business.

VOLUMNIA:
Awhiles be this innocent vestal self;


In [13]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 384
embd_size = 384
context_size = 64 #8 # 16 # 32 # 64 # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,828,161
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3453  val: 4.343
train: 1.6647  val: 1.82
train: 1.4375  val: 1.637
train: 1.3358  val: 1.573
train: 1.2692  val: 1.543
done!
elapsed: 396.50476908683777 

A says is he will enough valier Tellon,
And shall I proud unpierce to some Apollo.
I brill thee for them to them.

AUSINA:
I know the morning extremes it?

MENENIUS:
He's rightens of mine,
And in this will be foul, 'now full wounds.
I pray it said bow a-day's regior!
I hide thy words, will endaugh noise alone.
Thou uncless banish her bride.

Second Gentleman:
Here is my Aumerle, yet--book every one peers,
I'll do thinke, in this, and give him my kinsmen,
The county the part ston mine o'ercry,
As passad-lead, his sinks rume, may look fond
A golden approset with soldier.
I hands and knows young name in the duke.
I see, you sie the worldly rooks of preceition
To con

In [14]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 384
embd_size = 384
context_size = 128 #8 # 16 # 32 # 64 # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,852,737
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  128
device       =  cuda
use_bias_attn=  False
train: 4.3609  val: 4.357
train: 1.7876  val: 1.914
train: 1.4730  val: 1.681
train: 1.3386  val: 1.577
train: 1.2525  val: 1.548
done!
elapsed: 785.4520983695984 

Fall on them,
Either must in the crown of his accident and dispatch'd,
Since's letters no private, let us do them time,
And, ask your face, to where be rebels.

BISHOMON OF WARWISS
RO:
I move col't my heart, Can Petruchio it,
As you are who doseed!

YORK:
No? Marry, a good stoop bold proverty
Hath bold of at Rusal's bidding city,
I will are you passes, and alone.

WARWICK:
I cannot be off! Heah, when you do it?

Third Murderer:
I am sirrah, let's good kneel to day out.

DUKE VINCENTIO:
Requite, sir; no hot stay at your gates!

PAULINA:
I wis?

To Citizen:
Go to, sir; that you drow not, my such lord
Is burn to a either Lord Rastess, and we'll
kneel sheet not for 

In [15]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 384
embd_size = 384
context_size = 256 #8 # 16 # 32 # 64 # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,901,889
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3647  val: 4.359
train: 1.9514  val: 2.042
train: 1.5521  val: 1.734
train: 1.3938  val: 1.62
train: 1.2995  val: 1.565
done!
elapsed: 1673.3493466377258 

Yield that are the sea poor rock'd's son excellent,
He weary well gave of the spirit,
Yet now hove the wish doubted hour had warn
With his suspect, his hand accuption of the
bloody tidating: hath he better than to daughter
And did into lawful crumption and days?
The noble fools his entreat and excude;
More ornament with her blood or my partern hollow in
With tongue-palpon smiles, but my lands for as a cold,
That you shall not be his ears apted mine.

AUFIDIUS:
Pray your chaffer writes, we more anoid, would
Desert first in loving thee actor; never
His knee of renounce? Which we have old at half
And years, to dark for a rewarms foot,
Or hath piece her from and spi

In [16]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 384     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 16     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  163,505
head_num     =  6
block_num    =  6
head_size    =  384
embd_size    =  16
context_size =  256
device       =  cuda
use_bias_attn=  False


RuntimeError: The size of tensor a (16) must match the size of tensor b (384) at non-singleton dimension 2

In [17]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 8     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 16     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  20,425
head_num     =  6
block_num    =  6
head_size    =  8
embd_size    =  16
context_size =  256
device       =  cuda
use_bias_attn=  False


RuntimeError: The size of tensor a (16) must match the size of tensor b (8) at non-singleton dimension 2

In [18]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 16     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 16     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  22,881
head_num     =  6
block_num    =  6
head_size    =  16
embd_size    =  16
context_size =  256
device       =  cuda
use_bias_attn=  False


RuntimeError: The size of tensor a (12) must match the size of tensor b (16) at non-singleton dimension 2

In [19]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 18     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 16     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  24,791
head_num     =  6
block_num    =  6
head_size    =  18
embd_size    =  16
context_size =  256
device       =  cuda
use_bias_attn=  False


RuntimeError: The size of tensor a (16) must match the size of tensor b (18) at non-singleton dimension 2

In [20]:
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 18     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 18     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  29,405
head_num     =  6
block_num    =  6
head_size    =  18
embd_size    =  18
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3311  val: 4.32
train: 3.0563  val: 3.083
train: 2.7163  val: 2.721
train: 2.5912  val: 2.592
train: 2.5302  val: 2.529
done!
elapsed: 426.2003791332245 

An RUND thilr coweame ve hiss
Be masthat re,
MLTout mes, myib t wouor t!

WU:
MEang hiofps s Beathine pl thend I sin$l:
Helllof wis tad f d
SNAntourir t whed, ido beth frorxorers'dinthilllitary s s
hoo che fy r omd I o:
I ad y kielu;
Th I m, torabnd d bowin ay k sefou dfir y in ndour 'fooow't beret:
S, asvelllonod har besumathige aknis he d wemat ir golereyconold angieit f ciererat trer
fethe
Ars
Sofll beng.
Fome fre ses I An mased n p cy my afako-imoup
We
GUYAs t:

Y, syotathaver:
The murived ma thiseitis ther he, bindy ou; buto d,
ERouy anshes;
An iathace bldshesgelombe? gur, os t the tine r hindorice rtane wieeofcha inglomy on seastoorond s, and,

Che Th hves whenu

In [21]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)


# torch.cuda.memory.reset_max_memory_allocated(0)
torch.cuda.memory.empty_cache()

using more blocks with skip-connection-layernorm-beefed up!


In [22]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 32     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 32     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  80,641
head_num     =  6
block_num    =  6
head_size    =  32
embd_size    =  32
context_size =  256
device       =  cuda
use_bias_attn=  False


RuntimeError: The size of tensor a (30) must match the size of tensor b (32) at non-singleton dimension 2

In [23]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 36     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 36     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  101,513
head_num     =  6
block_num    =  6
head_size    =  36
embd_size    =  36
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3422  val: 4.334
train: 2.6777  val: 2.679
train: 2.5161  val: 2.518
train: 2.4446  val: 2.445
train: 2.3764  val: 2.386
done!
elapsed: 495.70260667800903 

DY:
Fat antih, mace'Sg gre therspome, bunort;
Tir mbear le pote Efis thesgrare.
On tond swh owad ke ge: sat; lee,
d theral'd y trefrot dof the skim's ang witlod Amecod,
And wwo fr othe y.

LOIDOL: sisis; se ha thoung-
A akere thor'ToLou pret iofard's.

Sh IOre INDY:
Re3 Gra po urre chtho mowe ak whe$alst wonorsour mor ary 3Nal bichis,
Lot I sacho that at ote; tolleyst;
y t fore shou he?
Gourde--byd y kigasite dovein,
To, e ha re wiff tom woru bleererd hiche: ap penig
Yot preesilon g wathin be the swifive, lowort mve
Sie ldnes, gr sintoat th mugeret blound We e,
Asou rthof ian
Hand id hismuk
HELLIUTHANE:

NIBS mo s conot ha gof I wasu th,
LINY haneaus, st Yotowngt i

In [24]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 72     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 72     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  374,033
head_num     =  6
block_num    =  6
head_size    =  72
embd_size    =  72
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3593  val: 4.358
train: 2.5095  val: 2.515
train: 2.3689  val: 2.382
train: 2.1617  val: 2.193
train: 1.9749  val: 2.04
done!
elapsed: 585.5581364631653 

AULO:
QUue new you's your counses, lord,
So defleamely praind hour somend me I
not the swell a our: sager yeal comistan.
For laviedys muved diowedult.

GLOUSTEr:
I never My mored! Ather, to thy, dot
Druach yould, thy decks if fairthnt: whand;
And shell oubun, to.

AGAMER:
Mivend, to but a pricein, and that and but,
He deecr give to thee flearas itend?

APUTICULUS:
Garstioute buty. hen sind thee you, arrice,
Thy, 'techer oit techer son have in beacid.

QUTRCHARS:
Priltho$
Y:
Ast faiol, nobtteles,
I nobliushene prore yours! was it entrorieiago!
To me head shall it brrove on anme.

Of SAMERLL:
He the Ead gayous droned say seack
To a oncaul userf boosen the schall spee.


In [25]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 144     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 144     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  1,432,289
head_num     =  6
block_num    =  6
head_size    =  144
embd_size    =  144
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3884  val: 4.382
train: 2.3967  val: 2.411
train: 2.0929  val: 2.146
train: 1.8464  val: 1.969
train: 1.6816  val: 1.847
done!
elapsed: 799.7265410423279 

With uppon shall keep them end mad.

BENVFSERDIA:
Broken begod, prove, that you pehar thou
Dest bid upon of cannot
Cliper of no poinur, Beam thath with a tilt
I had woed. Hearlys is not the on oxech souns
app'd heple; the now bastipese laumbosod: lestitay:
Had this stiller. Five shaws us you
Ed to the wimanaton and do wigh namisule bod
Than by'd sureity pared; if at all foifurlr
Your a teld soundour deemp; this threate it ye.

HASTINGS:
Thanks yet are heart onk yet, I tamkle have
Frands traniths with our spead sted quintasur,
To brack shelds:
A't sern, we accus, to be will, was and age
To die the curite budgge of eye?
Marry, who should ashoond,
To the be cince;


In [26]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 288     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 288     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  5,601,665
head_num     =  6
block_num    =  6
head_size    =  288
embd_size    =  288
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.2985  val: 4.299
train: 2.0821  val: 2.141
train: 1.6825  val: 1.845
train: 1.5079  val: 1.707
train: 1.4086  val: 1.627
done!
elapsed: 1348.598309993744 

A san! is he were excute was exance,
Musiciansfy suitories leave me attive;
Romeo be burge with the freewing it.

DUKE OF YORK:
But troth, whoe beseep it a bludy dispatch'd.

PETRGIS MOWBROP:
My crown without for me; What confound,
What could shining hat noble Agelo!
I hide thy widow'st most waulk noine along to thee;
Though I pray it bitter comes to doing into teach.

LADY:
You your both every?

LADY LARENCE:

MARCIUS:
With Darry still our days off God just count out to these,
Sir most are him in a Title be hitdred
Here unrrumed then down to him a brother, your
with must unseel's knows, whose and bid; I'll and wild
By their great and dertain than enemy
Of preci

In [27]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 400     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 400     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  10,708,065
head_num     =  6
block_num    =  6
head_size    =  400
embd_size    =  400
context_size =  256
device       =  cuda
use_bias_attn=  False


RuntimeError: The size of tensor a (396) must match the size of tensor b (400) at non-singleton dimension 2

In [28]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6
block_num = 6 # aka layers!
head_size = 396     #16 # 32 # 64 # 128 # 256 # 384
embd_size = 396     #16 # 32 # 64 # 128 # 256 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  10,524,953
head_num     =  6
block_num    =  6
head_size    =  396
embd_size    =  396
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3721  val: 4.377
train: 1.9475  val: 2.033
train: 1.5501  val: 1.742
train: 1.3864  val: 1.609
train: 1.2882  val: 1.561
done!
elapsed: 1966.0733542442322 

Fall one yet to be a money's in planet,
And barn'd in a kind, and they in a top for
By her supforan's condemny?

WARTELAND:
I give them come of the enjoy buil.

BIANCA:
I' true drunk and most country,
We have abherved hence; fear for face,
If swearch yours, batter warwing the our gars;
And, sir, runsXaring to his daughter's country!
Onfice and out tribs, to she is leaving with Christopher
Withhd watchful I contend and more repition.
Farewell, but a pirk flower, and stirute! Will, this
king, to great was poverture, and in worn with
forship, as we do? If any house scape spott,
Hermit at us 'Petruchio?' my sighs.

Second Lord:
He you and rach ales; but he many fo

In [29]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2
block_num = 6 # aka layers!
head_size = 396     #18 # 36 # 72 # 144 # 288 # 396 # 384
embd_size = 396     #18 # 36 # 72 # 144 # 288 # 396 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  10,524,953
head_num     =  2
block_num    =  6
head_size    =  396
embd_size    =  396
context_size =  256
device       =  cuda
use_bias_attn=  False


KeyboardInterrupt: 

In [30]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2
block_num = 6 # aka layers!
head_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
embd_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,901,889
head_num     =  2
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 49.81 MiB is free. Including non-PyTorch memory, this process has 9.28 GiB memory in use. Of the allocated memory 8.00 GiB is allocated by PyTorch, and 285.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [31]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2
block_num = 6 # aka layers!
head_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
embd_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,901,889
head_num     =  2
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 59.38 MiB is free. Including non-PyTorch memory, this process has 9.26 GiB memory in use. Of the allocated memory 8.03 GiB is allocated by PyTorch, and 241.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [32]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)

using more blocks with skip-connection-layernorm-beefed up!


In [33]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2
block_num = 6 # aka layers!
head_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
embd_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,901,889
head_num     =  2
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 44.44 MiB is free. Including non-PyTorch memory, this process has 9.28 GiB memory in use. Of the allocated memory 8.07 GiB is allocated by PyTorch, and 214.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [34]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)
model.blocks=None

using more blocks with skip-connection-layernorm-beefed up!


In [35]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)
model.blocks=None
model.position_embeddings = None 
model.token_embeddings = None

using more blocks with skip-connection-layernorm-beefed up!


In [36]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)
model.blocks=None
model.position_embeddings = None 
model.token_embeddings = None
del model

using more blocks with skip-connection-layernorm-beefed up!


In [37]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)
model.blocks=None
model.position_embeddings = None 
model.token_embeddings = None
dir(locals)

using more blocks with skip-connection-layernorm-beefed up!


NameError: name 'model' is not defined

In [38]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)

dir(locals)

using more blocks with skip-connection-layernorm-beefed up!


['__call__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__self__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__text_signature__']

In [39]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)

dir(__module__)

using more blocks with skip-connection-layernorm-beefed up!


NameError: name '__module__' is not defined

In [40]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)

dir(__sel__)

using more blocks with skip-connection-layernorm-beefed up!


NameError: name '__sel__' is not defined

In [41]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)

del __self__

using more blocks with skip-connection-layernorm-beefed up!


NameError: name '__self__' is not defined

In [42]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)

del __dict__

using more blocks with skip-connection-layernorm-beefed up!


NameError: name '__dict__' is not defined

In [43]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)
dir()

using more blocks with skip-connection-layernorm-beefed up!


['AttentionHead',
 'AttentionwithFFNetBlock',
 'B',
 'BigramModel',
 'BigramModelWithAttention',
 'C',
 'F',
 'FeedForward',
 'In',
 'Iterable',
 'LayerNorm',
 'MultiHeadAttention',
 'Out',
 'T',
 '_',
 '_38',
 '_VSCODE_hashlib',
 '_VSCODE_types',
 '__',
 '__VSCODE_compute_hash',
 '__VSCODE_wrap_run_cell',
 '__VSCODE_wrapped_run_cell',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_data_copy',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i12',
 '_i13',
 '_i14',
 '_i15',
 '_i16',
 '_i17',
 '_i18',
 '_i19',
 '_i2',
 '_i20',
 '_i21',
 '_i22',
 '_i23',
 '_i24',
 '_i25',
 '_i26',
 '_i27',
 '_i28',
 '_i29',
 '_i3',
 '_i30',
 '_i31',
 '_i32',
 '_i33',
 '_i34',
 '_i35',
 '_i36',
 '_i37',
 '_i38',
 '_i39',
 '_i4',
 '_i40',
 '_i41',
 '_i42',
 '_i43',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'a',
 'a_tril',
 'atoi',
 'b',
 'batch_size',
 'block_num',
 'block_size',

In [44]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2
block_num = 6 # aka layers!
head_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
embd_size = 384     #18 # 36 # 72 # 144 # 288 # 396 # 384
context_size = 256  #8  # 16 # 32 # 64  # 128 # 256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,901,889
head_num     =  2
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacty of 9.77 GiB of which 66.12 MiB is free. Including non-PyTorch memory, this process has 9.29 GiB memory in use. Of the allocated memory 8.07 GiB is allocated by PyTorch, and 224.71 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [45]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)
del x, del y

SyntaxError: invalid syntax (<ipython-input-45-6ec4fd7e91e8>, line 9)

In [46]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)
del x, 
del y

using more blocks with skip-connection-layernorm-beefed up!


/home/hossein/anaconda3/lib/python3.11/site-packages/torch/cuda/memory.py:329: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [47]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)
import gc
torch.cuda.memory.reset_max_memory_allocated(0)
gc.collect()

using more blocks with skip-connection-layernorm-beefed up!


611

In [1]:
# in the name of God the most compassionate the most merciful
# in this section we will have a look at how a GPT model works
# in this section we will implement attention module, and see 
# how it works and lean more about its (sel-attention, cross
# attention, multi-head attention, etc)
# 
# so how are we going about this. we need to create a language model
# and then progressively imporve it with attention mechanism.
# we will basically be creating a kind of chatgpt (minus its chat capability and
# its obviously great perormance :)) 
# to keep this as simple as possible and not lose track of the important concepts involved,
# we can use our initial bigram model as the base model and work on improving that with attention.
# so lets start
#
# first lets import the basic stuff 

# for type hints
from collections.abc import Iterable

import random 
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# lets setup the manual seeds or determinstic output
torch.manual_seed(255)
np.random.seed(255)
random.seed(255)

# first lets read the dataset, we are going to use tinyshalespear which
# is around 1million characters (around 1MB in size), and our model is
# supposed to create texts resembling this dataset.
dataset = []
with open('./tiny_shakespear.txt','r') as file:
    # this time we read the whole text as one big str
    dataset = file.read()
    
print(f'{dataset[:100]=}')
# now lets create our atoi and itoa dictionaries for mapping
# lets create our unique character list, which is infact our vocabulary or vocab for short
vocab_list = sorted(set(''.join(dataset)))
vocab_size = len(vocab_list)
# ! explain token
# lets create our mapping dictionaries, we are basically going to use them
# for tokenization, converting our input into tokens which here are characters 
# and ultimately their integer representations
# so tokenization simply put, refers to converting an input into a list of numbers based on some creteria
# here, we just use a simple index number to map our vocabulary characters, which represent all character
# our model can use to generate text, to integer representation. 
# for realworld applications, we usually use libraries such as sentecepeace by google which works on a subword
# level which means, it doesnt encode the whole word, neither does it encode based on individual characters, it
# use sth in between. 
# (from its github repo: https://github.com/google/sentencepiece )
# SentencePiece is a re-implementation of sub-word units, an effective way to alleviate the open vocabulary 
# problems in neural machine translation. SentencePiece supports two segmentation algorithms, 
# byte-pair-encoding (BPE) [Sennrich et al.] and unigram language model [Kudo.]. 
#
# (For the reference:  
# Byte Pair Encoding (BPE) is a subword tokenization technique used in natural language processing (NLP) 
# and text processing tasks. It is a data compression algorithm that splits words into subword units. 
# BPE is commonly used in tasks such as machine translation, text generation, and language modeling.
# The basic idea behind BPE is to iteratively merge the most frequent pairs of characters or subword units
# in a corpus to create a new subword vocabulary. This merging process is based on the statistical properties
# of the corpus, specifically the frequency of character or subword pairs.
#
# Here's a high-level overview of the BPE algorithm:
# 1. Initialize the vocabulary with all the characters or subwords in the corpus.
# 2. Calculate the frequency of each character or subword in the corpus.
# 3. While the desired vocabulary size or a maximum number of iterations is not reached:
#    - Find the most frequent pair of characters or subwords in the corpus.
#    - Merge the pair into a new subword unit by concatenating them.
#    - Update the corpus by replacing occurrences of the merged pair with the new subword unit.
#    - Update the vocabulary and frequency counts based on the new corpus.
# 4. The final vocabulary is the set of subword units obtained after the desired number of iterations 
#    or vocabulary size is reached.
#
# BPE allows for the representation of both known and unknown words in a corpus. It is effective in 
# handling out-of-vocabulary (OOV) words and reducing the vocabulary size, which can improve the 
# efficiency and performance of NLP models. By breaking down words into subword units, BPE can capture
# morphological and semantic information more effectively, especially for languages with complex word 
# formations and agglutinative structures.
# 
# tiktokenize repo explains BPE in rather friendlier way: 
# Models don't see text like you and I, instead they see a sequence of numbers (known as tokens). 
# Byte pair encoding (BPE) is a way of converting text into tokens. It has a couple desirable properties:
# It's reversible and lossless, so you can convert tokens back into the original text
# It works on arbitrary text, even text that is not in the tokeniser's training data
# It compresses the text: the token sequence is shorter than the bytes corresponding to the original text. 
# On average, in practice, each token corresponds to about 4 bytes.
# It attempts to let the model see common subwords. For instance, "ing" is a common subword in English, 
# so BPE encodings will often split "encoding" into tokens like "encod" and "ing" 
# (instead of e.g. "enc" and "oding"). Because the model will then see the "ing" token again and again 
# in different contexts, it helps models generalise and better understand grammar.
#
# 
# A unigram language model however, is a type of statistical language model that predicts the probability 
# of each word in a sequence independently, based solely on the frequency of occurrence of individual 
# words in the training data. It does not consider the context or the order of the words in the sequence.
# In a unigram language model, the probability of a particular word is estimated by counting the frequency 
# of that word in the training corpus and normalizing it by the total number of words in the corpus. 
# The probability of a sequence of words is then calculated by multiplying the probabilities of each 
# individual word in the sequence.
# For example, consider the sentence "I love to eat pizza." In a unigram language model, the probability
# of this sentence would be calculated as the product of the probabilities of each word which would be: 
# P(I) * P(love) * P(to) * P(eat) * P(pizza).
# Unigram models are the simplest form of language models and do not capture any contextual information
# or dependencies between words. They are often used as a baseline or reference model in natural language 
# processing tasks. While unigram models are not very accurate in capturing the complexities of natural 
# language, they can be computationally efficient and useful in tasks where the context is less important, 
# such as certain text classification tasks or language generation tasks where only the frequency of 
# individual words matters.
# 
#
# Openai/chatgpt for example uses its own tokenizer, called tiktoken (https://github.com/openai/tiktoken)
# there are other tokenizers as well.
# huggingface has a good article on tokenizers (recommend it): https://huggingface.co/docs/transformers/main/tokenizer_summary 
# its a given that, each model only works with the tokenizer by which it was used to train. 
# so BERT, DistilBERT, and Electra only work with wordpiece, while chatgpt uses titoken and we! use
# simple characters-indexes! also note that the choice of tokenizer obviously affects our vocab size and
# model overhead/performance as well. 
# for example lets first implement our tokenizer and then compare it with sth like tiktoken module
atoi = {c:n for n,c in enumerate(vocab_list)}
itoa = {n:c for c,n in atoi.items()}
print(f'{vocab_size=}, {vocab_list=}')
print(f'{atoi=}')
print(f'{itoa=}')
# lets also create two helper functions to convert a list of these to the other part
def encode(characters:Iterable[str] ):
    return [atoi[c] for c in characters]

def decode(token_lst:Iterable[int]):
    return [itoa[n] for n in token_lst]

# lets test these 
print(f'{encode(dataset[:10])}')
print(f'{decode(encode(dataset[:10]))}')

# now lets try tiktoken
try: 
    import tiktoken
except:
    import os 
    os.system('pip install tiktoken')
    import tiktoken
# lets use the tokenizer for gpt2 model, this line downloads the gpt2 tokenizer and allows us to use it
encoder_gpt = tiktoken.get_encoding('gpt2')
# lets view its vocab size:
print(f'{encoder_gpt.n_vocab=:,}') # prints encoder_gpt.n_vocab=50,257
# now lets see how the encoding looks like here
print(f'{encoder_gpt.encode(dataset[:10])=}') # prints [5962, 327, 8846] 
# which is very intresting! while for us its 10 numbers/tokens for 10 characters(vocab size 65),
# this is 3 here with vocab size of 50,000!
# so we can have a short vocab size, at the expense of a larger sequence size, 
# or a large vocabsize and smaller sequence size.
# so thats why in practice, these subwords tokenizers are used for real world applications. but for our case
# we stick to our primitive tokenizer to keep things as simple as possible. 
# 
# Ok, so far so good. now we need to create a dataset, like before we need to have an input/label pair
# our input is a series of characters, (a sequence of some length), and our label is the next character
# sicne we are using a simple bigram model, given a single character, we want the probablity of what comes
# next. but, we also want to incorporate attention, and we want a context for our prediction, we want to
# be able to look at the past, and look at the past characters, and based on that do sth. this is the essence
# of attention (although this is not accurate, but for now this is the case, we will elaborate on this and expand
# this metaphor and reasoning inshallah)
# 
# lets first tokenize the whle dataset or corpus as its usually called in nlp nomenclature!
data = encode(dataset)
# since we are using pytorch lets convert that to a tensor
data_tensor = torch.tensor(data)
print(data_tensor[:100])
# now lets create a train/val split 
train_length = int(0.9 * len(data_tensor))
# note that we do not shuffle the data here like before, because we are not dealing with a list of samples!
# like names, here we are dealing with the whole text, and if we shuffle it like that, we just destroy it
# making it into a batch of random characters! which would not be useable for us anymore. we are trying to
# learn the underlying semantic and relationships hidden in our data, and randomizing them like that just 
# destroys those information! 
# to see this in action try this
_data_copy = data.copy()
random.shuffle(_data_copy)
print(''.join(decode(_data_copy[:100])))
# prints:
# Osetow Pr 
# oa geEeM rhnalelrt   aAdt Ktsw pTpeGxli
# oasEliMe
# vsieeose
# etttbSdnctiras  hnr:yhtayiuCv g
# so we simple divide the data normally
train_data = data_tensor[:train_length]
val_data = data_tensor[train_length:] 
# note that since we are planning on creating a simple transformer model, we usually dont feed the whole dataset
# becaue its prohibitevly computation intensive, instead, what happens in practice is that we, grab chunks 
# of data from the dataset and feed it to the transformer. and these chunks, of course has a length, what length?
# we usually specify a maximum_length for the input on which our transformer model works.
# this maximum_length is usually refered to as block_size or context_size.
# we had previously used different context_sizes and this is not really that different,
# lets for example define a context_size of 8
# block_size and context_size are interchangable
block_size = context_size = 8 
print(f'{train_data[:block_size]=}')
# which prints:
# train_data[:block_size]=tensor([18, 47, 56, 57, 58,  1, 15, 47])
# 
# one intresting observation we can make here is that, as simple as this seemingly ordindary list of numbers
# looks, this sequence of numbers, actually contains several examples. 
# if you think about it, each number, is a token, representing a character (or word, subword, etc),
# and it shows what comes after what. basically not only it shows several pairs so to speak, it also
# shows, which characters are more likely to come, before a specific character comes later. 
# what we are actually going to do is that, we are going to train all of these characters simultaneously
# notice that in this example, we have 7 examples in a sequence of 8 characters:
# lets elaborate on this more. 
# 1-in the context of 18, the next character is 47
# 2-in the context of 18,47, the next character is 56
# 3-in the context of 18,47,56, the next character is 57
# 4-in the context of 18,47,56,57 the next character is 58
# 5-in the context of 18,47,56,57,58 the next character is 1
# 6-in the context of 18,47,56,57,58,1 the next character is 15
# 7-and finally, in the context of 18,47,56,57,58,1,15 the next character is 47
# so this is infact 8 7 individual example embedded in a single context, 
# since we want the xontext_size to be 8, then we should grab one more character to have 
# 8 contexts
# lets visualize this in example in code
# lets have a typical sequence of size 8 
x = train_data[:block_size]
y = train_data[1:block_size+1]
# what would be our label? we want the next character so it would be the previous locations +1
# basically offset the input by 1!
# thats why we started from 1, becasue the input started from 0, and since the input ended at block_size
# its label(next character) would obviously be block_size+1, pretty obvious right?
#
# lets print this for better understanding 
for i in range(block_size):
    # note that since we are using slice, x[:i+1], gives us 0 up to ith element (inclusive)
    # this should be obvious! but I said it in case you forgot!!
    print(f'input is {x[:i+1].tolist()} label is {y[i]}')
# prints 
# input is [18] label is 47
# input is [18, 47] label is 56
# input is [18, 47, 56] label is 57
# input is [18, 47, 56, 57] label is 58
# input is [18, 47, 56, 57, 58] label is 1
# input is [18, 47, 56, 57, 58, 1] label is 15
# input is [18, 47, 56, 57, 58, 1, 15] label is 47
# input is [18, 47, 56, 57, 58, 1, 15, 47] label is 58
# so as you can see we started from context_size of 1 up to context_size of 8.
# note that we do not do this simply for the efficancy aspect of it, but also, when we feed these to
# our transformer model, we are making sure that the transformer model gets used to see all combinations
# of our input as well (from the context size of 1 up tp the context size of 8). 
# this way not only it sees the whole context_size as we initially expected
# but also all sequences before it, and basically what consituted to make the sample. 
# this allows us to later on, at test time be able to create sequences as small as context_size of only 1
# up to the max_length which is our context_size of 8 in our case.
# ! recheck and elaborate to clear any confusion
# !so by doing this, the transformer can learn how to predict/create/genrate text up to context_size, and after
# !it reached thta, we have to truncate it, becausse the transformer model never recieves more than the
# !context_size as input when its predicting the next character.
# so far what we covered here was the time dimension of our input. we have a sequence, and each entery
# basically denotes a time t dimension, at which, a character is introduced. 
# another imporatnt aspect we need to take care of is the batch dimension, cuz we are going to feed 
# multiple examples at once, we use this to harness the gpu parallilization capalibity in pytorch
# as without it, training this simple model would take a lot of time on cpu! 
#
# so lets create a function that gives us a batch of the inputs rather than a single list/tensor!
# 
batch_size = 4 
# our block_size
context_size = 8 
# 
def get_batch(split, batch_size):
    #lets grab the data basedo on the split
    data = train_data if split =='train' else val_data
    # we want a batch of 4 of 8 characters (context-size). 
    # to make a batch we can grab 4 random indices as input and then expand them
    # by adding the next 8(context_size) characters to them, in order not to go past the 
    # last index, we subtract the length of data(last valid index) from context size 
    idxs = torch.randint(low=0, high=len(data)-context_size, size=(batch_size,))
    # we have our indices, so lets create our samples
    # x = [data[i:i+context_size] for i in idxs]
    # and for labels we do the same thing but offset it by 1 so each characters label
    # becomes the next character
    # y = [data[i+1:i+context_size+1] for i in idxs]
    #
    # print(f'{x=}')
    # print(f'{y=}')
    # prints: 
    # x=[tensor([58, 39, 49, 43,  1, 51, 63,  1]), tensor([53, 59, 41, 46,  5, 42,  1, 61]), tensor([47, 53, 52,  1, 39, 57,  1, 63]), tensor([39, 57, 58,  1, 51, 63,  1, 50])]
    # y=[tensor([39, 49, 43,  1, 51, 63,  1, 54]), tensor([59, 41, 46,  5, 42,  1, 61, 47]), tensor([53, 52,  1, 39, 57,  1, 63, 53]), tensor([57, 58,  1, 51, 63,  1, 50, 53])]
    #
    # as you can see we have a list of tensors. since we want a batch, we just stack them or concat them
    # on top of each other
    # using stack!
    # x = torch.stack(x)
    # y = torch.stack(y)
    # stack() is intrestingly implemented by concat(), so if we want, we can do the same using concat!
    #
    # by default conact, concatenates all the tensors as one large tensor!
    # we need a reshape/view to get the right shape
    # since we are using view, no copy,etc is done, and its an efficient operation
    # x = torch.concat(x,dim=0).view(batch_size, context_size)
    # y = torch.concat(y,dim=0).view(batch_size, context_size)
    # 
    # however if we add an extra dim to our inputs we can remove the extra reshape/view
    # x = torch.concat([t.unsqueeze(0) for t in x], dim=0)
    # y = torch.concat([t.unsqueeze(0) for t in y], dim=0)
    #
    # By unsqueezing, we ensure that the tensors have the same number of dimensions before 
    # concatenating them with torch.cat/concat/concatenate.(side tip: concat and concatenate are aliases for cat) 
    # This effectively replicates the behavior of torch.stack.
    # see torch.cat concatenates tensors along an existing dimension, and we only have 1 dimensional
    # tensors, so it will concatenate them along that, effectively making one large 1 dimensional vector
    # however, when we use unsqueeze on each tensor, using torch.unsqueeze(0), we are adding a new dimension
    # (dim 0) to that tensor, making it 1,x instead of the original shape of (x,).  
    # So, by unsqueezing each tensor along the desired dimension, which for us is the 0ths dimension (or row dim) 
    # and then concatenating them with torch.cat, we achieve the same result as torch.stack.
    # 
    # in practice we'd like to do this all in one go!
    x = torch.stack([data[i:i+context_size] for i in idxs])
    y = torch.stack([data[i+1:i+context_size+1] for i in idxs])
    
    return x,y

x,y  = get_batch('train',batch_size=4)
print(f'{x.shape=}\n{y.shape=}')
print(f'{x=}\n{y=}')
# which prints 
# x.shape=torch.Size([4, 8])
# y.shape=torch.Size([4, 8])
# tensor([[58, 39, 49, 43,  1, 51, 63,  1],
#         [53, 59, 41, 46,  5, 42,  1, 61],
#         [47, 53, 52,  1, 39, 57,  1, 63],
#         [39, 57, 58,  1, 51, 63,  1, 50]])
# tensor([[39, 49, 43,  1, 51, 63,  1, 54],
#         [59, 41, 46,  5, 42,  1, 61, 47],
#         [53, 52,  1, 39, 57,  1, 63, 53],
#         [57, 58,  1, 51, 63,  1, 50, 53]])
#
# now this is our batch of data, 4 samples with 8 characters, bascially 32 examples, lets see that as well
for b in range(batch_size):
    # this is our time dimension
    for t in range(context_size):
        print(f'{x[b,:t+1]} --> {y[b,t:t+1].item()}')
#        
# prints : 
# tensor([58]) --> 39
# tensor([58, 39]) --> 49
# tensor([58, 39, 49]) --> 43
# tensor([58, 39, 49, 43]) --> 1
# tensor([58, 39, 49, 43,  1]) --> 51
# tensor([58, 39, 49, 43,  1, 51]) --> 63
# tensor([58, 39, 49, 43,  1, 51, 63]) --> 1
# tensor([58, 39, 49, 43,  1, 51, 63,  1]) --> 54
# tensor([53]) --> 59
# tensor([53, 59]) --> 41
# tensor([53, 59, 41]) --> 46
# tensor([53, 59, 41, 46]) --> 5
# tensor([53, 59, 41, 46,  5]) --> 42
# tensor([53, 59, 41, 46,  5, 42]) --> 1
# tensor([53, 59, 41, 46,  5, 42,  1]) --> 61
# tensor([53, 59, 41, 46,  5, 42,  1, 61]) --> 47
# tensor([47]) --> 53
# tensor([47, 53]) --> 52
# tensor([47, 53, 52]) --> 1
# tensor([47, 53, 52,  1]) --> 39
# tensor([47, 53, 52,  1, 39]) --> 57
# tensor([47, 53, 52,  1, 39, 57]) --> 1
# tensor([47, 53, 52,  1, 39, 57,  1]) --> 63
# tensor([47, 53, 52,  1, 39, 57,  1, 63]) --> 53
# tensor([39]) --> 57
# tensor([39, 57]) --> 58
# tensor([39, 57, 58]) --> 1
# tensor([39, 57, 58,  1]) --> 51
# tensor([39, 57, 58,  1, 51]) --> 63
# tensor([39, 57, 58,  1, 51, 63]) --> 1
# tensor([39, 57, 58,  1, 51, 63,  1]) --> 50
# tensor([39, 57, 58,  1, 51, 63,  1, 50]) --> 53 
#
# so now that we have the data sorted out, lets create our model. 
# as we said, we are going to use a bigram model and later add attention mechanism to it. 
# to make things easier and more self contained, lets add all the required logic to this model
# like when we do a forward, we be able to calculate loss as well if we are given the targets
# so lets go
class BigramModel(nn.Module):
    def __init__(self, vocab_size) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        # our bigram model was nothing more than a 2d array of vocab_size, we can achive that using 
        # a single weight matrix or torch.Embedding. we use torch.Embedding to not reinvent the wheel!
        self.token_embedding = torch.nn.Embedding(vocab_size, vocab_size)
    
    # since we want to be able to calculate loss, if there are labels, we get Y as well
    def __call__(self, inputs:torch.Tensor, labels:torch.Tensor=None) -> torch.Tensor:
        logits = self.token_embedding(inputs)
        loss = None
        if labels is not None:
            # calculate loss 
            # print(f'{logits.shape=}') # prints (4,8,65)
            # note that both inputs and labels have the shape (B,T)
            # but logits has the shape (B,T,C) which C here equals vocab_size 
            # this is an issue for torch.crossentropy, as it expects the input
            # to be in the form of (B,C,T), that is, the channels/embeddings dimension
            # need to be right after the batch dimension or otherwise it wont work.
            # so we need to account for that.
            # we can go on permute the dimensions, like this and make crossentropy happy 
            # logits = logits.permute((0,2,1))
            # however, if for some reason the output of logits in the form of (4,8,65) is not ideal, 
            # maybe for example because really what it is 32 examples arranged in a (4,8) shape and 
            # we rather a normal 2d tensor of shape (32,65), 
            # we can instead, do just that, flatten the two dimensions into one and carry on!
            # we basically are concatenating the batch and time dimensions
            # into one dimension (effectively, stacking samples on top of each other), instead of having 
            # four compartments, each having 8 segments, we are going to have 1 long compartment with 32 segments/rows
            # for the lack of better words!!
            # so lets first get the shapes 
            # B,T,C = logits.shape
            # logits = logits.view(B*T,C)
            # and we also need to do the same for the labels 
            # labels = labels.view(B*T)
            # we could also do,
            # labels = labels.view(-1)
            # but thats not really needed, so we just simply permute logits temporarily so the logits shape
            # stays the same regardless of calculating the loss or not 
            loss = F.cross_entropy(logits.permute((0,2,1)), labels)
        return logits, loss
    
    # now lets also add a generate method to generate texts
    def generate(self, idx, max_token_count):
        # before we implement this lets review what we expect this method to do
        # we want this to generate some characters, given an initial character
        # we also want to be able to control the length of the generated text
        # so we take a max_token_count.
        # we take the initial index, 
        # feed it to the model, 
        # get the logits,
        # turn logits to probs,
        # use torch.multinomial to sample from our probablity distribution
        # get the new character index, and add it to a list to gradually 
        # -create our final text output
        # so lets go 
        # our idx should have the shape [B,T]
        assert len(idx.shape) == 2, f'idx.shape({idx.shape}) should have (b,t) form.'
        #
        # lets generate as many characters/idx as max_token_count specifies
        # this is basically specifies the time/sequence dimensions
        for i in range(max_token_count):
            # since we are defining a class method, to call the callable, we simply use self()
            logits, _ = self(idx)
            # to get the probablities 
            # usually we would simply do 
            # probs = torch.softmax(logits, dim=1)
            # but here, we want to only focus on the next character because this is what comes next
            # each time obviously, so we only take the last timestep to see what the model predicted
            logits = logits[:,-1,:] # this now becomes (B,C) instead of the initial (B,T,C)
            probs = torch.softmax(logits, dim=-1) 
            # now lets sample from it
            idx_next_char = torch.multinomial(probs, num_samples=1, replacement=True) # shape is (B,1)
            # now lets add this to the next input to be fed to the model 
            # since idx has the shape(batch, T), we should add this tothe second dimension
            # to the time dimension/ or sequence dimension. this as the loop goes on, 
            # creates the shape (B,T+1) 
            #this doesnt make sense for this particular model, becasue we are always checking
            # the next character given the previous one, so all the concatenation we are doing
            # is just useless. the reason we are implementing this like this, is to create a 
            # base, so that we can improve upon it when we add attention later on which will use
            # the history of previous characters.
            idx = torch.cat((idx, idx_next_char), dim=1) 
        
        # and finally when all is done return the idx which by now should have the whole output
        return idx
    
model = BigramModel(vocab_size)
out,loss = model(x,y)
print(f'{out.shape} {loss}')
# prints:
# torch.Size([32, 65]) 4.574285984039307
# the loss is good, we learned previously that we can evaluate a base loss provided our number of classes
# since we have 65 classes (our vocab_size or number of characters involved) the uniform probability for each class
# would be 1/65 =0.015384615, which if we take its negative log would turn out to be -ln(1/65) = 4.17438727, 
# which is pretty close to the loss we got here, signifying its a pretty decent value to begin with.
# also lets see the generate method at work
# lets create a dummy input, basically a batch of 1 and sequence of 1 of zero!
# we do this to generate a text from scratch
inputs = torch.zeros(size=(1,1),dtype=torch.int32)
output = model.generate(inputs, max_token_count=100)
print(f'{output=}')
print(''.join(decode(output.squeeze(0).tolist())))
# prints 
# wUbN:i :kLnOqHCQTG;.MbupOWH Xfi!MSUalaQppNNqIoWfmuSIIfmuZqf-3NLt-YkOc3vC-YkBfEHr3pJodwVY.nw.,r!&,Clt
# which is expected since our model is not trained yet! 
# lets train the model now and see how it works

batch_size = 32
context_size = 8
vocab_size = len(vocab_list)
max_iter = 20000

model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device=device)

print(f'{torch.__config__.show()}')
print(f'{device=}')
print(f'{model=}')

model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        print(f'{loss}')
# prints 
# 4.760574817657471
# 3.727213144302368
# 3.0139858722686768
# 2.6749324798583984
# 2.5884087085723877
# 2.5262675285339355
# 2.4420430660247803
# 2.4787800312042236
# 2.4680707454681396
# 2.564662456512451
# 2.51802659034729
# 2.562812328338623
# 2.422783613204956
# 2.5023772716522217
# 2.501049518585205
# 2.411632537841797
# 2.4028165340423584
# 2.4135568141937256
# 2.50952410697937
# 2.574878215789795
# relying on single batch loss is not a good idea to measure the performance of a model. moreover
# relying on training loss, is not good either, so it would be much better if we considered more batches
# for loss and even better we could also investivate the models performance on our validation set. 
# so lets do just this and define a function that calculates loss for training and validation sets alike
# but considers more batches for loss calculation

@torch.no_grad()
def evaluate_loss (iterations, device=None):
    results={}
    # before calculating the loss, lets switch to eval mode,although for our specific case this doesnt matter
    # but its goo practice, as later on, we will add layers that their behavior do change depending on traing
    # val mode.  
    model.eval()
    if device is None:
        # use the device assigned to what model params are assigned
        device = next(model.parameters()).device
    # since we already used no_grad decorator, we dont need to use no_grad context manager here
    # with torch.no_grad():
    for split in ['train','val']:
        losses = torch.zeros(size=(iterations,), device=device)
        for i in range(iterations):
            x,y = get_batch(split,batch_size)
            x,y = tuple(t.to(device) for t in (x,y))
            logits, loss = model(x,y)
            losses[i] += loss
        results[split] = losses.mean(0)
    # since we want to use this inside training loop, make sure we set the model back to train mode
    # incase we use layers such as batchnorm,etc that the require being trained!
    model.train()
    return results

loss= evaluate_loss(200)
print(f'{loss["train"]=} {loss["val"]=}')
# so now that our function seems to be working lets use it in the training loop :
model = BigramModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)
model = model.to(device=device)
model.train()
for i in range(max_iter):
    x,y = get_batch('train', batch_size=batch_size)
    x=x.to(device)
    y=y.to(device)
    logits, loss = model(x,y)
    # zero out the grad
    model.zero_grad(True)
    # do a backward pass
    loss.backward()
    # and do a single optimizer update
    optimizer.step()
    
    if i%1000==0:
        # now instead of simply printing loss, lets use our new function!
        loss= evaluate_loss(200)
        print(f'train_loss: {loss["train"].item():.4f},  val_loss: {loss["val"].item():.4f}')
# which results in :
# train_loss: 4.7491,  val_loss: 4.7430
# train_loss: 3.6673,  val_loss: 3.6670
# train_loss: 3.0460,  val_loss: 3.0511
# train_loss: 2.7335,  val_loss: 2.7492
# train_loss: 2.5948,  val_loss: 2.6156
# train_loss: 2.5292,  val_loss: 2.5470
# train_loss: 2.4982,  val_loss: 2.5194
# train_loss: 2.4807,  val_loss: 2.4991
# train_loss: 2.4737,  val_loss: 2.5036
# train_loss: 2.4650,  val_loss: 2.4949
# train_loss: 2.4648,  val_loss: 2.4910
# train_loss: 2.4626,  val_loss: 2.4836
# train_loss: 2.4570,  val_loss: 2.4893
# train_loss: 2.4517,  val_loss: 2.4823
# train_loss: 2.4508,  val_loss: 2.4819
# train_loss: 2.4564,  val_loss: 2.4839
# train_loss: 2.4555,  val_loss: 2.4802
# train_loss: 2.4503,  val_loss: 2.4927
# train_loss: 2.4522,  val_loss: 2.4905
# train_loss: 2.4572,  val_loss: 2.4888
#
# now lets check its output again after some training 
inputs = torch.zeros(size=(1,1), device=device, dtype=torch.int32)
output = model.generate(inputs, max_token_count=500)
print(''.join(decode(output.squeeze(0).tolist())))
# prints :
# Ane pod BUL:
#
# Oringe
# nfote s Rinsou oe the,
# An ETwe
#
# PUSENI bmilft!'d ate t Ifur
# Athomeg?
# Whagre,
# TCo belangead ne bl e, bednth ftor veso.
# US tla lin:
# Yourthiee he w whetitrd;
# Angoknghothartro ll heshethor hie douluk t, s se, denopit s h wom uspouct blyowie s'nos t prou gmesat
# Shd, tieithy.
# Asiour hofo bay avear hanousthaie
# Calonth wolulo.
#
# The hancondors pthighar:
# WAME outhaser d!
# S:
# h im t w s, inowo ORILOUCHoou isecetoubecompis
# Ay gnd, teve luthorea, therpeclsthevecthede fichim?
# PUSa VI aken 
# 
# which looks much better than the initial random output we got earlier.

dataset[:100]='First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
vocab_size=65, vocab_list=['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
atoi={'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't':

In [2]:
torch.manual_seed(255)
# now lets add attention mechanism to our base model. 
# attention mechanism at its core tries to take advantage of the rich information embedded
# in the sequence. for our work, this is specifically about the past history but in general 
# attention can utilize both past and future connections/sequence tokens. what we described 
# here just now is not exactly accurate, but gives us a foundation to build our intuition as
# we continue on. we will elaborate more of course and hopefully get it all.
# before we venture any further into the crux of the matter, let us learn about a technique
# thats used to efficiently implement attention mechanism.
# for this purpose,lets imagine we have a simple input like the following: 
# lets create and input of the following shape
B,T,C = (4,8,2)
# to make it more intuitive lets make a tensor with known numbers andthen reshape it
x = torch.arange(0,64,dtype=torch.float).view(B,T,C)
# imagine we have an input like what we encountered previously in our examples. in this sample 
# input, we have a batch of 4 samples, each having 8 sequences with each sequence having a vector of 2 values
# what we are planning to do is to provide a way by which each token can communicate with other 
# tokens. we have 8 tokens in our sequence. so we want our tokens to be able to communicate with
# all previous tokens that came before it. the reason we are only looking in the past token is 
# simply becasue the we are trying to perdict the future, so it only makes sense to look at the
# past and current timestamp and infer on what to do for the future.  
# so the easiest way to implement a kind of communication between tokens could be to sum or average the
# values of all previous tokens plus the current one as a way of taking into account their contribution
# to the final answer.
# that is, lets say if we are currently at token 5, we take the average of 
# the current token and all previous tokens before it, effectively making a feature vector that
# reflects our current status of the sequence so far, having taken all previous tokens/steps up to now.
# note that as you may also have thought, summing or averaging arent the best way to model such interations.
# in fact they are an extremely weak form of interaction between tokens,
# this kind of communicating is extremely lossy so to speak, that is we lose a great deal of information 
# concerning the underlying relationships between tokens, their arrangements,their implicit interactions,
# semantics, etc. but for now this is ok. we will later on see how to bring back such information.
# so now what we want to do, is to calculate the sum or average of all tokens up to the current token in 
# all batches at the same time.
# a naive way would be to do sth like this using a for loop:
results = torch.zeros(size=(B,T,C))
for b in range(B):
    for t in range(T):
        results[b,t] = x[b,:t+1].mean(0)
print(f'{results=}')
# which prints 
# results=tensor(
#        [[[ 0.,  1.],
#          [ 1.,  2.],
#          [ 2.,  3.],
#          [ 3.,  4.],
#          [ 4.,  5.],
#          [ 5.,  6.],
#          [ 6.,  7.],
#          [ 7.,  8.]],

#         [[16., 17.],
#          [17., 18.],
#          [18., 19.],
#          [19., 20.],
#          [20., 21.],
#          [21., 22.],
#          [22., 23.],
#          [23., 24.]],

#         [[32., 33.],
#          [33., 34.],
#          [34., 35.],
#          [35., 36.],
#          [36., 37.],
#          [37., 38.],
#          [38., 39.],
#          [39., 40.]],

#         [[48., 49.],
#          [49., 50.],
#          [50., 51.],
#          [51., 52.],
#          [52., 53.],
#          [53., 54.],
#          [54., 55.],
#          [55., 56.]]])
#
# You may find out that, some researchers refer to this operation here as BoW, or bag of words. 
# We are effectively averaging embeddings here and averaging embeddings can be considered a form of 
# Bag of Words (BoW) representation. In BoW, the focus is on the occurrence and frequency of words, 
# rather than their order or structure. By averaging embeddings, we are essentially treating each word 
# as an independent feature and capturing its representation in the form of a numerical vector.
#
# While averaging embeddings does not capture the exact frequency of each word, it does capture the 
# overall distribution and semantic information present in the text. Similar to BoW, this approach 
# disregards word order and focuses on the presence and representation of words. However, it should be
# noted that averaging embeddings may preserve some semantic relationships between words, which BoW 
# representations might not capture as effectively.
# 
# Side note: 
# Bag of Words (BoW) is a commonly used technique in natural language processing (NLP) for representing text
# as a numerical feature vector. It disregards the order and structure of words in a document and focuses only
# on their occurrence and frequency.
# In the BoW model, a document or a piece of text is represented as a "bag" (unordered set) of words, where 
# each word is treated as an independent feature. The presence or absence of words in the document is encoded
# as a binary value (0 or 1), and the frequency of each word is often used as the value in the feature vector.
# 
# Here's a step-by-step overview of the BoW process:
# 1. Tokenization: The text is split into individual words or tokens. Punctuation marks, whitespace, and other
#    special characters are usually removed or treated as separate tokens.
# 2. Vocabulary Creation: A vocabulary is created by taking all unique words from the entire corpus 
#    (collection of documents). Each unique word is assigned a unique index or position in the vocabulary.
# 3. Vectorization: Each document is represented as a feature vector, typically a one-hot encoding or a count
#    vector. In a one-hot encoding, each word in the vocabulary corresponds to a binary feature, and the vector
#    contains 1s in the positions where the word occurs and 0s elsewhere. In a count vector, the value at each 
#    position represents the frequency of the corresponding word in the document.
# 4. Classification or Analysis: The resulting feature vectors can be used as input to machine learning models
#    for tasks such as text classification, sentiment analysis, document clustering, or information retrieval.
# Needless to say, BoW has some limitations. It does not capture the semantic meaning or context of words, as
# it treats each word independently. It also ignores the grammar and word order. However, BoW is simple, 
# efficient, and can be a useful baseline representation for various NLP tasks.
# 
# so to recap one more time, we are basiaclly treating each timestep/sequence dimension, as a word, so we have
# 8 tokens/words, and we are averaging them (we are infact averaging their embeddings, but thats obvious!)
# so we got ourselves bow representation!
# now back to our discussion, if we  try to visualize the results we get 
print(x[0])
print(results[0])
# prints:
# x[0]=
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
#         [ 6.,  7.],
#         [ 8.,  9.],
#         [10., 11.],
#         [12., 13.],
#         [14., 15.]])
# results[0]=
# tensor([[0., 1.],
#         [1., 2.],
#         [2., 3.],
#         [3., 4.],
#         [4., 5.],
#         [5., 6.],
#         [6., 7.],
#         [7., 8.]])
# if you look closely, you'll notice that each row, contains the mean of all the rows before it
# consider the first 3 rows in x[0], 
# tensor([[ 0.,  1.],
#         [ 2.,  3.],
#         [ 4.,  5.],
# now refer to the 3rd row in results[0] which is :
#         [2., 3.],
# likewise, consider the last row in results which contains the average for all the rows in x:
# 0+2+4+6+8+10+12+14 = 56 which when divided by their count, 8, results in 7, 
# and this is the same for the second column, thus we get:
# results[0,7] = [7., 8.]])
# this all good but the problem is using for loops to calculate this is very inefficient, it
# happens that this operation can be efficiently calculated using matrix multiplication.
# lets learn this trick using an example: 
# suppose we have the following as the input: 
a = torch.ones(size=(3,3))
b = torch.randint(0,10,size=(3,2)).float()
c = a@b
print(f'{a=}')
print(f'{b=}')
print(f'{c=}\n----')
# prints
# a=tensor(
#        [[1., 1., 1.],
#         [1., 1., 1.],
#         [1., 1., 1.]])
# b=tensor(
#        [[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c=tensor(
#        [[20., 13.],
#         [20., 13.],
#         [20., 13.]])
# nothing fancy here, we have matrix multiplication, the first row of 'a' is dot-producted by first col
# of 'b', then sumed, it makes up the first col of first row in c. likewise the first row of 'a' dot 
# the second col of 'b', then summed the results, makes up the second col of first row in c. and this goes on for the rest of the matrixes. this is
# what we learned back in higheschool, so what is it exactly that we are learning exactly?
# if you look closely, you'll notice that, the c cols are actually the sum of all the rows in b!
#        [9]
# b[:,0]=[5] 
#        [6]
# 9+5+6 is 20! likewise, 
#        [3]
# b[:,1]=[5] 
#        [5]
# 3+5+5 is 13!
# hence c = [20., 13.] which is repeated obviously because the second and third rows of a are all 1s as well.
#           [20., 13.]  
#           [20., 13.] 
# I guess you are now starting to get where we are going with this, if we can some how alter the 'a' matrix,
# we may very well be able to achieve our goal! how you may ask? the answer is using torch.tril!
# torch.tril() is a function that returns a matrix from a given tensor, so that half of it set to zero,
# basially it creates a triangular tensor, where the right half is just zeros! lets see how it works, 
# lets apply it on 'a'
a_tril = torch.tril(a)
print(f'a_tril:\n{a_tril}')
# it prints
# a_tril=tensor(
#        [[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# as you can see the the right half is set to zero and we are left with a triangle shape of 1s! on the left side
# now if we do a@b this time we get:
c = a_tril@b 
print(f'b:\n{b}')
print(f'c:\n{c}')
# a_tril:
# tensor([[1., 0., 0.],
#         [1., 1., 0.],
#         [1., 1., 1.]])
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[ 9.,  3.],
#         [14.,  8.],
#         [20., 13.]])
# now if you look closely, you'll notice that, this time, each row in c, is effectively the sum of the previous
# rows in b, like the first row of 'a' is only 1 in the 0ths column, so the first row of b is copied in c intact
# (workout the math and see why). 
# the second row in 'a', now has two 1s in col 0 and 1 respectively, which effectively translates to summing the first
# two rows in b. (9+5 =14, 3+5=8). likewise, the third row in 'a' is all 1s, signfigying all rows in b will
# be summed which gives us (9+5+5=20, 3+5+5=13). 
# so basically we are doing sums here, becasue our tensor a is all ones. so if we want to somehow calculate the
# average, instead of sum, we can easily change 'a' by normalizing it so that the each row sums to 1 (i.e. all cols
# sum to 1), this way the end result will be the average (becasue the 'b' is multiplied by a fraction/scale and then summed)
# so if we scale 'a' by the sum of all its columns, we should get average instead
a = torch.ones(size=(3,3))
a = torch.tril(a)
a = a/a.sum(dim=1, keepdim=True)
print(f'a:\n{a}')
# a:
# tensor([[1.0000, 0.0000, 0.0000],
#         [0.5000, 0.5000, 0.0000],
#         [0.3333, 0.3333, 0.3333]])
#
# note that, now each row, sums to 1. the first row, the first element is 1, because the rest are 0s
# but in the second row, as there are two 1s, the probabality is divided between the two, each being 0.5
# likewise, in the third row, as there are 3 1s, the probablity is divided between all of them, making each
# to have the value 0.33
# and now if we try to multiply them, we get average as the result:
c = a@b 
print(f'calculating average:')
print(f'b:\n{b}')
print(f'c:\n{c}')
# we get 
# calculating average:
# b:
# tensor([[9., 3.],
#         [5., 5.],
#         [6., 5.]])
# c:
# tensor([[9.0000, 3.0000],
#         [7.0000, 4.0000],
#         [6.6667, 4.3333]])
# we see that, each row in c, is the average of all the rows before it. 
#
# so using this trick, we can take the incremental average of any matrix we like. 
# now that we learned the trick, lets go back and implement the bows for loops using this techique !
# prevbiously we had : 
# results = torch.zeros(size=(B,T,C))
# for b in range(B):
#     for t in range(T):
#         results[b,t] = x[b,:t+1].mean(0)
# which calculated the average for sequence dimensions (tokens) incrimentally
# lets do this now 
# we want a TxT weight becasue we want to average T timestep/tokens
weight =  torch.tril(torch.ones(size=(T,T)))
# remember to set keepdim=True, or otherwise, as we sum along dim=1, we lose that dim
# (it collapses, and then the broadcast will be wrong, it will infact make each column 
# have one probablity instead of each row, which is the exact opposite of what we want)
weight = weight/weight.sum(dim=1, keepdim=True)
print(f'weight:\n{weight}')
# and now we need to multiply this by x! 
bow_results= weight@x
print(f'bow_results:\n{bow_results}') 
# which prints 
# bow_results:
# shape: torch.Size([4, 8, 2])
# tensor([[[ 0.0000,  1.0000],
#          [ 1.0000,  2.0000],
#          [ 2.0000,  3.0000],
#          [ 3.0000,  4.0000],
#          [ 4.0000,  5.0000],
#          [ 5.0000,  6.0000],
#          [ 6.0000,  7.0000],
#          [ 7.0000,  8.0000]],

#         [[16.0000, 17.0000],
#          [17.0000, 18.0000],
#          [18.0000, 19.0000],
#          [19.0000, 20.0000],
#          [20.0000, 21.0000],
#          [21.0000, 22.0000],
#          [22.0000, 23.0000],
#          [23.0000, 24.0000]],

#         [[32.0000, 33.0000],
#          [33.0000, 34.0000],
#          [34.0000, 35.0000],
#          [35.0000, 36.0000],
#          [36.0000, 37.0000],
#          [37.0000, 38.0000],
#          [38.0000, 39.0000],
#          [39.0000, 40.0000]],

#         [[48.0000, 49.0000],
#          [49.0000, 50.0000],
#          [50.0000, 51.0000],
#          [51.0000, 52.0000],
#          [52.0000, 53.0000],
#          [53.0000, 54.0000],
#          [54.0000, 55.0000],
#          [55.0000, 56.0000]]])
# which gives us the same results as we expected.
# one more thing before we continue on, note that the weight matrix is TxT while 
# the input is (BxTxC). (T,T) and (B,T,C) are not compatible, so what happens is
# that (T,T) is reshaped and a batch dimension is added to (T,T),making it (1,T,T)
# and then this is broadcasted along the batch dimension (replicated) to become 
# (B,T,T), then this will be multiplied by the (B,T,C)( note that at this stage
# a@b will be a batch multiplication operation, if you set the batch dimension aside, youll
# see that the rest of the dimensions match up, we have (T,T) and (T,C) which will
# result in (T,C). now if we add the batches back in, we will endup with (B,T,C)
# so in practice, the multiplication is done B times and then results are stacked.
# the first batches will multiply each other (T,T)x(T,C) = (T,C)
# the second batches will then multiply each other as well, getting another (T,C)
# and this goes on until we get (B,T,C))
#
a = torch.arange(0,9).view(3,3)
# a = torch.ones((3,3)).long()
b = torch.arange(0,30).view(2,3,5)
print(f'a:\n{a}')
print(f'b:\n{b}')
c = a@b 
print(f'c=a@b:\n{c}')
# is equivalent to 
# add a batch dimension to a
a = a.view(1,*a.shape) # or a.unsqueeze(0)
print(f'a with batch dim:\n{a.shape=}')
# replicate along the batch dimension
a = torch.cat(tuple(a.clone() for i in range(len(b))), dim=0)
print(f'{a.shape=}')
print(f'a(after replication along dim=0):\n{a}')
print(f'b:\n{b}')
# and now we have a case of batch-multiplication, the batch is the same 
# and sub tensors also are compatible, we have (T,T) and (T,C) so we now
# individually multiply each sub-tensor
c2 = torch.zeros_like(b)
for i in range(b.shape[0]):
    c2[i,...] = a[i]@b[i] # (T,T) x (T,C) -> (T,C)
# and ultimatley the result will have the shape (B,T,C)
print((c==c2).all())
# so to recap this 
# When performing the matrix multiplication `c = a @ b` with the given tensors, 
# several broadcasting steps occur to align the dimensions properly. 
# Here's a how it happens:
# 1. Tensor a has the (shape: 3, 3):
# 2. Tensor b has the (shape: 2, 3, 5):

# 3. They are not compatible so we broadcast tensor a to match the shape of b. 
# tensor a is expanded to (1, 3, 3) to have a batch dimension, and replicated 
# along dim 0 to result in (2, 3, 3). now both tensors have a batch of 2, and 
# if we put aside the batch dimension for a second, we'll notice that the rest
# of the shapes are compatible (3,3) and (3,5). so when we bring back the batch 
# dimension, we see we have a case of batch multiplication, that is we have 2
# sets of compatible tensors that need to be multiplied together. so we use a 
# simple for loop, to do multiplication, and stack the results and we are done!
#
# now back to our discussion. as we just saw, the traingualr shape in our weighted
# sum matrix, allows that each token at t dimension, can only interact with the tokens
# before it.
# there is another way of implementing the same thing but a bit differently
# if you look closely you can see that, we are dealing with probablities, so
# we may verywell use softmax to simplify this further.
# first lets create our triangular weight matrix
tril_tensor = torch.tril(torch.ones(size=(T,T)))
# now lets create a mask
weight = torch.zeros_like(tril_tensor)
# now lets mask all the zeros to -inf, so when we do softmax, all those -infs
# become 0. (recall that exp^-inf is 0! while exp^inf is inf! so its important to 
# set -inf (and also exp^0 is 1))
# so effectively what happens here is that, we setting each entery in trail_tensor
# with zero to -inf, and leave the rest as zeros. when this tensor goes through softmax
# the 0s will be 1s and -infs will be 0s before they are normalized, when they are normalized
# the probablity of 1 will be split between all the enteries with the value of 1.
# so the first row has a single 1, so it will be 1.0, the second row has two 1s, so
# each one will take 0.5, the third will have 3 1s, so they each will become 0.333
# and so on.
weight=weight.masked_fill(tril_tensor==0, -torch.inf)
# now calculate the probs for each row, treating all cols as probs so their sum is 1
weight = weight.softmax(dim=-1)
bow_results2 = weight@x
print(f'{torch.all(bow_results==bow_results2)}')
# so you might ask, why would we want to make things more complicated like this to only
# achieve what we already achieved pretyy efficiently before?
# the answer is, this approach provides us with a flexibility that the previous ones
# wouldnt provide us withs. if you think about it for a moment, you'll notice that
# the 'weight' matrix can essentially be anything and not just zeros! 
# we have decoupled it from tril, which's job is to set the right half of the tensor
# to zero, so we actually get the expected behavior later on.(which is setting a constrain really (more layer on))
# if you havent yet figured it out, the 'wieght' matrix, can be truly a weight matrix
# which can show different strengths for each token, basically it can learn the interations
# between tokens and manifest them. currently it is us who sets it to all zeros, so we get
# uniform probablities, which inturn manifest itself as an average. but what if instead of 
# all zeros, we actually learn the values from the data itself? that would make sense, and 
# make it so that each token, can have a different connection(strength) to any other tokens
# now, and thus build semantic/meaningful relation. this is inafct what we are after. we want
# for tokens to learn associations and relations with other tokens based on the data present
# in the dataset, having uniform weight like what we initially did really is a far cry from
# what we intend, and therefore, we opt in to use this new approach that allows us to actually
# exploit this new capability. 
# This is infact the problem that attention solves, that is gathering
# information from the past but in a data driven manner.(side note, we are not limited to 'past' 
# information only per say, in this example, this is the case however, we will explain this 
# in more detail)), we will see how attention does this exactly in a moment. 
#
# but before we jump into attention implementation also note that the tril part, infact is a hard-constrain here that prevents tokens from
# the past from interacting with the tokens from the future. 
# so to recap here, basically the idea is, this triangular form, allows us to have 
# weighted aggregations of past elements. each element in the lower triangular part, 
# specifies, the degree by which it plays a rule in the said outcome. 
# that is how much of each element gets to fuse into this specific position(i.e. current token's)
# so now lets incorporate attention into our model
# Attention does its job by using two vectors called, key and query.
# basically every single token, emits two vectors called key and query, the query verctor
# as the name suggests, implies, what we are looking for, and the key vetcor, again as the
# name suggets, implies, the contents, what it contains. 
# the way we get our 'weights' for these tokens is we simply dotproduct them together, 
# the ones that yield a high output/value, signify they are related positively.
# so our query is dotproducted with all the token's(keys) and the outputs reveal their 
# closeness/relevancy/similarity so to speak(if they align so to speak, they result in larger number)
# so effectively, the ones with higher number, are more similar to the query, the highest, 
# obviously having the most relavancy/similarity to the query.s
# so lets implemenet this
# #%%
# we want to implement a single attention head, we can later use this to create 
# multi-attention-head which is basically several single attention heads working in 
# parallel. so how do we implement one! 
# first lets create some random input 
torch.manual_seed(255)
# lets specify the batch, context_size and vocab_size
B,T,C = 4,8,32
# lets create an input 
x = torch.randn(size=(B,T,C))
# we said that attention works with two vectors, key and query, lets implement them
# we can use nn.Linear to implement them but beore that, what are the dims of such vectors,
# we are dealing with text and thus our inputs are tokens/vocabs, so the first dim would be 
# our vocab_size to account for all tokens, the next dim, is sth called a head_sizes, which 
# specifies the size of the key/query output usually 16 is used a lot for head size so we 
# use that as well
head_size = 16
# lets not forget to set bias=False, so what it does is exactly dotproduct 
# (actually, some people still leave the bias enabled! 
# so we will test both cases later and see if it actually matters! and if so to what extend! ) 
key = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
# now lets get the output
k = key(x)      # shape : 4,8,16 or (B,T, head_size)
q = query(x)    # shape : 4,8,16 or (B,T, head_size)
 # so the dims arent compatible for batch-multiplication, so we need a transpose
 # k.T wouldnt work becasue we have a batch-dim, so instead we use .transpose and
 # explictily specify the dims we want to be transposed. 
 # this will result in 4,8,16 by 4,16,8 which would give us 4,8,8 which is (B,T,T)
 # really as the result
 #! check transpose result is it okto use 2,1 or -2,-1 or 1,2
weight_raw = q@k.transpose(2,1)
# now lets for a moment think about what is happening here, the key and query are applied
# on the input and each return an output of (B,T,head_size), they are in fact, processing
# all the tokens in the input, individually, simultaneously, all the same time. so each 
# token is both a query, and a key, and when we do a dotproduct, we are basically telling 
# it to reveal the relation/similarity/relevance of every token with every other tokens.
# and as we explained earlier, this is infact our weight matrix (which was initially zeros)
# but is now learned from the data!
# print(f'weight_raw\n{weight_raw}')
# now we can apply constrain on it so that tokens can only communicate with the past so 
# we use the tril trick now!
tril_constrain = torch.tril(torch.ones(size=(T,T)))
raw_wieghts_masked = weight_raw.masked_fill(tril_constrain==0, float('-inf'))
# apply sotmax to get probablity for each token
weight = raw_wieghts_masked.softmax(dim=2) # remember weight is (B,T,T)
# and finally we can apply our weight on the input (we called it raw for a reason, read on)
# (by the way this is also called self-attention!)
bow_raw = weight@x 
# lets print raw_weights and weights and have some intuitive observations 
print(f'weight_raw\n{weight_raw}')
print(f'raw_weights_masked: {raw_wieghts_masked}')
print(f'weight\n{weight}')
# prints
# raw_weights(unconstrained)
#weight_raw
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00, -1.7903e+00,  1.4650e-01, -1.4147e+00, -1.7452e+00, -4.5249e+00, -1.9763e+00,  1.8833e+00],
#          [-1.3490e+00,  9.4076e-01, -6.9690e-01,  7.9933e-01,  4.7634e-01,  4.1861e-01,  2.3663e-01,  4.5217e-01],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,  9.0092e-01,  1.4083e+00,  2.5488e+00,  1.6350e+00,  1.0048e+00],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01, -1.1213e+00,  2.7562e+00,  7.0136e-02, -2.0337e+00],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,  4.9236e+00,  3.9545e+00, -2.2600e-01],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01, -1.5979e+00,  8.4627e-01],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,  2.2338e+00],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00, -2.1323e+00, -1.3948e+00, -5.6973e-01, -2.7986e-01,  4.3579e+00,  4.8477e-01, -9.7559e-01],
#          [ 8.7786e-01, -2.3352e+00, -2.2988e+00,  1.0322e+00, -1.4231e-01,  5.5233e+00,  1.0819e+00, -2.4722e-01],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,  2.9387e-02, -5.6397e-01, -2.4025e-01, -4.2512e-02, -8.4302e-01],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01, -4.0429e+00, -1.0676e+00, -3.0694e+00, -1.6156e+00],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00, -4.5593e+00, -3.1755e-01, -1.0504e+00],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00, -3.5616e-01, -2.6951e+00],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,  4.9077e-02],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01, -4.7902e-01,  6.7590e-01, -2.3153e-02,  1.9763e-01,  4.4385e-01,  7.2986e-01, -3.8846e-01],
#          [-6.2810e-01, -8.3713e-01,  1.3468e-01, -5.1328e-01, -2.6653e-02,  3.4325e-01, -1.0015e+00, -1.0334e+00],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01, -1.9702e+00, -1.0641e+00,  1.7792e-01,  8.0194e-01, -1.5265e-01],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,  3.6759e-01,  1.0552e+00, -5.2949e-01, -1.7715e+00],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01, -3.0057e+00,  1.3784e+00, -1.2091e+00],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02, -1.4040e+00,  2.3786e+00],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00, -1.6068e+00],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<UnsafeViewBackward0>)
#
# raw_masked_wieghts(constrained):
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],

#         [[ 1.0570e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.3490e+00,  9.4076e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1173e-01,  7.0261e-01, -4.0204e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.8030e+00, -1.9162e+00,  4.6460e-01,  9.0675e-01,        -inf,        -inf,        -inf,        -inf],
#          [ 4.6742e-01, -2.6938e+00,  2.9156e+00,  2.5135e+00,  7.9011e-01,        -inf,        -inf,        -inf],
#          [ 1.0721e+00,  1.0566e+00,  1.4205e+00, -8.5941e-01,  7.1559e-01,  9.4798e-01,        -inf,        -inf],
#          [ 1.7478e-01, -1.3748e+00, -1.6473e+00, -1.9683e+00, -2.7323e-01, -6.0714e+00, -2.3567e+00,        -inf],
#          [ 5.0234e-01,  2.5284e+00, -2.0990e+00, -9.8971e-01,  1.9566e+00, -4.6640e+00, -1.0950e+00,  7.8260e-01]],

#         [[ 1.2140e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 8.7786e-01, -2.3352e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.0629e+00, -6.2475e-01, -3.5850e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-4.1248e-01,  2.9397e+00,  1.9516e+00,  8.3461e-01,        -inf,        -inf,        -inf,        -inf],
#          [-2.7064e-01,  2.5970e+00,  4.7168e-01, -2.2572e+00,  1.5132e+00,        -inf,        -inf,        -inf],
#          [ 1.2678e+00, -7.5552e-01, -5.5975e-01,  1.1335e+00, -7.4317e-01,  2.8864e+00,        -inf,        -inf],
#          [ 5.6708e-01, -1.3799e+00, -1.1335e+00, -1.7761e+00,  1.7449e+00,  5.7658e-01,  2.0551e+00,        -inf],
#          [-1.0118e+00, -5.8753e-01,  3.9512e-01,  9.0220e-01, -1.7156e+00,  9.8603e-01, -4.4348e-01,  1.9781e+00]],

#         [[-3.6428e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-6.2810e-01, -8.3713e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-2.3925e-01,  1.6580e+00,  6.6220e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.2114e-01, -8.1565e-01, -5.0441e-01,  1.0567e+00,        -inf,        -inf,        -inf,        -inf],
#          [ 3.3470e-01, -5.7964e-01, -1.1165e+00,  7.8448e-01,  7.9756e-01,        -inf,        -inf,        -inf],
#          [-4.0079e-01, -5.7271e-01,  4.4317e-01,  4.7532e-01,  5.2594e-01,  2.2570e-02,        -inf,        -inf],
#          [ 1.3713e-01,  5.4308e-01,  1.6898e+00, -1.6946e+00, -6.3463e-01,  5.4102e-01,  1.3484e+00,        -inf],
#          [ 1.9584e-01,  4.7083e-01,  3.8171e-01, -3.8883e-01,  2.3697e-01,  8.6615e-01, -3.8587e-01,  2.5004e+00]]],
#        grad_fn=<MaskedFillBackward0>)
# 
# weight (final- normalized)
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.1976e-02, 9.0802e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9773e-01, 6.0261e-01, 1.9966e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.4181e-02, 3.4422e-02, 3.7221e-01, 5.7918e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [4.6023e-02, 1.9501e-03, 5.3236e-01, 3.5612e-01, 6.3550e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.9494e-01, 1.9195e-01, 2.7619e-01, 2.8253e-02, 1.3648e-01, 1.7219e-01, 0.0000e+00, 0.0000e+00],
#          [4.5214e-01, 9.6007e-02, 7.3108e-02, 5.3035e-02, 2.8887e-01, 8.7621e-04, 3.5963e-02, 0.0000e+00],
#          [6.8046e-02, 5.1605e-01, 5.0474e-03, 1.5304e-02, 2.9133e-01, 3.8823e-04, 1.3775e-02, 9.0057e-02]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.6132e-01, 3.8675e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.1870e-01, 3.3895e-01, 4.4235e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.2894e-02, 6.5398e-01, 2.4345e-01, 7.9674e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [3.7332e-02, 6.5690e-01, 7.8427e-02, 5.1206e-03, 2.2222e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.3611e-01, 1.7995e-02, 2.1887e-02, 1.1900e-01, 1.8219e-02, 6.8680e-01, 0.0000e+00, 0.0000e+00],
#          [9.8946e-02, 1.4120e-02, 1.8065e-02, 9.5010e-03, 3.2130e-01, 9.9891e-02, 4.3817e-01, 0.0000e+00],
#          [2.3306e-02, 3.5621e-02, 9.5163e-02, 1.5801e-01, 1.1530e-02, 1.7183e-01, 4.1141e-02, 4.6340e-01]],
#
#         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.5207e-01, 4.4793e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.8708e-02, 6.5816e-01, 2.4313e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.8422e-01, 9.1987e-02, 1.2557e-01, 5.9822e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [2.0870e-01, 8.3642e-02, 4.8896e-02, 3.2723e-01, 3.3154e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [9.4141e-02, 7.9271e-02, 2.1893e-01, 2.2608e-01, 2.3782e-01, 1.4376e-01, 0.0000e+00, 0.0000e+00],
#          [7.8726e-02, 1.1815e-01, 3.7190e-01, 1.2607e-02, 3.6387e-02, 1.1790e-01, 2.6434e-01, 0.0000e+00],
#          [5.6646e-02, 7.4575e-02, 6.8217e-02, 3.1568e-02, 5.9024e-02, 1.1073e-01, 3.1662e-02, 5.6757e-01]]], grad_fn=<SoftmaxBackward0>)
#
# as you can see, our weight is initialized for each batch based on the input data, and each token has its own
# weight, that is they are not uniform!, to get a better understanding lets consider the first batch of 
# raw-weights[0], raw_masked_wieghts[0] and weight[0]:
#
# raw_weights[0]
# tensor([[[ 1.0082e+00,  6.7622e-01, -6.4624e-02,  6.1701e-01, -2.7026e-01,  5.0988e-01,  1.6494e-01,  3.3523e-02],
#          [ 1.3140e+00, -1.1289e-01,  3.8299e-01, -1.1825e+00, -1.5816e+00,  3.9671e-01, -4.1703e-01,  7.0794e-02],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,  8.2834e-01,  6.7349e-01,  1.3126e+00, -3.1857e-01, -5.0728e-01],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,  7.8265e-01, -9.2628e-01,  1.1426e+00, -3.1167e-01],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01, -2.1753e-02, -5.0537e-01, -6.1707e-02],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,  5.8941e-01, -1.2738e+00],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,  1.7111e-01],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
#
#
# raw_masked_wieghts[0]
# tensor([[[ 1.0082e+00,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 1.3140e+00, -1.1289e-01,        -inf,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [-1.9066e+00,  7.9071e-01, -7.3821e-01,        -inf,        -inf,        -inf,        -inf,        -inf],
#          [ 3.0213e+00,  1.9722e+00,  2.0852e-01, -1.2651e+00,        -inf,        -inf,        -inf,        -inf],
#          [-2.5292e+00, -7.1544e-01,  9.2379e-01,  3.6982e-03,  9.7670e-01,        -inf,        -inf,        -inf],
#          [ 9.9968e-01,  2.9016e+00,  1.2989e+00, -7.0930e-01, -8.8080e-01, -2.6495e-01,        -inf,        -inf],
#          [-7.1386e-01,  1.0064e+00, -5.4975e-01, -2.6423e-01, -1.8767e+00,  1.2148e+00, -8.6248e-01,        -inf],
#          [-5.8812e-01, -1.1953e+00,  5.5418e-01, -2.3265e+00,  8.7663e-01, -9.5643e-01,  3.2523e-01, -4.3643e-01]],
# 
# weight[0]
# tensor([[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [8.0641e-01, 1.9359e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [5.2476e-02, 7.7872e-01, 1.6880e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [7.0222e-01, 2.4596e-01, 4.2162e-02, 9.6594e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.1816e-02, 7.2474e-02, 3.7333e-01, 1.4877e-01, 3.9362e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#          [1.0348e-01, 6.9321e-01, 1.3957e-01, 1.8735e-02, 1.5783e-02, 2.9217e-02, 0.0000e+00, 0.0000e+00],
#          [5.7514e-02, 3.2129e-01, 6.7772e-02, 9.0167e-02, 1.7979e-02, 3.9571e-01, 4.9572e-02, 0.0000e+00],
#          [7.3912e-02, 4.0274e-02, 2.3164e-01, 1.2995e-02, 3.1978e-01, 5.1140e-02, 1.8424e-01, 8.6020e-02]],
#
# take the last token, which 8.6020e-02, notice this token, not only knows its content and own position in 
# the sequence(its the 8th token after all) but also knows which tokens comes before it and how much its 
# related to any of them.(basically when it knows which token comes before it, it creates its own query so
# to speak, and talks to every single previous token to find about which ones are more or less relavent to
# it and to what extend.) 
# For example,lets say, (since we are dealing with character level text generation!), the last token is a 
# vowel and says hey im a vowel and im looking for everyone else thats a vowel! and uses query to talk to 
# other tokens(keys). and intrestingly another token (lets say number 4) says im a vowel as well, and the 
# response is thus generate a larger number.(this is not a good example, words in sentence would make more sense
# !give a better example!)
# in our example, For the last token, it seems the 5th and 3rd token are particularly intresting/relavent/important,
# followed by the 7th token.
# likewise, this happens for every token, i.e. token number 4, says the same thing and searches in its past toekns
# so on and so forth! 
# so what happens next is that when we get a high relevancy scale /response in the weights, like we just described
# when we do a softmax it will assign a large probablity to them, and this instructs the network that, we need more
# information from them, effectively allowing for aggregating a lot of their information into our position(lets say e.g. 8th token. 
# and we happen to learn more about them this way.
# now in practice, we are not intrested in aggregating the x raw values per say, rather we want their information, 
# so instead of just using the raw values of x, we instead use a representation of them, so to speak. 
# this is achieved using a third vector known as, 'value' and is the last vector we use. 
# !explain when we say vector, note that, we are talking from the prespective of a token, (every token has some vector 
# !of information and it gets to aggregate information via a weighted sum from all the tokens that point to it, and it
# )# !in practice we use a matrix, and hence the linear module to implement this to run the operation for all tokens in parallell. 
# just like the key and query, we set its bias to False, so we only get a simple vector,
# and a dotproduct output (again this is the intuition, but how much adding a bias would
# affect this we will see later, for now we stick to the default no bias version)
value = torch.nn.Linear(C,head_size, bias=False) # produces (B,T,head_size) just like the other two key,query vectors
x_processed = value(x)
# this is not yet final final, we still need to do one more thing (read on!)
bow_final = weight@x_processed
#
#!check also note that, as we previously once pointed out, attention is a communication mechanism between tokens, and 
# by default there is nothing in this mechanism that provides a notion of space/position for tokens involved, i.e.
# by default these tokens/nodes/points, dont have any idea about where they are or how they are positioned (with resepect
# to others) etc, so we need to encode this information as well.(to better visualizing it, consider each token as 
# a node in a directed graph, each node has a connection to another node, (for example each node has a connection to
# itself, and another connection to other nodes,etc) and as you can see there is no notion of space here,the attention
# simply acts on a set of vectors in this graph, and thats why we need to encode them positionally as well, so they 
# have information about their position with regards to other tokens. 
# !check (compare this to the convolution case and images
# or even text where the spatial aspect of data is preserved, but in attention, as we just stated, there is no notion
# of position, its just a set of individual vectors being operated on)-note that the connection between nodes, do give
# us a sense of structure, but it may not be enough to infer the underlying semantic, imagine a case, where a word
# for example has several meaning, and may very well have high relevancy to some tokens at the same time, but without
# additional positional information, an ambiguous semantic can be infered between the tokens involved, however when
# positional information is also present, such ambiguity can be avoided)
# also note that, in attention, samples do not interact with each other at all. when we have a batch of 4, each sample
# is processed in isolation, but in parallell to other samples, based on our graph example earlier, we would have 4 
# graphs of 8 nodes for example for each input sample. 
#
# we said earlier that what we implemented here is known as self-attention, the reason it is called self attention
# is that the key and query and values are applied on the same input(the use the same source!), and hence the name,
# self attention.
# also note that, in our specific case, tokens/nodes are can not communicate with the future nodes, but in general
# this constraint can be removed (and infact is removed/not implemented for some applications) where its benificial
# to be able to communicate with all the tokens. one example is sentiment analysis, where you want all the tokens to
# able to communicate with eachother so you can get an accurate analysis. and for this case, we would use an encoder
# block, which is basically what we have here, minus the constraint section(tril/mask part), what we have implemented
# here is called a decoder block, where we are decoding bunch of tokens, and it makes sense that the previous tokens
# do not comunicate with the future ones(becasue they would give the answer! and it defeats the whole purpose here!),
# becasue its a given that only the previous tokens must be used to predict the future/next token, hence the filtering/constraint part to prevent tokens from 
# comunicating with the future nodes/tokens.
# !so far we explained about the self-attention, which we saw, is called that way solely for the fact that key, query
# and value use the same source. the attention mechanism as we briefly pointed out, is much more general and can be
# used in different ways. one of such ways, is what is used to create sth called cross-attention. 
# cross-attention basically refers to the case where we have an encoder/decoder blocks, in which the queries come from x
# but the key and value come from an external source and sometimes from the encoder block. so cross-attention is used
# when theres a separate source of information we would like to pool from and use it as well.
#
# so far we implemented the attention based on the original paper(there are some differences we get to later on)
# except the part where we need to divide by the sqrt of the head_size. that is called scaled-attention
# so lets talk about this, and see why its needed. 
# the reason we add this so called 'scale' to our computation, is that, without it, the probablities will be saturated
# and when we add this term to the mix, it will make the 'weight' matrix to be 'unit variance', when Q and K are unit variance
# and this allows softamx to stay diffuse and not saturate too much.
# in other words, if we simply multiply key and query like that, the variance of the resulting weight matrix will be
# around the head_size instead of 1 which is bad and makes optimization really hard.
# to see this effect consider the following example
k = torch.randn(size=(B,T,head_size))
q = torch.randn(size=(B,T,head_size))
w_unscaled = q@k.transpose(-2, -1) 
print(f'{k.var()=}')
print(f'{q.var()=}')
print(f'{w_unscaled.var()=}')
# now add the 1/sqrt(head_size)
w_scaled = q@k.transpose(-2, -1) * head_size**-0.5 
print(f'after applying 1/sqrt(head_size)')
# makes the weight variance 1!
print(f'{w_scaled.var()=}')
# why is it important? if you recall, the weight matrix is fed into softmax, so its really important, especially during
# initialization that weight matrix be fairly diffuse, if we look at weight matrix here, we'll notice that they are now
# fairly diffuse 
print(f'weight_scaled[0]:\n{w_scaled[0]}')
# prints 
# weight_scaled[0]:
# tensor([[ 0.3972, -2.0957, -0.5396, -1.4860, -0.8393,  0.6273, -0.0157,  0.6284],
#         [ 1.3588, -0.0440,  2.1040, -0.2796,  1.7797,  1.4362,  1.3843,  0.0368],
#         [ 0.8109,  1.7400,  1.3579,  0.2097,  1.7152, -1.1242, -0.4349, -0.5690],
#         [ 0.0513,  2.8303,  0.7332,  0.0041,  0.9688, -1.6174, -1.4255,  0.2869],
#         [-0.7819, -0.6691, -1.4017, -0.4155, -0.8568, -0.2704, -0.9453,  0.3763],
#         [ 0.4092, -0.3079,  0.8472, -1.1753, -0.7699,  0.5091,  0.6385,  0.9677],
#         [ 0.3043, -2.1402, -2.2582, -0.3573, -1.2838, -0.0260, -0.0513,  0.9151],
#         [ 0.4180, -2.8353, -0.6435,  0.4842, -3.0202,  1.7690,  1.8837, -0.4393]])
#
# when softmax is applied:
print(w_scaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.8026, 0.1974, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1901, 0.4814, 0.3285, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.0499, 0.8038, 0.0987, 0.0476, 0.0000, 0.0000, 0.0000, 0.0000],
#         [0.1989, 0.2226, 0.1070, 0.2869, 0.1845, 0.0000, 0.0000, 0.0000],
#         [0.2148, 0.1049, 0.3329, 0.0440, 0.0661, 0.2374, 0.0000, 0.0000],
#         [0.3027, 0.0263, 0.0233, 0.1562, 0.0618, 0.2176, 0.2121, 0.0000],
#         [0.0901, 0.0035, 0.0312, 0.0962, 0.0029, 0.3478, 0.3901, 0.0382]])
#
# 
# now compare it with the unscaled weight : 
print(f'weight_unscaled[0]:\n{w_unscaled[0]}')
# weight_unscaled[0]:
# tensor([[  1.5889,  -8.3829,  -2.1583,  -5.9440,  -3.3573,   2.5092,  -0.0629,  2.5135],
#         [  5.4350,  -0.1759,   8.4159,  -1.1183,   7.1189,   5.7446,   5.5373,  0.1472],
#         [  3.2435,   6.9600,   5.4317,   0.8388,   6.8607,  -4.4969,  -1.7396, -2.2761],
#         [  0.2051,  11.3214,   2.9327,   0.0165,   3.8754,  -6.4696,  -5.7019,  1.1475],
#         [ -3.1278,  -2.6763,  -5.6069,  -1.6621,  -3.4272,  -1.0816,  -3.7811,  1.5053],
#         [  1.6368,  -1.2317,   3.3888,  -4.7012,  -3.0795,   2.0364,   2.5541,  3.8710],
#         [  1.2171,  -8.5610,  -9.0327,  -1.4293,  -5.1353,  -0.1038,  -0.2053,  3.6603],
#         [  1.6719, -11.3411,  -2.5740,   1.9369, -12.0810,   7.0760,   7.5347, -1.7573]])
#
# when softmax is applied:
print(w_unscaled.masked_fill(torch.tril(torch.ones(T,T))==0,float('-inf')).softmax(dim=-1)[0])
# tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [9.9636e-01, 3.6442e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.9592e-02, 8.0566e-01, 1.7475e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.4865e-05, 9.9975e-01, 2.2738e-04, 1.2309e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2943e-01, 2.0328e-01, 1.0848e-02, 5.6050e-01, 9.5940e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00],
#         [1.2012e-01, 6.8207e-03, 6.9264e-01, 2.1236e-04, 1.0748e-03, 1.7913e-01, 0.0000e+00, 0.0000e+00],
#         [6.3261e-01, 3.5856e-05, 2.2371e-05, 4.4856e-02, 1.1023e-03, 1.6883e-01, 1.5254e-01, 0.0000e+00],
#         [1.7351e-03, 3.8710e-09, 2.4850e-05, 2.2616e-03, 1.8472e-09, 3.8572e-01, 6.1020e-01, 5.6236e-05]])
#
# as you can see, some values are very negative, while others are very positive, for example, we have both 11 and -11
# and this is a recipe for disaster! the reason it is a problem is that, with softamx, when numbers take very positive
# and nagative values, softmax actually converges towards one-hot-vector. basically the values shrink towards the max.
# this can be seen the above, but the example below should demonstrate it more clearly.
# lets apply softmax on a list of numbers that are close to each other, we see the probablities are difuse and normal
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2]),dim=-1)}')
# we get a diffused probablity out of softmax
# tensor([0.2562, 0.3822, 0.1717, 0.1898])
# however if we increase the magnitude of the numbers(sharpen them) (and thus the difference between them) by like multiplying by 
# a number like 8 (just to simulate the effect here and so we can compare it with the previous case)
print(f'{torch.softmax(torch.tensor([0.1,0.5,-0.3,-0.2])*8,dim=-1)}')
# we see that the softmax, starts to sharpen towards the max, shrinking others except the 
# largest number, and effectively
# converging toward a one-hot-encoded vector.
# tensor([0.0390, 0.9559, 0.0016, 0.0035])
# so we dont want these values to be extreme, especially during initialization or otherwise, softmax will be way too picky!
# and we are basically aggregating the information from a single node instead of multiple ones (becasue one has the largest
# nvalue, its as if our sequence length is 1! and we lose access to the wealth of information the past history offers)
# so we want the probablities to be diffuse and not peaked like the second example here.
# so this scaling is used to retain the variance at a good value especially at initialization.
# so now that we are finally finished the self attention head, lets implement it as a module and incorporate everything
# we just discussed here.
class AttentionHead(nn.Module):
    def __init__(self, context_size, embd_size, head_size=16, use_bias=False) -> None:
        super().__init__()
        # we need context_size or block_size for creating the tril constrain
        self.context_size = context_size
        # we want this as the input dim for our key,query and value, this is 
        # infact the input_dim, since we plan on using the embeddings, we named
        # it embd_size
        self.embd_size = embd_size
        # head_size is the output dimension of our attnetion
        # !note that usually the head_size is equal to embd_size
        self.head_size = head_size
        self.key = nn.Linear(embd_size, head_size, bias=use_bias)
        self.query = nn.Linear(embd_size, head_size, bias=use_bias)
        self.value = nn.Linear(embd_size, head_size, bias=use_bias)

        # use buffer for trail, since this is a buffer, we can use the self.register_buffer which is
        # inherited from nn.Module class, to add it as buffer to the module, so it can be saved in the
        # state_dict when we save the model.  tril doesnt change, and is not updated(its required_grad is false),
        # so it being saved in state_dict doesnt matter to us, but to demonstrate this feature of pytorch, 
        # we are using it, and its a good practice, since pytorch knows how to deal with it and wont include it
        # in the computaion graph anyway!
        # 
        # This is typically used to register a buffer that should not to be considered a model parameter. 
        # For example, BatchNorm's running_mean is not a parameter, but is part of the module's state. 
        # Buffers, by default, are persistent and will be saved alongside parameters. This
        # behavior can be changed by setting persistent to False. The only difference between a persistent
        # buffer and a non-persistent buffer is that the latter will not be a part of this module's state_dict.
        # tril allows us to impose constrain by creating a lower traingualr matrix and setting
        # the rest entries to zero, effectively preventing tokens of the future from communicating with the past
        # each token can only communicate with the previous tokens that came before it.
        self.register_buffer('tril', torch.tril(torch.ones(context_size,context_size)))

    def __call__(self, inputs:torch.Tensor) -> torch.Tensor:
        #! check the shapes!! 
        B,T,C = inputs.shape # 4,8,16
        # print(f'{inputs.shape=}')
        # create the weight by using k,q,v
        k = self.key(inputs)    #! (B,T,C) or (B,E,16)? (4,8,16)
        q = self.query(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        v = self.value(inputs)  #! (B,T,C) or (B,E,16)? (4,8,16)
        # create the weight matrix and scale it by 1/sqrt(head_size) to keep weight unit variance 
        weight = q@k.transpose(-2,-1)* self.head_size**-0.5 # !(B,E,E)
        # weight_C = q@k.transpose(-2,-1)* C**-0.5 # !(B,E,E)
        # print(f'weight.var: {weight.var().item():.4f}')
        # print(f'weight_C.var: {weight_C.var().item():.4f}')
        # print(f'x:{tuple(inputs.shape)} k:{tuple(k.shape)} q:{tuple(q.shape)} v:{tuple(v.shape)} w:{tuple(weight.shape)} hs:{self.head_size}')
        # apply the tril constrain - (this makes this a decoder block!)
        # important note: notice we used tril[:T,:T] and not simply tril
        # this is because, when the input has a small context_size < self.context_size
        # like when we wantto generate inputs with sth like zeros((1,1)) which says
        # there is a single token (context_size 1) of 0 as the begining of the sequence
        # when this is input, the tril by default makes a context_size,context_size matrix
        # which will be different than the input context_size which is 1 e.g. or 2 e.g.
        # and it will fail becasue our weight would be 1,1,1 or 1,2,2, but trail is 1,8,8
        # and clearly this will cause an error. for this reason, we always create the 
        # tril check dynamcally by explicitly specifying the context_size based on the 
        # current input context_size so in case the context_size is smaller, tril is resized
        # dynamically accordingly. 
        # print(f'tril[:T,:T]==0: {(self.tril[:T,:T]==0).shape}')
        # print(f'tril==0: {(self.tril==0).shape}')
        weight = weight.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        # weight2 = weight.masked_fill(self.tril==0,float('-inf'))
        # print(f'{weight.shape=}')
        # print(f'{weight2.shape=}')
        # note that we are using batch, so instead of hardcodin 2,
        # we use -1 to refer to the last dim
        weight = weight.softmax(dim=-1)
        # finally apply the weight on the v
        bow = weight@v  # !(B,E,16)
        return bow        

# now lets add this to our model 
        
# so to recap, we first calculated the relavancy between all tokens against eachother, then constrained them so that 
# each token can only use the information from/interact with its past tokens. then since we needed probablity distribution so
# we then normalized it and then used that to pickout which tokens information(in the past) to aggregate/use with the inputs to 
# achieve our goal.(which is to predict the next character based on everything seen so far!)
# we dont want to aggregate inputs value (with respect to the weight matrix), we instead would like to use their representation
# so we use a new layer to do this, its called value, and we instead use its output instead of inputs raw value.
#
# 
#
# before we jump in and add the attention module, lets review our base model and see how we can 
# improve/prepare it before we incorporate the attention. 
# class BigramModelWithAttention(nn.Module):
#     def __init__(self, vocab_size, embd_size) -> None:
#         super().__init__()
#         self.vocab_size = vocab_size
#         self.embd_size = embd_size
#         # unlike the previous model, lets decouple the final logits from 
#         # the number of embeddings, because we are using attentions, and
#         # we want to have multiple operations inbetween obviously.
#         self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
#         # in order to get the final logits, we need a linea layer at end
#         self.fc = torch.nn.Linear(embd_size, vocab_size)
#        
#     def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
#         out = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
#         logits = self.fc(out)         # has the shape (B,T,C) c is vocabsize
#         loss = None
#         if labels is not None:
#             # recall that crossentropy likes its input to be B,C,T and we are B,T,C
#             # so lets permute and make it happy!
#             loss = F.cross_entropy(logits.permute(0,2,1), labels)
#         return logits, loss
#    
#     def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
#         # lets generate an output as long as num_max_token
#         for i in range(max_token_count):
#             # make sure idx is 2d
#             assert len(idxs) >1, f"idx.shape '({tuple(idxs.shape)})' is invalid. it must have the form (B,T)"
#             # now lets feed it to the model and sample from the probablities it produces
#             preds,_ = self(idxs)
#             # convert to probs 
#             probs = preds.softmax(dim=1)
#             # since we are bigram still, lets only get the last token as the next token predicted!
#             probs = probs[:,-1,:]
#             # now lets sample from it 
#             new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
#             # now concatenate the new token to the previous one and feed it back to the model
#             # for the next round of prediction
#             # also remember that we are creating a sequence, so we concat them at dim=1 to get 
#             # a longer sequence (we are gradually increasing the sequence length from 1 up to
#             # max_token_count)
#             idxs = torch.cat((idxs,new_idx), dim=1)
#            
#         return idxs

# this works fine, however, we can do better. here we just encoded the tokens, but
# as we already explained, attention has no notion of position or spatial structure like conv e.g.
# so we need to also incorporate the position information for each token as well
# since we are using the the attention head, we need more arguments
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, head_size, device, use_bias_att=False) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.embd_size = embd_size
        self.context_size = context_size
        # for gpu/cpu acceleration during training
        self.device = device
        # unlike the previous model, lets decouple the final logits from 
        # the number of embeddings, because we are using attentions, and
        # we want to have multiple operations inbetween obviously.
        self.embeddings = torch.nn.Embedding(vocab_size, embd_size)
        # lets now add the positional embedding as well 
        # using this, we try to retain the position embedding for our tokens
        # up to the current token in context_size
        self.position_embd = torch.nn.Embedding(context_size, embd_size)
        # add a self-attention head
        self.head = AttentionHead(context_size, embd_size, head_size, use_bias=use_bias_att)
        
        # in order to get the final logits, we need a linea layer at end
        # since we now have attention before this layer, the output dim of attention which is
        # head_size will be used here
        self.fc = torch.nn.Linear(head_size, vocab_size)
        
    def __call__(self, inputs:torch.Tensor, labels=None) -> torch.Tensor:
        # lets grab the shapes, since we will be using them 
        B,T = inputs.shape
        token_embeddings = self.embeddings(inputs) # has the shape (B,T,E) e is embd_size
        # since we have a position_embedding lets use that as well
        # and notice that we didnt use the inputs, but rather torch.arange(T)
        # this means, for each input, as we process it, we also get embeddings up to
        # the current token count as well, if the inputs has 3 tokens currently, we
        # will creeate position embeddings for 0,1 and 2, and for the next input
        # this continues likewise. this results in (T,E)
        position_embeddings = self.position_embd(torch.arange(T,device=self.device))
        # print(f'pos_embd:{position_embeddings.shape}')
        # and lets add the two embeddings together
        # this effectively gives us, not only the token embeddings(identity)
        # but also its position in the sequence. note that this doesnt really
        # help in a bigram model, but when it comes to attention it really does!
        embeddings = token_embeddings + position_embeddings
        # now lets feed this to attention head
        out_attention = self.head(embeddings)
        # lets feed this embedding to our fc at the end instead
        logits = self.fc(out_attention)         # has the shape (B,T,C) c is vocabsize
        loss = None
        if labels is not None:
            # recall that crossentropy likes its input to be B,C,T and we are B,T,C
            # so lets permute and make it happy!
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss
    
    def generate(self, idxs, max_token_count)-> list[torch.Tensor]:
        # lets generate an output as long as num_max_token
        for i in range(max_token_count):
            # print(f'{i}/{max_token_count}) idxs: {tuple(idxs.shape)}')
            # make sure idx is 2d
            assert idxs.ndim >1, f"idx.shape '({tuple(idxs.shape)})' is invalid({idxs.ndim}). it must have the form (B,T)"
            # now lets feed it to the model and sample from the probablities it produces
            # but since, we now have postional embeddings as well, we can no longer have 
            # more than block_size/contex_size in, becasue if our idx is more than context_size
            # our positional embedding will go out of scope and error out
            # so here we are basically getting as many as context_size
            # note that we dont destroy the idxs! each time we get the last context_size tokens
            # from it and feed it to the model to generate the next token, and keep going
            # if we replace the idxs by sth like idx=idx[:,-self.context_size:], we would
            # only create a sequence of only self.context_size, no matter how many iterations
            # we do, we just repeat the same sequence again and again!
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are bigram still, lets only get the last token as the next token predicted!
            logits = logits[:,-1,:]
            # convert to probs 
            probs = logits.softmax(dim=-1)
            # now lets sample from it 
            new_idx = torch.multinomial(probs, num_samples=1, replacement=True)
            # now concatenate the new token to the previous one and feed it back to the model
            # for the next round of prediction
            # also remember that we are creating a sequence, so we concat them at dim=1 to get 
            # a longer sequence (we are gradually increasing the sequence length from 1 up to
            # max_token_count)
            idxs = torch.cat((idxs,new_idx), dim=-1)
            
            
        return idxs
    
# now lets test this and see if it works 
x,y = get_batch('train',4)
model = BigramModelWithAttention(vocab_size, 
                                 context_size, 
                                 embd_size=16,
                                 head_size=16,
                                 device='cpu',
                                 use_bias_att=False)
# model.cuda()
# x,y = (t.cuda() for t in zip(x,y))
logits,loss = model(x,y)
print(f'{logits.shape=} {loss=:.4f}')
# and now we can train this : 
device='cpu'
batch_size = 32
head_size = 16
embd_size = 16 
context_size = 8
vocab_size = len(vocab_list)
max_iter = 5000
# only to test the effect of bias in k,q,v calculations
use_bias_attn=False
# attention requires much lower lr compared to plain bigram model
lr = 1e-3
model = BigramModelWithAttention(vocab_size=vocab_size,
                                 context_size=context_size,
                                 embd_size=embd_size,
                                 head_size=head_size,
                                 device=device,
                                 use_bias_att=use_bias_attn)

param_count = sum([p.nelement() for p in model.parameters()])
print(f'param count:  {param_count:,}')
print(f'head size:    {head_size}')
print(f'embd size:    {embd_size}')
print(f'context size: {context_size}')

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
model = model.to(device)
# set model to train mode explicitly
model.train()
for i in range(max_iter):
    # get the batch
    x,y = get_batch('train', batch_size=batch_size)
    # feed the model and get the logits
    logits, loss = model(x,y)
    # evaluate the model
    if i%1000==0:
        losses=evaluate_loss(100, device)
        print(f'train: {losses["train"]:.4f}  val: {losses["val"]:.4f}')
    # zeroout_grads
    model.zero_grad(True)
    loss.backward() 
    optimizer.step()
print(f'done!')

# now lets try its output
input = torch.zeros(size=(1,1)).int()
output = model.generate(input, 500).squeeze(0).tolist()
print(f"{''.join(decode(output))}")
# prints
#here are a few tries:
# param count:  3,041
# head size:    16
# embd size:    16
# context size: 8
# train: 4.2551  val: 4.2533
# train: 2.7175  val: 2.7354
# train: 2.5917  val: 2.5681
# train: 2.5299  val: 2.5265
# train: 2.4684  val: 2.4759
# done!

# Wonedimy
# Y:
# ARUS:
# LAnouver; pof th, sthe the mae,
# Wor ut barrioreche savarduncou?

# hiler bathakm,
# I O:
# AK:
# Acat ED:
# et alliro dow wowilou ttherss; whest owur, garsacwe Ie hithaceyidous sou f'ze iwot ry tohien fm.
# h.

# Ar'm-ore tr Fofhou
# Mr ojous omerrastheasncy yous Wowuce whor tu I I:
# Hbeotth spat fel woro bel, aneicudidy ktiferrd thout ngord. th Iid aral, be'nd.

# Pal tels Ms eimu Me myo on I teasthe del,
# AChinout,
# Wowure. pu tcher muss-renon; wingh ocet im
# r Ddr,
# Do ne thu,
# LI ir't
# Operst ry wu

# param count:  5,745
# head size:    16
# embd size:    32
# context size: 32
# train: 4.1374  val: 4.1414
# train: 2.6264  val: 2.6210
# train: 2.5100  val: 2.5092
# train: 2.4434  val: 2.4578
# train: 2.4140  val: 2.4252
# done!

# Thee to an tal my,
# Thake ced arce chart bre le be bizoresf thak des garg
# Lur Wentescaln
# Hooco beins pord sth ds hom span gr; ther. My Why ort thallils ofrof not tos gas hyine ind hand
# Thid One the shond nd,
# AFenosee omn st hy no fe.
# GEwank, iand!
# The sbe qurss yod hem snel.
# CLerle,
# Whow Vou bem!
# Theewourtoave, shougen t:
# Ber hangn toutince mor ed,
# AD Scheapy thatho ABEN:
# AD:
# SUEDithy nde ndel mmy Tongusate hfolit, gintoher, hat gwe tiet what whingimst hpoourde wowopal,
# The by br des qO: I no os,

# try2: decreasing context size
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

#try 3: decreasing embd_size, but keeping context_size 32
# param count:  2,265
# head size:    16
# embd size:    8
# context size: 32
# train: 4.2202  val: 4.2158
# train: 2.9840  val: 3.0093
# train: 2.7667  val: 2.7621
# train: 2.6355  val: 2.6411
# train: 2.6070  val: 2.6077
# done!

# o
# TI:
# BOGot be
# DOus ssh.
# Bea hashren v:
# ANULINII feso sid izeis by tofuso te I be fnowe?
# Wledu ha m, thope w,
# ARUTI land med cu
# SI bve fme
# ARIThilaserothan omyhee
# Oina c;
# UCAngose.
# Terve t thirre my tthunulpthemaf wof.

#  corlur yoNE:
# NHy w te hen cod, tonos vengo veschete tors t ithaimeqlutreqRoosoye chin y h thuasafaous.
# ANCathelo-
# The?
# Shen br uyh sbe ans I:
# Gouthyykey m Authirou ae tovas ber ut lnt:,

# TAMENI sf rousinthsin h'howenthor sg.
# hidandhe waresd!

# I f
# Se loner mrdin gsvette n thest?

# try 4: decreasing head_size, keeping embd_size and context_size the same
# param count:  4,457
# head size:    8
# embd size:    32
# context size: 32
# train: 4.3416  val: 4.3409
# train: 2.7354  val: 2.7333
# train: 2.6400  val: 2.6309
# train: 2.5716  val: 2.5692
# train: 2.5217  val: 2.5182
# done!

# IF
# S:
# To ofldickeens we fofane for o warus
# Herdliion bis, te,
# The ta, d ounor;
# Ad h ank; sor Cito-'d turses tierlllle,
# Whe pen, tint yt Ging sino leles fes hesr sst.
# Than hinean.


# Nofl an udn f hak,
# Wh stharovernes;
# And meno scerr tth Gbl y hy I t stit pand sy, pefif.
# US dik thith be tagag, harlovenppsh!
#  cegshil,
# Wheanst thiges eprine sy wabat tha.

# Wheas
# Wit:
# I
# Thouw'arsangort,
# Giseal?

# Wh; mo tands t's.

# NRNULECADSAxave!

# ANNDIEI Fh Fod meveeas pee be,
# Bes arner,

# Woo t'le.


# VIPORLEOG
# CAOO:

# try 5:
# param count:  4,977
# head size:    16
# embd size:    32
# context size: 8
# train: 4.2156  val: 4.2269
# train: 2.5848  val: 2.6063
# train: 2.4926  val: 2.4850
# train: 2.4587  val: 2.4601
# train: 2.4199  val: 2.4242
# done!

# DANGUTTA:
# Bert hatire on the wo A? sewas ker haywins wet.

# mousill akerd fous sther Le fke sth,, bepeere gn ssst I oun amand serewos, cobear song.


# AMEfediengg yit othesshint.

# Anc:
# A must shan ghety ak is, shet.

# MEO:
# Tom yot stths!
# NCIBus hand akt imy berdr,
# Thulll id mere thassut,
# Moureleray, fous mante. QBO decencepous ariveed hart;
# I; bifo wthen chant tht tho hy youn,
# Grolomy, ucel.

# BRIINE ERIYht I arum arw t; odeat ngher bly wehe foru thas sot our:
# Bn; sshiutithemowith her fourpler thant

#
# which looks depressing not gonna lie! but its better than the simple bigram model we 
# built earlier, anyway, can we improve it? certainly! 
# for example we could use dropout on our attention! we could use normalization layers
# and we could use multiple heads instead of just one! which takes us to the next
# subject which is multi-head attention! which is  really nothing except several normal!
# attention blocks run in parallell! 
# so lets implement these and see how much we can improve upon this

results=tensor([[[ 0.,  1.],
         [ 1.,  2.],
         [ 2.,  3.],
         [ 3.,  4.],
         [ 4.,  5.],
         [ 5.,  6.],
         [ 6.,  7.],
         [ 7.,  8.]],

        [[16., 17.],
         [17., 18.],
         [18., 19.],
         [19., 20.],
         [20., 21.],
         [21., 22.],
         [22., 23.],
         [23., 24.]],

        [[32., 33.],
         [33., 34.],
         [34., 35.],
         [35., 36.],
         [36., 37.],
         [37., 38.],
         [38., 39.],
         [39., 40.]],

        [[48., 49.],
         [49., 50.],
         [50., 51.],
         [51., 52.],
         [52., 53.],
         [53., 54.],
         [54., 55.],
         [55., 56.]]])
tensor([[ 0.,  1.],
        [ 2.,  3.],
        [ 4.,  5.],
        [ 6.,  7.],
        [ 8.,  9.],
        [10., 11.],
        [12., 13.],
        [14., 15.]])
tensor([[0., 1.],
        [1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.],
        [5., 6.],
        [6., 7.],
        [7., 8.]])
a=tens

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_head, head_size, embd_size, context_size,bias_attn=False) -> None:
        super().__init__()
        self.num_head = num_head
        self.head_size = head_size
        # we assume the head_size is already split between the num_heads and thus we dont split
        # it again
        # assert head_size//num_head == 0, f'head_size({head_size}) must be divisable by head_num({num_head})'
        # we need to create n heads so lets do it 
        # self.heads = [AttentionHead(context_size, 
        #                             embd_size, 
        #                             head_size, 
        #                             bias_attn) for _ in range(num_head)]
        # but we can also use pytorch's nn.ModuleList which is a better equivalent than list
        # note that, nn.Sequential cant be used, becasue it runs the modules in succesion
        # i.e. serially, one after the other (feeds the output of the previous module to 
        # the next module, etc) which is not what we want. we want to calculate each head
        # independetly and aggregate their outputs so, either a python list or torch moudle list
        # can be used
        # sidenote: this module that we are building, is also known as, masked multi-attention-head
        # becasue we are using the single-head self-attention which uses a constrain we imposed
        # by masking if you recall that!
        self.heads = torch.nn.ModuleList(AttentionHead(context_size, 
                                         embd_size, 
                                         head_size,
                                         bias_attn) for _ in range(num_head))
      
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # note that head_size is usually the embd_size, so it covers the whole 
        # embeddings obviously, and note that each head will work on a portion
        # of the given head_size, if e.g. we have head_size/embd_size = 16
        # if we had 1 head, the head_size would be 16. however if we had 2 heads
        # we had to split the head_size in half for each head, and later on
        # we had to concat them to get the full-size head_size/embd_size.
        # therefore here, we need to concat their results along the cols
        # so multi-head-attention is akin to group convolution, and thus the head_size
        # and num_head must align properly.
        outputs = torch.cat([head(inputs) for head in self.heads], dim=-1)
        return outputs
# lets test 
# note that we set the split value for head_size based on our num_head
m = MultiHeadAttention(num_head=4, head_size=16//4, embd_size=16, context_size=8, bias_attn=False)
logits = m(torch.randn(size=(1,8,16)))
print(f'{logits.shape=}')
# 
# now lets use this in our model
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 
# now lets train this model and see how it performs this time
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# outputs : 
# param_count  =  3,041
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2041  val: 4.204
# train: 2.6895  val: 2.685
# train: 2.5449  val: 2.528
# train: 2.4570  val: 2.477
# train: 2.4238  val: 2.433
# done!

# I'd In mame
# Ast owu hapland.

# Mayl thew youres?
# IRisire dis lhend gitim whorder, sphorgunct Beye skek boutrig!

# I, sho sciltr not bpreathl harr'd doess elowt cof or iny poovigre.
# Thalrit?

# HThary?

# Lerim he bamat:
# Thiet hey, krepably bose bour cave grosr benemece wleir hon'g.

# MADIUMI but th Mond im's veoo tit. IDf met Cerve's wue rrer hiftren'tr'd:
# Anoks:
# HOt
# We I phess chi bouine mares yon ther; tlotirtlinl therot:
# Io
# Wh per,
# Fous thayigso ceany pate.

# Annd wosh fong uat
# Now ICUSo CCgodertiove
#
# trying with larger embedding size seems to improve the results: 
# param_count  =  7,553
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1828  val: 4.186
# train: 2.4703  val: 2.472
# train: 2.3649  val: 2.367
# train: 2.3009  val: 2.32
# train: 2.2713  val: 2.288
# done!

# Cly Toste?
# IOK:
# Whan his me' tis it I ond ber'thse?

# PO:
# Theray delling, dothoinerk an to the, or
# E VINGON: rocO:
# Thes win now thatin knir:
# Wigh fy, bund werar is wet my;
# I ene:
# JUu'd sow prut hat amver'd ENDUE VUF E VO?

# Se gings nom and.
# CArwak mandesplay beesire brit mand.

# MED:
# Thou ased ith is whis wentemprt hits this in ther.
# Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
# LUT: am le the wids my thousecarto my haty wind WLIERME:
# Wher a and
# Whesh to wyt wir

# compared to single head attention, with the same hyper parameters, train: 2.4684  val: 2.4759
# and now we got train: 2.4238  val: 2.433 which is better(we also got train: 2.2713  val: 2.288 
# with just increasing the embedding_size, so playing with parameters even blindly making model bigger
# seems to give us a boost), so we had improvements, but we still need a long way ahead of us!
# 
# Ok, to improve upon our results, there are couple of more things we need to add to our attention 
# block. if you look at the paper, we'll see a few concepts that we havent talked about or implemented
# yet, including the feed-forward(position-wise feedforward network) and layernorm, so lets talk about 
# them.
# feed-forward network or as its called in the paper,'position-wise feedforward network', is simply
# a "fully connected network which is applied to each position separately and identically", this 
# consists of two linear layer with a relu activation function inbetween. 
# so its a linear layer with a relu activation function followed by another linear layer basically. 
# you may also hear this network be refered to as 'computation after communication' so to speak!
# signifying the fact that it runs a computation after the attention module.
# the idea behind this extra addition is simple, to provide a higher representation out of attention work
# this network, causes every single token to have a nonlinear transformation and achieve a higher abstraction
# possibly yielding new information benficial to the task. to put it in casual way, it can be seen as though
# the attention gatheres some stats/information, and this step is akin to looking into it and thinking about it
# comming up with some new findings, that is , if attention part is refered to as communication part, this is
# the computation part/ or thinking part for the lack of a better word.
# lets implement this network in our bigram model and see if this seemingly simple change, affects us at all
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # multihead attention instead of a single head attention block
        # note that the head_size is going to be divided between the 
        # heads, so ultimately we have the same number for head_size
        # globally (basically each head gets its share of head_size
        # which is head_size//num_head, but at the end since we have
        # num_head heads, their output makes us the whole head_size) 
        # ! check -> usually head_size == embd_size so each head works on the 
        # ! whole embedding 
        self.multi_head_attention = MultiHeadAttention(num_head, head_size//num_head, embd_size, context_size,bias_attn)
        # now let us create the feedforward network, which basically is a linear layer with relu
        # followed by another linear layer! for this so called network, the in_features and out_features
        # are simply the same, and is embd_size, because its a sandwich layer between our attention output
        # and the last layer. this layer usually is (embedsize,embed_size) if you recall head_size is usally
        # equal to embed_size. also note that in the paper the feedforward network is two linear layers with
        # a relu (one linear layer with a relu plus another linear layer).if you look closely you'll notice 
        # we already have a last layer after the attention, so we dont need to put a nother linear layer
        # afterward, becasue that one linear layer suffices (multiple linear layers, dont provide
        # any higher abstraction anyway, so its prefectly fine)
        # but on the other hand the paper's implementation has another change, it states 
        # the inner layers dim are increased by 4x, so to be faithful to the paper we also 
        # add the second linear layer, with the suggested change, this wont change the output 
        # shape, so we are fine. we just added an extra linear projection layer. 
        # this additional projection operation actually improves the result,however if we simply 
        # use the same dim linear layer (i.e. have sth like linear(head_size, head_size)) adds nothing
        # to the representational power of the network and you wont see anything substantial vs if you
        # completely remove this layer and only use linear/relu only.
        # why this works is becasue, we increase the nonlinear output neurons of the first layer by 4,
        # increasing its representational capacity, and then use the second linear layer to get the output
        # size compatible for the next layer (doing a linear projection), and hence our improvements lie
        # in the nonlinearity this addition provides. 
        self.feedforwardnet = nn.Sequential(nn.Linear(embd_size, head_size*4), nn.ReLU(),
                                            nn.Linear(head_size*4, head_size))
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets feed them to our multihead attention block 
        out = self.multi_head_attention(embds_combilned)
        # add our new addition, feedforwardnet
        out = self.feedforwardnet(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using feedforwardnet added')
torch.manual_seed(255)
random.seed(255)

head_num = 4
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size, 
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints : 
# using feedforwardnet added
# param_count  =  5,169
# head_num     =  4
# head_size    =  16
# embd_size    =  16
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1975  val: 4.198
# train: 2.6222  val: 2.634
# train: 2.4919  val: 2.481
# train: 2.4320  val: 2.435
# train: 2.3979  val: 2.392
# done!

# Selwils iur.

# AnNdt
# By thak, yue, nein ende vat Rout'w haler,
# The hot hes? is, whas bles. Yig,
# ROm
# QUS:
# Ay Whigice rea:
# Therer a youst um mith dins I dorseancer by gou:
# I sot the, aserer tom sne arobfs-Rid,
# This sas whe ou,
# Acet Yark no.

# OT:
# Fely ip you pel hivet seatly. hial pand,
# Thand repwat,
# I tarve ther se this ten thio drord dot tho,
# Rulee res le, il fet!
# Nord Wotiilet homom for my woue cuve wid.

# Angilene.

# Fhivy, ih:
# Boumlke uthereaerf
# Whe ithe hares.

# Whos sthik thenth
# Tait:
# She tove w

# the loss didnt get better! but maybe if we increase the context_size, and embedding_size, it 
# perorm better?
# using a larger embedding_size and thus larger model did improve the result, 
#
# using feedforwardnet added
# param_count  =  15,905
# head_num     =  4
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.1569  val: 4.159
# train: 2.3885  val: 2.396
# train: 2.2789  val: 2.312
# train: 2.2064  val: 2.249
# train: 2.1599  val: 2.198
# done!

# To get fer arast deve lad, I' fale are you up this riany;
# Weareien,
# Tho Guser, me ter is! Do lothe, to ling wo whe Pist to deave mat not, in and- air besface, y to fron; use ler weet herefalf thop an der pretrier
# nesly toide
# Thats ous.

# RENT:
# I for the haw to, to noss teave ler shat earelss ater, want Heanct herry deeter erry, heave masere dacke; nochat onsemy ning Meib,--?

# LUCLUER:
# If Feropeake mo shat onsivoole washy the well beWarquent;
# I on well he ome it you lare to lo to kit youser,
# Weves

# the larger network, seems to be doing much better than its counter part from before
# the one without the ffnet, we got train-loss: 2.1599  val-loss: 2.198 here whereas 
# previously we got train-loss: 2.2713  val-loss: 2.288. so we have improvements despite the 
# text still not being that good(but still better than before). so we need more enhancements)

logits.shape=torch.Size([1, 8, 16])
param_count  =  7,553
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1828  val: 4.186
train: 2.4703  val: 2.472
train: 2.3649  val: 2.367
train: 2.3009  val: 2.32
train: 2.2713  val: 2.288
done!

Cly Toste?
IOK:
Whan his me' tis it I ond ber'thse?

PO:
Theray delling, dothoinerk an to the, or
E VINGON: rocO:
Thes win now thatin knir:
Wigh fy, bund werar is wet my;
I ene:
JUu'd sow prut hat amver'd ENDUE VUF E VO?

Se gings nom and.
CArwak mandesplay beesire brit mand.

MED:
Thou ased ith is whis wentemprt hits this in ther.
Whis are mald thanle wive itis the menjuth weat dert, nen you vimt siomB ighichtr
LUT: am le the wids my thousecarto my haty wind WLIERME:
Wher a and
Whesh to wyt wir
using feedforwardnet added
param_count  =  15,905
head_num     =  4
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.1569  val: 4.1

In [4]:
# the next improvement, would be to use more of these, as its shown in the paper, if one block
# works, then adding more should work better right? we had single head, it improved our condition, 
# so we used more heads, and now lets more multi-attention heads! to make things easier, lets
# make a module out of it the same way we created previous modules. 

# first lets create our ffnet as a seprate block, so we use it after the attention
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        return self.ffnet(self.attn(inputs))

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks!')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks!
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.2087  val: 4.211
# train: 2.6084  val: 2.606
# train: 2.3737  val: 2.39
# train: 2.2865  val: 2.307
# train: 2.2087  val: 2.237
# done!

# FAUKIN YRE:
# Dato':
# wis.

# BERDEWEE:

# Piwit ancon; awt
# Emuke this his fred not pe,
# Har,
# And him my bird-wray sluf. Ile:
# With tulel wik, antegatorg.

# EREN VELV:
# Whath a meane, ones
# sheirss,
# Bose til
# Ilf bodtund ba,
# Aripce nerod is the blilshil hit hit mef?

# GESINCE:
# Sard, with the deoorrnsicominle thid: as.

# ANRE:
# Hy prall anges what micciths delre
# To tike comf hon ebat'd mur. Kin'sa, Fharth:
# Af'?

# ROASA:
# Ect mily fledd.
# SHricest.
# At dey bompe novor where lisg.

# KETILNINCE:
# There gay his my nefonk
#
# as you can see, we got wrose results than before! despite making the network larger, our loss
# really didnt improve as we expected. 
# is our initial hypothesis that having more blocks and higher nonlinearity/representation is benificial
# wrong? or is there something else thats causing the issue? 
# as you might have guessed, its the latter. we are basically creating more layers, and with
# more layers, we face training issues that we discussed earlier. 
# so how should we tackle this, we cant use BN, becasue BatchNOrm, accumulates the statistics
# from different samples, this is a no no for us. we dont want other samples to interfer with 
# our sample (or basically each other!) in anyway, so what should we do? 
# the paper utilizes two mechanisms or operations to tackle this issue. one being skip-connections
# (also known as residual connction) and the other, layer-normalization. 
# you should be familiar with skip-connections as they are the founding factor or resenets
# and have been extremely influential. so to cut a long story short, skip connections are simply
# connections from input skipping the operations involved in the block they reside and directly 
# being added to the output and then returned the result.(basically F(x) + x)
# as we know, the gradients are distributed equally when they reach addition, so input gets the gradients
# without being weakened due to large depth of the network. 
# so before we implement the layer normalization part, lets see howmuch of a change adding skip-connection
# causes and whether it proves our initial hypothesis about depth and gradient signal weakening or not
# we add this to our AttentionBlock (but we could add this to any submodule)

using more blocks!
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.2087  val: 4.211
train: 2.6084  val: 2.606
train: 2.3737  val: 2.39
train: 2.2865  val: 2.307
train: 2.2087  val: 2.237
done!

FAUKIN YRE:
Dato':
wis.

BERDEWEE:

Piwit ancon; awt
Emuke this his fred not pe,
Har,
And him my bird-wray sluf. Ile:
With tulel wik, antegatorg.

EREN VELV:
Whath a meane, ones
sheirss,
Bose til
Ilf bodtund ba,
Aripce nerod is the blilshil hit hit mef?

GESINCE:
Sard, with the deoorrnsicominle thid: as.

ANRE:
Hy prall anges what micciths delre
To tike comf hon ebat'd mur. Kin'sa, Fharth:
Af'?

ROASA:
Ect mily fledd.
SHricest.
At dey bompe novor where lisg.

KETILNINCE:
There gay his my nefonk 


In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_features, bias=True) -> None:
        super().__init__()
        # since this is only used with attention, the in/out features are the same
        # and usually the embedding size in our case but the paper states that the
        # inner layer dims are increased 4 times, so lets also reflect this change here.
        # as we saw earlier, this improves our results.
        self.n_features = n_features
        self.block = nn.Sequential(nn.Linear(in_features=n_features, out_features=n_features*4, bias=bias),
                                   nn.ReLU(inplace=True),
                                   nn.Linear(in_features=n_features*4, out_features=n_features, bias=bias))
    def forward(self, inputs):
        return self.block(inputs)
    
# lets add a skip-connection to this block
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        out = self.attn(inputs) + inputs
        out =  self.ffnet(out)  + inputs
        return out

# now lets add this to our base model and see how it performs this time: 
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints 
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5636  val: 4.553
# train: 2.2666  val: 2.281
# train: 2.1330  val: 2.183
# train: 2.0708  val: 2.14
# train: 2.0204  val: 2.109
# done!

# Zunt will jucked's whreef onst's so
# wippplalct, sawter.

# Firfor have deal pere,
# Shal,
# thich many.
# Frig.

# Thall have.

# WANWHAM:
# Sad king theave,
# Cnomlen nare, near was?


# CKINGlLOROMBOENGBETH:
# Dich loverd und by, may;
# ''Kh
# Godie the bline plock's minstell the denelsbad, with think
# What
# sicond lead this undrut, too, the but me my se ming'ld;
# Shing
# To tiot comford, engelan this it's heare: hear them the vinclamil.
# Is done, the stronced,
# Lion?
# Hartow where lisg.

# DOLILINA:
# Lord ISe and
# Lud'd nef hel

# and test with one skip-connection on the 'output only' to prove or assumption on skip-connections
# using more blocks with skip-connection
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.5616  val: 4.552
# train: 2.3196  val: 2.322
# train: 2.1862  val: 2.221
# train: 2.1175  val: 2.163
# train: 2.0640  val: 2.137
# done!

# FARY VI:
# A-bame 'tries.

# BERGEWES:

# PABLLO:
# Yon; aws
# Then ath-pmate for, not pe,
# PARILA:
# So man.

# FORY.

# TAsUS:
# I cenpine nou, lad king theak,
# Anny, In nartine-'

# An yeant, of ladie ust, good ting lover that by, may;
# And hordie the
# llils plove ond sofful and heave
# Take with the deo,
# And mort leath and usir the dom the but me my crombasill;
# Shim are, con comlmork engeland your, trabe,
# Beth:
# Yor?

# CORIAR EFt mily fle--spoles, strence'd ono do now, will?
# Mursice,
# Ntormorest
# The juge ernus'd neforki

# as you can see, it greatly improved our results, and the text also got much better!
# as we already pointed out, having skip-connections per modules, help much more than a single 
# per module output only, nevertheless, we notice, using skip-connection really improved our results.
# now lets add the second operation, layernorm. layernorm(https://arxiv.org/abs/1607.06450) is a normalization layer, just like batchnormalization
# came a year later than batchnormalization paper, but the difference between them is that, unline batchnormalization
# it works on a per sample basis and does not involve using othersamples to normalize a specific sample (basically
# samples dont affect eachother)
# As pytorch docs puts it:
# "Unlike Batch Normalization and Instance Normalization, 
#  which applies scalar scale and bias for each entire channel/plane with the affine option,
#  Layer Normalization applies per-element scale and bias with elementwise_affine"
#! check and expand
# Layer normalization (LayerNorm) and batch normalization (BatchNorm) are both normalization techniques commonly used in deep learning models. While they aim to normalize the input data, they differ in their operation, goal, and outcome. Here's a detailed comparison:
# 1. Operation:
#    - LayerNorm: LayerNorm normalizes the input tensor across the feature dimension (last dimension) independently for each example in the batch. It calculates the mean and variance along the feature dimension and applies a normalization operation.
#    - BatchNorm: BatchNorm normalizes the input tensor across the batch dimension (first dimension) as well as the feature dimension. It calculates the mean and variance along the batch dimension and applies a normalization operation.
# 2. Goal:
#    - LayerNorm: The goal of LayerNorm is to normalize the activations of each individual example in the batch. It aims to reduce the internal covariate shift, helping the network converge faster and generalize better.
#    - BatchNorm: The goal of BatchNorm is to normalize the activations across the batch dimension. It aims to reduce the effects of internal covariate shift, stabilize the network during training, and improve generalization by reducing overfitting.
# 3. Outcome:
#    - LayerNorm: LayerNorm ensures that the mean of each feature across each example is zero and the standard deviation is one. It preserves the relative relationships between the features within each example.
#    - BatchNorm: BatchNorm ensures that the mean of each feature across the entire batch is zero and the standard deviation is one. It introduces dependencies between examples in the batch during training but can be disabled during inference to allow independent predictions.
# 4. Applicability:
#    - LayerNorm: LayerNorm is commonly used in recurrent neural networks (RNNs), such as LSTMs and GRUs, where the normalization is applied along the time steps (sequence length) dimension.
#    - BatchNorm: BatchNorm is typically used in convolutional neural networks (CNNs) and fully connected layers, where the normalization is applied across the batch dimension.
# 5. Training vs. Inference:
#    - LayerNorm: LayerNorm behaves the same during training and inference. It normalizes each example independently, making it suitable for both training and inference.
#    - BatchNorm: BatchNorm behaves differently during training and inference. During training, it normalizes the activations across the batch dimension. During inference, the statistics (mean and variance) are usually calculated using a running average from the training phase, and normalization is applied based on these fixed statistics.
# In summary, LayerNorm and BatchNorm are both normalization techniques, but they differ in terms of the dimension over which normalization is applied, the goal of normalization, and the outcome. LayerNorm normalizes each example independently along the feature dimension, while BatchNorm normalizes across the batch dimension as well. They serve different purposes and are commonly used in different types of neural networks.

# the implementation is similar to the batchnormalization, and it does not require calculating running_mean/var
# we can use the pytorch module just fine, but since its really similar to BN, lets implement it here 
# 

using more blocks with skip-connection
param_count  =  38,753
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.5636  val: 4.553
train: 2.2666  val: 2.281
train: 2.1330  val: 2.183
train: 2.0708  val: 2.14
train: 2.0204  val: 2.109
done!

Zunt will jucked's whreef onst's so
wippplalct, sawter.

Firfor have deal pere,
Shal,
thich many.
Frig.

Thall have.

WANWHAM:
Sad king theave,
Cnomlen nare, near was?


CKINGlLOROMBOENGBETH:
Dich loverd und by, may;
''Kh
Godie the bline plock's minstell the denelsbad, with think
What
sicond lead this undrut, too, the but me my se ming'ld;
Shing
To tiot comford, engelan this it's heare: hear them the vinclamil.
Is done, the stronced,
Lion?
Hartow where lisg.

DOLILINA:
Lord ISe and
Lud'd nef hel


In [6]:
class LayerNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.eps = eps
        # alpha_ln_gain
        # note that since we are inheriting from nn.Module, and are in pytorch teritory
        # we need to use nn.Parameter to mark our parameters trainable! otherwise 
        # pytorch will ignore them, even if we set the requires_grad as true!
        self.alpha_ln_gain = torch.nn.Parameter(torch.ones(size=(1,in_features)))
        # beta_ln_bias
        self.beta_ln_bias = torch.nn.Parameter(torch.zeros(size=(1,in_features)))
    
    def forward(self, inputs):
        # calculate the mean and var
        # we calculate the mean for the last dims, that is feature dims (we are trying to normalize the features)
        # so we specify what we consider feature dims.
        #
        # note: initially I wrote 
        # dims = tuple(i for i in range(inputs.ndim-1,0,-1)) which didnt cause any errors, and infact
        # resulted in a very low loss, however the text generation seemed kind of random, and differed
        # from the pytorch's Layernorm version(0.60 vs 1.90 of pytorch's which is a huge diference). 
        # Further investigation revealed the issue which was caused by the dims involved. basically 
        # We needed to only use the last column which would be (-1) in our specific case. basicallys
        # what we want is to calculate mean/var for the dims that belong to features, e.g. in a 2d case like
        # (B,C), we want to mean/var on dim=1. Having worked with BN, we might also assume the same here 
        # that is if we have sth like (B,T,C) we would want to treat, B,T as batch and aggregate the mean/var
        # along dim=(1,2). however, as we learned earlier, LayerNorm does not involve other samples at all, 
        # and infact if we do this, we'll see despite the loss decreasing rapidly, our text generation are not
        # good at all! so when it comes to LayerNorm we always normalize the feature dimensions, and in our case
        # its just the last diminsion. 
        # what was causing the issue here was this exact issue, since we were calculating the dims dynamically
        # based on the inputs shape, for 2d inputs(ignoring batch) this would work as expected, but for 3d+, 
        # it would aggregate other samples stats and therefore creating the discrepency in the output between 
        # ours and pytorch's.s 
        # (we were calculating the mean/var for dims=(1,2) while Pytorch was only calculating it on the last dim, 
        # and hence the difference. (the output  shows that involving other samples adversly affect our output
        # and hence  why BN is not used and instead LN is used.) 
        # this happened becasue we dynamically tried to infer the dims by looking at the input shape
        # the behavior for 2d shapes will be the same, however, for the 3d shapes, our results differ.
        # we can define the aggregation along feature dims in Pytorch, so if we wanted to get the same 
        # behavior in pytorch we had to write sth like this: 
        # self.ln1 = nn.LayerNorm((context_size,head_size)) 
        # that is instead of useing a single number denoting the dimension's size, we specify the dim's size
        # explicticly. 
        # obviously since we are coding this for our usecase, we dont bother getting the input shape here
        # and use -1 to get the job done, otherwise, we would be getting the dim's size as input just like
        # pytorch and based on the dim's size, decide how to do mean/var. 
        # side note: note that if we use sth self.ln1 = nn.LayerNorm((context_size,head_size))  in our code, 
        # during text generation, we no longer can start with context_size of 1, we must always start with
        # sth like initial_token = torch.zeros(size=(1,8)).int() to get it working otherwise it'd complain
        # about shape mismatch which is expected because unlike our method, its static (while ours dynamically
        # would calculate the mean/var based on the inputsize) anyway, this shouldnt be an issue, becasue we
        # dont need to do this as not only it doesnt benifit us but also creates more hassle!.
        # side note2: aggregation in LN, may be benificial in other domains, as BN was, but for us, now it isnt
        # so we only make the one that works with the last dim! 
        # here's the wrong snipped that would create the wrong result for 3d inputs.
        #dims = tuple(i for i in range(inputs.ndim-1,0,-1)) 
        # print(f'{dims=}')
        dims = -1
        mean = inputs.mean(dim=dims, keepdim=True)
        # to get the same outputas pytorch's, we use the biased version as well
        var = inputs.var(dim=dims, keepdim=True, unbiased=False)
        # ​ (x−E[x])​
        # ----------    ∗ γ + β
        # sqrt(Var[x]+ϵ)
        # 
        xhat = (inputs - mean)/torch.sqrt(var+self.eps)  
        out = xhat * self.alpha_ln_gain + self.beta_ln_bias
        return out

# lets test it 
x = torch.randn(size = (3,8,300))
ln = LayerNorm(300)
ln_torch = nn.LayerNorm(300)
out = ln(x)
out_torch = ln_torch(x)
# now we expect that each sample/row now to have mean=0 and var=1
# (peviously for batchnorm this was the opposite, the mean/var 
# were computed for the whole samples,for each columns so that
# out[:,0] would be mean=0 var=1, likewise out[:,1] and etc )
# for layernorm however, we need out[0,:] to be mean=0 var=1 now
print(f"our's: {out[0,:].mean().item(), out[0,:].var().item()}")
print(f"torch: {out_torch[0,:].mean().item(), out_torch[0,:].var().item()}")
# ok now lets add this to our AttentionwithFFNetBlock and test it 
#

our's: (2.38418573772492e-09, 1.0004067420959473)
torch: (9.53674295089968e-09, 1.0004067420959473)


In [7]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

# and now lets train with the new change and see how it performs:
print(f'using more blocks with skip-connection-layernorm')
torch.manual_seed(255)
random.seed(255)

head_num = 4
block_num = 3
head_size = 32
embd_size = 32
context_size = 8 
vocab_size = len(vocab_list)
device = 'cpu'
use_bias_attn = False

lr = 0.001
batch_size = 32
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
# lets see how this model fairs now and what it generates 
initial_token = torch.zeros(size=(1,1)).int()
output = model.generate(initial_token, max_token_count=500).squeeze().tolist()
print(f"{''.join(decode(output))}")
# prints
# ----------------
#using more blocks with skip-connection-layernorm(using ours!-the wrong layernorm implementation)
# see the note at the end.
# param_count  =  38,753
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3881  val: 4.381
# train: 1.5804  val: 1.594
# train: 1.0614  val: 1.102
# train: 0.8326  val: 0.8784
# train: 0.7059  val: 0.7436
# done!
#
#
#
# VNIUKIVUTADES:'Kow seof on tith, owiw that thasawtde us athly hakef Bot pele,
# I al,
# I windis,
# And i-wradtsl! u slenerugh talelad kied togange,
# norl wink
# Tint ar in yoroqusendsld elust, gh hote woll brath hiss,
# A yoce nerdd verelLe I'lampl hat thisseflled
# Shell wharderw by in belourrns mominle thid;
# Aulir,
# Antoor plall'aege myane momo'thead bust ofting cow mol, esth'ang s. mit'sa, arak h:
# I thouge;
# Shat tam
# BONose,
# Low she storncexey an pot owoe whe?
# My sted-s't foot song
# qoulixe?
# Lur'deae:
# fol
#
#--------------
#
# using more blocks with skip-connection-layernorm(using pytorch's)
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2358  val: 2.242
# train: 2.1059  val: 2.154
# train: 2.0323  val: 2.098
# train: 1.9834  val: 2.068
# done!
#
# Fontive that mad's wise:
# But tith, luidien:
# Yord awto-must this have deat perefed al,
# thing is this is.
#
# PARUABE:
# O.
#
# WANWHAR:
# Yow king, what wirny.
# No nare, near in yeane, of lady. Whe, ghue tinst of this him,
# Murspose him dis the
# thine plock, this the pandse:
# Which will contend
# Whiresim thele this:
# Wutire
# A too, the but me wicHe micciodsh him are ting comf honamble's mur. Kin'sabhanke hear rend the thy the
# But seades, the strenced,
# Liven. How earse?
# Mursin.
#
# Nt from sing, I'll be
# Lur'd neforri
#
# --------------
# our layernorm fixed: 
# using more blocks with skip-connection-layernorm
# param_count  =  39,201
# head_num     =  4
# block_num    =  3
# head_size    =  32
# embd_size    =  32
# context_size =  8
# device       =  cpu
# use_bias_attn=  False
# train: 4.3906  val: 4.385
# train: 2.2373  val: 2.244
# train: 2.1070  val: 2.155
# train: 2.0339  val: 2.102
# train: 1.9812  val: 2.069
# done!

# For the that mad'
# With of on tith, luidie anct, saws
# Eule at botht if thou
# Wlefted' you windian.

# FLO-wike sluke slen:
# I have, ladik, anteraves,
# norl wink
# the bard a yeane, of lady. Whe, goshot I
# thy brath his what:
# Nown hondin the
# blinsh, shat this the pandself livadet brives nefors.
# ISIUS:
# Aren that usir the of me
# he hange wicHe mirs,
# Behat!


# TOLAT:
# But ford, engelan this it'sabhanke hear rend these this in usse,--
# Lord, strence'er thy?
# In woes. IDandry,
# Soset frongs neved
# With haul'd nefonk 
#
#
#
# for some reason, our layernorm achieves a much lower loss, much quicker, however the text seems to be worse
# than pytorch's for some reason and I have no idea what is causing this! 
# ok there were two issues, 1.our gamma and beta werent trained! becasue we forgot to wrap them in nn.Parameter
# and the second reason was, we were aggregating the last two dims like BN, whereas we should have only used the
# last dim, as we should not involve other samples in normalization. (I explained this thoroughly in the LayerNorm class)
# 
# now how can we improve more? we implemented the paper, basically we implemented transormer from scratch
# and what remains is to test with different hyperparameters to see how well it can generate texts similar 
# to our dataset. (we implemented the decore version, but the encoder as we explained earlier is the same
# without the constrains! well talk about this in a moment )
# so lets increase the model size now and see how much improvement we can get 
# 

using more blocks with skip-connection-layernorm
param_count  =  39,201
head_num     =  4
block_num    =  3
head_size    =  32
embd_size    =  32
context_size =  8
device       =  cpu
use_bias_attn=  False
train: 4.3906  val: 4.385
train: 2.2373  val: 2.244
train: 2.1070  val: 2.155
train: 2.0339  val: 2.102
train: 1.9812  val: 2.069
done!

For the that mad'
With of on tith, luidie anct, saws
Eule at botht if thou
Wlefted' you windian.

FLO-wike sluke slen:
I have, ladik, anteraves,
norl wink
the bard a yeane, of lady. Whe, goshot I
thy brath his what:
Nown hondin the
blinsh, shat this the pandself livadet brives nefors.
ISIUS:
Aren that usir the of me
he hange wicHe mirs,
Behat!


TOLAT:
But ford, engelan this it'sabhanke hear rend these this in usse,--
Lord, strence'er thy?
In woes. IDandry,
Soset frongs neved
With haul'd nefonk 


In [8]:
print(f'using more blocks with skip-connection-layernorm-beefed up!')
import time
import torch
torch.manual_seed(255)
random.seed(255)

torch.cuda.memory.reset_max_memory_allocated(0)

using more blocks with skip-connection-layernorm-beefed up!


/home/hossein/anaconda3/lib/python3.11/site-packages/torch/cuda/memory.py:329: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [9]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2        #2 #4  #6
block_num = 6 # aka layers!
head_size = 384     #18 #36 #72 #144 #288 #396 #384
embd_size = 384     #18 #36 #72 #144 #288 #396 #384
context_size = 256  #8  #16 #32 #64  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,901,889
head_num     =  2
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3868  val: 4.391
train: 1.9088  val: 1.995
train: 1.5507  val: 1.74
train: 1.3815  val: 1.617
train: 1.2835  val: 1.566
done!
elapsed: 1334.4600734710693 

DUKE VINCENTIO:
Sir, good master;
You but repebiling, and with a fire,
Engraced on the brat of death. It would
My dishop may your from the creyent I'll give Poducy,
To undo who from heaven do you: sister to have,
The king's sister'd. O, pretty Ratcliff!
I will make you to suppose him, he so!

PECHES:
Why he's it, he's are that so?

BRANA:
I say thou stay o'er hallowst it;
And now on his of justice, you do it.

LEONTES:
Go, have you for me? put do your honour;
For, pluck your silf. But, Hold, he makes me Lady.

CARDIPUS:
But have I not at the whereon, but shall discurr,
I'll have knightful. Let me book to live me.

CARDIPSABELLA:
O sake, amen! What us jon is of t

In [10]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 4        #2 #4  #6
block_num = 6 # aka layers!
head_size = 384     #18 #36 #72 #144 #288 #396 #384
embd_size = 384     #18 #36 #72 #144 #288 #396 #384
context_size = 256  #8  #16 #32 #64  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  9,901,889
head_num     =  4
block_num    =  6
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3916  val: 4.391
train: 1.9366  val: 2.026
train: 1.5327  val: 1.722
train: 1.3753  val: 1.608
train: 1.2819  val: 1.557
done!
elapsed: 1463.4444839954376 

AUTOLYCUS:
Marcy temperded the duke, then!
So fair came, and first some lay than
We are welcome are sugar to becomes,
And I may defysember you at usurp.
O God! why, says Misarian! Atter, to tell
The Duke of Norfolk; therefore, what's made an party
I the head, to you, made it and breathed,
Shich manarried to be pition of you.
Even then: and your vanity
By wit right in noise ear not one right;
And I'll trick, and foreign with you have
never soon, but that you behind himself wits,
Of our skits, too tapst for you fight you,
My good fency a liand tides with a haple to Rrave
And change for Curiolanus: which saying down the
Lancaster
And wait out after since to chall 

In [11]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6        #2 #4  #6
block_num = 1 # aka layers!# 1 # 2# 4# 6
head_size = 384     #18 #36 #72 #144 #288 #396 #384
embd_size = 384     #18 #36 #72 #144 #288 #396 #384
context_size = 256  #8  #16 #32 #64  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  1,774,529
head_num     =  6
block_num    =  1
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3219  val: 4.32
train: 2.2471  val: 2.282
train: 1.8389  val: 1.95
train: 1.6342  val: 1.804
train: 1.5447  val: 1.733
done!
elapsed: 311.1164629459381 

We mours wish rewken wit when of brought of
it makes, thou art that a do thy good, I my livined
by ruptiect not it pluner off their up,
Bexarge you shired in otherily earn, all slip-not be some
deaths for a made.

ISABELLA:
Faith, proclately?

MARCUTIUS:
No. She billo,
And old set nor usay die for our will and up,
Sir worth versue, to will not's lookingdren,
This fals thus plearful of canswer swook of For of it threy again?

WARm Bound you stand black, whethere your own.
Glouch lour gracer: yet up from heard,
For dequmetisure. thor eyes to be
A't serve. Bolinghands, that it was true go
To did the guain hite my merchancts of anhouse
Bintain. Chaudionession such;
Mu

In [12]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6        #2 #4  #6
block_num = 2 # aka layers!# 1 # 2# 4# 6
head_size = 384     #18 #36 #72 #144 #288 #396 #384
embd_size = 384     #18 #36 #72 #144 #288 #396 #384
context_size = 256  #8  #16 #32 #64  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  3,400,001
head_num     =  6
block_num    =  2
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.3344  val: 4.329
train: 2.1957  val: 2.228
train: 1.8599  val: 1.968
train: 1.6712  val: 1.834
train: 1.5534  val: 1.737
done!
elapsed: 587.169285774231 

A say! O, he we steel fullar eye execuse.

Butchas this not
poor men to say Romeo. I berl we arm to fridle
And me retume him the their tongue exbance
With stilly discon his ever side eath.

COMINIUS:
No, good for seen can cond the days raight sain:
He a-darglare pirce hip to glaw dow'd thou,
'I
Then it say shat, upoulitiong sabries,
Whet's you weast comising thre:
There of youghty timet? I verren
Leteens one that th't end nathis,
Ever His righty borth headless,
Line the coums; all brown morthin imake,
End but it sinch he earmile wearfullanter
And appy sue shy frown of squeunse find:
And, whee name be with shonot bed, le a

Tell my a ved royal than ene; for prewis

In [13]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6        #2 #4  #6
block_num = 4 # aka layers!# 1 # 2# 4# 6
head_size = 384     #18 #36 #72 #144 #288 #396 #384
embd_size = 384     #18 #36 #72 #144 #288 #396 #384
context_size = 256  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  6,650,945
head_num     =  6
block_num    =  4
head_size    =  384
embd_size    =  384
context_size =  256
device       =  cuda
use_bias_attn=  False
train: 4.2913  val: 4.286
train: 2.0428  val: 2.101
train: 1.6427  val: 1.807
train: 1.4664  val: 1.669
train: 1.3726  val: 1.604
done!
elapsed: 1144.0807490348816 

Fatal no dismiss next so rich, spoken together.

LUCENTIO:
Life, and you take the gentle constreman.

AUTOTH:
Balit with the peace afrizence of commend;
Where as I see could not retire the way
ta'en your honour peevised, or of my father
By I Have but your kneel cour;
For that our shands on desirers small kill you
'Tis eye, think ye! and are, woo't tells,
Holloses fly time will drown. Go What chrown
3 your brother bad grows, more batter one, who's good
Plecedgeness of heigh! What will be must usurp
Apptainted to fastic in worn with to shad,
Which do senten him; hast puts to of draw hath
Do such hold
Is but with teign widow his versel.
If consue I both him never 

In [14]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 6        #2 #4  #6
block_num = 6 # aka layers!# 1 # 2# 4# 6
head_size = 36     #18 #36 #72 #144 #288 #396 #384
embd_size = 36     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  94,601
head_num     =  6
block_num    =  6
head_size    =  36
embd_size    =  36
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3978  val: 4.4
train: 2.6556  val: 2.661
train: 2.4406  val: 2.449
train: 2.3151  val: 2.329
train: 2.2201  val: 2.24
done!
elapsed: 81.56748342514038 

Avepren att in bof sthorsouly, Fo 'Ton,
We sthacony fe leace wellw's: lo sthe's the
Buck.

CERCEve MARD:
Yurdoutt your ery bewarsIf me hy thell kno'st.
Rher mim you hour, al butant ting, ulared!wh me arertul
Thy thithe ho as trightest ollit prurren sthis,
pos has than lickint's hond thin hormp--mced,
Vit Whorisor thy earnte, lis oorce,
pat een hollor copis thilest won prok mad;
Corue mis woms fordes the thet gave he sim.

WALINO:
Ande om rot wom athese hen frie.

Sould hat fror wopert preim ancante a won muters.

NolTger Songil, the wat orel e.
Q

ORCIS:
I hor nlorputt I Het silly hobe aste you,
AMy, she to dard fome? Thad machte, Crin whe:
Mawachth youllatw's not faint 

In [15]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2        #2 #4  #6
block_num = 2 # aka layers!# 1 # 2# 4# 6
head_size = 144     #18 #36 #72 #144 #288 #396 #384
embd_size = 144     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  487,073
head_num     =  2
block_num    =  2
head_size    =  144
embd_size    =  144
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3068  val: 4.31
train: 2.2355  val: 2.253
train: 2.0030  val: 2.051
train: 1.8785  val: 1.959
train: 1.7911  val: 1.902
done!
elapsed: 40.65671443939209 

And litesh'd humped, my finds
Be may like theder'st my makes of would thing pideatis to parvone in go plitience;
Ye lour untake will and my dong from than hath a the hate buse:
A'll this in a my dieloose; why royad I cam.

GLOUCENCE;
Then makes,
Ris with if York's food far youn now this o's' fror to the his flows.

BOSTER:
That nowknid us dight words in we connly and Venon,
Were are replact's
Are not undent.

LORgUC:
Now I havess dine it from facoul, speactor.

PREXENVE BOMINGULED:
The cusing to fegiss to more the of night not
th disish
Of not of quilard's. There brew.
So saureto be the time.
Which Haint this wite; chark whom condselj:
And, you and suphesen age: whe

In [16]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 2        #2 #4  #6
block_num = 4 # aka layers!# 1 # 2# 4# 6
head_size = 144     #18 #36 #72 #144 #288 #396 #384
embd_size = 144     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  945,857
head_num     =  2
block_num    =  4
head_size    =  144
embd_size    =  144
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3297  val: 4.33
train: 2.1424  val: 2.167
train: 1.8964  val: 1.971
train: 1.7457  val: 1.872
train: 1.6492  val: 1.807
done!
elapsed: 78.7065634727478 


QUEEN MEN EDWARD:
What, my lord.

CORIOLANNA:
In this such praul
And do a reI'ld Nay! how ill thee? The thoughts stons,
And tine something dids I would be you somens selse
For not it one him.

LUCIO:
What devent we all'd, preir to a he
counsed of go: I sendien well a him;
Frang I home with you dead, nay too good: the I bloke,
That his the treaps here tthere, in with not woulds and
such tell binish can me any a queen
Even shill sweep is now.

BUCKINGHAM:
VIUS:
Greach is duke of a kind as pray!

SOF MARGAvoula.

ASTIS:
I wrething outher weet he son.

LEONTES:
An pelshall they to in to but wistane the shappin a rain
To queening natity prage enger comise me them loves! 

In [17]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 4        #2 #4  #6
block_num = 2 # aka layers!# 1 # 2# 4# 6
head_size = 144     #18 #36 #72 #144 #288 #396 #384
embd_size = 144     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  487,073
head_num     =  4
block_num    =  2
head_size    =  144
embd_size    =  144
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3477  val: 4.352
train: 2.2615  val: 2.274
train: 2.0259  val: 2.071
train: 1.8859  val: 1.967
train: 1.7862  val: 1.9
done!
elapsed: 49.76547884941101 

He like hand obther hefuld; nather, 't DisBut duke,
I, my so but ond niend dignee, then? Loor majesty.

ExANGUS:
I Go save;
Witild! it whom thou fir knot, Coniughte,
Therous, no stank as no'll
wores sear and whos my wishat Eds; whet?

THANGJOUS:
I heady dito nour streems is hapbay,
Loot'd know tor not and speak af you
STOLET:
And say him argoot, in blowNod:
O whill ble hest bed ressres oflice, han the bears.

VIBUORGBEOLIEL:
Hest by Gow healt, but who fe it.

KIOL ALoULIO:

ANCENEN:
TGA:
My bequrs that, it myray thuse: ap,
My lawdere hou spead to sonds grothad swill
Is you reacliadionble of suis jetty
No se, und therier. Tit'st crise that then that bloveNod.

CHARD C

In [18]:
class AttentionwithFFNetBlock(nn.Module):
    def __init__(self, context_size, embd_size, num_head, head_size, bias_attn=False ) -> None:
        super().__init__()
        self.head_size = head_size
        self.context_size = context_size
        self.num_head = num_head
        self.embd_size = embd_size
        self.bias_attn = bias_attn
        
        self.attn = MultiHeadAttention(num_head=num_head,
                                       head_size=head_size//num_head,
                                       embd_size=embd_size, 
                                       context_size=context_size,
                                       bias_attn=bias_attn)
        # note as a reminder, this is being applied
        # after an attention module, and attention module's input is embd_size, while its 
        # output-dim is head_size (we know they are usually the same, but to be flexible
        # we always use head_size to be on the safe side))
        self.ffnet = FeedForward(embd_size, head_size)
        # lets add the layernorm and to be sure our implementation works lets also test 
        # with pytorch's layernorm
        # self.ln1 = nn.LayerNorm(head_size)
        # self.ln2 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        self.ln2 = LayerNorm(head_size)

    def forward(self, inputs:torch.Tensor) -> torch.Tensor:
        # since we have two blocks, we add skip-connection to both of them here
        # we could aggerate them as one and a add the skip connection to their output
        # but this is less benificial than creating seprate skip-connections for each block
        # to see this in action, uncomment this and comment the latter part and run the test
        # out = self.attn(inputs)
        # out =  self.ffnet(out)
        # return out + inputs 
        # note that when it comes to applying normalization, there are two ways of going about it
        # 1.normalize the outputs of the attention block
        # 2.normalize the inputs before going to the attention block
        # the first one is the one the paper initially used, but later on it was shown that actually
        # normalzing the input yields better result, so we test both cases here
        # so check and see how it affects it! (personally,however, I found the first approach yielding midly
        # better loss, but I only tested it on small scale, so we will test this more)
        # method 1:
        # out = self.ln1(self.attn(inputs)) + inputs
        # out = self.ln2(self.ffnet(out))  + inputs
        # method 2:
        out = self.attn(self.ln1(inputs)) + inputs
        out =  self.ffnet(self.ln2(out))  + inputs
        return out

# we also need to add layernorm at the end of the block before feeding it to the nextlayer
class BigramModelWithAttention(nn.Module):
    def __init__(self, vocab_size, context_size, embd_size, num_head, head_size, num_blocks, device='cpu', bias_attn=False) -> None:
        super().__init__()
        self.vocab_size=  vocab_size
        self.context_size = context_size
        self.embd_size = embd_size
        self.num_head = num_head
        self.head_size = head_size
        # now lets add the number of blocks 
        self.num_blocks = num_blocks
        self.device = device
        # token/character embeddings
        self.token_embeddings = nn.Embedding(vocab_size, embd_size)
        # position embeddings
        self.position_embeddings = nn.Embedding(context_size, embd_size)
        # now lets use attention blocks instead of a multi-head-attention and ffnet
        # note that for this to work properly serialy, the output of this needs to be
        # the same as its input (which is embd_size) which by default for our case should
        # be ok.
        self.blocks = nn.Sequential(*[AttentionwithFFNetBlock(context_size=context_size, 
                                              embd_size=embd_size,
                                              num_head=num_head,
                                              head_size=head_size,
                                              bias_attn=bias_attn)
                                     for _ in range(num_blocks)])
        # lets test both layers
        # self.ln1 = nn.LayerNorm(head_size)
        self.ln1 = LayerNorm(head_size)
        
        # finally the output fc layer 
        self.fc = nn.Linear(head_size, vocab_size)
        
    def forward(self, inputs:torch.Tensor, labels:torch.Tensor=None)->torch.Tensor:
        B,T = inputs.shape
        # print(f'{inputs.shape=} {self.context_size=} {self.embd_size=}')
        token_embds = self.token_embeddings(inputs)
        # dont forget, our positional embd only involves the token position information
        # if we didnt have self.device we coulds use model's parameters device to dynamically
        # set the proper device here which would be:
        # device = next(self.parameters()).device
        # and this way, setting model.cuda() or model.cpu() would do the trick here, but
        # currently, we have to explictily set model.device in order to update this snippets
        # device or otherwise it will fail becasue of mistamtching devices (if we train on 
        # gpu and intend on running on cpu imediately! of course we can always save states
        # and load a cpu only model, but we need to be consistent in our code)
        position_embds = self.position_embeddings(torch.arange(T,device=self.device))
        embds_combilned = token_embds + position_embds
        # now lets have several multi-head-attentions instead of 1, one after the other
        out = self.blocks (embds_combilned)
        # normalize by layernorm 
        out = self.ln1(out)
        # and finally the logits 
        logits = self.fc(out)
        loss = None
        if labels is not None:
            # remember the cross entropy wanted its input in B,C,T while ours is in (B,T,C)
            loss = F.cross_entropy(logits.permute(0,2,1), labels)
        return logits, loss 

    def generate(self, idxs, max_token_count):
        assert idxs.ndim>1 , f'idxs.ndim({idxs.ndim}) must be 2 (in the form of (B,T))'
        for i in range (max_token_count):
            # we keep feeding the input to the model and get the next character
            # but since we use positional embeddings, we are limited to context_size
            # of tokens at anygiven time to feed the network or we face an error 
            # so we always tke the last T tokens from our input. we use negative
            # slicing, so if we have less than context_size, we only grab that many
            # otherwise, we get an error, becasue obviously at the begining we may 
            # start from a single token, denoting context_size of 1, while our model
            # expects like full context_size (e.g. 8 or more)
            idxs_cropped = idxs[:, -self.context_size:]
            logits,_ = self(idxs_cropped)
            # since we are after the next character only and we have to choose among 
            # context_size number of tokens, we get the last one and treat it as the next
            # character to calculate its probablity to sample from 
            logits = logits[:,-1,:] 
            # calulate the probs
            probs = logits.softmax(dim=-1)
            # sample the next character/token 
            idx_token_next = torch.multinomial(probs, num_samples=1, replacement=True)
            # add this to our existing tokens in idxs 
            idxs = torch.cat((idxs, idx_token_next), dim=-1)
            
        return idxs 

In [21]:
torch.cuda.memory.empty_cache()
start = time.time()
head_num = 1        #2 #4  #6
block_num = 4 # aka layers!# 1 # 2# 4# 6
head_size = 144     #18 #36 #72 #144 #288 #396 #384
embd_size = 144     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  945,857
head_num     =  1
block_num    =  4
head_size    =  144
embd_size    =  144
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3584  val: 4.364
train: 2.1029  val: 2.133
train: 1.8546  val: 1.941
train: 1.7210  val: 1.858
train: 1.6386  val: 1.8
done!
elapsed: 72.06510305404663 

Go joy on so, sound wisext ho have, come you foul same let exeder how the house.

DUCHESTESSS OF Reto me, sir, you sends. He's to affend, good it
The andser for me, who Grown; but it with his greater grial.
Why, forgen thinks, king-ig! Vilying be good and gentlone of fond
But sweet come, antench nor whose, where some world the Gaford this not we there in how that woak tongland;
O half be done of thou knew's to how spitter and;
Buch is the diself to
Do and have he
woe: a the way.

LLAUS:
My softer much o'e senter noing often meap!
The paress you hamidger as care to news:
Comis death look is thiness of a Clanuons Nor saint hoporseming.

LUCIO:

CORIOLANUS:
Why, Pome th

In [22]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
start = time.time()
head_num = 1        #2 #4  #6
block_num = 4 # aka layers!# 1 # 2# 4# 6
head_size = 144     #18 #36 #72 #144 #288 #396 #384
embd_size = 144     #18 #36 #72 #144 #288 #396 #384
context_size = 128  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  955,073
head_num     =  1
block_num    =  4
head_size    =  144
embd_size    =  144
context_size =  128
device       =  cuda
use_bias_attn=  False
train: 4.3672  val: 4.366
train: 2.2106  val: 2.229
train: 1.8922  val: 1.976
train: 1.7387  val: 1.873
train: 1.6464  val: 1.807
done!
elapsed: 138.7252631187439 

Mose to--
And Fringelencle we mave am bey my than in to mourt unbort Bhome the
Than would divogarl to upon back-bles.

KING RICHARG'd ICHive no lesselved,
And law new sin'tis of his talken
To daught him for ell of this me;
Even virtity as this besirgy to hom right as
Senaty, both droth made trums? O dubt press all muster the
Epourahter: in and you
A seem chame friends civent, a daughtol,
For die the siges: foul. O lettle you draid him,
And faultible drive; me was anoth doth misols repearl.
By see'er the jiet I to lettle,
Purn rest Richard to see of were can their reclatess.

Third:
I'll Bling, and Cacest an-peaced you mine talk'd him.

HENRY OF
If Mymen:
So rest i

In [23]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
start = time.time()
head_num = 1        #2 #4  #6
block_num = 6 # aka layers!# 1 # 2# 4# 6s
head_size = 144     #18 #36 #72 #144 #288 #396 #384
embd_size = 144     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  1,404,641
head_num     =  1
block_num    =  6
head_size    =  144
embd_size    =  144
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3412  val: 4.341
train: 2.0551  val: 2.092
train: 1.7727  val: 1.89
train: 1.6304  val: 1.789
train: 1.5450  val: 1.725
done!
elapsed: 105.23357772827148 

And wife wepty and me father boats, begued
By ruight with so; puturn not miverunk,--but, be atter Record.

LEONTES:
Had such many at those show thee accarione liers,
Ins as are told Rosours: they show, loht.
For they reshnet with such'd, gentlemen
But so with him excluns. Polous a proaty.
Morioum and dim'd vise the butch
dise to can encasters his dimstral from the
thou end worthem
your conforced and me there of days
And ne'er sle; you were mun; and love alisted then,
And proud alack and mark; youth be was me; oncefor
Your to this not that bread theremory,
Our up and decarence in gender meet
As the swords up ward falst title dark,
Swear'd thy luncher thou make my 

In [24]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
start = time.time()
head_num = 1        #2 #4  #6
block_num = 4 # aka layers!# 1 # 2# 4# 6s
head_size = 288     #18 #36 #72 #144 #288 #396 #384
embd_size = 288     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  3,716,417
head_num     =  1
block_num    =  4
head_size    =  288
embd_size    =  288
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3670  val: 4.371
train: 1.8191  val: 1.917
train: 1.5996  val: 1.775
train: 1.4953  val: 1.688
train: 1.4340  val: 1.637
done!
elapsed: 155.11348509788513 

Why, sir, you must not lug him.

First Citizen:
I no hear good so?

CORIOLANUS:
O God's a crown our charbes, prithes
Or the hurt of good and kilSTy 'Capitoe.

HENRY KING RICHARD III:
Marry, but me boy! Volsce and punish weals!

ANTIGONUS:
And he crides, do men, some were halming
For my sirs; it is the Vensentenance's pex
Wates to diving use Somerset, his wounvant:
The honemies Claudio?

AUTOLYCUS:
Good you are spripen for their honour,
I remaintains enver itself.

KINGBROKE:
What comfort? giveness as the cheriptess true:
Breath estempned is follower indeed's disster.

LUCIO:
No, they belly help? and call-hough did?

JULIET:
To be it with the strong sigh any empr

In [25]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
# between embd/bluck_num, increase block_num to achieve better result with less param
# with the same embd_size.(6 blocks o 144 embd_size > 4 blocks of 288 embd_size) or not!
# more embd_size with same block_num, perorms much better at the cost of twice param count
# but if lower param count is required, then block_num with same embd_size is better
# 
start = time.time()
head_num = 2        #2 #4  #6
block_num = 4 # aka layers!# 1 # 2# 4# 6s
head_size = 288     #18 #36 #72 #144 #288 #396 #384
embd_size = 288     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  3,716,417
head_num     =  2
block_num    =  4
head_size    =  288
embd_size    =  288
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.2768  val: 4.288
train: 1.8409  val: 1.926
train: 1.6026  val: 1.766
train: 1.4950  val: 1.68
train: 1.4276  val: 1.631
done!
elapsed: 168.80567741394043 

Ingeance chaste as the trivio, Or thy orse,
Where Cutivage to our bratuage away:
Or I stuty.

First Citizen:
The fit sovereign with the doubtle husband,
May sundon broash life, and keeping from and lost,
Before, if again'd be, cannot piercing sea:
weeping: you love's thing to so; my deep my lexft.

First Murderer:
I did our broother withappy down,
If thee, frest, being tales how first troubless laves they
To make was graceFs these daughter-threadement:
And legselches the lermit? what says how act,
Yet mays I return.

HENRY PERjYOURE:
Giolance? Balhoods my one.

GLOUCESTER:
Do thunder action tell.
Is set must be thought makewhat lighing
Salce hath letter courtain:

In [26]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
# between embd/bluck_num, increase block_num to achieve better result with less param
# with the same embd_size.(6 blocks o 144 embd_size > 4 blocks of 288 embd_size) or not!
# more embd_size with same block_num, perorms much better at the cost of twice param count
# but if lower param count is required, then block_num with same embd_size is better
# head_num doesnt impose much overhead, and no param increase, when all params are sorted out
# try increasing head_hum, you'll notice when it stops beniiting. so start with 1, adjust other
# params, and then play with head_num.
start = time.time()
head_num = 1        #2 #4  #6
block_num = 4 # aka layers!# 1 # 2# 4# 6s
head_size = 396     #18 #36 #72 #144 #288 #396 #384
embd_size = 396     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  6,991,841
head_num     =  1
block_num    =  4
head_size    =  396
embd_size    =  396
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3351  val: 4.327
train: 1.7144  val: 1.853
train: 1.5186  val: 1.708
train: 1.4237  val: 1.632
train: 1.3645  val: 1.588
done!
elapsed: 261.12746500968933 

To fires and all, home! would she with
As he is not falrel venge
That sets found it watch standes chaius Clarence?
With that every before Lord Hold frogot?

Second Murderer:
Well, my Lends, you will, by my service:
With our father--buftle.

HENRY BOLINGBROKE:
Make more fellow most little; be put them house
He alike her night me usure name to fights.

CORIOLANUS:
Was chaer then do shripe the swordsiries,
Still remember of empte
For on the helmed absence of unbaries:
Tell me told me the whopes seen public eagle
That hate beggled thee have tooth, sir.

Lord Murderer:
Farewell, if you'll presently him,
As they to see whose 'shaped rag in Nob,
Like to her by't. Look 

In [27]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
# between embd/bluck_num, increase block_num to achieve better result with less param
# with the same embd_size.(6 blocks o 144 embd_size > 4 blocks of 288 embd_size) or not!
# more embd_size with same block_num, perorms much better at the cost of twice param count
# but if lower param count is required, then block_num with same embd_size is better
# head_num doesnt impose much overhead, and no param increase, when all params are sorted out
# try increasing head_hum, you'll notice when it stops benifiting or is not just worth it,
# so start with 1, adjust other params, and then play with head_num.
start = time.time()
head_num = 1        #2 #4  #6
block_num = 6 # aka layers!# 1 # 2# 4# 6s
head_size = 288     #18 #36 #72 #144 #288 #396 #384
embd_size = 288     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  5,546,369
head_num     =  1
block_num    =  6
head_size    =  288
embd_size    =  288
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3546  val: 4.353
train: 1.7420  val: 1.869
train: 1.5220  val: 1.713
train: 1.4194  val: 1.617
train: 1.3619  val: 1.585
done!
elapsed: 230.01764106750488 

Shepherd:
So preserve and yourself and in this truth?
Amen, the second of secraff.

EDWARD:
That apollory'd! protesperous.

JULIET:
With the resuest home; and thy
That which your stooporous; to wail.

TRANGS OF ELIO:
But in pellar, send unto my power, sport that with theIt:
And let's not they speech, false them's will
while wondult still move the dies,
His sinceasements;
We unbukes then I that it.

DUCHESS OF YORquiet,--

ANGELO:
Sir, how frecure you, good lie you, my lord;
Be lusts and the wind, many rich my help!
What something worthy exile.
More than a man too seem.

First Senator:
I would have so goes before him? when he
When 'tis medication call'd as I
hear

In [28]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
# between embd/bluck_num, increase block_num to achieve better result with less param
# with the same embd_size.(6 blocks of 144 embd_size > 4 blocks of 288 embd_size) or not!
# more embd_size with same block_num, performs much better at the cost of twice param count
# but if lower param count is required, then block_num with same embd_size is better(some times
# higher block_num achives comparable result with fewerr param counts)
# head_num doesnt impose much overhead, and no param increase, when all params are sorted out
# try increasing head_hum, you'll notice when it stops benifiting or is not just worth it,
# so start with 1, adjust other params, and then play with head_num.
start = time.time()
head_num = 1        #2 #4  #6
block_num = 6 # aka layers!# 1 # 2# 4# 6s
head_size = 300     #18 #36 #72 #144 #288 #396 #384
embd_size = 300     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  6,015,065
head_num     =  1
block_num    =  6
head_size    =  300
embd_size    =  300
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.3292  val: 4.34
train: 1.7512  val: 1.873
train: 1.5285  val: 1.71
train: 1.4240  val: 1.622
train: 1.3626  val: 1.582
done!
elapsed: 256.6072313785553 


THASTINGS:
I cannot then, like me to elucy.

NORTHUMBERd WILLO:
Hold, for not, hath hee, sir! I will fought,
Knock my patricians me house, my court'sic;
Her known chaolicy poor piercement that,
For one will you brother fiery will no;
As thou art all to comes of your kin?
Down, my both against a face! wold their York!

CLIFFORD:
Come, partly spoke be not off; thou who,
And shepherd are sprune to her hearts men
Their partal heads oft your foes and bleaters elf,
wet must death by lend again befound,
When provide as sold he bounds to my lies;
His restor thousand us slaughter, and my fault!
Why well, my lord, my trumber of general standles.
Nay, man, so, that ament, I 

In [29]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
# between embd/bluck_num, increase block_num to achieve better result with less param
# with the same embd_size.(6 blocks of 144 embd_size > 4 blocks of 288 embd_size) or not!
# more embd_size with same block_num, performs much better at the cost of twice param count
# but if lower param count is required, then block_num with same embd_size is better(some times
# higher block_num achives comparable result with fewerr param counts)
# head_num doesnt impose much overhead, and no param increase, when all params are sorted out
# try increasing head_hum, you'll notice when it stops benifiting or is not just worth it,
# so start with 1, adjust other params, and then play with head_num.
start = time.time()
head_num = 4        #2 #4  #6
block_num = 6 # aka layers!# 1 # 2# 4# 6s
head_size = 288     #18 #36 #72 #144 #288 #396 #384
embd_size = 288     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  5,546,369
head_num     =  4
block_num    =  6
head_size    =  288
embd_size    =  288
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.4196  val: 4.421
train: 1.7971  val: 1.906
train: 1.5277  val: 1.711
train: 1.4134  val: 1.617
train: 1.3463  val: 1.577
done!
elapsed: 280.61063385009766 


Second Murderer:
The noble of his cocks and honour, I
Thanks you. Deceive this oracle, the dishal, ourself,
Send me more almous. And thou mayst most ae;
And secret sing is confirm with these hencejour'd
Go, but that sweet revenged, Walihed, for Richard and our
good washing; but seit o' time the duke, I know,
accusatured, familia, I will give desert,
No; what there news are fouler doubt issufferement
With feel not to be and his begot.

CORIOLANUS:
O, my lord,
I'ld help you: before to you?

BENVOLIO:
No, that of the shore? what masters him without?
'A reject in it must other Lady
Here cut our land in my childish full libe,
But he that your morry speech. The monum

In [30]:
torch.cuda.memory.empty_cache()
# head_num < block_num , increase block_num over head_num
# context_size doesnt afffect loss much than embd/block num
# between embd/bluck_num, increase block_num to achieve better result with less param
# with the same embd_size.(6 blocks of 144 embd_size > 4 blocks of 288 embd_size) or not!
# more embd_size with same block_num, performs much better at the cost of twice param count
# but if lower param count is required, then block_num with same embd_size is better(some times
# higher block_num achives comparable result with fewerr param counts)
# head_num doesnt impose much overhead, and no param increase, when all params are sorted out
# try increasing head_hum, you'll notice when it stops benifiting or is not just worth it,
# so start with 1, adjust other params, and then play with head_num.
start = time.time()
head_num = 2        #2 #4  #6
block_num = 6 # aka layers!# 1 # 2# 4# 6s
head_size = 288     #18 #36 #72 #144 #288 #396 #384
embd_size = 288     #18 #36 #72 #144 #288 #396 #384
context_size = 64  #8  #16 #32 #64*  #128 #256
vocab_size = len(vocab_list)
device = 'cuda'
use_bias_attn = False

lr = 0.0001
batch_size = 128
max_iter = 5000
eval_period = 1000
model = BigramModelWithAttention(vocab_size=vocab_size, 
                                 context_size=context_size, 
                                 embd_size=embd_size,
                                 num_head=head_num, 
                                 head_size=head_size,
                                 num_blocks=block_num,
                                 device=device,
                                 bias_attn=use_bias_attn)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)
param_count = sum(p.nelement() for p in model.parameters())

print(f'param_count  =  {param_count:,}')
print(f'head_num     =  {head_num}')
print(f'block_num    =  {block_num}')
print(f'head_size    =  {head_size}')
print(f'embd_size    =  {embd_size}')
print(f'context_size =  {context_size}')
print(f'device       =  {device}')
print(f'use_bias_attn=  {use_bias_attn}')

for i in range(max_iter):
    # read a batch 
    x, y = get_batch('train', batch_size=batch_size)
    x,y= tuple(t.to(device) for t in (x,y))
    logits, loss = model(x,y)
    
    # calculate the smoother loss on multiple batches on train/val splits
    if i%eval_period == 0:
        losses = evaluate_loss(200, device)
        print(f"train: {losses['train']:.4f}  val: {losses['val']:.4}")
    # zero-out gradients 
    model.zero_grad()
    # do a backward pass 
    loss.backward()
    # do a single optimization step 
    optimizer.step()

print(f'done!')
print(f'elapsed: {time.time() - start} ')

# lets see how this model fairs now and what it generates 
model.cuda()
#model.device ='cuda'
initial_token = torch.zeros(size=(1,1)).int().cuda()
output = model.generate(initial_token, max_token_count=1000).squeeze().tolist()
print(f"{''.join(decode(output))}")
# which prints 
# using more blocks with skip-connection-layernorm-beefed up!
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
#
# What, my Lady of Angelo?
#
# BUCKINGHAM:
# At at the untimely of doth bed, much,
# I'll be a kind his worth sea of the
# live mone conclaim my sorrow and amblitude
# Dut jot the watching of goos, Caius York,
# Whom hath power tire mild, I will leave you tell the
# is it subjects.

# GLOUCESTER:
# So rass, swing play his manner great his love
# Death is cloid of his fine.

# POLIXENES:
# No.

# Ghost offence, sea, when we he is name
# Where it were become me one years.

# TLBONA:
# Who, is he can you for the horse,
# That he clave upon him. Grim to't her.

# BRUTUS:
# It be more fair climits night rest thinke I
# not, and what all worse this.

# SAETER:
# Sir you are you must you wed?
# Doth mendles when you have once descries them,
# The harm of cullent you cursue for this:
# Resole he must be abused you.

# Clown:
# Kill, by my your highne, and that all bitter thee.

# JULIET:
# Gaunt, I'll make thee.
# Untired my brother birisment buriest!
# I pray, minister in my eight; it is it he
# not fastic that that fant for let thou
# of thine, or sleepter.
#
# second try
# param_count  =  9,901,889
# head_num     =  6
# block_num    =  6
# head_size    =  384
# embd_size    =  384
# context_size =  256
# device       =  cuda
# use_bias_attn=  False
# train: 4.3845  val: 4.388
# train: 1.9630  val: 2.045
# train: 1.5727  val: 1.755
# train: 1.4054  val: 1.632
# train: 1.3029  val: 1.57
# done!
#
# DUKE VINCENTIO:
# Scent the Faulia.

# KING EDWARD:
# Sweet, good conceive these are to be.

# CAPULET:
# Not gentle Menenius, here well they pend lives and I
# Rengeath one much buried.

# HARCID:
# O, God I'll: Bidish, Brobet, not--
# Hath said, quotest me, is a done.

# CORIOLANUS:
# Then, no proud heret! what sat withile: woen so,
# ha, wary thou comest, why thought that art thou nelsest;
# yet fights on his dearth-hoad yoking hered.

# KING EGBRY:
# Very Grey, my lord?

# CORIOLANUS:
# The people.
# Why let, sir, who have been sweet, Let to thee.

# BARBIUNCENA:
# In to the rest, my blood; Warwick's house.
# Take you in his worth train'd then he mone.

# ROMEO:
# Those wasted damnable, Eauton,
# Like, and brisothe comes do I nay,
# and that you know I were, and you lead us
# I tept brail, if such enemi: knack'd love from
# Unsuing person, and never. O, you should woman joys;
# But in this seven strem one of mine honestes
# Their woman to die: nay, sir, and were back
# if the imploteful of late, our chante faults
# He that wretche with this.
#----------------------------------
# which is remarkably better than all of our previous outputs. so as we increased our model capacity
# we witnessed much better results. 
# side note:note that our positinal emebdding's need to be placed on the cpu or gpu explicitly 
# after model instantiation, or otherwise, as its set separately as the model, simply doing model.cpu()
# or model.cuda() wouldnt do it. so here I simply used cuda. (or we have to set the device in forward
# dynamically sth like device = next(model.parameters()).device)
# this was gpt! lets talk about the models, glue activiation ufnction, efficiancy , chatgpt vs us, 
# document completer vs chatgptetc 

param_count  =  5,546,369
head_num     =  2
block_num    =  6
head_size    =  288
embd_size    =  288
context_size =  64
device       =  cuda
use_bias_attn=  False
train: 4.2760  val: 4.272
train: 1.7579  val: 1.873
train: 1.5127  val: 1.694
train: 1.4118  val: 1.614
train: 1.3468  val: 1.576
done!
elapsed: 245.41499209403992 

The track, as he dising true. O, he is lend.

WARWICK:
Thy dropter and creates to conclude,
Did much with some vestury of lamen,
I'll brought in him grace a Copitol?

DUKE VINCENTIO:
Now, I thousand mouthst he speak-to't
As he neck of her body furthee to myself
Have brother our life, or mointy obscured
Is peace welcome and tends of removiin, which
Then, sir Tybalt's carry. What, our son.

AUTOLYCUS:
Turn in brother, then where sell.

CLARENCE:
My lord, is made be encounterned!' you, he hath a lead.

Servingnated of Edward: you are a maid gave us to kill your
your croachin kinsman father; thanks thou yet?

DUKE OF AUMERLE:
But truth.

HENRY BOLINGBROKE:
Dy, as So